# EESM LUT numerical analysis

Reproduce the numerical experiments with a local ZIP containing `Parameters.mat` and `Tables.mat`. The source data are available at `https://doi.org/10.5281/zenodo.18822007`, but no automatic dataset download is performed.



## Libraries

In [ ]:
import os
# Configure cuBLAS before importing PyTorch (restart the kernel after changing this).
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import sys
import math
import json
import random
import itertools
import time
import warnings
import shutil
import hashlib
import platform
import subprocess
import importlib.metadata as importlib_metadata
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import scipy.io as sio
from scipy.interpolate import CubicSpline, RegularGridInterpolator
from scipy.linalg import expm
from scipy.integrate import solve_ivp
from scipy.stats import wilcoxon, mannwhitneyu
import matplotlib.pyplot as plt
from IPython.display import display
import torch
import torch.nn as nn
from torch.nn.utils.parametrizations import spectral_norm
from torch.utils.data import DataLoader, TensorDataset
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna==4.8.0'])
    import optuna
from optuna.samplers import TPESampler
warnings.filterwarnings('default')

plt.ioff()


## Analysis settings

In [ ]:
MASTER_SEED = 20260909
SEED = MASTER_SEED
np.random.seed(MASTER_SEED)
random.seed(MASTER_SEED)
torch.manual_seed(MASTER_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MASTER_SEED)
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)  # Backend fallbacks are reported in the run manifest
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FULL_RUN = True
RUN_LOSO = True
RUN_EXTRA_MODE_CHECK = True
RUN_TIMING = True
DT = 5e-05
STATE_SCALE = np.array([30.0, 30.0, 0.35], dtype=float)
REPORT_VOLTAGE_FEASIBILITY = True
SPLIT_MODE = 'spatial_block_guarded'
SPATIAL_NBINS = 5
SPATIAL_GUARD_A = 75.0
SPATIAL_SEARCH_ITERS = 20000 if FULL_RUN else 2000
NEURAL_MAX_PERTURB_NORM = 0.75
LONG_PERIODS = 1.0
LIP_COORD_EPS = 0.001
LIP_COORD_SAMPLES = 300 if FULL_RUN else 80
CROSSED_BOOTSTRAP_B = 50000 if FULL_RUN else 2000
BOOTSTRAP_RNG_REPEATS = 5 if FULL_RUN else 2
RUN_DIRECT_RHS_JACOBIAN_AUDIT = True
RUN_OFFGRID_JACOBIAN_AUDIT = True
RUN_REFERENCE_PRECISION_AUDIT = True
RUN_INDEPENDENT_REFERENCE_AUDIT = True
RUN_SOLVER_STEP_ATLAS = True
RUN_EXACT_WITHIN_CELL_JACOBIAN_AUDIT = True
RUN_OFFGRID_VOLTAGE_STABILITY_AUDIT = True
RUN_ROLLOUT_CELL_CONTACT_AUDIT = True
RUN_DERIVATIVE_CONSISTENT_LOCAL_BASELINE = True
RUN_LEAVE_ONE_SEED_OUT_BOOTSTRAP = True
RUN_MISMATCH_STABILITY_COUPLING = True
RUN_OFFGRID_INTRACELL_ROBUSTNESS = True
RUN_DERIVATIVE_CONSISTENT_TRAJECTORY_AUDIT = True
RUN_SYNTHETIC_NONCOMMUTATION_EXAMPLE = True
RUN_DERIVATIVE_CONSISTENT_CONDITIONING_AUDIT = True
RUN_EXHAUSTIVE_INTRACELL_AUDIT = True
RUN_DC_BOUNDARY_REFERENCE_AUDIT = True
RUN_INTERPOLATION_FAMILY_SENSITIVITY = True
RUN_GRID_RESOLUTION_SENSITIVITY = True
RUN_EVENT_ALIGNED_REFERENCE_AUDIT = True
INTERPOLATION_SENSITIVITY_METHOD = 'cubic'
GRID_COARSEN_FACTORS = (1, 2, 3)
EVENT_ALIGNED_MAX_SCENARIOS = 30 if FULL_RUN else 6
EVENT_ALIGNED_MAX_RESTARTS = 500
EVENT_ALIGNED_NUDGE_REL = 1e-10
EXPORT_SELECTION_MANIFESTS = True
FIGURE_DPI = 600
EXPORT_AUDIT_EVIDENCE = False  # Separate optional detailed evidence archive
VERIFY_INPUT_MD5 = True
DIRECT_RHS_JAC_EPS = [0.0003, 0.001, 0.003]
DIRECT_RHS_JAC_PRIMARY_EPS = 0.001
OFFGRID_JAC_PRIMARY_EPS = 0.001
SPECTRAL_ABSCISSA_NEAR_ZERO_THRESHOLDS = [1.0, 5.0, 10.0]
HPO_COMMON_SEED = SEED + 1000
DOP853_MIN_MARGIN_NORMALIZED = 0.1
DOP853_RTOL = 1e-10
DOP853_ATOL = np.array([1e-09, 1e-09, 1e-11], dtype=float)
if FULL_RUN:
    N_TRIALS = 25
    HPO_EPOCHS = 40
    FINAL_EPOCHS = 100
    PRIMARY_SEEDS = 8
    ABLATION_SEEDS = 4
    SAMPLES_PER_OP = 5
    ONE_STEP_REF_SUBSTEPS = 20
    HPO_TRAIN_OPS = 1500
    HPO_VAL_OPS = 500
    TRAIN_ROLLOUT_OPS = 800
    VAL_ROLLOUT_OPS = 400
    MAIN_HORIZON = 80
    ROBUST_HORIZON = 40
    ROLLOUT_REF_SUBSTEPS = 10
    ROBUST_MODEL_SEEDS = PRIMARY_SEEDS
    LOSO_EPOCHS = 60
    LOSO_SEEDS = 1
    FULL_MAP_EPS = 0.001
    EPS_SENSITIVITY = [0.0001, 0.0003, 0.001, 0.003]
    EPS_SENSITIVITY_SCENARIOS = 1500
    FINITE_TIME_STEPS = 20
    FINITE_TIME_MODEL_SEEDS = PRIMARY_SEEDS
    REFERENCE_AUDIT_TARGET_VALID = 100
    REFERENCE_AUDIT_MAX_ATTEMPTS = 300
    OFFGRID_JACOBIAN_POINTS = 5000
    OFFGRID_ROBUST_CELLS = 1000
    DERIVATIVE_CONSISTENT_TRAJECTORY_BASES = 30
    DERIVATIVE_CONSISTENT_REF_SUBSTEPS = 20
    DC_REFERENCE_AUDIT_SUBSTEPS = (10, 20, 40)
    DC_DOP853_MAX_SCENARIOS = 60
    EXHAUSTIVE_INTRACELL_BATCH = 8192
    TIMING_REPEATS = 200
    ROLLOUT_UPDATES = 3
    ROLLOUT_BATCH = 192
else:
    N_TRIALS = 2
    HPO_EPOCHS = 4
    FINAL_EPOCHS = 8
    PRIMARY_SEEDS = 2
    ABLATION_SEEDS = 1
    SAMPLES_PER_OP = 2
    ONE_STEP_REF_SUBSTEPS = 4
    HPO_TRAIN_OPS = 120
    HPO_VAL_OPS = 60
    TRAIN_ROLLOUT_OPS = 80
    VAL_ROLLOUT_OPS = 40
    MAIN_HORIZON = 12
    ROBUST_HORIZON = 8
    ROLLOUT_REF_SUBSTEPS = 3
    ROBUST_MODEL_SEEDS = 1
    LOSO_EPOCHS = 5
    LOSO_SEEDS = 1
    FULL_MAP_EPS = 0.001
    EPS_SENSITIVITY = [0.0001, 0.0003, 0.001, 0.003]
    EPS_SENSITIVITY_SCENARIOS = 1500
    FINITE_TIME_STEPS = 20
    FINITE_TIME_MODEL_SEEDS = PRIMARY_SEEDS
    REFERENCE_AUDIT_TARGET_VALID = 10
    REFERENCE_AUDIT_MAX_ATTEMPTS = 30
    OFFGRID_JACOBIAN_POINTS = 500
    OFFGRID_ROBUST_CELLS = 120
    DERIVATIVE_CONSISTENT_TRAJECTORY_BASES = 8
    DERIVATIVE_CONSISTENT_REF_SUBSTEPS = 8
    DC_REFERENCE_AUDIT_SUBSTEPS = (4, 8, 12)
    DC_DOP853_MAX_SCENARIOS = 12
    EXHAUSTIVE_INTRACELL_BATCH = 4096
    TIMING_REPEATS = 50
    ROLLOUT_UPDATES = 1
    ROLLOUT_BATCH = 64
    RUN_LOSO = False
OFFGRID_ROBUST_FRACTIONS = (0.25, 0.75)
DERIVATIVE_CONSISTENT_PERTURB_CELL_FRACTION = 0.15
DERIVATIVE_CONSISTENT_DIRECTIONS = ((1.0, -1.0, 1.0), (-1.0, 1.0, -1.0))
SELECTION_PROTOCOL_VERSION = 'balanced_selection'
PRIMARY_OFFGRID_SAMPLE_SEED = SEED + 184
INTRACELL_PARENT_SELECTION_MODE = 'full_candidate_pool_balanced_even_rank'
DC_BASE_SELECTION_MODE = 'speed_x_deployed_hurwitz_stratified_mismatch_even_rank'
SYNTHETIC_EXAMPLE_GRID_POINTS = 161 if FULL_RUN else 41
SYNTHETIC_GRID_REFINEMENT_POINTS = (21, 41, 81, 161) if FULL_RUN else (21, 41)
DC_BOUNDARY_POLICY = 'half_open_right_internal_knots_last_cell_at_upper_boundary'
DC_INTERFACE_TOL_NORMALIZED = 1e-10
DC_DOP853_RTOL = 1e-10
DC_DOP853_ATOL = np.array([1e-09, 1e-09, 1e-11], dtype=float)
DC_DOP853_MAX_STEP = DT / (40 if FULL_RUN else 12)
BASE_DIR = Path(os.environ.get('EESM_WORKDIR', '/content' if Path('/content').exists() else str(Path.cwd())))
BASE_DIR.mkdir(parents=True, exist_ok=True)
CODE_VERSION = '2026.10.18-v2.3'
import tempfile
from datetime import datetime as _RunDateTime
_results_workspace = tempfile.TemporaryDirectory(prefix='eesm_results_work_')
RESULTS_OUT = BASE_DIR / ('EESM_LUT_Results_' + _RunDateTime.now().strftime('%Y%m%d_%H%M%S_%f'))
OUT = Path(_results_workspace.name) / 'analysis'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
MOD = OUT / 'models'
for d in [OUT, FIG, TAB, MOD]:
    d.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('Full run:', FULL_RUN)


## Local ZIP input

Provide exactly one ZIP archive; direct MAT uploads and automatic network downloads are disabled. For local Python execution, set `EESM_INPUT_ZIP=/absolute/path/to/data.zip`. Expected file checksums identify the source-data version.


In [ ]:
import io
import zipfile

DATA = BASE_DIR / 'eesm_data'
DATA.mkdir(parents=True, exist_ok=True)
REQUIRED_FILES = ('Parameters.mat', 'Tables.mat')
OPTIONAL_FILES = ()
ACCEPTED_FILES = set(REQUIRED_FILES) | set(OPTIONAL_FILES)
MD5_EXPECTED = {
    'Parameters.mat': '8c94ef1c92d47215f7143d79cac08b6d',
    'Tables.mat': '9846d57760e4375ac59b508450e5b1e1',
}


def open_input_archive():
    try:
        from google.colab import files as colab_files
    except ImportError:
        name = os.environ.get('EESM_INPUT_ZIP', '').strip()
        if name:
            path = Path(name).expanduser()
        else:
            archives = sorted(Path.cwd().glob('*.zip'))
            if len(archives) != 1:
                raise FileNotFoundError(
                    'Set EESM_INPUT_ZIP to a local ZIP path, or keep exactly '
                    'one ZIP file in the working directory.'
                )
            path = archives[0]
        if not path.is_file() or path.suffix.lower() != '.zip':
            raise FileNotFoundError(f'Local ZIP not found: {path}')
        print(f'Input ZIP: {path}')
        return zipfile.ZipFile(path, 'r')

    print('Select a local ZIP archive containing Parameters.mat and Tables.mat.')
    uploaded = colab_files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one ZIP archive.')
    name, content = next(iter(uploaded.items()))
    if Path(name).suffix.lower() != '.zip':
        raise ValueError('The selected file must be a ZIP archive.')
    print(f'Input ZIP: {name}')
    return zipfile.ZipFile(io.BytesIO(content), 'r')


# Reject missing or repeated MAT files before writing anything to disk.
with open_input_archive() as archive:
    selected = {}
    for info in archive.infolist():
        if info.is_dir():
            continue
        name = Path(info.filename.replace('\\', '/')).name
        if name not in ACCEPTED_FILES:
            continue
        if name in selected:
            raise ValueError(f'Duplicate file in ZIP: {name}')
        if info.file_size > 100 * 1024 * 1024:
            raise ValueError(f'Unexpectedly large MAT file: {name}')
        selected[name] = info

    missing = sorted(set(REQUIRED_FILES) - set(selected))
    if missing:
        raise FileNotFoundError('Missing ZIP entries: ' + ', '.join(missing))

    for name in ACCEPTED_FILES:
        (DATA / name).unlink(missing_ok=True)
    for name, info in sorted(selected.items()):
        with archive.open(info, 'r') as source, (DATA / name).open('wb') as dest:
            shutil.copyfileobj(source, dest)
        if (DATA / name).stat().st_size != info.file_size:
            raise IOError(f'Incomplete extraction: {name}')

if VERIFY_INPUT_MD5:
    for name in REQUIRED_FILES:
        digest = hashlib.md5((DATA / name).read_bytes()).hexdigest()
        if digest != MD5_EXPECTED[name]:
            raise ValueError(f'Unexpected checksum for {name}: {digest}')

print('FEA files:', ', '.join(sorted(selected)))

INPUT_CHECKSUMS = {name: hashlib.sha256((DATA / name).read_bytes()).hexdigest() for name in REQUIRED_FILES}
print('Input SHA-256:', INPUT_CHECKSUMS)


## Model and analysis functions

In [ ]:
def load_data(data_dir):
    P = sio.loadmat(Path(data_dir) / 'Parameters.mat', squeeze_me=True)
    T = sio.loadmat(Path(data_dir) / 'Tables.mat', squeeze_me=True)
    out = {}
    for k in ['id_vec', 'iq_vec', 'if_vec', 'n_vec']:
        out[k] = np.asarray(T[k], dtype=float).ravel()
    out['psid'] = np.asarray(T['IdIqIf_Fluxd_grid'], dtype=float)
    out['psiq'] = np.asarray(T['IdIqIf_Fluxq_grid'], dtype=float)
    out['psif'] = np.asarray(T['IdIqIf_Fluxf_grid'], dtype=float)
    out['Tem'] = np.asarray(T['IdIqIf_Tem_grid'], dtype=float)
    out['Pfe'] = np.asarray(T['IdIqIfN_Pfe_grid'], dtype=float)
    out['Rs'] = float(P['R_s'])
    out['Rf'] = float(P['R_f'])
    out['p'] = int(P['p'])
    out['ismax'] = float(P['i_s_max'])
    out['ifmax'] = float(P['i_f_max'])
    out['udc'] = float(P['u_dc'])
    return out

def derivative_grids(D, method='edge2'):
    iq, idv, ifv = (D['iq_vec'], D['id_vec'], D['if_vec'])
    fluxes = [D['psid'], D['psiq'], D['psif']]
    allg = []
    for arr in fluxes:
        if method in ('edge1', 'edge2'):
            eo = 1 if method == 'edge1' else 2
            giq, gid, gif = np.gradient(arr, iq, idv, ifv, edge_order=eo)
        elif method == 'spline':
            giq = CubicSpline(iq, arr, axis=0, bc_type='natural').derivative()(iq)
            gid = CubicSpline(idv, arr, axis=1, bc_type='natural').derivative()(idv)
            gif = CubicSpline(ifv, arr, axis=2, bc_type='natural').derivative()(ifv)
        else:
            raise ValueError(method)
        allg.append((gid, giq, gif))
    L = np.empty(D['psid'].shape + (3, 3), float)
    for r in range(3):
        for c in range(3):
            L[..., r, c] = allg[r][c]
    return L

def local_A(L, Rs, Rf, p, n_rpm):
    we = 2 * np.pi * p * n_rpm / 60.0
    Omega = np.array([[0, we, 0], [-we, 0, 0], [0, 0, 0]], float)
    R = np.diag([Rs, Rs, Rf])
    return np.linalg.solve(L, -R + Omega @ L)

def dtcrit_from_eigs(eigs):
    if np.any(np.real(eigs) >= 0):
        return 0.0
    return float(np.min(-2 * np.real(eigs) / np.abs(eigs) ** 2))

def build_atlas(D, Lgrid, dt=DT):
    I_lim = np.sqrt(3) * D['ismax']
    mask = D['iq_vec'][:, None] ** 2 + D['id_vec'][None, :] ** 2 <= I_lim ** 2 + 1e-12
    rows = []
    op = []
    failed = []
    for n in D['n_vec']:
        for k, iff in enumerate(D['if_vec']):
            for j, iq in enumerate(D['iq_vec']):
                for i, id_ in enumerate(D['id_vec']):
                    if not mask[j, i]:
                        continue
                    L = Lgrid[j, i, k]
                    if not np.all(np.isfinite(L)):
                        failed.append((float(n), float(iff), float(id_), float(iq), 'nonfinite_L'))
                        continue
                    try:
                        A = local_A(L, D['Rs'], D['Rf'], D['p'], n)
                    except np.linalg.LinAlgError:
                        failed.append((float(n), float(iff), float(id_), float(iq), 'singular_L'))
                        continue
                    ev = np.linalg.eigvals(A)
                    dc = dtcrit_from_eigs(ev)
                    rho = float(np.max(np.abs(np.linalg.eigvals(np.eye(3) + dt * A))))
                    rec = dict(n_rpm=float(n), if_A=float(iff), id_A=float(id_), iq_A=float(iq), j=j, i=i, k=k, dtcrit_s=dc, rho_euler=rho, cont_stable=bool(np.max(np.real(ev)) < 0), cond_L=float(np.linalg.cond(L)))
                    op.append({**rec, 'A': A, 'L': L})
                    rows.append(rec)
    if failed:
        failed_df = pd.DataFrame(failed, columns=['n_rpm', 'if_A', 'id_A', 'iq_A', 'reason'])
        failed_df.to_csv(TAB / 'table00_atlas_failures.csv', index=False)
        raise RuntimeError(f"Atlas construction failed at {len(failed)} operating point(s). See {TAB / 'table00_atlas_failures.csv'}.")
    return (pd.DataFrame(rows), op, mask)

class TorchFEA:

    def __init__(self, D, Lgrid, device=DEVICE, dtype=torch.float32):
        self.device = device
        self.dtype = dtype
        self.idv = torch.tensor(D['id_vec'], dtype=dtype, device=device)
        self.iqv = torch.tensor(D['iq_vec'], dtype=dtype, device=device)
        self.ifv = torch.tensor(D['if_vec'], dtype=dtype, device=device)
        self.nvec = torch.tensor(D['n_vec'], dtype=dtype, device=device)
        self.id0 = float(D['id_vec'][0])
        self.iq0 = float(D['iq_vec'][0])
        self.if0 = float(D['if_vec'][0])
        self.did = float(D['id_vec'][1] - D['id_vec'][0])
        self.diq = float(D['iq_vec'][1] - D['iq_vec'][0])
        self.dif = float(D['if_vec'][1] - D['if_vec'][0])
        self.nid = len(D['id_vec'])
        self.niq = len(D['iq_vec'])
        self.nif = len(D['if_vec'])
        self.flux = torch.tensor(np.stack([D['psid'], D['psiq'], D['psif']], axis=-1), dtype=dtype, device=device)
        self.L = torch.tensor(Lgrid, dtype=dtype, device=device)
        self.Rs = float(D['Rs'])
        self.Rf = float(D['Rf'])
        self.p = int(D['p'])

    def in_domain(self, x, tol=1e-07):
        return (x[:, 0] >= self.id0 - tol) & (x[:, 0] <= float(self.idv[-1]) + tol) & (x[:, 1] >= self.iq0 - tol) & (x[:, 1] <= float(self.iqv[-1]) + tol) & (x[:, 2] >= self.if0 - tol) & (x[:, 2] <= float(self.ifv[-1]) + tol)

    def _interp(self, V, x):
        uid = (x[:, 0] - self.id0) / self.did
        uiq = (x[:, 1] - self.iq0) / self.diq
        uif = (x[:, 2] - self.if0) / self.dif
        eps = 1e-06
        uid = torch.clamp(uid, 0, self.nid - 1 - eps)
        uiq = torch.clamp(uiq, 0, self.niq - 1 - eps)
        uif = torch.clamp(uif, 0, self.nif - 1 - eps)
        i0 = torch.floor(uid).long()
        q0 = torch.floor(uiq).long()
        f0 = torch.floor(uif).long()
        i1 = i0 + 1
        q1 = q0 + 1
        f1 = f0 + 1
        wi = uid - i0.to(uid.dtype)
        wq = uiq - q0.to(uiq.dtype)
        wf = uif - f0.to(uif.dtype)
        tail = (1,) * (V.ndim - 3)
        wi = wi.reshape((-1,) + tail)
        wq = wq.reshape((-1,) + tail)
        wf = wf.reshape((-1,) + tail)
        c000 = V[q0, i0, f0]
        c001 = V[q0, i0, f1]
        c010 = V[q0, i1, f0]
        c011 = V[q0, i1, f1]
        c100 = V[q1, i0, f0]
        c101 = V[q1, i0, f1]
        c110 = V[q1, i1, f0]
        c111 = V[q1, i1, f1]
        c00 = c000 * (1 - wi) + c010 * wi
        c01 = c001 * (1 - wi) + c011 * wi
        c10 = c100 * (1 - wi) + c110 * wi
        c11 = c101 * (1 - wi) + c111 * wi
        c0 = c00 * (1 - wq) + c10 * wq
        c1 = c01 * (1 - wq) + c11 * wq
        return c0 * (1 - wf) + c1 * wf

    def cell_indices(self, x):
        uid = (x[:, 0] - self.id0) / self.did
        uiq = (x[:, 1] - self.iq0) / self.diq
        uif = (x[:, 2] - self.if0) / self.dif
        valid = self.in_domain(x)
        ii = torch.clamp(torch.floor(uid).long(), 0, self.nid - 2)
        qq = torch.clamp(torch.floor(uiq).long(), 0, self.niq - 2)
        ff = torch.clamp(torch.floor(uif).long(), 0, self.nif - 2)
        return (qq, ii, ff, valid)

    def cell_interface_mask(self, x, tol=DC_INTERFACE_TOL_NORMALIZED):
        u = [(x[:, 0] - self.id0) / self.did, (x[:, 1] - self.iq0) / self.diq, (x[:, 2] - self.if0) / self.dif]
        ns = [self.nid, self.niq, self.nif]
        masks = []
        for uu, nn in zip(u, ns):
            nearest = torch.round(uu)
            internal = (nearest > 0) & (nearest < nn - 1)
            masks.append(internal & (torch.abs(uu - nearest) <= tol))
        return masks[0] | masks[1] | masks[2]

    def boundary_policy_name(self):
        return DC_BOUNDARY_POLICY

    def flux_jacobian_at(self, x, interface_tol=1e-10, return_interior_mask=False):
        uid = (x[:, 0] - self.id0) / self.did
        uiq = (x[:, 1] - self.iq0) / self.diq
        uif = (x[:, 2] - self.if0) / self.dif
        eps = 1e-09
        uidc = torch.clamp(uid, 0, self.nid - 1 - eps)
        uiqc = torch.clamp(uiq, 0, self.niq - 1 - eps)
        uifc = torch.clamp(uif, 0, self.nif - 1 - eps)
        i0 = torch.floor(uidc).long()
        q0 = torch.floor(uiqc).long()
        f0 = torch.floor(uifc).long()
        i1 = i0 + 1
        q1 = q0 + 1
        f1 = f0 + 1
        wi = uidc - i0.to(uidc.dtype)
        wq = uiqc - q0.to(uiqc.dtype)
        wf = uifc - f0.to(uifc.dtype)
        c000 = self.flux[q0, i0, f0]
        c001 = self.flux[q0, i0, f1]
        c010 = self.flux[q0, i1, f0]
        c011 = self.flux[q0, i1, f1]
        c100 = self.flux[q1, i0, f0]
        c101 = self.flux[q1, i0, f1]
        c110 = self.flux[q1, i1, f0]
        c111 = self.flux[q1, i1, f1]
        didv = ((1 - wq) * (1 - wf))[:, None] * (c010 - c000) + ((1 - wq) * wf)[:, None] * (c011 - c001) + (wq * (1 - wf))[:, None] * (c110 - c100) + (wq * wf)[:, None] * (c111 - c101)
        diqv = ((1 - wi) * (1 - wf))[:, None] * (c100 - c000) + ((1 - wi) * wf)[:, None] * (c101 - c001) + (wi * (1 - wf))[:, None] * (c110 - c010) + (wi * wf)[:, None] * (c111 - c011)
        difv = ((1 - wq) * (1 - wi))[:, None] * (c001 - c000) + ((1 - wq) * wi)[:, None] * (c011 - c010) + (wq * (1 - wi))[:, None] * (c101 - c100) + (wq * wi)[:, None] * (c111 - c110)
        J = torch.stack([didv / self.did, diqv / self.diq, difv / self.dif], dim=2)
        strict = self.in_domain(x) & (wi > interface_tol) & (wi < 1 - interface_tol) & (wq > interface_tol) & (wq < 1 - interface_tol) & (wf > interface_tol) & (wf < 1 - interface_tol)
        return (J, strict) if return_interior_mask else J

    def flux_at(self, x):
        return self._interp(self.flux, x)

    def L_at(self, x):
        return self._interp(self.L, x)

    def equilibrium_voltage(self, base, n_rpm):
        psi = self.flux_at(base)
        we = 2 * math.pi * self.p * n_rpm / 60.0
        return torch.stack([self.Rs * base[:, 0] - we * psi[:, 1], self.Rs * base[:, 1] + we * psi[:, 0], self.Rf * base[:, 2]], dim=1)

    def rhs(self, x, n_rpm, ueq):
        psi = self.flux_at(x)
        L = self.L_at(x)
        we = 2 * math.pi * self.p * n_rpm / 60.0
        b = torch.stack([ueq[:, 0] - self.Rs * x[:, 0] + we * psi[:, 1], ueq[:, 1] - self.Rs * x[:, 1] - we * psi[:, 0], ueq[:, 2] - self.Rf * x[:, 2]], dim=1)
        return torch.linalg.solve(L, b.unsqueeze(-1)).squeeze(-1)

def exact_equilibrium_rhs_jacobian(fea, x, n_rpm):
    L = fea.L_at(x)
    Jpsi, strict = fea.flux_jacobian_at(x, return_interior_mask=True)
    we = 2 * math.pi * fea.p * n_rpm / 60.0
    B = torch.zeros_like(Jpsi)
    B[:, 0, :] = we[:, None] * Jpsi[:, 1, :]
    B[:, 1, :] = -we[:, None] * Jpsi[:, 0, :]
    B[:, 0, 0] -= fea.Rs
    B[:, 1, 1] -= fea.Rs
    B[:, 2, 2] -= fea.Rf
    A = torch.linalg.solve(L, B)
    return (A, Jpsi, strict)

def problem_cell_mask_torch(fea, x, n_rpm, lookup):
    if lookup is None:
        return torch.zeros(x.shape[0], dtype=torch.bool, device=x.device)
    q, i, f, valid = fea.cell_indices(x)
    nidx = torch.argmin(torch.abs(n_rpm[:, None] - fea.nvec[None, :]), dim=1)
    return valid & lookup[nidx, q, i, f]

def rk4_step(fea, x, n, ueq, h, substeps=1, return_domain_contact=False):
    hh = h / substeps
    y = x
    contact = torch.zeros(y.shape[0], dtype=torch.bool, device=y.device) if return_domain_contact else None
    if return_domain_contact:
        contact |= ~fea.in_domain(y)
    for _ in range(substeps):
        k1 = fea.rhs(y, n, ueq)
        y2 = y + 0.5 * hh * k1
        if return_domain_contact:
            contact |= ~fea.in_domain(y2)
        k2 = fea.rhs(y2, n, ueq)
        y3 = y + 0.5 * hh * k2
        if return_domain_contact:
            contact |= ~fea.in_domain(y3)
        k3 = fea.rhs(y3, n, ueq)
        y4 = y + hh * k3
        if return_domain_contact:
            contact |= ~fea.in_domain(y4)
        k4 = fea.rhs(y4, n, ueq)
        y = y + hh / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
        if return_domain_contact:
            contact |= ~fea.in_domain(y)
    return (y, contact) if return_domain_contact else y

def heun_step(fea, x, n, ueq, h):
    k1 = fea.rhs(x, n, ueq)
    k2 = fea.rhs(x + h * k1, n, ueq)
    return x + 0.5 * h * (k1 + k2)

def rk4_macro(fea, x, n, ueq, h):
    return rk4_step(fea, x, n, ueq, h, 1)

def euler_step(fea, x, n, ueq, h):
    return x + h * fea.rhs(x, n, ueq)

def make_op_arrays(ops, D, fea):
    base = np.array([[o['id_A'], o['iq_A'], o['if_A']] for o in ops], np.float32)
    n = np.array([o['n_rpm'] for o in ops], np.float32)
    with torch.no_grad():
        bt = torch.tensor(base, device=fea.device)
        nt = torch.tensor(n, device=fea.device)
        u = fea.equilibrium_voltage(bt, nt).cpu().numpy().astype(np.float32)
    return (base, n, u)

def split_ops_random(ops, eligible_idx, seed=SEED):
    rng = np.random.default_rng(seed)
    strata = {}
    for gi in eligible_idx:
        o = ops[int(gi)]
        key = (int(o['n_rpm']), float(o['if_A']), int(o['rho_euler'] > 1))
        strata.setdefault(key, []).append(int(gi))
    tr = []
    va = []
    te = []
    for ids in strata.values():
        ids = np.array(ids)
        rng.shuffle(ids)
        m = len(ids)
        ntr = max(1, int(round(0.6 * m)))
        nva = max(1, int(round(0.2 * m)))
        if ntr + nva >= m:
            ntr = max(1, m - 2)
            nva = 1
        tr += ids[:ntr].tolist()
        va += ids[ntr:ntr + nva].tolist()
        te += ids[ntr + nva:].tolist()
    return (np.array(tr), np.array(va), np.array(te))

def split_ops_spatial_guarded(ops, eligible_idx, nbins=5, guard_A=75.0, search_iters=20000, seed=SEED):
    eligible_idx = np.asarray(eligible_idx, dtype=int)
    rows = []
    for gi in eligible_idx:
        o = ops[int(gi)]
        rows.append((int(gi), float(o['id_A']), float(o['iq_A']), float(o['n_rpm']), float(o['if_A']), int(o['rho_euler'] > 1)))
    e = pd.DataFrame(rows, columns=['op_index', 'id_A', 'iq_A', 'n_rpm', 'if_A', 'unstable'])
    coords = e[['id_A', 'iq_A']].drop_duplicates().sort_values(['iq_A', 'id_A']).reset_index(drop=True)
    xmin, xmax = (float(coords.id_A.min()), float(coords.id_A.max()))
    ymin, ymax = (float(coords.iq_A.min()), float(coords.iq_A.max()))
    xedges = np.linspace(xmin - 1e-09, xmax + 1e-09, int(nbins) + 1)
    yedges = np.linspace(ymin - 1e-09, ymax + 1e-09, int(nbins) + 1)
    coords['xb'] = np.digitize(coords.id_A.to_numpy(), xedges[1:-1])
    coords['yb'] = np.digitize(coords.iq_A.to_numpy(), yedges[1:-1])
    coords['spatial_block'] = (coords.xb + int(nbins) * coords.yb).astype(int)
    coord_key = {(float(r.id_A), float(r.iq_A)): i for i, r in coords.iterrows()}
    e['coord_index'] = [coord_key[float(a), float(b)] for a, b in zip(e.id_A, e.iq_A)]
    e = e.merge(coords[['id_A', 'iq_A', 'spatial_block']], on=['id_A', 'iq_A'], how='left')
    strata = sorted(e[['n_rpm', 'if_A', 'unstable']].drop_duplicates().itertuples(index=False, name=None))
    sidx = {k: i for i, k in enumerate(strata)}
    C = np.zeros((len(coords), len(strata)), dtype=int)
    for r in e.itertuples(index=False):
        C[int(r.coord_index), sidx[r.n_rpm, r.if_A, r.unstable]] += 1
    total_strata = C.sum(axis=0)
    total_ops = int(C.sum())
    xy = coords[['id_A', 'iq_A']].to_numpy(float)
    dist = np.sqrt(np.sum((xy[:, None, :] - xy[None, :, :]) ** 2, axis=2))
    barr = coords.spatial_block.to_numpy(int)
    blocks = np.unique(barr)
    n_val = max(1, int(round(0.2 * len(blocks))))
    n_test = max(1, int(round(0.2 * len(blocks))))
    rng = np.random.default_rng(seed)
    best = None
    target = {'train': 0.42, 'validation': 0.14, 'test': 0.2}
    for _ in range(int(search_iters)):
        perm = rng.permutation(blocks)
        vb = set((int(x) for x in perm[:n_val]))
        tb = set((int(x) for x in perm[n_val:n_val + n_test]))
        trb = set((int(x) for x in perm[n_val + n_test:]))
        tm = np.isin(barr, list(trb))
        vm = np.isin(barr, list(vb))
        sm = np.isin(barr, list(tb))
        held = vm | sm
        trkeep = tm & (np.min(dist[:, held], axis=1) > float(guard_A))
        vkeep = vm & (np.min(dist[:, sm], axis=1) > float(guard_A))
        masks = {'train': trkeep, 'validation': vkeep, 'test': sm}
        counts = {k: int(C[m].sum()) for k, m in masks.items()}
        if counts['train'] < 0.32 * total_ops or counts['validation'] < 0.07 * total_ops or counts['test'] < 0.12 * total_ops:
            continue
        loss = 0.0
        for k, m in masks.items():
            v = C[m].sum(axis=0)
            fr = max(counts[k] / total_ops, 1e-09)
            exp = fr * total_strata
            loss += float(np.mean(((v - exp) / np.maximum(exp, 1.0)) ** 2))
            loss += 0.35 * ((fr - target[k]) / max(target[k], 1e-09)) ** 2
        for k in ('validation', 'test'):
            loss += 5.0 * float(np.mean(C[masks[k]].sum(axis=0) == 0))
        if best is None or loss < best[0]:
            best = (loss, trb, vb, tb, masks, counts)
    if best is None:
        raise RuntimeError('Could not construct the spatially guarded split. Reduce SPATIAL_GUARD_A or increase search iterations.')
    _, trb, vb, tb, masks, counts = best
    partition = np.full(len(coords), 'guard', dtype=object)
    partition[masks['train']] = 'train'
    partition[masks['validation']] = 'validation'
    partition[masks['test']] = 'test'
    coords['partition'] = partition
    cmap = {(float(r.id_A), float(r.iq_A)): str(r.partition) for r in coords.itertuples(index=False)}
    bmap = {(float(r.id_A), float(r.iq_A)): int(r.spatial_block) for r in coords.itertuples(index=False)}
    e['partition'] = [cmap[float(a), float(b)] for a, b in zip(e.id_A, e.iq_A)]
    tr = e.loc[e.partition == 'train', 'op_index'].to_numpy(int)
    va = e.loc[e.partition == 'validation', 'op_index'].to_numpy(int)
    te = e.loc[e.partition == 'test', 'op_index'].to_numpy(int)
    gd = e.loc[e.partition == 'guard', 'op_index'].to_numpy(int)
    op_block = {int(r.op_index): int(r.spatial_block) for r in e.itertuples(index=False)}

    def mindist(pa, pb):
        A = coords.loc[coords.partition == pa, ['id_A', 'iq_A']].to_numpy(float)
        B = coords.loc[coords.partition == pb, ['id_A', 'iq_A']].to_numpy(float)
        if len(A) == 0 or len(B) == 0:
            return np.nan
        return float(np.sqrt(np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=2)).min())
    audit = []
    for name, ids in [('train', tr), ('validation', va), ('test', te), ('guard', gd)]:
        c = coords[coords.partition == name]
        audit.append({'partition': name, 'operating_points': len(ids), 'current_plane_points': len(c), 'spatial_blocks': int(c.spatial_block.nunique()) if len(c) else 0})
    audit_df = pd.DataFrame(audit)
    distance_df = pd.DataFrame([{'train_validation_min_current_plane_distance_A': mindist('train', 'validation'), 'train_test_min_current_plane_distance_A': mindist('train', 'test'), 'validation_test_min_current_plane_distance_A': mindist('validation', 'test'), 'guard_threshold_A': float(guard_A), 'grid_blocks': int(len(blocks)), 'search_objective': float(best[0])}])
    return (tr, va, te, gd, audit_df, distance_df, coords[['id_A', 'iq_A', 'spatial_block', 'partition']].copy(), op_block)

def dominant_direction(A):
    S = np.diag(STATE_SCALE)
    An = np.linalg.solve(S, A @ S)
    M = np.eye(3) + DT * An
    vals, vecs = np.linalg.eig(M)
    v = vecs[:, np.argmax(np.abs(vals))]
    r = np.real(v)
    if np.linalg.norm(r) < 1e-08:
        r = np.imag(v)
    r = r / np.linalg.norm(r)
    return r.astype(np.float32)

class ResidualNet(nn.Module):

    def __init__(self, width=96, depth=3, beta=0.2, certified=True):
        super().__init__()
        self.beta = float(beta)
        layers = []
        dims = [8] + [int(width)] * int(depth) + [3]
        for li, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            m = nn.Linear(a, b)
            if certified:
                m = spectral_norm(m, n_power_iterations=5)
            layers.append(m)
            if li < len(dims) - 2:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.beta * self.net(z)

def features_torch(fea, x, base, n, ueq, xmu, xsd):
    rhs = fea.rhs(x, n, ueq)
    xn = (x - base) / torch.as_tensor(STATE_SCALE, dtype=x.dtype, device=x.device)
    dxn = DT * rhs / torch.as_tensor(STATE_SCALE, dtype=x.dtype, device=x.device)
    raw = torch.cat([xn, dxn, (n / 9000.0).unsqueeze(1), (base[:, 2] / 10.0).unsqueeze(1)], dim=1)
    return ((raw - xmu) / xsd, rhs)

def corrected_step(model, fea, x, base, n, ueq, xmu, xsd):
    z, rhs = features_torch(fea, x, base, n, ueq, xmu, xsd)
    corr = model(z) * torch.as_tensor(STATE_SCALE, dtype=x.dtype, device=x.device)
    return x + DT * rhs + corr

def rollout_loss(model, fea, base, n, ueq, x0n, xmu, xsd, ref_traj):
    scale = torch.as_tensor(STATE_SCALE, dtype=base.dtype, device=base.device)
    x = base + x0n * scale
    loss = 0.0
    H = ref_traj.shape[1] - 1
    for t in range(H):
        x = corrected_step(model, fea, x, base, n, ueq, xmu, xsd)
        diff = (x - ref_traj[:, t + 1]) / scale
        loss = loss + torch.mean(diff * diff)
    return loss / H

def generate_one_step_samples(fea, ops, base_all, n_all, ueq_all, op_ids, samples_per_op=4, seed=SEED, ref_substeps=10, batch_size=4096):
    rng = np.random.default_rng(seed)
    gids = np.repeat(np.asarray(op_ids, dtype=int), samples_per_op)
    x0n = rng.normal(0, 0.45, size=(len(gids), 3)).astype(np.float32)
    x0n = np.clip(x0n, -1, 1)
    raw_all = []
    y_all = []
    kept_gids = []
    kept_x0n = []
    rejected = 0
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    for st in range(0, len(gids), batch_size):
        sl = slice(st, min(st + batch_size, len(gids)))
        gi = gids[sl]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        xn = torch.tensor(x0n[sl], device=fea.device)
        x = base + xn * scale
        with torch.no_grad():
            initial_ok = fea.in_domain(x)
            rhs = fea.rhs(x, n, u)
            xe = x + DT * rhs
            xr, ref_contact = rk4_step(fea, x, n, u, DT, ref_substeps, return_domain_contact=True)
            dxn = DT * rhs / scale
            raw = torch.cat([xn, dxn, (n / 9000.0).unsqueeze(1), (base[:, 2] / 10.0).unsqueeze(1)], 1)
            y = (xr - xe) / scale
            keep = initial_ok & ~ref_contact
        raw_all.append(raw[keep].cpu().numpy())
        y_all.append(y[keep].cpu().numpy())
        kept_gids.append(gi[keep.cpu().numpy()])
        kept_x0n.append(x0n[sl][keep.cpu().numpy()])
        rejected += int((~keep).sum().item())
    if rejected:
        print(f'Excluded {rejected} one-step sample(s) that contacted the FEA table boundary.')
    return (np.vstack(raw_all).astype(np.float32), np.vstack(y_all).astype(np.float32), np.concatenate(kept_gids), np.vstack(kept_x0n).astype(np.float32))

def make_rollout_bank(fea, ops, base_all, n_all, ueq_all, gids, x0n, horizon=20, ref_substeps=10, batch_size=4096, problem_cell_lookup=None):
    gids = np.asarray(gids, dtype=int)
    x0n = np.asarray(x0n, dtype=np.float32)
    assert len(gids) == len(x0n)
    refs = []
    contacts = []
    first_contacts = []
    problem_contacts = []
    problem_endpoint_counts = []
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    lookup = problem_cell_lookup.to(fea.device) if problem_cell_lookup is not None else None
    for st in range(0, len(gids), batch_size):
        en = min(st + batch_size, len(gids))
        gi = gids[st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        x = base + torch.tensor(x0n[st:en], device=fea.device) * scale
        traj = [x.cpu().numpy()]
        contact = ~fea.in_domain(x)
        first = torch.where(contact, torch.zeros_like(contact, dtype=torch.int64), torch.full_like(contact, -1, dtype=torch.int64))
        p0 = problem_cell_mask_torch(fea, x, n, lookup)
        pcontact = p0.clone()
        pcount = p0.to(torch.int64)
        with torch.no_grad():
            for t in range(horizon):
                x, step_contact = rk4_step(fea, x, n, u, DT, ref_substeps, return_domain_contact=True)
                new = step_contact & ~contact
                first[new] = t + 1
                contact |= step_contact
                ps = problem_cell_mask_torch(fea, x, n, lookup)
                pcontact |= ps
                pcount += ps.to(torch.int64)
                traj.append(x.cpu().numpy())
        refs.append(np.stack(traj, axis=1))
        contacts.append(contact.cpu().numpy())
        first_contacts.append(first.cpu().numpy())
        problem_contacts.append(pcontact.cpu().numpy())
        problem_endpoint_counts.append(pcount.cpu().numpy())
    return {'gids': gids, 'x0n': x0n, 'ref': np.concatenate(refs, axis=0).astype(np.float32), 'horizon': horizon, 'ref_domain_contact': np.concatenate(contacts), 'ref_first_domain_contact_step': np.concatenate(first_contacts), 'ref_nonhurwitz_cell_center_contact': np.concatenate(problem_contacts), 'ref_nonhurwitz_cell_center_endpoint_count': np.concatenate(problem_endpoint_counts), 'problem_cell_contact_definition': 'macro-step endpoint lies in a cell whose exact cell-center equilibrium RHS Jacobian is non-Hurwitz'}

def bank_tensors(bank, base_all, n_all, ueq_all, device):
    gi = bank['gids']
    return (torch.tensor(base_all[gi], device=device), torch.tensor(n_all[gi], device=device), torch.tensor(ueq_all[gi], device=device), torch.tensor(bank['x0n'], device=device), torch.tensor(bank['ref'], device=device))

def eval_bank(model, fea, bank, base_all, n_all, ueq_all, xmu_np, xsd_np, batch_size=4096):
    model.eval()
    rms = []
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    xmu = torch.tensor(xmu_np, device=fea.device)
    xsd = torch.tensor(xsd_np, device=fea.device)
    gi_all = bank['gids']
    for st in range(0, len(gi_all), batch_size):
        en = min(st + batch_size, len(gi_all))
        gi = gi_all[st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        x = base + torch.tensor(bank['x0n'][st:en], device=fea.device) * scale
        ref = torch.tensor(bank['ref'][st:en], device=fea.device)
        se = torch.zeros(len(gi), device=fea.device)
        with torch.no_grad():
            for t in range(bank['horizon']):
                x = corrected_step(model, fea, x, base, n, u, xmu, xsd)
                d = (x - ref[:, t + 1]) / scale
                se += torch.sum(d * d, dim=1)
        rms.append(torch.sqrt(se / (bank['horizon'] * 3)).cpu().numpy())
    return np.concatenate(rms)

def eval_classical_bank(fea, bank, base_all, n_all, ueq_all, method='euler', batch_size=4096):
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    out = []
    gi_all = bank['gids']
    for st in range(0, len(gi_all), batch_size):
        en = min(st + batch_size, len(gi_all))
        gi = gi_all[st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        x = base + torch.tensor(bank['x0n'][st:en], device=fea.device) * scale
        ref = torch.tensor(bank['ref'][st:en], device=fea.device)
        se = torch.zeros(len(gi), device=fea.device)
        with torch.no_grad():
            for t in range(bank['horizon']):
                if method == 'euler':
                    x = euler_step(fea, x, n, u, DT)
                elif method == 'heun':
                    x = heun_step(fea, x, n, u, DT)
                elif method == 'rk4':
                    x = rk4_macro(fea, x, n, u, DT)
                else:
                    raise ValueError(method)
                d = (x - ref[:, t + 1]) / scale
                se += torch.sum(d * d, dim=1)
        out.append(torch.sqrt(se / (bank['horizon'] * 3)).cpu().numpy())
    return np.concatenate(out)

def train_model(params, certified, fea, Xtr, Ytr, Xv, Yv, train_bank, val_bank, base_all, n_all, ueq_all, xmu_np, xsd_np, epochs=30, seed=SEED, rollout_updates=2, rollout_batch=128, patience=10, selection_metric='rollout'):
    torch.manual_seed(seed)
    model = ResidualNet(params['width'], params['depth'], params['beta'], certified).to(fea.device)
    opt = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    ds = TensorDataset(torch.tensor(Xtr), torch.tensor(Ytr))
    dl = DataLoader(ds, batch_size=params['batch_size'], shuffle=True)
    xmu = torch.tensor(xmu_np, device=fea.device)
    xsd = torch.tensor(xsd_np, device=fea.device)
    rng = np.random.default_rng(seed + 17)
    best = np.inf
    state = None
    stale = 0
    hist = []
    for ep in range(epochs):
        model.train()
        losses = []
        for xb, yb in dl:
            xb = xb.to(fea.device)
            yb = yb.to(fea.device)
            opt.zero_grad()
            pred = model(xb)
            loss = torch.mean((pred - yb) ** 2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            opt.step()
            losses.append(float(loss.detach().cpu()))
        lam = float(params.get('lambda_rollout', 0.0))
        H = int(params.get('rollout_horizon', 0))
        if lam > 0 and H > 0:
            for _ in range(rollout_updates):
                ids = rng.choice(len(train_bank['gids']), size=min(rollout_batch, len(train_bank['gids'])), replace=False)
                gi = train_bank['gids'][ids]
                base = torch.tensor(base_all[gi], device=fea.device)
                n = torch.tensor(n_all[gi], device=fea.device)
                u = torch.tensor(ueq_all[gi], device=fea.device)
                x0n = torch.tensor(train_bank['x0n'][ids], device=fea.device)
                ref = torch.tensor(train_bank['ref'][ids, :H + 1], device=fea.device)
                opt.zero_grad()
                rloss = rollout_loss(model, fea, base, n, u, x0n, xmu, xsd, ref)
                (lam * rloss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
                opt.step()
        if ep % 3 == 0 or ep == epochs - 1:
            vmse = float(np.mean((model(torch.tensor(Xv, device=fea.device)).detach().cpu().numpy() - Yv) ** 2))
            vr = eval_bank(model, fea, val_bank, base_all, n_all, ueq_all, xmu_np, xsd_np, batch_size=2048)
            rollout_med = float(np.median(vr))
            score = float(rollout_med + 0.05 * np.sqrt(vmse)) if selection_metric == 'rollout' else float(vmse)
            hist.append((ep, np.mean(losses), vmse, rollout_med, score))
            if score < best - 1e-08:
                best = score
                state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale = 0
            else:
                stale += 3
            if stale >= patience:
                break
    model.load_state_dict(state)
    model.to(fea.device).eval()
    return (model, best, pd.DataFrame(hist, columns=['epoch', 'train_mse', 'val_one_step_mse', 'val_rollout_median_rmse', 'score']))

def empirical_full_map_rho(model, fea, bank, base_all, n_all, ueq_all, xmu_np, xsd_np, eps=0.0001, batch_size=2048, include_euler=True):
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    xmu = torch.tensor(xmu_np, device=fea.device)
    xsd = torch.tensor(xsd_np, device=fea.device)
    rhos = []
    rhos_e = []
    for st in range(0, len(bank['gids']), batch_size):
        en = min(st + batch_size, len(bank['gids']))
        gi = bank['gids'][st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        xn0 = torch.tensor(bank['x0n'][st:en], device=fea.device)
        cols = []
        cols_e = []
        with torch.no_grad():
            for d in range(3):
                e = torch.zeros_like(xn0)
                e[:, d] = eps

                def phi(xn, net=True):
                    x = base + xn * scale
                    if net:
                        y = corrected_step(model, fea, x, base, n, u, xmu, xsd)
                    else:
                        y = euler_step(fea, x, n, u, DT)
                    return (y - base) / scale
                pp = phi(xn0 + e, True)
                pm = phi(xn0 - e, True)
                cols.append(((pp - pm) / (2 * eps)).cpu().numpy())
                if include_euler:
                    pp = phi(xn0 + e, False)
                    pm = phi(xn0 - e, False)
                    cols_e.append(((pp - pm) / (2 * eps)).cpu().numpy())
        J = np.stack(cols, axis=2)
        rhos.append(np.max(np.abs(np.linalg.eigvals(J)), axis=1))
        if include_euler:
            Je = np.stack(cols_e, axis=2)
            rhos_e.append(np.max(np.abs(np.linalg.eigvals(Je)), axis=1))
    return (np.concatenate(rhos), np.concatenate(rhos_e) if include_euler else None)

def scenario_bank_ids(ops, op_ids, amplitudes=(0.25, 0.75), random_dirs=2, seed=SEED):
    gids = []
    dirs = []
    amps = []
    labels = []
    for gi in op_ids:
        rng = np.random.default_rng(seed + int(gi))
        dom = dominant_direction(ops[int(gi)]['A'])
        dlist = [('dom+', dom), ('dom-', -dom), ('id', np.array([1, 0, 0], np.float32)), ('iq', np.array([0, 1, 0], np.float32)), ('if', np.array([0, 0, 1], np.float32))]
        for r in range(random_dirs):
            v = rng.normal(size=3)
            v = v / np.linalg.norm(v)
            dlist.append((f'rand{r + 1}', v.astype(np.float32)))
        for lab, v in dlist:
            for a in amplitudes:
                gids.append(int(gi))
                dirs.append(v)
                amps.append(float(a))
                labels.append(lab)
    x0n = np.asarray(dirs, np.float32) * np.asarray(amps, np.float32)[:, None]
    return (np.array(gids, int), x0n, np.array(labels), np.array(amps))

def architectural_lipschitz_bound(model):
    product = 1.0
    sigmas = []
    for mod in model.net:
        if isinstance(mod, nn.Linear):
            W = mod.weight.detach().cpu().numpy()
            sig = float(np.linalg.svd(W, compute_uv=False)[0])
            sigmas.append(sig)
            product *= sig
    return (float(model.beta * product), sigmas)

def empirical_residual_jacobian_norms(model, Xstd, n_samples=250, seed=SEED):
    rng = np.random.default_rng(seed)
    ids = rng.choice(len(Xstd), size=min(n_samples, len(Xstd)), replace=False)
    vals = []
    model = model.to(DEVICE).eval()
    for idx in ids:
        x = torch.tensor(Xstd[idx:idx + 1], dtype=torch.float32, device=DEVICE, requires_grad=True)

        def f(inp):
            return model(inp).squeeze(0)
        J = torch.autograd.functional.jacobian(f, x).squeeze(1).detach().cpu().numpy()
        vals.append(float(np.linalg.svd(J, compute_uv=False)[0]))
    return np.asarray(vals)

def cluster_stats(df, diff_col, cluster_cols=('n_rpm', 'if_A'), B=3000, seed=SEED):
    c = df.groupby(list(cluster_cols))[diff_col].median().reset_index()
    vals = c[diff_col].to_numpy(float)
    if len(vals) >= 3 and (not np.allclose(vals, 0)):
        W, p = wilcoxon(vals)
    else:
        W, p = (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(B):
        z = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.median(z))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {'clusters': len(vals), 'median_cluster_difference': float(np.median(vals)), 'ci_low': float(lo), 'ci_high': float(hi), 'wilcoxon_W': float(W) if np.isfinite(W) else np.nan, 'p_value': float(p) if np.isfinite(p) else np.nan}


## Operating points and data splits

In [ ]:
D = load_data(DATA)
L_edge2 = derivative_grids(D, 'edge2')
atlas, ops, nominal_mask = build_atlas(D, L_edge2, DT)
fea = TorchFEA(D, L_edge2, device=DEVICE)
base_all, n_all, ueq_all = make_op_arrays(ops, D, fea)
summary = pd.DataFrame([{'pole_pairs': D['p'], 'dc_link_V': D['udc'], 'Rs_ohm': D['Rs'], 'Rf_ohm': D['Rf'], 'phase_current_limit_rms_A': D['ismax'], 'field_current_max_A': D['ifmax'], 'nominal_id_iq_grid_points': int(nominal_mask.sum()), 'local_models': len(atlas), 'continuous_time_hurwitz_fraction': float(atlas.cont_stable.mean()), 'minimum_hcrit_us': float(atlas.dtcrit_s.min() * 1000000.0), 'maximum_hcrit_us': float(atlas.dtcrit_s.max() * 1000000.0)}])
summary.to_csv(TAB / 'table01_dataset_audit.csv', index=False)
normalization_scales = pd.DataFrame([{'state': 'i_d', 'scale_A': float(STATE_SCALE[0])}, {'state': 'i_q', 'scale_A': float(STATE_SCALE[1])}, {'state': 'i_f', 'scale_A': float(STATE_SCALE[2])}])
normalization_scales.to_csv(TAB / 'table01b_state_normalization_scales.csv', index=False)
margin = NEURAL_MAX_PERTURB_NORM * STATE_SCALE
id_lo, id_hi = (float(D['id_vec'][0]), float(D['id_vec'][-1]))
iq_lo, iq_hi = (float(D['iq_vec'][0]), float(D['iq_vec'][-1]))
if_lo, if_hi = (float(D['if_vec'][0]), float(D['if_vec'][-1]))
eligible = np.array([idx for idx, o in enumerate(ops) if o['if_A'] in (2.0, 4.0, 6.0, 8.0) and id_lo + margin[0] <= o['id_A'] <= id_hi - margin[0] and (iq_lo + margin[1] <= o['iq_A'] <= iq_hi - margin[1]) and (if_lo + margin[2] <= o['if_A'] <= if_hi - margin[2])], dtype=int)
if SPLIT_MODE == 'spatial_block_guarded':
    train_ops, val_ops, test_ops, guard_ops, spatial_split_audit, spatial_distance_audit, spatial_coord_partition, OP_BLOCK = split_ops_spatial_guarded(ops, eligible, SPATIAL_NBINS, SPATIAL_GUARD_A, SPATIAL_SEARCH_ITERS, SEED)
else:
    train_ops, val_ops, test_ops = split_ops_random(ops, eligible, SEED)
    guard_ops = np.array([], dtype=int)
    OP_BLOCK = {int(gi): -1 for gi in eligible}
    spatial_split_audit = pd.DataFrame([{'partition': 'train', 'operating_points': len(train_ops)}, {'partition': 'validation', 'operating_points': len(val_ops)}, {'partition': 'test', 'operating_points': len(test_ops)}])
    spatial_distance_audit = pd.DataFrame([{'guard_threshold_A': 0.0}])
    spatial_coord_partition = pd.DataFrame()
print('Neural operating points after interior-margin audit:', len(eligible))
print('Train / validation / test / guard:', len(train_ops), len(val_ops), len(test_ops), len(guard_ops))
spatial_split_audit.to_csv(TAB / 'table05a_spatial_partition_audit.csv', index=False)
spatial_distance_audit.to_csv(TAB / 'table05b_spatial_partition_distances.csv', index=False)
if len(spatial_coord_partition):
    spatial_coord_partition.to_csv(TAB / 'table05c_spatial_coordinate_partition.csv', index=False)
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    for part in ['train', 'validation', 'test', 'guard']:
        q = spatial_coord_partition[spatial_coord_partition.partition == part]
        if len(q):
            ax.scatter(q.id_A, q.iq_A, s=16, label=part, alpha=0.8)
    ax.set_xlabel('$i_d$ (A)')
    ax.set_ylabel('$i_q$ (A)')
    ax.set_aspect('equal', 'box')
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')


## Knot and cell-interior Jacobians

In [ ]:
def direct_rhs_jacobian_fd(fea_model, base_np, n_np, eps_normalized, batch_size=1024):
    jacobians, schemes = ([], [])
    for st in range(0, len(base_np), batch_size):
        en = min(st + batch_size, len(base_np))
        x0 = torch.tensor(base_np[st:en], dtype=fea_model.dtype, device=fea_model.device)
        n = torch.tensor(n_np[st:en], dtype=fea_model.dtype, device=fea_model.device)
        with torch.no_grad():
            u = fea_model.equilibrium_voltage(x0, n)
            f0 = fea_model.rhs(x0, n, u)
            cols = []
            code = torch.full((len(x0), 3), 9, dtype=torch.int8, device=fea_model.device)
            for d in range(3):
                delta = float(eps_normalized) * float(STATE_SCALE[d])
                xp = x0.clone()
                xp[:, d] += delta
                xm = x0.clone()
                xm[:, d] -= delta
                vp = fea_model.in_domain(xp)
                vm = fea_model.in_domain(xm)
                fp = fea_model.rhs(xp, n, u)
                fm = fea_model.rhs(xm, n, u)
                col = torch.full_like(f0, float('nan'))
                both = vp & vm
                forward = vp & ~vm
                backward = vm & ~vp
                col[both] = (fp[both] - fm[both]) / (2.0 * delta)
                col[forward] = (fp[forward] - f0[forward]) / delta
                col[backward] = (f0[backward] - fm[backward]) / delta
                code[both, d] = 0
                code[forward, d] = 1
                code[backward, d] = -1
                cols.append(col)
            jacobians.append(torch.stack(cols, dim=2).cpu().numpy())
            schemes.append(code.cpu().numpy())
    return (np.concatenate(jacobians, axis=0), np.concatenate(schemes, axis=0))

def _explicit_stability_polynomial_local(z, method):
    if method == 'Euler':
        return 1.0 + z
    if method == 'Heun':
        return 1.0 + z + 0.5 * z * z
    if method == 'RK4':
        return 1.0 + z + 0.5 * z * z + z ** 3 / 6.0 + z ** 4 / 24.0
    raise ValueError(method)

def _critical_step_local(eigs, method, h_cap=0.02, bisect_iter=70):
    eigs = np.asarray(eigs, dtype=complex)
    if np.any(~np.isfinite(eigs)) or np.any(np.real(eigs) >= 0):
        return 0.0
    if method == 'Euler':
        return dtcrit_from_eigs(eigs)

    def stable(h):
        return bool(np.max(np.abs(_explicit_stability_polynomial_local(h * eigs, method))) < 1.0)
    lo, hi = (0.0, 1e-09)
    while hi < h_cap and stable(hi):
        lo, hi = (hi, 2.0 * hi)
    if hi >= h_cap and stable(h_cap):
        return float(h_cap)
    hi = min(hi, h_cap)
    for _ in range(int(bisect_iter)):
        mid = 0.5 * (lo + hi)
        if stable(mid):
            lo = mid
        else:
            hi = mid
    return float(lo)

def _summarize_linearization_frame(df, group_label):
    rows = []
    for eps, g in df.groupby('eps_normalized'):
        scopes = [('all', g)]
        if 'if_A' in g:
            interior = g[g.if_A.isin([2.0, 4.0, 6.0, 8.0])]
            boundary = g[g.if_A.isin([0.0, 10.0])]
            if len(interior):
                scopes.append(('interior_field_levels', interior))
            if len(boundary):
                scopes.append(('field_boundaries', boundary))
        for scope, z in scopes:
            rec = {'analysis': group_label, 'scope': scope, 'eps_normalized': float(eps), 'points': int(len(z)), 'median_relative_A_difference': float(z.relative_frobenius_A_difference.median()), 'p95_relative_A_difference': float(z.relative_frobenius_A_difference.quantile(0.95)), 'maximum_relative_A_difference': float(z.relative_frobenius_A_difference.max()), 'model_hurwitz_fraction': float(z.model_hurwitz.mean()), 'rhs_fd_hurwitz_fraction': float(z.rhs_fd_hurwitz.mean()), 'hurwitz_classification_agreement_fraction': float((z.model_hurwitz == z.rhs_fd_hurwitz).mean())}
            for method in ['Euler', 'Heun', 'RK4']:
                tag = method.lower()
                rec[f'{tag}_median_relative_hcrit_difference'] = float(z[f'relative_hcrit_difference_{tag}'].replace([np.inf, -np.inf], np.nan).median())
                rec[f'{tag}_stable50_agreement_fraction'] = float(z[f'stable50_agreement_{tag}'].mean())
            rows.append(rec)
    return pd.DataFrame(rows)

def _exact_deployed_jacobians_numpy(fea_model, x_np, n_np, batch_size=2048):
    Aall = []
    Jall = []
    strict_all = []
    for st in range(0, len(x_np), batch_size):
        en = min(st + batch_size, len(x_np))
        xt = torch.tensor(x_np[st:en], dtype=fea_model.dtype, device=fea_model.device)
        nt = torch.tensor(n_np[st:en], dtype=fea_model.dtype, device=fea_model.device)
        with torch.no_grad():
            A, Jpsi, strict = exact_equilibrium_rhs_jacobian(fea_model, xt, nt)
        Aall.append(A.cpu().numpy())
        Jall.append(Jpsi.cpu().numpy())
        strict_all.append(strict.cpu().numpy())
    return (np.concatenate(Aall), np.concatenate(Jall), np.concatenate(strict_all))

def _local_A_from_flux_jacobian(Jpsi, Rs, Rf, p, n_rpm):
    try:
        return local_A(Jpsi, Rs, Rf, p, n_rpm)
    except np.linalg.LinAlgError:
        return np.full((3, 3), np.nan, float)
rhs_jacobian_audit = pd.DataFrame()
rhs_jacobian_summary = pd.DataFrame()
if RUN_DIRECT_RHS_JACOBIAN_AUDIT:
    fea64 = TorchFEA(D, L_edge2, device=torch.device('cpu'), dtype=torch.float64)
    base64 = np.asarray(base_all, dtype=np.float64)
    n64 = np.asarray(n_all, dtype=np.float64)
    rows = []
    for eps in DIRECT_RHS_JAC_EPS:
        Jfd, schemes = direct_rhs_jacobian_fd(fea64, base64, n64, eps)
        for gi, (J, scode) in enumerate(zip(Jfd, schemes)):
            Amodel = np.asarray(ops[gi]['A'], dtype=float)
            eig_model = np.linalg.eigvals(Amodel)
            eig_fd = np.linalg.eigvals(J) if np.all(np.isfinite(J)) else np.full(3, np.nan, complex)
            rec = {'linearization_location': 'grid_knot', 'eps_normalized': float(eps), 'op_index': int(gi), 'n_rpm': float(ops[gi]['n_rpm']), 'if_A': float(ops[gi]['if_A']), 'id_A': float(ops[gi]['id_A']), 'iq_A': float(ops[gi]['iq_A']), 'relative_frobenius_A_difference': float(np.linalg.norm(J - Amodel, 'fro') / max(np.linalg.norm(J, 'fro'), 1e-15)), 'model_hurwitz': bool(np.max(np.real(eig_model)) < 0), 'rhs_fd_hurwitz': bool(np.max(np.real(eig_fd)) < 0) if np.all(np.isfinite(eig_fd)) else False, 'difference_scheme_id': int(scode[0]), 'difference_scheme_iq': int(scode[1]), 'difference_scheme_if': int(scode[2])}
            for method in ['Euler', 'Heun', 'RK4']:
                tag = method.lower()
                hm = _critical_step_local(eig_model, method)
                hd = _critical_step_local(eig_fd, method) if np.all(np.isfinite(eig_fd)) else np.nan
                rec[f'hcrit_model_{tag}_s'] = hm
                rec[f'hcrit_rhs_fd_{tag}_s'] = hd
                rec[f'relative_hcrit_difference_{tag}'] = abs(hd - hm) / max(abs(hd), 1e-15) if np.isfinite(hd) and hd > 0 else np.nan
                rec[f'stable50_model_{tag}'] = bool(DT < hm)
                rec[f'stable50_rhs_fd_{tag}'] = bool(DT < hd) if np.isfinite(hd) else False
                rec[f'stable50_agreement_{tag}'] = bool((DT < hm) == (DT < hd)) if np.isfinite(hd) else False
            rows.append(rec)
    rhs_jacobian_audit = pd.DataFrame(rows)
    rhs_jacobian_summary = _summarize_linearization_frame(rhs_jacobian_audit, 'grid_knot_symmetric_fd')
    rhs_jacobian_audit.to_csv(TAB / 'table04a_grid_knot_rhs_fd_consistency_all.csv', index=False)
    rhs_jacobian_summary.to_csv(TAB / 'table04b_grid_knot_rhs_fd_consistency_summary.csv', index=False)

def make_offgrid_cell_center_samples(D, max_points, seed, apply_current_limit=True):
    idm = 0.5 * (np.asarray(D['id_vec'][:-1]) + np.asarray(D['id_vec'][1:]))
    iqm = 0.5 * (np.asarray(D['iq_vec'][:-1]) + np.asarray(D['iq_vec'][1:]))
    ifm = 0.5 * (np.asarray(D['if_vec'][:-1]) + np.asarray(D['if_vec'][1:]))
    rlim = np.sqrt(3.0) * float(D['ismax'])
    first_if, last_if = (float(ifm[0]), float(ifm[-1]))
    rows = []
    for ni, n in enumerate(np.asarray(D['n_vec'], float)):
        for fk, iff in enumerate(ifm):
            region = 'edge_adjacent_field_cell' if np.isclose(iff, first_if) or np.isclose(iff, last_if) else 'central_field_cell'
            for qj, iq in enumerate(iqm):
                for ii, idd in enumerate(idm):
                    current_ok = idd * idd + iq * iq <= rlim * rlim
                    if apply_current_limit and (not current_ok):
                        continue
                    rows.append((idd, iq, iff, n, region, ni, qj, ii, fk, current_ok))
    q = pd.DataFrame(rows, columns=['id_A', 'iq_A', 'if_A', 'n_rpm', 'field_cell_region', 'n_index', 'q_cell', 'id_cell', 'if_cell', 'current_limited_center'])
    if len(q) <= int(max_points):
        return q.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    chosen = []
    groups = list(q.groupby(['n_rpm', 'if_A']).groups.items())
    quota = max(1, int(max_points) // max(1, len(groups)))
    for _, idx in groups:
        a = np.asarray(list(idx), dtype=int)
        take = min(quota, len(a))
        chosen.extend(rng.choice(a, size=take, replace=False).tolist())
    chosen = list(dict.fromkeys(chosen))
    if len(chosen) < int(max_points):
        remain = np.setdiff1d(np.arange(len(q)), np.asarray(chosen, dtype=int), assume_unique=False)
        take = min(int(max_points) - len(chosen), len(remain))
        if take:
            chosen.extend(rng.choice(remain, size=take, replace=False).tolist())
    return q.iloc[chosen[:int(max_points)]].reset_index(drop=True)

def _exact_offgrid_frame(fea_model, frame):
    x = frame[['id_A', 'iq_A', 'if_A']].to_numpy(float)
    n = frame.n_rpm.to_numpy(float)
    with torch.no_grad():
        xt = torch.tensor(x, dtype=fea_model.dtype, device=fea_model.device)
        nt = torch.tensor(n, dtype=fea_model.dtype, device=fea_model.device)
        L = fea_model.L_at(xt).cpu().numpy()
        u = fea_model.equilibrium_voltage(xt, nt).cpu().numpy()
    Aexact, Jpsi, strict = _exact_deployed_jacobians_numpy(fea_model, x, n)
    out = frame.copy().reset_index(drop=True)
    vmag = np.sqrt(u[:, 0] ** 2 + u[:, 1] ** 2)
    out['u_dq_eq_V'] = vmag
    out['voltage_feasible_spwm'] = vmag <= float(D['udc']) * np.sqrt(3.0 / 8.0) + 1e-12
    out['voltage_feasible_svpwm'] = vmag <= float(D['udc']) / np.sqrt(2.0) + 1e-12
    rows = []
    for k in range(len(out)):
        Am = local_A(L[k], D['Rs'], D['Rf'], D['p'], n[k])
        Ae = Aexact[k]
        em = np.linalg.eigvals(Am)
        ee = np.linalg.eigvals(Ae)
        Ac = _local_A_from_flux_jacobian(Jpsi[k], D['Rs'], D['Rf'], D['p'], n[k]) if RUN_DERIVATIVE_CONSISTENT_LOCAL_BASELINE else np.full((3, 3), np.nan)
        ec = np.linalg.eigvals(Ac) if np.all(np.isfinite(Ac)) else np.full(3, np.nan, complex)
        rec = {'sample_index': int(k), 'strict_cell_interior': bool(strict[k]), 'relative_frobenius_model_vs_exact': float(np.linalg.norm(Ae - Am, 'fro') / max(np.linalg.norm(Ae, 'fro'), 1e-15)), 'model_spectral_abscissa_sinv': float(np.max(np.real(em))), 'rhs_exact_spectral_abscissa_sinv': float(np.max(np.real(ee))), 'model_hurwitz': bool(np.max(np.real(em)) < 0), 'rhs_exact_hurwitz': bool(np.max(np.real(ee)) < 0), 'hurwitz_classification_agreement': bool((np.max(np.real(em)) < 0) == (np.max(np.real(ee)) < 0)), 'consistent_baseline_available': bool(np.all(np.isfinite(Ac))), 'consistent_baseline_spectral_abscissa_sinv': float(np.max(np.real(ec))) if np.all(np.isfinite(ec)) else np.nan, 'consistent_baseline_hurwitz': bool(np.max(np.real(ec)) < 0) if np.all(np.isfinite(ec)) else False, 'Dpsi_condition_number': float(np.linalg.cond(Jpsi[k])) if np.all(np.isfinite(Jpsi[k])) else np.nan}
        for method in ['Euler', 'Heun', 'RK4']:
            tag = method.lower()
            hm = _critical_step_local(em, method)
            he = _critical_step_local(ee, method)
            hc = _critical_step_local(ec, method) if np.all(np.isfinite(ec)) else np.nan
            rec[f'hcrit_model_{tag}_s'] = hm
            rec[f'hcrit_rhs_exact_{tag}_s'] = he
            rec[f'hcrit_consistent_{tag}_s'] = hc
            rec[f'relative_hcrit_difference_{tag}'] = abs(he - hm) / max(abs(he), 1e-15) if he > 0 else np.nan
            rec[f'stable50_model_{tag}'] = bool(DT < hm)
            rec[f'stable50_rhs_exact_{tag}'] = bool(DT < he)
            rec[f'stable50_agreement_{tag}'] = bool((DT < hm) == (DT < he))
        rows.append(rec)
    return (pd.concat([out, pd.DataFrame(rows)], axis=1), Aexact, Jpsi)
offgrid_exact_df = pd.DataFrame()
offgrid_jacobian_audit = pd.DataFrame()
offgrid_jacobian_summary = pd.DataFrame()
UNSTABLE_EXACT_CELL_LOOKUP_NP = None
UNSTABLE_EXACT_CELL_LOOKUP_TORCH = None
if RUN_OFFGRID_JACOBIAN_AUDIT:
    if 'fea64' not in globals():
        fea64 = TorchFEA(D, L_edge2, device=torch.device('cpu'), dtype=torch.float64)
    off = make_offgrid_cell_center_samples(D, OFFGRID_JACOBIAN_POINTS, PRIMARY_OFFGRID_SAMPLE_SEED, apply_current_limit=True)
    offgrid_exact_df, Aexact, Jpsi = _exact_offgrid_frame(fea64, off)
    offgrid_exact_df.to_csv(TAB / 'table04h_offgrid_exact_within_cell_jacobian_all.csv', index=False)
    exact_rows = []
    for scope, z in [('all', offgrid_exact_df), ('central_field_cells', offgrid_exact_df[offgrid_exact_df.field_cell_region == 'central_field_cell']), ('edge_adjacent_field_cells', offgrid_exact_df[offgrid_exact_df.field_cell_region == 'edge_adjacent_field_cell'])]:
        pos = z[~z.rhs_exact_hurwitz]
        exact_rows.append({'scope': scope, 'points': int(len(z)), 'median_relative_A_difference': float(z.relative_frobenius_model_vs_exact.median()), 'p95_relative_A_difference': float(z.relative_frobenius_model_vs_exact.quantile(0.95)), 'maximum_relative_A_difference': float(z.relative_frobenius_model_vs_exact.max()), 'rhs_exact_hurwitz_fraction': float(z.rhs_exact_hurwitz.mean()), 'rhs_exact_nonhurwitz_points': int((~z.rhs_exact_hurwitz).sum()), 'rhs_exact_positive_alpha_median_sinv': float(pos.rhs_exact_spectral_abscissa_sinv.median()) if len(pos) else np.nan, 'rhs_exact_positive_alpha_p95_sinv': float(pos.rhs_exact_spectral_abscissa_sinv.quantile(0.95)) if len(pos) else np.nan, 'euler_stable50_agreement_fraction': float(z.stable50_agreement_euler.mean()), 'heun_stable50_agreement_fraction': float(z.stable50_agreement_heun.mean()), 'rk4_stable50_agreement_fraction': float(z.stable50_agreement_rk4.mean()), 'consistent_baseline_available_fraction': float(z.consistent_baseline_available.mean()), 'consistent_baseline_hurwitz_fraction_on_available': float(z.loc[z.consistent_baseline_available, 'consistent_baseline_hurwitz'].mean()) if z.consistent_baseline_available.any() else np.nan})
    offgrid_exact_summary = pd.DataFrame(exact_rows)
    offgrid_exact_summary.to_csv(TAB / 'table04i_offgrid_exact_within_cell_summary.csv', index=False)
    vf_rows = []
    for domain, mask in [('current_limited', np.ones(len(offgrid_exact_df), bool)), ('two_level_spwm_linear', offgrid_exact_df.voltage_feasible_spwm.to_numpy(bool)), ('two_level_svpwm_linear', offgrid_exact_df.voltage_feasible_svpwm.to_numpy(bool))]:
        z = offgrid_exact_df.loc[mask]
        vf_rows.append({'domain': domain, 'speed_scope': 'all', 'points': int(len(z)), 'rhs_exact_nonhurwitz_points': int((~z.rhs_exact_hurwitz).sum()), 'rhs_exact_nonhurwitz_fraction': float((~z.rhs_exact_hurwitz).mean()) if len(z) else np.nan, 'median_positive_alpha_sinv': float(z.loc[~z.rhs_exact_hurwitz, 'rhs_exact_spectral_abscissa_sinv'].median()) if (~z.rhs_exact_hurwitz).any() else np.nan})
        for speed, q in z.groupby('n_rpm'):
            vf_rows.append({'domain': domain, 'speed_scope': f'{int(speed)}rpm', 'points': int(len(q)), 'rhs_exact_nonhurwitz_points': int((~q.rhs_exact_hurwitz).sum()), 'rhs_exact_nonhurwitz_fraction': float((~q.rhs_exact_hurwitz).mean()) if len(q) else np.nan, 'median_positive_alpha_sinv': float(q.loc[~q.rhs_exact_hurwitz, 'rhs_exact_spectral_abscissa_sinv'].median()) if (~q.rhs_exact_hurwitz).any() else np.nan})
    offgrid_voltage_stability_summary = pd.DataFrame(vf_rows)
    offgrid_voltage_stability_summary.to_csv(TAB / 'table04j_offgrid_exact_voltage_feasibility.csv', index=False)
    alpha = offgrid_exact_df.rhs_exact_spectral_abscissa_sinv.to_numpy(float)
    alpha_rows = []
    alpha_rows.append({'threshold_type': 'positive', 'threshold_sinv': 0.0, 'points': int(np.sum(alpha >= 0)), 'fraction': float(np.mean(alpha >= 0))})
    for thr in SPECTRAL_ABSCISSA_NEAR_ZERO_THRESHOLDS:
        alpha_rows.append({'threshold_type': 'positive_near_zero', 'threshold_sinv': float(thr), 'points': int(np.sum((alpha >= 0) & (alpha < float(thr)))), 'fraction': float(np.mean((alpha >= 0) & (alpha < float(thr))))})
        alpha_rows.append({'threshold_type': 'absolute_near_zero', 'threshold_sinv': float(thr), 'points': int(np.sum(np.abs(alpha) < float(thr))), 'fraction': float(np.mean(np.abs(alpha) < float(thr)))})
    spectral_abscissa_threshold_summary = pd.DataFrame(alpha_rows)
    spectral_abscissa_threshold_summary.to_csv(TAB / 'table04k_offgrid_exact_spectral_abscissa_thresholds.csv', index=False)
    xoff = offgrid_exact_df[['id_A', 'iq_A', 'if_A']].to_numpy(float)
    noff = offgrid_exact_df.n_rpm.to_numpy(float)
    fd_rows = []
    fd_summary_rows = []
    for eps in DIRECT_RHS_JAC_EPS:
        Jfd, schemes = direct_rhs_jacobian_fd(fea64, xoff, noff, eps)
        fd_h = []
        fd_alpha = []
        rel = []
        class_agree = []
        for k, (J, scode) in enumerate(zip(Jfd, schemes)):
            ee = np.linalg.eigvals(Aexact[k])
            ed = np.linalg.eigvals(J)
            r = float(np.linalg.norm(J - Aexact[k], 'fro') / max(np.linalg.norm(Aexact[k], 'fro'), 1e-15))
            hfd = bool(np.max(np.real(ed)) < 0)
            hexact = bool(np.max(np.real(ee)) < 0)
            rec = {'eps_normalized': float(eps), 'sample_index': int(k), 'n_rpm': float(noff[k]), 'if_A': float(xoff[k, 2]), 'id_A': float(xoff[k, 0]), 'iq_A': float(xoff[k, 1]), 'relative_frobenius_fd_vs_exact': r, 'rhs_fd_spectral_abscissa_sinv': float(np.max(np.real(ed))), 'rhs_exact_spectral_abscissa_sinv': float(np.max(np.real(ee))), 'rhs_fd_hurwitz': hfd, 'rhs_exact_hurwitz': hexact, 'fd_exact_hurwitz_agreement': bool(hfd == hexact), 'difference_scheme_id': int(scode[0]), 'difference_scheme_iq': int(scode[1]), 'difference_scheme_if': int(scode[2])}
            fd_rows.append(rec)
            fd_h.append(hfd)
            fd_alpha.append(rec['rhs_fd_spectral_abscissa_sinv'])
            rel.append(r)
            class_agree.append(hfd == hexact)
        fd_summary_rows.append({'eps_normalized': float(eps), 'points': len(xoff), 'fd_nonhurwitz_points': int(np.sum(~np.asarray(fd_h, bool))), 'exact_nonhurwitz_points': int((~offgrid_exact_df.rhs_exact_hurwitz).sum()), 'fd_exact_hurwitz_agreement_fraction': float(np.mean(class_agree)), 'median_relative_fd_vs_exact': float(np.median(rel)), 'p95_relative_fd_vs_exact': float(np.quantile(rel, 0.95))})
    offgrid_jacobian_audit = pd.DataFrame(fd_rows)
    offgrid_jacobian_summary = pd.DataFrame(fd_summary_rows)
    offgrid_jacobian_audit.to_csv(TAB / 'table04d_offgrid_rhs_jacobian_consistency_all.csv', index=False)
    offgrid_jacobian_summary.to_csv(TAB / 'table04e_offgrid_rhs_jacobian_consistency_summary.csv', index=False)
    detail_rows = []
    for (speed, iff), z in offgrid_exact_df.groupby(['n_rpm', 'if_A']):
        detail_rows.append({'n_rpm': float(speed), 'if_A': float(iff), 'points': int(len(z)), 'field_cell_region': str(z.field_cell_region.iloc[0]), 'median_relative_A_difference': float(z.relative_frobenius_model_vs_exact.median()), 'rhs_exact_hurwitz_fraction': float(z.rhs_exact_hurwitz.mean()), 'euler_stable50_agreement_fraction': float(z.stable50_agreement_euler.mean()), 'rhs_exact_spectral_abscissa_median_sinv': float(z.rhs_exact_spectral_abscissa_sinv.median()), 'rhs_exact_spectral_abscissa_p95_sinv': float(z.rhs_exact_spectral_abscissa_sinv.quantile(0.95))})
    offgrid_speed_field_summary = pd.DataFrame(detail_rows)
    offgrid_speed_field_summary.to_csv(TAB / 'table04f_offgrid_hurwitz_by_speed_field.csv', index=False)
    nonhurwitz = offgrid_exact_df[~offgrid_exact_df.rhs_exact_hurwitz]
    nonhurwitz_summary = pd.DataFrame([{'points': int(len(nonhurwitz)), 'fraction': float(len(nonhurwitz) / len(offgrid_exact_df)), 'spectral_abscissa_median_sinv': float(nonhurwitz.rhs_exact_spectral_abscissa_sinv.median()) if len(nonhurwitz) else np.nan, 'spectral_abscissa_p95_sinv': float(nonhurwitz.rhs_exact_spectral_abscissa_sinv.quantile(0.95)) if len(nonhurwitz) else np.nan, 'spectral_abscissa_max_sinv': float(nonhurwitz.rhs_exact_spectral_abscissa_sinv.max()) if len(nonhurwitz) else np.nan, 'edge_adjacent_fraction': float((nonhurwitz.field_cell_region == 'edge_adjacent_field_cell').mean()) if len(nonhurwitz) else np.nan}])
    nonhurwitz_summary.to_csv(TAB / 'table04g_offgrid_nonhurwitz_summary.csv', index=False)
    all_cells = make_offgrid_cell_center_samples(D, 10 ** 9, SEED + 185, apply_current_limit=False)
    complete_exact, _, _ = _exact_offgrid_frame(fea64, all_cells)
    complete_exact.to_csv(TAB / 'table04l_complete_cell_center_exact_classification.csv', index=False)
    lookup = np.zeros((len(D['n_vec']), len(D['iq_vec']) - 1, len(D['id_vec']) - 1, len(D['if_vec']) - 1), dtype=bool)
    for r in complete_exact.itertuples(index=False):
        lookup[int(r.n_index), int(r.q_cell), int(r.id_cell), int(r.if_cell)] = not bool(r.rhs_exact_hurwitz)
    UNSTABLE_EXACT_CELL_LOOKUP_NP = lookup
    UNSTABLE_EXACT_CELL_LOOKUP_TORCH = torch.tensor(lookup, dtype=torch.bool)
    if RUN_DERIVATIVE_CONSISTENT_LOCAL_BASELINE:
        dc_rows = []
        for domain, mask in [('all', np.ones(len(offgrid_exact_df), bool)), ('two_level_spwm_linear', offgrid_exact_df.voltage_feasible_spwm.to_numpy(bool)), ('two_level_svpwm_linear', offgrid_exact_df.voltage_feasible_svpwm.to_numpy(bool))]:
            z = offgrid_exact_df.loc[mask]
            a = z[z.consistent_baseline_available]
            dc_rows.append({'domain': domain, 'points': int(len(z)), 'available_points': int(len(a)), 'Dpsi_condition_number_median': float(a.Dpsi_condition_number.median()) if len(a) else np.nan, 'consistent_baseline_hurwitz_fraction': float(a.consistent_baseline_hurwitz.mean()) if len(a) else np.nan, 'deployed_exact_hurwitz_fraction': float(z.rhs_exact_hurwitz.mean()) if len(z) else np.nan, 'interpolated_L_hurwitz_fraction': float(z.model_hurwitz.mean()) if len(z) else np.nan})
        derivative_consistent_summary = pd.DataFrame(dc_rows)
        derivative_consistent_summary.to_csv(TAB / 'table04m_derivative_consistent_local_baseline.csv', index=False)
    if RUN_DIRECT_RHS_JACOBIAN_AUDIT and len(rhs_jacobian_audit):
        b = rhs_jacobian_audit[np.isclose(rhs_jacobian_audit.eps_normalized, DIRECT_RHS_JAC_PRIMARY_EPS) & rhs_jacobian_audit.if_A.isin([0.0, 10.0])]
        e = offgrid_exact_df[offgrid_exact_df.field_cell_region == 'edge_adjacent_field_cell']
        boundary_adjacent_summary = pd.DataFrame([{'location': 'boundary_grid_knots_if0_if10', 'points': len(b), 'derivative_convention': 'boundary-safe finite difference at a knot', 'median_relative_A_difference': float(b.relative_frobenius_A_difference.median())}, {'location': 'adjacent_cell_centers_if1_if9', 'points': len(e), 'derivative_convention': 'exact derivative inside trilinear cells', 'median_relative_A_difference': float(e.relative_frobenius_model_vs_exact.median())}])
        boundary_adjacent_summary.to_csv(TAB / 'table04n_boundary_vs_adjacent_cell_derivative_convention.csv', index=False)
    g = offgrid_exact_df.copy()
    q = g[g.model_hurwitz & g.rhs_exact_hurwitz & (g.hcrit_model_euler_s > 0) & (g.hcrit_rhs_exact_euler_s > 0)].copy()
    fig, axs = plt.subplots(1, 2, figsize=(10.2, 4.2))
    alpha = g.rhs_exact_spectral_abscissa_sinv.to_numpy(float)
    alpha = alpha[np.isfinite(alpha)]
    axs[0].hist(alpha, bins=50)
    axs[0].axvline(0.0, linestyle='--', linewidth=1.0)
    axs[0].set_xlabel('Exact within-cell RHS spectral abscissa $\\alpha$ (s$^{-1}$)')
    axs[0].set_ylabel('Off-grid equilibria')
    axs[0].grid(alpha=0.2)
    axs[0].text(0.03, 0.97, f'non-Hurwitz: {(~g.rhs_exact_hurwitz).sum()} / {len(g)}', transform=axs[0].transAxes, va='top')
    if len(q):
        axs[1].scatter(q.hcrit_model_euler_s * 1000000.0, q.hcrit_rhs_exact_euler_s * 1000000.0, s=8, alpha=0.35)
        lo = min(float((q.hcrit_model_euler_s * 1000000.0).min()), float((q.hcrit_rhs_exact_euler_s * 1000000.0).min()))
        hi = max(float((q.hcrit_model_euler_s * 1000000.0).max()), float((q.hcrit_rhs_exact_euler_s * 1000000.0).max()))
        axs[1].plot([lo, hi], [lo, hi], 'k--', lw=0.8)
        axs[1].set_xscale('log')
        axs[1].set_yscale('log')
    axs[1].set_xlabel('$h_{crit}$ from interpolated $L_\\Delta$ ($\\mu$s)')
    axs[1].set_ylabel('$h_{crit}$ from exact within-cell RHS Jacobian ($\\mu$s)')
    axs[1].grid(alpha=0.2, which='both')
    axs[1].text(0.03, 0.97, f'jointly Hurwitz: {len(q)} / {len(g)}', transform=axs[1].transAxes, va='top')
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')


## Analytical benchmark

In [ ]:
synthetic_example_summary = pd.DataFrame()
synthetic_example_grid = pd.DataFrame()

def _toy_flux_values(xg, yg):
    a, b, c, d, e, f, g, h = (0.411, -0.691, -1.377, -1.45, 0.94, 1.238, 0.32, 0.688)
    X, Y = np.meshgrid(np.asarray(xg, float), np.asarray(yg, float), indexing='ij')
    p1 = 2.0 * X + 0.3 * Y + a * X ** 2 + b * X * Y + c * Y ** 2 + d * X ** 2 * Y
    p2 = 0.2 * X + 1.5 * Y + e * X ** 2 + f * X * Y + g * Y ** 2 + h * X * Y ** 2
    return np.stack([p1, p2], axis=2)

def _toy_fd_jacobian_grid(values, xg, yg):
    J = np.zeros(values.shape[:2] + (2, 2), float)
    for out_idx in range(2):
        J[:, :, out_idx, 0] = np.gradient(values[:, :, out_idx], xg, axis=0, edge_order=2)
        J[:, :, out_idx, 1] = np.gradient(values[:, :, out_idx], yg, axis=1, edge_order=2)
    return J

def _toy_bilinear_flux_jacobian(values, xg, yg, x, y):
    i = int(np.clip(np.searchsorted(xg, x) - 1, 0, len(xg) - 2))
    j = int(np.clip(np.searchsorted(yg, y) - 1, 0, len(yg) - 2))
    x0, x1 = (float(xg[i]), float(xg[i + 1]))
    y0, y1 = (float(yg[j]), float(yg[j + 1]))
    tx = (float(x) - x0) / (x1 - x0)
    ty = (float(y) - y0) / (y1 - y0)
    v00, v10 = (values[i, j], values[i + 1, j])
    v01, v11 = (values[i, j + 1], values[i + 1, j + 1])
    d_dx = ((1 - ty) * (v10 - v00) + ty * (v11 - v01)) / (x1 - x0)
    d_dy = ((1 - tx) * (v01 - v00) + tx * (v11 - v10)) / (y1 - y0)
    return np.stack([d_dx, d_dy], axis=1)

def _toy_bilinear_matrix_interp(M, xg, yg, x, y):
    i = int(np.clip(np.searchsorted(xg, x) - 1, 0, len(xg) - 2))
    j = int(np.clip(np.searchsorted(yg, y) - 1, 0, len(yg) - 2))
    x0, x1 = (float(xg[i]), float(xg[i + 1]))
    y0, y1 = (float(yg[j]), float(yg[j + 1]))
    tx = (float(x) - x0) / (x1 - x0)
    ty = (float(y) - y0) / (y1 - y0)
    return (1 - tx) * (1 - ty) * M[i, j] + tx * (1 - ty) * M[i + 1, j] + (1 - tx) * ty * M[i, j + 1] + tx * ty * M[i + 1, j + 1]
if RUN_SYNTHETIC_NONCOMMUTATION_EXAMPLE:
    xg = np.array([-1.0, 0.0, 1.0])
    yg = np.array([-1.0, 0.0, 1.0])
    psi_grid = _toy_flux_values(xg, yg)
    Dh_grid = _toy_fd_jacobian_grid(psi_grid, xg, yg)
    Rtoy = 0.3 * np.eye(2)
    Otoy = np.array([[0.0, 1.0], [-1.0, 0.0]])

    def _scan_toy_cell(npts):
        pts = (np.arange(int(npts), dtype=float) + 0.5) / float(npts)
        rows = []
        for xx in pts:
            for yy in pts:
                J = _toy_bilinear_flux_jacobian(psi_grid, xg, yg, xx, yy)
                L = _toy_bilinear_matrix_interp(Dh_grid, xg, yg, xx, yy)
                if abs(np.linalg.det(J)) < 1e-12 or abs(np.linalg.det(L)) < 1e-12:
                    continue
                Ac = np.linalg.solve(J, -Rtoy + Otoy @ J)
                Ad = np.linalg.solve(L, -Rtoy + Otoy @ J)
                alpha_c = float(np.max(np.real(np.linalg.eigvals(Ac))))
                alpha_d = float(np.max(np.real(np.linalg.eigvals(Ad))))
                rel = float(np.linalg.norm(L - J, 'fro') / max(np.linalg.norm(J, 'fro'), 1e-15))
                rows.append({'grid_points_per_axis': int(npts), 'x': float(xx), 'y': float(yy), 'relative_operator_mismatch_fro': rel, 'alpha_consistent': alpha_c, 'alpha_deployed': alpha_d, 'consistent_hurwitz': bool(alpha_c < 0.0), 'deployed_hurwitz': bool(alpha_d < 0.0), 'hurwitz_flip': bool((alpha_c < 0.0) != (alpha_d < 0.0))})
        return pd.DataFrame(rows)
    refinement_rows = []
    finest = None
    for npts in SYNTHETIC_GRID_REFINEMENT_POINTS:
        z = _scan_toy_cell(npts)
        if finest is None or int(npts) == max(SYNTHETIC_GRID_REFINEMENT_POINTS):
            finest = z
        refinement_rows.append({'grid_points_per_axis': int(npts), 'evaluated_midpoints': int(len(z)), 'hurwitz_flip_fraction': float(z.hurwitz_flip.mean()) if len(z) else np.nan, 'consistent_hurwitz_fraction': float(z.consistent_hurwitz.mean()) if len(z) else np.nan, 'deployed_hurwitz_fraction': float(z.deployed_hurwitz.mean()) if len(z) else np.nan})
    synthetic_example_grid_refinement = pd.DataFrame(refinement_rows)
    synthetic_example_grid_refinement['change_from_previous'] = synthetic_example_grid_refinement.hurwitz_flip_fraction.diff()
    synthetic_example_grid = finest.copy()
    x0, y0 = (0.25, 0.25)
    J0 = _toy_bilinear_flux_jacobian(psi_grid, xg, yg, x0, y0)
    L0 = _toy_bilinear_matrix_interp(Dh_grid, xg, yg, x0, y0)
    Ac0 = np.linalg.solve(J0, -Rtoy + Otoy @ J0)
    Ad0 = np.linalg.solve(L0, -Rtoy + Otoy @ J0)
    evc = np.linalg.eigvals(Ac0)
    evd = np.linalg.eigvals(Ad0)
    _, V0 = np.linalg.eig(Ac0)
    bf_radius0 = float(np.linalg.cond(V0) * np.linalg.norm(Ad0 - Ac0, 2))
    alpha_c0 = float(np.max(np.real(evc)))
    alpha_d0 = float(np.max(np.real(evd)))
    synthetic_example_summary = pd.DataFrame([{'map_type': 'constructed analytic polynomial flux map; numerical midpoint scan of the bilinear cell', 'grid_definition': 'x,y in {-1,0,1}; bilinear reconstruction on [0,1]^2', 'evaluation_x': x0, 'evaluation_y': y0, 'relative_operator_mismatch_fro': float(np.linalg.norm(L0 - J0, 'fro') / np.linalg.norm(J0, 'fro')), 'alpha_consistent': alpha_c0, 'alpha_deployed': alpha_d0, 'consistent_hurwitz': bool(alpha_c0 < 0.0), 'deployed_hurwitz': bool(alpha_d0 < 0.0), 'bauer_fike_radius': bf_radius0, 'bauer_fike_sufficient_margin': float(-(alpha_c0 + bf_radius0)), 'reported_scan_points_per_axis': int(max(SYNTHETIC_GRID_REFINEMENT_POINTS)), 'reported_dense_cell_hurwitz_flip_fraction': float(synthetic_example_grid.hurwitz_flip.mean()), 'note': 'Constructed analytic map with numerical midpoint-grid scan; not an EESM validation case.'}])
    synthetic_example_summary.to_csv(TAB / 'table00_synthetic_noncommutation_example.csv', index=False)
    synthetic_example_grid.to_csv(TAB / 'table00b_synthetic_noncommutation_grid.csv', index=False)
    synthetic_example_grid_refinement.to_csv(TAB / 'table00c_synthetic_grid_refinement.csv', index=False)
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    zz = synthetic_example_grid.pivot(index='y', columns='x', values='hurwitz_flip').astype(float)
    im = ax.imshow(zz.to_numpy(), origin='lower', aspect='auto', extent=[0.0, 1.0, 0.0, 1.0])
    fig.colorbar(im, ax=ax, label='Hurwitz classification differs (0/1)')
    ax.scatter([x0], [y0], marker='x', s=70, label='reported example point')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('Constructed non-commutation map (finest midpoint scan)')
    ax.legend()
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')


## Representation mismatch and cell-interior scan

In [ ]:
representation_coupling_df = pd.DataFrame()
representation_coupling_summary = pd.DataFrame()
intracell_robustness_df = pd.DataFrame()
intracell_robustness_summary = pd.DataFrame()
intracell_cell_persistence = pd.DataFrame()
derivative_consistent_conditioning_all = pd.DataFrame()
derivative_consistent_conditioning_summary = pd.DataFrame()
intracell_parent_selection_manifest = pd.DataFrame()
intracell_candidate_pool_by_stratum = pd.DataFrame()
intracell_design_retention_summary = pd.DataFrame()
exhaustive_intracell_points = pd.DataFrame()
exhaustive_intracell_cell_summary = pd.DataFrame()
exhaustive_intracell_summary = pd.DataFrame()

def _safe_spectral_abscissa(A):
    if not np.all(np.isfinite(A)):
        return np.nan
    return float(np.max(np.real(np.linalg.eigvals(A))))

def _balanced_deterministic_subset(frame, nmax, group_cols=('n_rpm', 'if_A')):
    q = frame.copy().reset_index(drop=True)
    if len(q) <= int(nmax):
        return q
    groups = list(q.groupby(list(group_cols), sort=True))
    per = max(1, int(nmax) // max(1, len(groups)))
    chosen = []
    for _, g in groups:
        g = g.sort_values([c for c in ['q_cell', 'id_cell', 'if_cell', 'id_A', 'iq_A'] if c in g.columns])
        if len(g) <= per:
            chosen.extend(g.index.tolist())
        else:
            pos = np.linspace(0, len(g) - 1, per, dtype=int)
            chosen.extend(g.index.to_numpy()[pos].tolist())
    chosen = list(dict.fromkeys(chosen))
    if len(chosen) < int(nmax):
        remain = [i for i in q.index if i not in set(chosen)]
        need = min(int(nmax) - len(chosen), len(remain))
        if need:
            pos = np.linspace(0, len(remain) - 1, need, dtype=int)
            chosen.extend(np.asarray(remain, dtype=int)[pos].tolist())
    return q.loc[chosen[:int(nmax)]].reset_index(drop=True)

def _representation_coupling_frame(fea_model, frame, A_deployed, J_flux):
    x = frame[['id_A', 'iq_A', 'if_A']].to_numpy(float)
    n = frame.n_rpm.to_numpy(float)
    with torch.no_grad():
        xt = torch.tensor(x, dtype=fea_model.dtype, device=fea_model.device)
        L_interp = fea_model.L_at(xt).cpu().numpy()
    rows = []
    for k in range(len(frame)):
        Lm = np.asarray(L_interp[k], float)
        Jp = np.asarray(J_flux[k], float)
        Ad = np.asarray(A_deployed[k], float)
        Ac = _local_A_from_flux_jacobian(Jp, D['Rs'], D['Rf'], D['p'], n[k])
        Am = local_A(Lm, D['Rs'], D['Rf'], D['p'], n[k])
        rel_D = float(np.linalg.norm(Lm - Jp, 'fro') / max(np.linalg.norm(Jp, 'fro'), 1e-15))
        rel_A_dc = float(np.linalg.norm(Ad - Ac, 'fro') / max(np.linalg.norm(Ac, 'fro'), 1e-15))
        rel_A_map = float(np.linalg.norm(Ad - Am, 'fro') / max(np.linalg.norm(Ad, 'fro'), 1e-15))
        alpha_d = _safe_spectral_abscissa(Ad)
        alpha_c = _safe_spectral_abscissa(Ac)
        alpha_m = _safe_spectral_abscissa(Am)
        bf_radius = np.nan
        eigvec_cond = np.nan
        bf_cert = False
        if np.all(np.isfinite(Ac)) and np.all(np.isfinite(Ad)):
            try:
                _, V = np.linalg.eig(Ac)
                eigvec_cond = float(np.linalg.cond(V))
                bf_radius = float(eigvec_cond * np.linalg.norm(Ad - Ac, 2))
                bf_cert = bool(np.isfinite(alpha_c) and np.isfinite(bf_radius) and (alpha_c + bf_radius < 0.0))
            except np.linalg.LinAlgError:
                pass
        rows.append({'sample_index': int(k), 'n_rpm': float(n[k]), 'if_A': float(frame.iloc[k].if_A), 'id_A': float(frame.iloc[k].id_A), 'iq_A': float(frame.iloc[k].iq_A), 'relative_D_operator_mismatch_fro': rel_D, 'relative_A_deployed_vs_consistent_fro': rel_A_dc, 'relative_A_map_vs_deployed_fro': rel_A_map, 'alpha_map_sinv': alpha_m, 'alpha_deployed_sinv': alpha_d, 'alpha_consistent_sinv': alpha_c, 'alpha_deployed_minus_consistent_sinv': float(alpha_d - alpha_c), 'map_hurwitz': bool(alpha_m < 0.0), 'deployed_hurwitz': bool(alpha_d < 0.0), 'consistent_hurwitz': bool(alpha_c < 0.0), 'consistent_to_deployed_hurwitz_flip': bool((alpha_c < 0.0) != (alpha_d < 0.0)), 'eigvector_condition_consistent': eigvec_cond, 'bauer_fike_radius_sinv': bf_radius, 'bauer_fike_hurwitz_certificate': bf_cert, 'certificate_margin_sinv': float(-(alpha_c + bf_radius)) if np.isfinite(alpha_c) and np.isfinite(bf_radius) else np.nan})
    return pd.DataFrame(rows)

def _make_intracell_factorial_points(base_cells):
    idv = np.asarray(D['id_vec'], float)
    iqv = np.asarray(D['iq_vec'], float)
    ifv = np.asarray(D['if_vec'], float)
    design = [('center', 0.5, 0.5, 0.5)]
    for a, b, c in itertools.product(OFFGRID_ROBUST_FRACTIONS, repeat=3):
        design.append((f'factorial_{a:.2f}_{b:.2f}_{c:.2f}', float(a), float(b), float(c)))
    rlim = np.sqrt(3.0) * float(D['ismax'])
    rows = []
    for parent_id, r in enumerate(base_cells.itertuples(index=False)):
        ii, qj, fk = (int(r.id_cell), int(r.q_cell), int(r.if_cell))
        for label, fi, fq, ff in design:
            idd = idv[ii] + fi * (idv[ii + 1] - idv[ii])
            iq = iqv[qj] + fq * (iqv[qj + 1] - iqv[qj])
            iff = ifv[fk] + ff * (ifv[fk + 1] - ifv[fk])
            if idd * idd + iq * iq > rlim * rlim + 1e-12:
                continue
            rows.append({'parent_cell_id': int(parent_id), 'local_design': label, 'local_fraction_id': fi, 'local_fraction_iq': fq, 'local_fraction_if': ff, 'id_A': float(idd), 'iq_A': float(iq), 'if_A': float(iff), 'n_rpm': float(r.n_rpm), 'n_index': int(r.n_index), 'q_cell': qj, 'id_cell': ii, 'if_cell': fk})
    return pd.DataFrame(rows)
if RUN_MISMATCH_STABILITY_COUPLING and RUN_OFFGRID_JACOBIAN_AUDIT and len(offgrid_exact_df):
    representation_coupling_df = _representation_coupling_frame(fea64, offgrid_exact_df, Aexact, Jpsi)
    for col in ['field_cell_region', 'voltage_feasible_spwm', 'voltage_feasible_svpwm']:
        if col in offgrid_exact_df:
            representation_coupling_df[col] = offgrid_exact_df[col].to_numpy()
    flip = representation_coupling_df.consistent_to_deployed_hurwitz_flip.to_numpy(bool)
    cert = representation_coupling_df.bauer_fike_hurwitz_certificate.to_numpy(bool)
    actual_h = representation_coupling_df.deployed_hurwitz.to_numpy(bool)
    representation_coupling_summary = pd.DataFrame([{'points': int(len(representation_coupling_df)), 'spearman_D_mismatch_vs_alpha_deployed': float(representation_coupling_df.relative_D_operator_mismatch_fro.corr(representation_coupling_df.alpha_deployed_sinv, method='spearman')), 'spearman_D_mismatch_vs_alpha_shift': float(representation_coupling_df.relative_D_operator_mismatch_fro.corr(representation_coupling_df.alpha_deployed_minus_consistent_sinv, method='spearman')), 'median_D_mismatch_flip': float(representation_coupling_df.loc[flip, 'relative_D_operator_mismatch_fro'].median()) if flip.any() else np.nan, 'median_D_mismatch_no_flip': float(representation_coupling_df.loc[~flip, 'relative_D_operator_mismatch_fro'].median()) if (~flip).any() else np.nan, 'hurwitz_flip_points': int(flip.sum()), 'hurwitz_flip_fraction': float(flip.mean()), 'bauer_fike_certified_hurwitz_points': int(cert.sum()), 'bauer_fike_certified_hurwitz_fraction': float(cert.mean()), 'certificate_false_safe_points': int(np.sum(cert & ~actual_h)), 'consistent_alpha_median_sinv': float(representation_coupling_df.alpha_consistent_sinv.median()), 'deployed_alpha_median_sinv': float(representation_coupling_df.alpha_deployed_sinv.median())}])
    representation_coupling_df.to_csv(TAB / 'table04o_representation_mismatch_stability_coupling_all.csv', index=False)
    representation_coupling_summary.to_csv(TAB / 'table04p_representation_mismatch_stability_coupling_summary.csv', index=False)
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    z0 = representation_coupling_df[~representation_coupling_df.consistent_to_deployed_hurwitz_flip]
    z1 = representation_coupling_df[representation_coupling_df.consistent_to_deployed_hurwitz_flip]
    ax.scatter(z0.relative_D_operator_mismatch_fro, z0.alpha_deployed_sinv, s=10, alpha=0.3, label='No Hurwitz flip')
    ax.scatter(z1.relative_D_operator_mismatch_fro, z1.alpha_deployed_sinv, s=18, alpha=0.75, marker='x', label='Hurwitz flip')
    ax.axhline(0.0, linestyle='--', linewidth=1.0)
    ax.set_xlabel('$\\|I(D_h\\psi)-D(I\\psi)\\|_F/\\|D(I\\psi)\\|_F$')
    ax.set_ylabel('Deployed spectral abscissa $\\alpha$ (s$^{-1}$)')
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')
if RUN_DERIVATIVE_CONSISTENT_CONDITIONING_AUDIT and RUN_OFFGRID_JACOBIAN_AUDIT and len(offgrid_exact_df):
    psi_scale_dc = np.array([np.max(np.abs(D['psid'])), np.max(np.abs(D['psiq'])), np.max(np.abs(D['psif']))], dtype=float)
    current_scale_dc = np.array([np.sqrt(3.0) * D['ismax'], np.sqrt(3.0) * D['ismax'], D['ifmax']], dtype=float)
    Psi_inv_dc = np.diag(1.0 / np.maximum(psi_scale_dc, 1e-15))
    I_scale_dc = np.diag(current_scale_dc)
    xdc = offgrid_exact_df[['id_A', 'iq_A', 'if_A']].to_numpy(float)
    with torch.no_grad():
        Ldc = fea64.L_at(torch.tensor(xdc, dtype=fea64.dtype, device=fea64.device)).cpu().numpy()
    crows = []
    for k in range(len(offgrid_exact_df)):
        Jk = np.asarray(Jpsi[k], float)
        Lk = np.asarray(Ldc[k], float)
        sJ = np.linalg.svd(Jk, compute_uv=False)
        sL = np.linalg.svd(Lk, compute_uv=False)
        Jhat = Psi_inv_dc @ Jk @ I_scale_dc
        Lhat = Psi_inv_dc @ Lk @ I_scale_dc
        shJ = np.linalg.svd(Jhat, compute_uv=False)
        shL = np.linalg.svd(Lhat, compute_uv=False)
        crows.append({'sample_index': int(k), 'n_rpm': float(offgrid_exact_df.iloc[k].n_rpm), 'if_A': float(offgrid_exact_df.iloc[k].if_A), 'id_A': float(offgrid_exact_df.iloc[k].id_A), 'iq_A': float(offgrid_exact_df.iloc[k].iq_A), 'hurwitz_flip': bool(offgrid_exact_df.iloc[k].consistent_baseline_hurwitz != offgrid_exact_df.iloc[k].rhs_exact_hurwitz), 'D_Ipsi_cond2_raw': float(sJ[0] / sJ[-1]), 'I_Dhpsi_cond2_raw': float(sL[0] / sL[-1]), 'D_Ipsi_cond2_dimensionless': float(shJ[0] / shJ[-1]), 'I_Dhpsi_cond2_dimensionless': float(shL[0] / shL[-1]), 'D_Ipsi_sigma_min_dimensionless': float(shJ[-1]), 'I_Dhpsi_sigma_min_dimensionless': float(shL[-1])})
    derivative_consistent_conditioning_all = pd.DataFrame(crows)
    derivative_consistent_conditioning_all.to_csv(TAB / 'table04v_derivative_consistent_conditioning_all.csv', index=False)
    sr = []
    for scope, z in [('all', derivative_consistent_conditioning_all), ('hurwitz_flip', derivative_consistent_conditioning_all[derivative_consistent_conditioning_all.hurwitz_flip]), ('no_hurwitz_flip', derivative_consistent_conditioning_all[~derivative_consistent_conditioning_all.hurwitz_flip])]:
        sr.append({'scope': scope, 'points': int(len(z)), 'D_Ipsi_raw_cond2_median': float(z.D_Ipsi_cond2_raw.median()) if len(z) else np.nan, 'D_Ipsi_raw_cond2_p95': float(z.D_Ipsi_cond2_raw.quantile(0.95)) if len(z) else np.nan, 'D_Ipsi_scaled_cond2_median': float(z.D_Ipsi_cond2_dimensionless.median()) if len(z) else np.nan, 'D_Ipsi_scaled_cond2_p95': float(z.D_Ipsi_cond2_dimensionless.quantile(0.95)) if len(z) else np.nan, 'I_Dhpsi_scaled_cond2_median': float(z.I_Dhpsi_cond2_dimensionless.median()) if len(z) else np.nan, 'I_Dhpsi_scaled_cond2_p95': float(z.I_Dhpsi_cond2_dimensionless.quantile(0.95)) if len(z) else np.nan, 'scaled_cond2_ratio_median_D_Ipsi_over_I_Dhpsi': float(np.median(z.D_Ipsi_cond2_dimensionless / np.maximum(z.I_Dhpsi_cond2_dimensionless, 1e-15))) if len(z) else np.nan})
    derivative_consistent_conditioning_summary = pd.DataFrame(sr)
    derivative_consistent_conditioning_summary.to_csv(TAB / 'table04w_derivative_consistent_conditioning_summary.csv', index=False)
    fig, ax = plt.subplots(figsize=(5.8, 3.8))
    ax.hist(np.log10(derivative_consistent_conditioning_all.D_Ipsi_cond2_dimensionless), bins=30, alpha=0.55, label='$D(I\\psi)$')
    ax.hist(np.log10(derivative_consistent_conditioning_all.I_Dhpsi_cond2_dimensionless), bins=30, alpha=0.55, label='$I(D_h\\psi)$')
    ax.set_xlabel('$\\log_{10}\\kappa_2$ (dimensionless scaling)')
    ax.set_ylabel('Off-grid centers')
    ax.grid(axis='y', alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')
if RUN_OFFGRID_INTRACELL_ROBUSTNESS and RUN_OFFGRID_JACOBIAN_AUDIT and len(offgrid_exact_df):
    all_parent_candidates = make_offgrid_cell_center_samples(D, 10 ** 9, PRIMARY_OFFGRID_SAMPLE_SEED, apply_current_limit=True)
    base_cells = _balanced_deterministic_subset(all_parent_candidates, OFFGRID_ROBUST_CELLS)
    base_cells = base_cells.copy()
    base_cells['selection_order'] = np.arange(len(base_cells), dtype=int)
    base_cells['candidate_pool_size'] = int(len(all_parent_candidates))
    base_cells['selection_protocol_version'] = SELECTION_PROTOCOL_VERSION
    base_cells['selection_mode'] = INTRACELL_PARENT_SELECTION_MODE
    cand_counts = all_parent_candidates.groupby(['n_rpm', 'if_A'], sort=True).size().rename('candidate_cells')
    sel_counts = base_cells.groupby(['n_rpm', 'if_A'], sort=True).size().rename('selected_cells')
    intracell_candidate_pool_by_stratum = pd.concat([cand_counts, sel_counts], axis=1).fillna(0).reset_index()
    intracell_candidate_pool_by_stratum['selected_cells'] = intracell_candidate_pool_by_stratum.selected_cells.astype(int)
    intracell_candidate_pool_by_stratum['selection_fraction'] = intracell_candidate_pool_by_stratum.selected_cells / intracell_candidate_pool_by_stratum.candidate_cells
    intracell_candidate_pool_by_stratum.to_csv(TAB / 'table04q5_intracell_candidate_pool_by_stratum.csv', index=False)
    if EXPORT_SELECTION_MANIFESTS:
        intracell_parent_selection_manifest = base_cells.copy()
        intracell_parent_selection_manifest.to_csv(TAB / 'table04q0_intracell_parent_selection_manifest.csv', index=False)
    robust_points = _make_intracell_factorial_points(base_cells)
    requested_design_points = int(len(base_cells) * 9)
    retained_design_points = int(len(robust_points))
    intracell_design_retention_summary = pd.DataFrame([{'parent_cells': int(len(base_cells)), 'requested_center_plus_factorial_points': requested_design_points, 'retained_inside_current_limit': retained_design_points, 'filtered_by_current_limit': int(requested_design_points - retained_design_points), 'retained_fraction': float(retained_design_points / requested_design_points) if requested_design_points else np.nan, 'note': 'Filtering occurs because the circular stator-current limit can cut strict-interior factorial points from otherwise admissible interpolation cells.'}])
    intracell_design_retention_summary.to_csv(TAB / 'table04q4_intracell_design_retention_summary.csv', index=False)
    xr = robust_points[['id_A', 'iq_A', 'if_A']].to_numpy(float)
    nr = robust_points.n_rpm.to_numpy(float)
    Adep_r, Jpsi_r, strict_r = _exact_deployed_jacobians_numpy(fea64, xr, nr)
    with torch.no_grad():
        Lr = fea64.L_at(torch.tensor(xr, dtype=fea64.dtype, device=fea64.device)).cpu().numpy()
    rrows = []
    for k in range(len(robust_points)):
        Am = local_A(Lr[k], D['Rs'], D['Rf'], D['p'], nr[k])
        Ac = _local_A_from_flux_jacobian(Jpsi_r[k], D['Rs'], D['Rf'], D['p'], nr[k])
        ad = _safe_spectral_abscissa(Adep_r[k])
        am = _safe_spectral_abscissa(Am)
        ac = _safe_spectral_abscissa(Ac)
        rrows.append({'sample_index': int(k), 'strict_cell_interior': bool(strict_r[k]), 'relative_D_operator_mismatch_fro': float(np.linalg.norm(Lr[k] - Jpsi_r[k], 'fro') / max(np.linalg.norm(Jpsi_r[k], 'fro'), 1e-15)), 'alpha_map_sinv': am, 'alpha_deployed_sinv': ad, 'alpha_consistent_sinv': ac, 'map_hurwitz': bool(am < 0), 'deployed_hurwitz': bool(ad < 0), 'consistent_hurwitz': bool(ac < 0)})
    intracell_robustness_df = pd.concat([robust_points.reset_index(drop=True), pd.DataFrame(rrows)], axis=1)
    intracell_robustness_df.to_csv(TAB / 'table04q_intracell_offgrid_robustness_all.csv', index=False)
    summary_rows = []
    for scope, z in [('all_interior_design_points', intracell_robustness_df), ('cell_centers', intracell_robustness_df[intracell_robustness_df.local_design == 'center']), ('off_center_factorial_points', intracell_robustness_df[intracell_robustness_df.local_design != 'center'])]:
        summary_rows.append({'scope': scope, 'points': int(len(z)), 'parent_cells': int(z.parent_cell_id.nunique()) if len(z) else 0, 'strict_interior_fraction': float(z.strict_cell_interior.mean()) if len(z) else np.nan, 'deployed_nonhurwitz_points': int((~z.deployed_hurwitz).sum()) if len(z) else 0, 'deployed_nonhurwitz_fraction': float((~z.deployed_hurwitz).mean()) if len(z) else np.nan, 'consistent_nonhurwitz_fraction': float((~z.consistent_hurwitz).mean()) if len(z) else np.nan, 'median_D_operator_mismatch': float(z.relative_D_operator_mismatch_fro.median()) if len(z) else np.nan, 'p95_D_operator_mismatch': float(z.relative_D_operator_mismatch_fro.quantile(0.95)) if len(z) else np.nan})
    intracell_robustness_summary = pd.DataFrame(summary_rows)
    intracell_robustness_summary.to_csv(TAB / 'table04q1_intracell_offgrid_robustness_summary.csv', index=False)
    per_cell = []
    for pid, z in intracell_robustness_df.groupby('parent_cell_id'):
        center = z[z.local_design == 'center']
        offc = z[z.local_design != 'center']
        per_cell.append({'parent_cell_id': int(pid), 'n_rpm': float(z.n_rpm.iloc[0]), 'center_available': bool(len(center)), 'center_nonhurwitz': bool((~center.deployed_hurwitz).iloc[0]) if len(center) else False, 'valid_design_points': int(len(z)), 'requested_design_points': 9, 'design_retention_fraction': float(len(z) / 9.0), 'all_valid_nonhurwitz_fraction': float((~z.deployed_hurwitz).mean()) if len(z) else np.nan, 'offcenter_points': int(len(offc)), 'offcenter_nonhurwitz_fraction': float((~offc.deployed_hurwitz).mean()) if len(offc) else np.nan, 'any_offcenter_nonhurwitz': bool((~offc.deployed_hurwitz).any()) if len(offc) else False, 'all_offcenter_nonhurwitz': bool((~offc.deployed_hurwitz).all()) if len(offc) else False})
    intracell_cell_persistence = pd.DataFrame(per_cell)
    intracell_cell_persistence.to_csv(TAB / 'table04q2_intracell_cell_level_persistence.csv', index=False)
    center_bad = intracell_cell_persistence[intracell_cell_persistence.center_nonhurwitz]
    persistence_summary = pd.DataFrame([{'sampled_parent_cells': int(len(intracell_cell_persistence)), 'center_nonhurwitz_cells': int(len(center_bad)), 'center_nonhurwitz_with_any_offcenter_nonhurwitz_fraction': float(center_bad.any_offcenter_nonhurwitz.mean()) if len(center_bad) else np.nan, 'center_nonhurwitz_median_offcenter_nonhurwitz_fraction': float(center_bad.offcenter_nonhurwitz_fraction.median()) if len(center_bad) else np.nan, 'cells_with_any_nonhurwitz_interior_point_fraction': float(intracell_cell_persistence.any_offcenter_nonhurwitz.mean()) if len(intracell_cell_persistence) else np.nan, 'equal_cell_weighted_mean_nonhurwitz_fraction_all_valid_design_points': float(intracell_cell_persistence.all_valid_nonhurwitz_fraction.mean()) if len(intracell_cell_persistence) else np.nan, 'median_cell_design_retention_fraction': float(intracell_cell_persistence.design_retention_fraction.median()) if len(intracell_cell_persistence) else np.nan}])
    persistence_summary.to_csv(TAB / 'table04q3_intracell_persistence_summary.csv', index=False)
    if RUN_EXHAUSTIVE_INTRACELL_AUDIT:
        exhaustive_points = _make_intracell_factorial_points(all_parent_candidates)
        xe = exhaustive_points[['id_A', 'iq_A', 'if_A']].to_numpy(float)
        ne = exhaustive_points.n_rpm.to_numpy(float)
        Adep_e, Jpsi_e, strict_e = _exact_deployed_jacobians_numpy(fea64, xe, ne, batch_size=int(EXHAUSTIVE_INTRACELL_BATCH))
        we = 2 * np.pi * float(D['p']) * ne / 60.0
        Omega = np.zeros((len(ne), 3, 3), float)
        Omega[:, 0, 1] = we
        Omega[:, 1, 0] = -we
        Rmat = np.diag([float(D['Rs']), float(D['Rs']), float(D['Rf'])])
        Bc = -Rmat[None, :, :] + np.matmul(Omega, Jpsi_e)
        try:
            Ac_e = np.linalg.solve(Jpsi_e, Bc)
        except np.linalg.LinAlgError:
            Ac_e = np.full_like(Adep_e, np.nan)
            for kk in range(len(Jpsi_e)):
                try:
                    Ac_e[kk] = np.linalg.solve(Jpsi_e[kk], Bc[kk])
                except np.linalg.LinAlgError:
                    pass
        alpha_dep = np.max(np.real(np.linalg.eigvals(Adep_e)), axis=1)
        finite_cons = np.all(np.isfinite(Ac_e), axis=(1, 2))
        alpha_cons = np.full(len(Ac_e), np.nan, float)
        if finite_cons.any():
            alpha_cons[finite_cons] = np.max(np.real(np.linalg.eigvals(Ac_e[finite_cons])), axis=1)
        exhaustive_intracell_points = exhaustive_points.copy()
        exhaustive_intracell_points['strict_cell_interior'] = strict_e.astype(bool)
        exhaustive_intracell_points['alpha_deployed_sinv'] = alpha_dep
        exhaustive_intracell_points['alpha_consistent_sinv'] = alpha_cons
        exhaustive_intracell_points['deployed_hurwitz'] = alpha_dep < 0
        exhaustive_intracell_points['consistent_hurwitz'] = alpha_cons < 0
        exhaustive_intracell_points.to_csv(TAB / 'table04q8_exhaustive_intracell_point_level.csv', index=False)
        ecell = []
        for pid, z in exhaustive_intracell_points.groupby('parent_cell_id', sort=True):
            center = z[z.local_design == 'center']
            offc = z[z.local_design != 'center']
            ecell.append({'parent_cell_id': int(pid), 'n_rpm': float(z.n_rpm.iloc[0]), 'if_A_center': float(center.if_A.iloc[0]) if len(center) else np.nan, 'valid_design_points': int(len(z)), 'design_retention_fraction': float(len(z) / 9.0), 'center_nonhurwitz': bool((~center.deployed_hurwitz).iloc[0]) if len(center) else False, 'all_valid_nonhurwitz_fraction': float((~z.deployed_hurwitz).mean()) if len(z) else np.nan, 'offcenter_nonhurwitz_fraction': float((~offc.deployed_hurwitz).mean()) if len(offc) else np.nan, 'any_nonhurwitz_valid_point': bool((~z.deployed_hurwitz).any()) if len(z) else False, 'any_offcenter_nonhurwitz': bool((~offc.deployed_hurwitz).any()) if len(offc) else False, 'consistent_nonhurwitz_valid_point': bool((~z.consistent_hurwitz).any()) if len(z) else False})
        exhaustive_intracell_cell_summary = pd.DataFrame(ecell)
        exhaustive_intracell_cell_summary.to_csv(TAB / 'table04q7_exhaustive_intracell_cell_level.csv', index=False)
        center_bad = exhaustive_intracell_cell_summary[exhaustive_intracell_cell_summary.center_nonhurwitz]
        requested = int(len(all_parent_candidates) * 9)
        retained = int(len(exhaustive_intracell_points))
        exhaustive_intracell_summary = pd.DataFrame([{'admissible_parent_cells': int(len(all_parent_candidates)), 'requested_design_points': requested, 'retained_design_points_inside_current_limit': retained, 'filtered_by_current_limit': int(requested - retained), 'retained_design_fraction': float(retained / requested) if requested else np.nan, 'deployed_nonhurwitz_points': int((~exhaustive_intracell_points.deployed_hurwitz).sum()), 'deployed_nonhurwitz_point_fraction': float((~exhaustive_intracell_points.deployed_hurwitz).mean()), 'consistent_nonhurwitz_points': int((~exhaustive_intracell_points.consistent_hurwitz).sum()), 'consistent_nonhurwitz_point_fraction': float((~exhaustive_intracell_points.consistent_hurwitz).mean()), 'cells_with_any_nonhurwitz_valid_point_fraction': float(exhaustive_intracell_cell_summary.any_nonhurwitz_valid_point.mean()), 'equal_cell_weighted_mean_nonhurwitz_fraction': float(exhaustive_intracell_cell_summary.all_valid_nonhurwitz_fraction.mean()), 'center_nonhurwitz_cells': int(len(center_bad)), 'center_nonhurwitz_with_any_offcenter_nonhurwitz_fraction': float(center_bad.any_offcenter_nonhurwitz.mean()) if len(center_bad) else np.nan, 'median_cell_design_retention_fraction': float(exhaustive_intracell_cell_summary.design_retention_fraction.median()), 'note': 'Exhaustive over every admissible interpolation-cell center in the current-limited candidate pool; fractions remain properties of this finite cell design, not continuous-domain measures.'}])
        exhaustive_intracell_summary.to_csv(TAB / 'table04q6_exhaustive_intracell_summary.csv', index=False)
    if EXPORT_SELECTION_MANIFESTS:
        selection_protocol = {'version': SELECTION_PROTOCOL_VERSION, 'primary_offgrid_5000': {'candidate_definition': 'all current-limited cell centers at five speeds and field-cell centers', 'sample_seed': int(PRIMARY_OFFGRID_SAMPLE_SEED), 'sampling': 'balanced random sample by speed x field-cell center', 'requested_points': int(OFFGRID_JACOBIAN_POINTS)}, 'strict_interior_parent_cells': {'candidate_pool': 'complete admissible cell-center candidate set', 'candidate_pool_cells': int(len(all_parent_candidates)), 'selection_mode': INTRACELL_PARENT_SELECTION_MODE, 'requested_parent_cells': int(OFFGRID_ROBUST_CELLS), 'within_cell_design': 'center plus eight 2^3 points at local fractions 0.25/0.75', 'current_limit_filtering': 'design points outside the circular stator-current limit are retained in the manifest count as filtered and excluded from Jacobian statistics', 'exhaustive_full_candidate_pool_audit': bool(RUN_EXHAUSTIVE_INTRACELL_AUDIT)}, 'derivative_consistent_trajectory_bases': {'selection_mode': DC_BASE_SELECTION_MODE, 'requested_bases': int(DERIVATIVE_CONSISTENT_TRAJECTORY_BASES), 'directions': [list(v) for v in DERIVATIVE_CONSISTENT_DIRECTIONS], 'perturbation_cell_fraction': float(DERIVATIVE_CONSISTENT_PERTURB_CELL_FRACTION)}}
        (OUT / 'selection_protocol.json').write_text(json.dumps(selection_protocol, indent=2), encoding='utf-8')
    fig, ax = plt.subplots(figsize=(5.8, 3.8))
    ax.hist(intracell_cell_persistence.offcenter_nonhurwitz_fraction.dropna().to_numpy(), bins=np.linspace(0, 1, 18))
    ax.set_xlabel('Within-cell off-center non-Hurwitz fraction')
    ax.set_ylabel('Sampled interpolation cells')
    ax.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')


## Interpolation and grid sensitivity

In [ ]:
interpolation_family_all = pd.DataFrame()
interpolation_family_summary = pd.DataFrame()
grid_resolution_sensitivity_all = pd.DataFrame()
grid_resolution_sensitivity_summary = pd.DataFrame()

def _rgi_eval_with_derivative(interp, pts_qidif, nu, axis_spacing=None):
    try:
        return np.asarray(interp(pts_qidif, nu=nu), float)
    except (TypeError, ValueError):
        if axis_spacing is None:
            raise
        d = int(np.argmax(nu))
        h = max(abs(float(axis_spacing[d])) * 1e-05, 1e-09)
        pp = np.asarray(pts_qidif, float).copy()
        pm = np.asarray(pts_qidif, float).copy()
        pp[:, d] += h
        pm[:, d] -= h
        return (np.asarray(interp(pp), float) - np.asarray(interp(pm), float)) / (2.0 * h)

def _equilibrium_A_from_denominator_and_flux_jac(Lden, Jpsi, n_rpm, D):
    we = 2.0 * np.pi * float(D['p']) * float(n_rpm) / 60.0
    Omega = np.array([[0.0, we, 0.0], [-we, 0.0, 0.0], [0.0, 0.0, 0.0]], float)
    R = np.diag([float(D['Rs']), float(D['Rs']), float(D['Rf'])])
    return np.linalg.solve(np.asarray(Lden, float), -R + Omega @ np.asarray(Jpsi, float))

def _build_tensor_product_interpolators(Dloc, Lgridloc, method='cubic'):
    axes = (np.asarray(Dloc['iq_vec'], float), np.asarray(Dloc['id_vec'], float), np.asarray(Dloc['if_vec'], float))
    flux = np.stack([Dloc['psid'], Dloc['psiq'], Dloc['psif']], axis=-1)
    flux_i = [RegularGridInterpolator(axes, flux[..., r], method=method, bounds_error=True) for r in range(3)]
    L_i = [[RegularGridInterpolator(axes, Lgridloc[..., r, c], method=method, bounds_error=True) for c in range(3)] for r in range(3)]
    return (axes, flux_i, L_i)

def _evaluate_interpolation_family(Dloc, Lgridloc, frame, method='cubic'):
    axes, flux_i, L_i = _build_tensor_product_interpolators(Dloc, Lgridloc, method=method)
    pts_idqif = frame[['id_A', 'iq_A', 'if_A']].to_numpy(float)
    pts_qidif = np.column_stack([pts_idqif[:, 1], pts_idqif[:, 0], pts_idqif[:, 2]])
    spacing = [np.median(np.diff(a)) for a in axes]
    N = len(frame)
    J = np.empty((N, 3, 3), float)
    for r in range(3):
        J[:, r, 0] = _rgi_eval_with_derivative(flux_i[r], pts_qidif, (0, 1, 0), spacing)
        J[:, r, 1] = _rgi_eval_with_derivative(flux_i[r], pts_qidif, (1, 0, 0), spacing)
        J[:, r, 2] = _rgi_eval_with_derivative(flux_i[r], pts_qidif, (0, 0, 1), spacing)
    L = np.empty((N, 3, 3), float)
    for r in range(3):
        for c in range(3):
            L[:, r, c] = np.asarray(L_i[r][c](pts_qidif), float)
    rows = []
    for k, rr in enumerate(frame.itertuples(index=False)):
        rec = {'sample_index': int(getattr(rr, 'sample_index', k)), 'n_rpm': float(rr.n_rpm), 'id_A': float(rr.id_A), 'iq_A': float(rr.iq_A), 'if_A': float(rr.if_A), 'interpolation_family': str(method)}
        try:
            Adep = _equilibrium_A_from_denominator_and_flux_jac(L[k], J[k], rr.n_rpm, Dloc)
            Acon = _equilibrium_A_from_denominator_and_flux_jac(J[k], J[k], rr.n_rpm, Dloc)
            ed = np.linalg.eigvals(Adep)
            ec = np.linalg.eigvals(Acon)
            rel = float(np.linalg.norm(L[k] - J[k], 'fro') / max(np.linalg.norm(J[k], 'fro'), 1e-15))
            rec.update({'operator_mismatch_rel_fro': rel, 'deployed_alpha_sinv': float(np.max(np.real(ed))), 'consistent_alpha_sinv': float(np.max(np.real(ec))), 'deployed_hurwitz': bool(np.max(np.real(ed)) < 0), 'consistent_hurwitz': bool(np.max(np.real(ec)) < 0), 'hurwitz_flip': bool((np.max(np.real(ed)) < 0) != (np.max(np.real(ec)) < 0)), 'deployed_hcrit_euler_s': float(dtcrit_from_eigs(ed)), 'consistent_hcrit_euler_s': float(dtcrit_from_eigs(ec)), 'cond_interpolated_Dhpsi': float(np.linalg.cond(L[k])), 'cond_D_interp_psi': float(np.linalg.cond(J[k])), 'valid': True})
        except (np.linalg.LinAlgError, ValueError, FloatingPointError) as exc:
            rec.update({'valid': False, 'failure': repr(exc)})
        rows.append(rec)
    return pd.DataFrame(rows)

def _subset_current_plane_grid(Dloc, factor):
    factor = int(factor)
    if factor <= 1:
        return {k: v.copy() if isinstance(v, np.ndarray) else v for k, v in Dloc.items()}
    iq_idx = np.arange(0, len(Dloc['iq_vec']), factor, dtype=int)
    id_idx = np.arange(0, len(Dloc['id_vec']), factor, dtype=int)
    if iq_idx[-1] != len(Dloc['iq_vec']) - 1:
        iq_idx = np.r_[iq_idx, len(Dloc['iq_vec']) - 1]
    if id_idx[-1] != len(Dloc['id_vec']) - 1:
        id_idx = np.r_[id_idx, len(Dloc['id_vec']) - 1]
    out = {}
    for k, v in Dloc.items():
        out[k] = v.copy() if isinstance(v, np.ndarray) else v
    out['iq_vec'] = np.asarray(Dloc['iq_vec'])[iq_idx]
    out['id_vec'] = np.asarray(Dloc['id_vec'])[id_idx]
    for k in ['psid', 'psiq', 'psif', 'Tem']:
        a = np.asarray(Dloc[k])
        out[k] = a[iq_idx, :, :][:, id_idx, :]
    if 'Pfe' in Dloc:
        a = np.asarray(Dloc['Pfe'])
        out['Pfe'] = a[iq_idx, :, :, :][:, id_idx, :, :]
    return out

def _cell_center_jac_and_interpolated_L(Dloc, Lgridloc, q, i, f):
    iqv, idv, ifv = (Dloc['iq_vec'], Dloc['id_vec'], Dloc['if_vec'])
    did = float(idv[i + 1] - idv[i])
    diq = float(iqv[q + 1] - iqv[q])
    dif = float(ifv[f + 1] - ifv[f])
    V = np.stack([Dloc['psid'], Dloc['psiq'], Dloc['psif']], axis=-1)
    c000 = V[q, i, f]
    c001 = V[q, i, f + 1]
    c010 = V[q, i + 1, f]
    c011 = V[q, i + 1, f + 1]
    c100 = V[q + 1, i, f]
    c101 = V[q + 1, i, f + 1]
    c110 = V[q + 1, i + 1, f]
    c111 = V[q + 1, i + 1, f + 1]
    didv = 0.25 * (c010 - c000 + (c011 - c001) + (c110 - c100) + (c111 - c101)) / did
    diqv = 0.25 * (c100 - c000 + (c101 - c001) + (c110 - c010) + (c111 - c011)) / diq
    difv = 0.25 * (c001 - c000 + (c011 - c010) + (c101 - c100) + (c111 - c110)) / dif
    J = np.stack([didv, diqv, difv], axis=1)
    Lc = (Lgridloc[q, i, f] + Lgridloc[q, i, f + 1] + Lgridloc[q, i + 1, f] + Lgridloc[q, i + 1, f + 1] + Lgridloc[q + 1, i, f] + Lgridloc[q + 1, i, f + 1] + Lgridloc[q + 1, i + 1, f] + Lgridloc[q + 1, i + 1, f + 1]) / 8.0
    return (J, Lc)

def _grid_resolution_cell_center_audit(Dsource, factors):
    all_rows = []
    sum_rows = []
    I_lim = np.sqrt(3.0) * float(Dsource['ismax'])
    for fac in factors:
        Dc = _subset_current_plane_grid(Dsource, int(fac))
        if min(len(Dc['iq_vec']), len(Dc['id_vec']), len(Dc['if_vec'])) < 3:
            continue
        Lc = derivative_grids(Dc, 'edge2')
        rows = []
        for n in Dc['n_vec']:
            for f in range(len(Dc['if_vec']) - 1):
                iff = 0.5 * (Dc['if_vec'][f] + Dc['if_vec'][f + 1])
                for q in range(len(Dc['iq_vec']) - 1):
                    iq = 0.5 * (Dc['iq_vec'][q] + Dc['iq_vec'][q + 1])
                    for i in range(len(Dc['id_vec']) - 1):
                        id_ = 0.5 * (Dc['id_vec'][i] + Dc['id_vec'][i + 1])
                        if id_ * id_ + iq * iq > I_lim * I_lim + 1e-12:
                            continue
                        rec = {'coarsen_factor_id_iq': int(fac), 'n_rpm': float(n), 'if_A': float(iff), 'id_A': float(id_), 'iq_A': float(iq), 'q_cell': q, 'id_cell': i, 'if_cell': f}
                        try:
                            J, Lmap = _cell_center_jac_and_interpolated_L(Dc, Lc, q, i, f)
                            Adep = _equilibrium_A_from_denominator_and_flux_jac(Lmap, J, n, Dc)
                            Acon = _equilibrium_A_from_denominator_and_flux_jac(J, J, n, Dc)
                            ed = np.linalg.eigvals(Adep)
                            ec = np.linalg.eigvals(Acon)
                            rec.update({'valid': True, 'operator_mismatch_rel_fro': float(np.linalg.norm(Lmap - J, 'fro') / max(np.linalg.norm(J, 'fro'), 1e-15)), 'deployed_alpha_sinv': float(np.max(np.real(ed))), 'consistent_alpha_sinv': float(np.max(np.real(ec))), 'deployed_hurwitz': bool(np.max(np.real(ed)) < 0), 'consistent_hurwitz': bool(np.max(np.real(ec)) < 0), 'hurwitz_flip': bool((np.max(np.real(ed)) < 0) != (np.max(np.real(ec)) < 0))})
                        except np.linalg.LinAlgError as exc:
                            rec.update({'valid': False, 'failure': repr(exc)})
                        rows.append(rec)
        z = pd.DataFrame(rows)
        all_rows.append(z)
        v = z[z.valid].copy() if len(z) else z
        sum_rows.append({'coarsen_factor_id_iq': int(fac), 'id_nodes': int(len(Dc['id_vec'])), 'iq_nodes': int(len(Dc['iq_vec'])), 'if_nodes': int(len(Dc['if_vec'])), 'requested_valid_cell_centers': int(len(z)), 'numerically_valid_cell_centers': int(len(v)), 'deployed_nonhurwitz_points': int((~v.deployed_hurwitz).sum()) if len(v) else 0, 'deployed_nonhurwitz_fraction': float((~v.deployed_hurwitz).mean()) if len(v) else np.nan, 'consistent_nonhurwitz_points': int((~v.consistent_hurwitz).sum()) if len(v) else 0, 'consistent_nonhurwitz_fraction': float((~v.consistent_hurwitz).mean()) if len(v) else np.nan, 'hurwitz_flip_fraction': float(v.hurwitz_flip.mean()) if len(v) else np.nan, 'median_operator_mismatch_rel_fro': float(v.operator_mismatch_rel_fro.median()) if len(v) else np.nan, 'p95_operator_mismatch_rel_fro': float(v.operator_mismatch_rel_fro.quantile(0.95)) if len(v) else np.nan, 'note': 'Finite cell-center design at the stated retained grid resolution; not a continuous-domain prevalence estimate.'})
    return (pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame(), pd.DataFrame(sum_rows))
if RUN_INTERPOLATION_FAMILY_SENSITIVITY and RUN_OFFGRID_JACOBIAN_AUDIT and len(offgrid_exact_df):
    cubic_df = _evaluate_interpolation_family(D, L_edge2, offgrid_exact_df, method=INTERPOLATION_SENSITIVITY_METHOD)
    interpolation_family_all = cubic_df.copy()
    valid = cubic_df[cubic_df.valid].copy()
    tri_cons = offgrid_exact_df.consistent_baseline_hurwitz.to_numpy(bool) if 'consistent_baseline_hurwitz' in offgrid_exact_df.columns else np.ones(len(offgrid_exact_df), bool)
    tri_row = {'interpolation_family': 'trilinear_primary', 'points': int(len(offgrid_exact_df)), 'valid_points': int(len(offgrid_exact_df)), 'deployed_nonhurwitz_points': int((~offgrid_exact_df.rhs_exact_hurwitz).sum()), 'deployed_nonhurwitz_fraction': float((~offgrid_exact_df.rhs_exact_hurwitz).mean()), 'consistent_nonhurwitz_points': int((~tri_cons).sum()), 'consistent_nonhurwitz_fraction': float((~tri_cons).mean()), 'hurwitz_flip_fraction': float(np.mean(offgrid_exact_df.rhs_exact_hurwitz.to_numpy(bool) != tri_cons)), 'median_operator_mismatch_rel_fro': float(representation_coupling_df.relative_D_operator_mismatch_fro.median()) if len(representation_coupling_df) else np.nan, 'p95_operator_mismatch_rel_fro': float(representation_coupling_df.relative_D_operator_mismatch_fro.quantile(0.95)) if len(representation_coupling_df) else np.nan, 'interpretation': 'Primary piecewise-trilinear representation.'}
    cub_row = {'interpolation_family': str(INTERPOLATION_SENSITIVITY_METHOD), 'points': int(len(cubic_df)), 'valid_points': int(len(valid)), 'deployed_nonhurwitz_points': int((~valid.deployed_hurwitz).sum()) if len(valid) else 0, 'deployed_nonhurwitz_fraction': float((~valid.deployed_hurwitz).mean()) if len(valid) else np.nan, 'consistent_nonhurwitz_points': int((~valid.consistent_hurwitz).sum()) if len(valid) else 0, 'consistent_nonhurwitz_fraction': float((~valid.consistent_hurwitz).mean()) if len(valid) else np.nan, 'hurwitz_flip_fraction': float(valid.hurwitz_flip.mean()) if len(valid) else np.nan, 'median_operator_mismatch_rel_fro': float(valid.operator_mismatch_rel_fro.median()) if len(valid) else np.nan, 'p95_operator_mismatch_rel_fro': float(valid.operator_mismatch_rel_fro.quantile(0.95)) if len(valid) else np.nan, 'interpretation': 'Smooth tensor-product interpolation-family sensitivity; not physical ground truth.'}
    interpolation_family_summary = pd.DataFrame([tri_row, cub_row])
    interpolation_family_all.to_csv(TAB / 'table04aa_interpolation_family_sensitivity_all.csv', index=False)
    interpolation_family_summary.to_csv(TAB / 'table04ab_interpolation_family_sensitivity_summary.csv', index=False)
if RUN_GRID_RESOLUTION_SENSITIVITY:
    grid_resolution_sensitivity_all, grid_resolution_sensitivity_summary = _grid_resolution_cell_center_audit(D, GRID_COARSEN_FACTORS)
    grid_resolution_sensitivity_all.to_csv(TAB / 'table04ac_grid_resolution_sensitivity_all.csv', index=False)
    grid_resolution_sensitivity_summary.to_csv(TAB / 'table04ad_grid_resolution_sensitivity_summary.csv', index=False)


## Derivative-consistent trajectories

In [ ]:
dc_trajectory_all = pd.DataFrame()
dc_trajectory_paired = pd.DataFrame()
dc_trajectory_summary = pd.DataFrame()
dc_reference_pair_summary = pd.DataFrame()
dc_trajectory_speed_summary = pd.DataFrame()
dc_reference_speed_summary = pd.DataFrame()
dc_base_selection_manifest = pd.DataFrame()
dc_reference_convergence_all = pd.DataFrame()
dc_reference_convergence_summary = pd.DataFrame()
dc_reference_convergence_by_speed = pd.DataFrame()
dc_hurwitz_stratum_summary = pd.DataFrame()
dc_speed_hurwitz_stratum_summary = pd.DataFrame()

class DerivativeConsistentFEA(TorchFEA):

    def rhs(self, x, n_rpm, ueq):
        psi = self.flux_at(x)
        Jpsi = self.flux_jacobian_at(x)
        we = 2 * math.pi * self.p * n_rpm / 60.0
        b = torch.stack([ueq[:, 0] - self.Rs * x[:, 0] + we * psi[:, 1], ueq[:, 1] - self.Rs * x[:, 1] - we * psi[:, 0], ueq[:, 2] - self.Rf * x[:, 2]], dim=1)
        return torch.linalg.solve(Jpsi, b.unsqueeze(-1)).squeeze(-1)

def _cell_change_count(fea_model, x_old, x_new):
    q0, i0, f0, v0 = fea_model.cell_indices(x_old)
    q1, i1, f1, v1 = fea_model.cell_indices(x_new)
    changed = (q0 != q1) | (i0 != i1) | (f0 != f1)
    return changed & v0 & v1

def _highres_reference_boundary_audit(fea_model, x0, n, u, nsteps, substeps):
    x = x0.clone()
    B = len(x)
    traj = [x.detach().cpu().numpy()]
    contact = ~fea_model.in_domain(x)
    crossings = torch.zeros(B, dtype=torch.int64, device=x.device)
    interface_hits = fea_model.cell_interface_mask(x).to(torch.int64)
    hh = float(DT) / int(substeps)
    with torch.no_grad():
        for _ in range(int(nsteps)):
            for _ in range(int(substeps)):
                xprev = x
                k1 = fea_model.rhs(x, n, u)
                y2 = x + 0.5 * hh * k1
                contact |= ~fea_model.in_domain(y2)
                interface_hits += fea_model.cell_interface_mask(y2).to(torch.int64)
                k2 = fea_model.rhs(y2, n, u)
                y3 = x + 0.5 * hh * k2
                contact |= ~fea_model.in_domain(y3)
                interface_hits += fea_model.cell_interface_mask(y3).to(torch.int64)
                k3 = fea_model.rhs(y3, n, u)
                y4 = x + hh * k3
                contact |= ~fea_model.in_domain(y4)
                interface_hits += fea_model.cell_interface_mask(y4).to(torch.int64)
                k4 = fea_model.rhs(y4, n, u)
                x = x + hh / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
                contact |= ~fea_model.in_domain(x)
                interface_hits += fea_model.cell_interface_mask(x).to(torch.int64)
                crossings += _cell_change_count(fea_model, xprev, x).to(torch.int64)
            traj.append(x.detach().cpu().numpy())
    return (np.stack(traj, axis=1), contact.detach().cpu().numpy(), crossings.detach().cpu().numpy(), interface_hits.detach().cpu().numpy())

def _dc_flux_and_jac_numpy(x, D):
    x = np.asarray(x, float)
    idv = np.asarray(D['id_vec'], float)
    iqv = np.asarray(D['iq_vec'], float)
    ifv = np.asarray(D['if_vec'], float)
    did = float(idv[1] - idv[0])
    diq = float(iqv[1] - iqv[0])
    dif = float(ifv[1] - ifv[0])
    uid = np.clip((x[0] - idv[0]) / did, 0.0, len(idv) - 1 - 1e-12)
    uiq = np.clip((x[1] - iqv[0]) / diq, 0.0, len(iqv) - 1 - 1e-12)
    uif = np.clip((x[2] - ifv[0]) / dif, 0.0, len(ifv) - 1 - 1e-12)
    i0, q0, f0 = (int(np.floor(uid)), int(np.floor(uiq)), int(np.floor(uif)))
    i1, q1, f1 = (i0 + 1, q0 + 1, f0 + 1)
    wi, wq, wf = (uid - i0, uiq - q0, uif - f0)
    V = np.stack([D['psid'], D['psiq'], D['psif']], axis=-1)
    c000 = V[q0, i0, f0]
    c001 = V[q0, i0, f1]
    c010 = V[q0, i1, f0]
    c011 = V[q0, i1, f1]
    c100 = V[q1, i0, f0]
    c101 = V[q1, i0, f1]
    c110 = V[q1, i1, f0]
    c111 = V[q1, i1, f1]
    c00 = c000 * (1 - wi) + c010 * wi
    c01 = c001 * (1 - wi) + c011 * wi
    c10 = c100 * (1 - wi) + c110 * wi
    c11 = c101 * (1 - wi) + c111 * wi
    c0 = c00 * (1 - wq) + c10 * wq
    c1 = c01 * (1 - wq) + c11 * wq
    psi = c0 * (1 - wf) + c1 * wf
    didv = (1 - wq) * (1 - wf) * (c010 - c000) + (1 - wq) * wf * (c011 - c001) + wq * (1 - wf) * (c110 - c100) + wq * wf * (c111 - c101)
    diqv = (1 - wi) * (1 - wf) * (c100 - c000) + (1 - wi) * wf * (c101 - c001) + wi * (1 - wf) * (c110 - c010) + wi * wf * (c111 - c011)
    difv = (1 - wq) * (1 - wi) * (c001 - c000) + (1 - wq) * wi * (c011 - c010) + wq * (1 - wi) * (c101 - c100) + wq * wi * (c111 - c110)
    J = np.stack([didv / did, diqv / diq, difv / dif], axis=1)
    return (psi, J)

def _dc_rhs_numpy(t, x, n_rpm, ueq, D, contact_state=None):
    if contact_state is not None:
        lo = np.array([D['id_vec'][0], D['iq_vec'][0], D['if_vec'][0]], float)
        hi = np.array([D['id_vec'][-1], D['iq_vec'][-1], D['if_vec'][-1]], float)
        if np.any(x < lo - 1e-07) or np.any(x > hi + 1e-07):
            contact_state['domain_contact'] = True
        u = np.array([(x[0] - D['id_vec'][0]) / (D['id_vec'][1] - D['id_vec'][0]), (x[1] - D['iq_vec'][0]) / (D['iq_vec'][1] - D['iq_vec'][0]), (x[2] - D['if_vec'][0]) / (D['if_vec'][1] - D['if_vec'][0])], float)
        ncoord = np.array([len(D['id_vec']), len(D['iq_vec']), len(D['if_vec'])], int)
        nearest = np.round(u)
        d = []
        for jj in range(3):
            if 0 < nearest[jj] < ncoord[jj] - 1:
                d.append(abs(u[jj] - nearest[jj]))
        if d:
            contact_state['min_interface_distance_normalized'] = min(contact_state.get('min_interface_distance_normalized', np.inf), float(np.min(d)))
    psi, J = _dc_flux_and_jac_numpy(x, D)
    we = 2 * np.pi * float(D['p']) * float(n_rpm) / 60.0
    b = np.array([ueq[0] - D['Rs'] * x[0] + we * psi[1], ueq[1] - D['Rs'] * x[1] - we * psi[0], ueq[2] - D['Rf'] * x[2]], float)
    return np.linalg.solve(J, b)

def _trajectory_rmse(a, b):
    d = (np.asarray(a) - np.asarray(b)) / STATE_SCALE[None, :]
    return float(np.sqrt(np.mean(d * d)))

def _select_dc_trajectory_bases(frame, n_total):
    q = frame[frame.consistent_baseline_available].copy().reset_index(drop=True)
    q['dc_case'] = np.where(q.rhs_exact_hurwitz, 'deployed_hurwitz', 'deployed_nonhurwitz')
    q['rank_metric'] = q.relative_frobenius_model_vs_exact.astype(float)
    q['_candidate_row'] = np.arange(len(q), dtype=int)
    q = q.sort_values(['n_rpm', 'dc_case', 'rank_metric', 'if_A', 'id_A', 'iq_A', 'sample_index'], kind='mergesort').reset_index(drop=True)
    q['candidate_group_size'] = q.groupby(['n_rpm', 'dc_case'])['sample_index'].transform('size')
    q['candidate_rank_within_group'] = q.groupby(['n_rpm', 'dc_case']).cumcount()
    q['candidate_rank_fraction'] = np.where(q.candidate_group_size > 1, q.candidate_rank_within_group / np.maximum(q.candidate_group_size - 1, 1), 0.0)
    q['selection_tier'] = ''
    selected = []
    group_keys = list(q.groupby(['n_rpm', 'dc_case'], sort=True))
    per = max(1, int(n_total) // max(1, len(group_keys)))
    for _, g in group_keys:
        take = min(per, len(g))
        if take:
            pos = np.unique(np.linspace(0, len(g) - 1, take, dtype=int))
            idx = g.index.to_numpy()[pos].tolist()
            selected.extend(idx)
            q.loc[idx, 'selection_tier'] = 'speed_x_hurwitz_even_mismatch_rank'
    selected = list(dict.fromkeys(selected))
    if len(selected) < int(n_total):
        remain = q.loc[~q.index.isin(selected)].copy()
        remain = remain.sort_values(['n_rpm', 'dc_case', 'rank_metric', 'if_A', 'id_A', 'iq_A', 'sample_index'], kind='mergesort')
        need = min(int(n_total) - len(selected), len(remain))
        if need:
            pos = np.unique(np.linspace(0, len(remain) - 1, need, dtype=int))
            idx = remain.index.to_numpy()[pos].tolist()
            selected.extend(idx)
            q.loc[idx, 'selection_tier'] = 'deterministic_global_fill'
    out = q.loc[selected[:int(n_total)]].copy().reset_index(drop=True)
    out['selection_order'] = np.arange(len(out), dtype=int)
    out['selection_protocol_version'] = SELECTION_PROTOCOL_VERSION
    out['selection_mode'] = DC_BASE_SELECTION_MODE
    return out

def _macro_step_with_contact(fea_model, x, n, u, h, method):
    contact = ~fea_model.in_domain(x)
    if method == 'euler':
        y = x + h * fea_model.rhs(x, n, u)
        contact |= ~fea_model.in_domain(y)
        return (y, contact)
    if method == 'heun':
        k1 = fea_model.rhs(x, n, u)
        y2 = x + h * k1
        contact |= ~fea_model.in_domain(y2)
        k2 = fea_model.rhs(y2, n, u)
        y = x + 0.5 * h * (k1 + k2)
        contact |= ~fea_model.in_domain(y)
        return (y, contact)
    if method == 'rk4':
        k1 = fea_model.rhs(x, n, u)
        y2 = x + 0.5 * h * k1
        contact |= ~fea_model.in_domain(y2)
        k2 = fea_model.rhs(y2, n, u)
        y3 = x + 0.5 * h * k2
        contact |= ~fea_model.in_domain(y3)
        k3 = fea_model.rhs(y3, n, u)
        y4 = x + h * k3
        contact |= ~fea_model.in_domain(y4)
        k4 = fea_model.rhs(y4, n, u)
        y = x + h / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
        contact |= ~fea_model.in_domain(y)
        return (y, contact)
    raise ValueError(method)

def _highres_reference(fea_model, x0, n, u, nsteps, substeps):
    x = x0.clone()
    traj = [x.detach().cpu().numpy()]
    contact = ~fea_model.in_domain(x)
    with torch.no_grad():
        for _ in range(int(nsteps)):
            x, c = rk4_step(fea_model, x, n, u, DT, substeps=int(substeps), return_domain_contact=True)
            contact |= c
            traj.append(x.detach().cpu().numpy())
    return (np.stack(traj, axis=1), contact.detach().cpu().numpy())

def _solver_rmse_against_reference(fea_model, x0, n, u, ref, method):
    x = x0.clone()
    scale = torch.tensor(STATE_SCALE, dtype=x.dtype, device=x.device)
    se = torch.zeros(len(x), dtype=x.dtype, device=x.device)
    contact = ~fea_model.in_domain(x)
    with torch.no_grad():
        for t in range(ref.shape[1] - 1):
            x, c = _macro_step_with_contact(fea_model, x, n, u, DT, method)
            contact |= c
            r = torch.tensor(ref[:, t + 1, :], dtype=x.dtype, device=x.device)
            d = (x - r) / scale
            se += torch.sum(d * d, dim=1)
    rmse = torch.sqrt(se / ((ref.shape[1] - 1) * 3.0))
    return (rmse.detach().cpu().numpy(), contact.detach().cpu().numpy())
if RUN_DERIVATIVE_CONSISTENT_TRAJECTORY_AUDIT and RUN_OFFGRID_JACOBIAN_AUDIT and len(offgrid_exact_df):
    fea_deployed_dc = TorchFEA(D, L_edge2, device=torch.device('cpu'), dtype=torch.float64)
    fea_consistent_dc = DerivativeConsistentFEA(D, L_edge2, device=torch.device('cpu'), dtype=torch.float64)
    bases = _select_dc_trajectory_bases(offgrid_exact_df, DERIVATIVE_CONSISTENT_TRAJECTORY_BASES)
    if EXPORT_SELECTION_MANIFESTS:
        dc_base_selection_manifest = bases.copy()
        dc_base_selection_manifest.to_csv(TAB / 'table04r0_derivative_consistent_base_selection_manifest.csv', index=False)
    did = float(D['id_vec'][1] - D['id_vec'][0])
    diq = float(D['iq_vec'][1] - D['iq_vec'][0])
    dif = float(D['if_vec'][1] - D['if_vec'][0])
    cell_step = np.array([did, diq, dif], float)
    rlim = np.sqrt(3.0) * float(D['ismax'])
    all_rows = []
    pair_rows = []
    scenario_id = 0
    for speed, gb in bases.groupby('n_rpm', sort=True):
        base_np = gb[['id_A', 'iq_A', 'if_A']].to_numpy(float)
        B0 = len(base_np)
        if B0 == 0:
            continue
        nsteps = max(1, int(round(60.0 / (float(D['p']) * float(speed)) / DT)))
        for dir_index, dvec in enumerate(DERIVATIVE_CONSISTENT_DIRECTIONS):
            dvec = np.asarray(dvec, float)
            x0_np = base_np + DERIVATIVE_CONSISTENT_PERTURB_CELL_FRACTION * cell_step[None, :] * dvec[None, :]
            valid0 = (x0_np[:, 0] >= float(D['id_vec'][0])) & (x0_np[:, 0] <= float(D['id_vec'][-1])) & (x0_np[:, 1] >= float(D['iq_vec'][0])) & (x0_np[:, 1] <= float(D['iq_vec'][-1])) & (x0_np[:, 2] >= float(D['if_vec'][0])) & (x0_np[:, 2] <= float(D['if_vec'][-1])) & (x0_np[:, 0] ** 2 + x0_np[:, 1] ** 2 <= rlim ** 2 + 1e-12)
            if not np.any(valid0):
                continue
            g = gb.loc[valid0].reset_index(drop=True)
            base_v = base_np[valid0]
            x0_v = x0_np[valid0]
            B = len(g)
            bt = torch.tensor(base_v, dtype=torch.float64)
            nt = torch.full((B,), float(speed), dtype=torch.float64)
            with torch.no_grad():
                ut = fea_deployed_dc.equilibrium_voltage(bt, nt)
            x0t = torch.tensor(x0_v, dtype=torch.float64)
            ref_dep, ref_dep_contact = _highres_reference(fea_deployed_dc, x0t, nt, ut, nsteps, DERIVATIVE_CONSISTENT_REF_SUBSTEPS)
            con_refs = {}
            con_contacts = {}
            con_crossings = {}
            con_interface_hits = {}
            for ss in sorted(set((int(v) for v in DC_REFERENCE_AUDIT_SUBSTEPS))):
                rr, cc, xxc, iih = _highres_reference_boundary_audit(fea_consistent_dc, x0t, nt, ut, nsteps, ss)
                con_refs[ss] = rr
                con_contacts[ss] = cc
                con_crossings[ss] = xxc
                con_interface_hits[ss] = iih
            primary_ss = int(DERIVATIVE_CONSISTENT_REF_SUBSTEPS)
            if primary_ss not in con_refs:
                rr, cc, xxc, iih = _highres_reference_boundary_audit(fea_consistent_dc, x0t, nt, ut, nsteps, primary_ss)
                con_refs[primary_ss] = rr
                con_contacts[primary_ss] = cc
                con_crossings[primary_ss] = xxc
                con_interface_hits[primary_ss] = iih
            ref_con = con_refs[primary_ss]
            ref_con_contact = con_contacts[primary_ss]
            audit_ss = sorted(con_refs)
            lo_ss, hi_ss = (audit_ss[0], audit_ss[-1])
            mid_ss = min(audit_ss, key=lambda v: abs(v - primary_ss))
            conv_lo_hi = np.sqrt(np.mean(((con_refs[lo_ss] - con_refs[hi_ss]) / STATE_SCALE[None, None, :]) ** 2, axis=(1, 2)))
            conv_mid_hi = np.sqrt(np.mean(((con_refs[mid_ss] - con_refs[hi_ss]) / STATE_SCALE[None, None, :]) ** 2, axis=(1, 2)))
            cross_rmse = np.sqrt(np.mean(((ref_dep - ref_con) / STATE_SCALE[None, None, :]) ** 2, axis=(1, 2)))
            solver_results = {}
            for model_name, model, ref in [('deployed', fea_deployed_dc, ref_dep), ('derivative_consistent', fea_consistent_dc, ref_con)]:
                for method in ['euler', 'heun', 'rk4']:
                    rmse, contact = _solver_rmse_against_reference(model, x0t, nt, ut, ref, method)
                    solver_results[model_name, method] = (rmse, contact)
            for b in range(B):
                sid = scenario_id
                scenario_id += 1
                common_ref_strict = bool(not ref_dep_contact[b] and (not ref_con_contact[b]))
                pair_rows.append({'scenario_id': sid, 'base_sample_index': int(g.iloc[b].sample_index), 'n_rpm': float(speed), 'if_A': float(g.iloc[b].if_A), 'id_A': float(g.iloc[b].id_A), 'iq_A': float(g.iloc[b].iq_A), 'direction_index': int(dir_index), 'direction_id': float(dvec[0]), 'direction_iq': float(dvec[1]), 'direction_if': float(dvec[2]), 'macro_steps_one_electrical_period': int(nsteps), 'base_rhs_exact_hurwitz': bool(g.iloc[b].rhs_exact_hurwitz), 'base_consistent_hurwitz': bool(g.iloc[b].consistent_baseline_hurwitz), 'base_relative_A_difference': float(g.iloc[b].relative_frobenius_model_vs_exact), 'reference_deployed_domain_contact': bool(ref_dep_contact[b]), 'reference_consistent_domain_contact': bool(ref_con_contact[b]), 'common_reference_strict': common_ref_strict, 'cross_model_reference_rmse': float(cross_rmse[b]), 'x0_id_A': float(x0_v[b, 0]), 'x0_iq_A': float(x0_v[b, 1]), 'x0_if_A': float(x0_v[b, 2]), 'ueq_d_V': float(ut[b, 0].item()), 'ueq_q_V': float(ut[b, 1].item()), 'ueq_f_V': float(ut[b, 2].item()), 'dc_boundary_policy': DC_BOUNDARY_POLICY, 'dc_primary_reference_substeps': int(primary_ss), 'dc_reference_low_substeps': int(lo_ss), 'dc_reference_high_substeps': int(hi_ss), 'dc_reference_low_vs_high_rmse': float(conv_lo_hi[b]), 'dc_reference_primary_vs_high_rmse': float(conv_mid_hi[b]), 'dc_reference_high_domain_contact': bool(con_contacts[hi_ss][b]), 'dc_reference_high_cell_crossings': int(con_crossings[hi_ss][b]), 'dc_reference_high_interface_hits': int(con_interface_hits[hi_ss][b])})
                for model_name in ['deployed', 'derivative_consistent']:
                    ref_contact = ref_dep_contact[b] if model_name == 'deployed' else ref_con_contact[b]
                    for method in ['euler', 'heun', 'rk4']:
                        rmse, contact = solver_results[model_name, method]
                        all_rows.append({'scenario_id': sid, 'base_sample_index': int(g.iloc[b].sample_index), 'n_rpm': float(speed), 'direction_index': int(dir_index), 'model_representation': model_name, 'solver': method, 'solver_rmse_vs_own_highres_reference': float(rmse[b]), 'reference_domain_contact': bool(ref_contact), 'solver_domain_contact': bool(contact[b]), 'common_reference_strict': common_ref_strict, 'cross_model_reference_rmse': float(cross_rmse[b])})
    dc_trajectory_all = pd.DataFrame(all_rows)
    dc_reference_pair_summary = pd.DataFrame(pair_rows)
    dc_trajectory_all.to_csv(TAB / 'table04r_derivative_consistent_trajectory_all.csv', index=False)
    dc_reference_pair_summary.to_csv(TAB / 'table04s_derivative_consistent_reference_pair_all.csv', index=False)
    if RUN_DC_BOUNDARY_REFERENCE_AUDIT and len(dc_reference_pair_summary):
        conv_rows = []
        candidates = dc_reference_pair_summary.head(int(DC_DOP853_MAX_SCENARIOS)).copy()
        for rr in candidates.itertuples(index=False):
            x0 = np.array([rr.x0_id_A, rr.x0_iq_A, rr.x0_if_A], float)
            ueq = np.array([rr.ueq_d_V, rr.ueq_q_V, rr.ueq_f_V], float)
            n_rpm = float(rr.n_rpm)
            nsteps = int(rr.macro_steps_one_electrical_period)
            t_eval = np.arange(nsteps + 1, dtype=float) * DT
            state = {'domain_contact': False, 'min_interface_distance_normalized': np.inf}
            try:
                sol = solve_ivp(lambda t, y: _dc_rhs_numpy(t, y, n_rpm, ueq, D, state), (0.0, float(t_eval[-1])), x0, method='DOP853', t_eval=t_eval, rtol=DC_DOP853_RTOL, atol=DC_DOP853_ATOL, max_step=float(DC_DOP853_MAX_STEP))
                success = bool(sol.success and sol.y.shape[1] == len(t_eval))
            except Exception as exc:
                sol = None
                success = False
                state['exception'] = repr(exc)
            xt = torch.tensor(x0[None, :], dtype=torch.float64)
            nt1 = torch.tensor([n_rpm], dtype=torch.float64)
            ut1 = torch.tensor(ueq[None, :], dtype=torch.float64)
            refs = {}
            contacts = {}
            crossings = {}
            for ss in sorted(set((int(v) for v in DC_REFERENCE_AUDIT_SUBSTEPS))):
                rrc, ccc, xxc, _ = _highres_reference_boundary_audit(fea_consistent_dc, xt, nt1, ut1, nsteps, ss)
                refs[ss] = rrc[0]
                contacts[ss] = bool(ccc[0])
                crossings[ss] = int(xxc[0])
            ssv = sorted(refs)
            low, high = (ssv[0], ssv[-1])
            primary = int(DERIVATIVE_CONSISTENT_REF_SUBSTEPS)
            if primary not in refs:
                primary = min(ssv, key=lambda v: abs(v - primary))
            dop_rmse = np.nan
            dop_max = np.nan
            if success:
                dop = sol.y.T
                dd = (refs[high] - dop) / STATE_SCALE[None, :]
                dop_rmse = float(np.sqrt(np.mean(dd * dd)))
                dop_max = float(np.max(np.linalg.norm(dd, axis=1)))
            conv_rows.append({'scenario_id': int(rr.scenario_id), 'n_rpm': n_rpm, 'base_rhs_exact_hurwitz': bool(rr.base_rhs_exact_hurwitz), 'low_substeps': int(low), 'primary_substeps': int(primary), 'high_substeps': int(high), 'low_vs_high_normalized_rmse': _trajectory_rmse(refs[low], refs[high]), 'primary_vs_high_normalized_rmse': _trajectory_rmse(refs[primary], refs[high]), 'low_domain_contact': bool(contacts[low]), 'primary_domain_contact': bool(contacts[primary]), 'high_domain_contact': bool(contacts[high]), 'high_reference_cell_crossings': int(crossings[high]), 'dop853_success': bool(success), 'dop853_domain_contact': bool(state.get('domain_contact', False)), 'dop853_nfev': int(sol.nfev) if success else np.nan, 'dop853_min_interface_distance_normalized': float(state.get('min_interface_distance_normalized', np.nan)), 'high_rk4_vs_dop853_normalized_rmse': dop_rmse, 'high_rk4_vs_dop853_max_normalized_state_error': dop_max, 'boundary_policy': DC_BOUNDARY_POLICY, 'dop853_rtol': float(DC_DOP853_RTOL), 'dop853_max_step_s': float(DC_DOP853_MAX_STEP)})
        dc_reference_convergence_all = pd.DataFrame(conv_rows)
        dc_reference_convergence_all.to_csv(TAB / 'table04z_derivative_consistent_reference_convergence_all.csv', index=False)
        valid = dc_reference_convergence_all[dc_reference_convergence_all.dop853_success & ~dc_reference_convergence_all.dop853_domain_contact & ~dc_reference_convergence_all.high_domain_contact].copy()
        dc_reference_convergence_summary = pd.DataFrame([{'audited_scenarios': int(len(dc_reference_convergence_all)), 'dop853_and_high_rk4_strict_scenarios': int(len(valid)), 'boundary_policy': DC_BOUNDARY_POLICY, 'substeps_tested': '/'.join((str(v) for v in sorted(set(DC_REFERENCE_AUDIT_SUBSTEPS)))), 'median_low_vs_high_rmse': float(dc_reference_convergence_all.low_vs_high_normalized_rmse.median()) if len(dc_reference_convergence_all) else np.nan, 'p95_low_vs_high_rmse': float(dc_reference_convergence_all.low_vs_high_normalized_rmse.quantile(0.95)) if len(dc_reference_convergence_all) else np.nan, 'median_primary_vs_high_rmse': float(dc_reference_convergence_all.primary_vs_high_normalized_rmse.median()) if len(dc_reference_convergence_all) else np.nan, 'p95_primary_vs_high_rmse': float(dc_reference_convergence_all.primary_vs_high_normalized_rmse.quantile(0.95)) if len(dc_reference_convergence_all) else np.nan, 'median_high_rk4_vs_dop853_rmse_strict': float(valid.high_rk4_vs_dop853_normalized_rmse.median()) if len(valid) else np.nan, 'p95_high_rk4_vs_dop853_rmse_strict': float(valid.high_rk4_vs_dop853_normalized_rmse.quantile(0.95)) if len(valid) else np.nan, 'max_high_rk4_vs_dop853_rmse_strict': float(valid.high_rk4_vs_dop853_normalized_rmse.max()) if len(valid) else np.nan, 'median_high_reference_cell_crossings': float(dc_reference_convergence_all.high_reference_cell_crossings.median()) if len(dc_reference_convergence_all) else np.nan}])
        dc_reference_convergence_summary.to_csv(TAB / 'table04za_derivative_consistent_reference_convergence_summary.csv', index=False)
        bs = []
        for speed, z in dc_reference_convergence_all.groupby('n_rpm', sort=True):
            zv = z[z.dop853_success & ~z.dop853_domain_contact & ~z.high_domain_contact]
            bs.append({'n_rpm': float(speed), 'audited_scenarios': int(len(z)), 'strict_dop853_pairs': int(len(zv)), 'median_primary_vs_high_rmse': float(z.primary_vs_high_normalized_rmse.median()) if len(z) else np.nan, 'median_high_rk4_vs_dop853_rmse_strict': float(zv.high_rk4_vs_dop853_normalized_rmse.median()) if len(zv) else np.nan, 'median_high_reference_cell_crossings': float(z.high_reference_cell_crossings.median()) if len(z) else np.nan})
        dc_reference_convergence_by_speed = pd.DataFrame(bs)
        dc_reference_convergence_by_speed.to_csv(TAB / 'table04zb_derivative_consistent_reference_convergence_by_speed.csv', index=False)
    piv = dc_trajectory_all.pivot_table(index=['scenario_id', 'solver'], columns='model_representation', values='solver_rmse_vs_own_highres_reference', aggfunc='first').reset_index()
    flags = dc_trajectory_all.groupby(['scenario_id', 'solver']).agg(any_reference_contact=('reference_domain_contact', 'max'), any_solver_contact=('solver_domain_contact', 'max'), common_reference_strict=('common_reference_strict', 'min'), cross_model_reference_rmse=('cross_model_reference_rmse', 'first')).reset_index()
    dc_trajectory_paired = piv.merge(flags, on=['scenario_id', 'solver'], how='left')
    dc_trajectory_paired['paired_strict'] = dc_trajectory_paired.common_reference_strict & ~dc_trajectory_paired.any_reference_contact.astype(bool) & ~dc_trajectory_paired.any_solver_contact.astype(bool)
    dc_trajectory_paired['consistent_minus_deployed_rmse'] = dc_trajectory_paired.derivative_consistent - dc_trajectory_paired.deployed
    scenario_meta = dc_reference_pair_summary[['scenario_id', 'base_sample_index', 'n_rpm', 'if_A', 'id_A', 'iq_A', 'direction_index', 'base_rhs_exact_hurwitz', 'base_consistent_hurwitz', 'base_relative_A_difference']].drop_duplicates('scenario_id')
    dc_trajectory_paired = dc_trajectory_paired.merge(scenario_meta, on='scenario_id', how='left')
    dc_trajectory_paired.to_csv(TAB / 'table04t_derivative_consistent_trajectory_paired.csv', index=False)
    sum_rows = []
    for solver, zall in dc_trajectory_paired.groupby('solver'):
        z = zall[zall.paired_strict].copy()
        sum_rows.append({'solver': solver, 'all_paired_scenarios': int(len(zall)), 'strict_paired_scenarios': int(len(z)), 'strict_pair_fraction': float(len(z) / len(zall)) if len(zall) else np.nan, 'median_rmse_deployed': float(z.deployed.median()) if len(z) else np.nan, 'median_rmse_derivative_consistent': float(z.derivative_consistent.median()) if len(z) else np.nan, 'median_consistent_minus_deployed_rmse': float(z.consistent_minus_deployed_rmse.median()) if len(z) else np.nan, 'consistent_lower_rmse_fraction': float((z.derivative_consistent < z.deployed).mean()) if len(z) else np.nan, 'median_cross_model_reference_rmse': float(z.cross_model_reference_rmse.median()) if len(z) else np.nan, 'p95_cross_model_reference_rmse': float(z.cross_model_reference_rmse.quantile(0.95)) if len(z) else np.nan})
    dc_trajectory_summary = pd.DataFrame(sum_rows)
    dc_trajectory_summary.to_csv(TAB / 'table04u_derivative_consistent_trajectory_summary.csv', index=False)
    speed_rows = []
    for (solver, speed), zall in dc_trajectory_paired.groupby(['solver', 'n_rpm'], sort=True):
        z = zall[zall.paired_strict].copy()
        speed_rows.append({'solver': solver, 'n_rpm': float(speed), 'all_paired_scenarios': int(len(zall)), 'strict_paired_scenarios': int(len(z)), 'strict_pair_fraction': float(len(z) / len(zall)) if len(zall) else np.nan, 'median_rmse_deployed': float(z.deployed.median()) if len(z) else np.nan, 'median_rmse_derivative_consistent': float(z.derivative_consistent.median()) if len(z) else np.nan, 'consistent_lower_rmse_fraction': float((z.derivative_consistent < z.deployed).mean()) if len(z) else np.nan, 'median_cross_model_reference_rmse': float(z.cross_model_reference_rmse.median()) if len(z) else np.nan, 'p95_cross_model_reference_rmse': float(z.cross_model_reference_rmse.quantile(0.95)) if len(z) else np.nan})
    dc_trajectory_speed_summary = pd.DataFrame(speed_rows)
    dc_trajectory_speed_summary.to_csv(TAB / 'table04x_derivative_consistent_trajectory_by_speed.csv', index=False)
    ref_rows = []
    for speed, z in dc_reference_pair_summary.groupby('n_rpm', sort=True):
        zs = z[z.common_reference_strict].copy()
        ref_rows.append({'n_rpm': float(speed), 'all_reference_pairs': int(len(z)), 'common_reference_strict_pairs': int(len(zs)), 'common_reference_strict_fraction': float(len(zs) / len(z)) if len(z) else np.nan, 'median_cross_model_reference_rmse_strict': float(zs.cross_model_reference_rmse.median()) if len(zs) else np.nan, 'p95_cross_model_reference_rmse_strict': float(zs.cross_model_reference_rmse.quantile(0.95)) if len(zs) else np.nan})
    dc_reference_speed_summary = pd.DataFrame(ref_rows)
    dc_reference_speed_summary.to_csv(TAB / 'table04y_derivative_consistent_reference_by_speed.csv', index=False)
    hrows = []
    shrows = []
    for (solver, hflag), zall in dc_trajectory_paired.groupby(['solver', 'base_rhs_exact_hurwitz'], sort=True):
        z = zall[zall.paired_strict].copy()
        hrows.append({'solver': solver, 'base_deployed_hurwitz': bool(hflag), 'all_paired_scenarios': int(len(zall)), 'strict_paired_scenarios': int(len(z)), 'strict_pair_fraction': float(len(z) / len(zall)) if len(zall) else np.nan, 'median_rmse_deployed': float(z.deployed.median()) if len(z) else np.nan, 'median_rmse_derivative_consistent': float(z.derivative_consistent.median()) if len(z) else np.nan, 'consistent_lower_rmse_fraction': float((z.derivative_consistent < z.deployed).mean()) if len(z) else np.nan, 'median_cross_model_reference_rmse': float(z.cross_model_reference_rmse.median()) if len(z) else np.nan})
    for (solver, speed, hflag), zall in dc_trajectory_paired.groupby(['solver', 'n_rpm', 'base_rhs_exact_hurwitz'], sort=True):
        z = zall[zall.paired_strict].copy()
        shrows.append({'solver': solver, 'n_rpm': float(speed), 'base_deployed_hurwitz': bool(hflag), 'all_paired_scenarios': int(len(zall)), 'strict_paired_scenarios': int(len(z)), 'consistent_lower_rmse_fraction': float((z.derivative_consistent < z.deployed).mean()) if len(z) else np.nan, 'median_cross_model_reference_rmse': float(z.cross_model_reference_rmse.median()) if len(z) else np.nan})
    dc_hurwitz_stratum_summary = pd.DataFrame(hrows)
    dc_speed_hurwitz_stratum_summary = pd.DataFrame(shrows)
    dc_hurwitz_stratum_summary.to_csv(TAB / 'table04xc_derivative_consistent_trajectory_by_hurwitz_stratum.csv', index=False)
    dc_speed_hurwitz_stratum_summary.to_csv(TAB / 'table04xd_derivative_consistent_trajectory_by_speed_hurwitz_stratum.csv', index=False)
    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    positive_vals = []
    for solver, z in dc_trajectory_paired[dc_trajectory_paired.paired_strict].groupby('solver'):
        zz = z[(z.deployed > 0) & (z.derivative_consistent > 0)]
        if len(zz):
            ax.scatter(zz.deployed, zz.derivative_consistent, s=18, alpha=0.55, label=solver.upper())
            positive_vals.extend(zz.deployed.tolist())
            positive_vals.extend(zz.derivative_consistent.tolist())
    if positive_vals:
        lo, hi = (min(positive_vals), max(positive_vals))
        ax.plot([lo, hi], [lo, hi], linestyle='--', linewidth=0.9)
        ax.set_xscale('log')
        ax.set_yscale('log')
    ax.set_xlabel('Solver RMSE — deployed representation')
    ax.set_ylabel('Solver RMSE — derivative-consistent representation')
    ax.grid(alpha=0.2, which='both')
    ax.legend()
    fig.tight_layout()
    plt.close(fig)
    plt.close('all')


## Paired comparison and local mode check

In [ ]:
def paired_nonlinear_table(frame, require_full=True):
    needed = {'scenario_id', 'solver', 'paired_strict', 'base_rhs_exact_hurwitz',
              'cross_model_reference_rmse'}
    absent = needed - set(frame.columns)
    if absent:
        raise ValueError('Missing trajectory columns: ' + ', '.join(sorted(absent)))
    df = frame.copy()
    df['solver'] = df['solver'].astype(str).str.lower()
    df['paired_strict'] = df['paired_strict'].astype(bool)
    df['base_rhs_exact_hurwitz'] = df['base_rhs_exact_hurwitz'].astype(bool)
    methods = ('euler', 'heun', 'rk4')
    results = []
    for method in methods:
        q = df.loc[df.solver.eq(method)]
        if q.duplicated(['scenario_id']).any():
            raise ValueError('Duplicate trajectory scenario: ' + method)
        strict = q.loc[q.paired_strict]
        unstable = q.loc[~q.base_rhs_exact_hurwitz]
        usable_unstable = unstable.loc[unstable.paired_strict]
        results.append({
            'Method': {'euler': 'Euler', 'heun': 'Heun', 'rk4': 'RK4'}[method],
            'Paired scenarios': int(q.scenario_id.nunique()),
            'Strict pairs': int(strict.scenario_id.nunique()),
            'Non-Hurwitz bases': int(unstable.scenario_id.nunique()),
            'Strict non-Hurwitz': int(usable_unstable.scenario_id.nunique()),
            'Median cross-model reference RMSE': float(strict.cross_model_reference_rmse.median()) if len(strict) else np.nan,
        })
    out = pd.DataFrame(results)
    if require_full:
        exp_n = {'Euler': 27, 'Heun': 29, 'RK4': 29}
        exp_unstable = {'Euler': 3, 'Heun': 4, 'RK4': 4}
        exp_median = {'Euler': 0.2351, 'Heun': 0.2366, 'RK4': 0.2366}
        for row in out.itertuples(index=False):
            name = row[0]
            if int(row[1]) != 60 or int(row[3]) != 30 or int(row[2]) != exp_n[name] or int(row[4]) != exp_unstable[name]:
                raise AssertionError('Paired-count check failed: ' + name)
            if not np.isclose(float(row[5]), exp_median[name], atol=0.00012):
                raise AssertionError('Reference-difference check failed: ' + name)
    return out


def eigenmode_diagnostic(fea_data, Lgrid, cases, max_cases=12, stop_time_s=0.0004,
                          perturbation_fraction=0.002, sample_count=41):
    import torch
    from scipy.linalg import eig
    if stop_time_s <= 0 or perturbation_fraction <= 0:
        raise ValueError('Positive test duration and perturbation required')
    required = {'id_A', 'iq_A', 'if_A', 'n_rpm', 'rhs_exact_hurwitz', 'rhs_exact_spectral_abscissa_sinv'}
    if not required.issubset(set(cases.columns)):
        raise ValueError('Off-grid audit columns are missing')
    candidates = cases[(~cases.rhs_exact_hurwitz.astype(bool)) &
                       (cases.rhs_exact_spectral_abscissa_sinv.astype(float) > 5)].copy()
    candidates = candidates.sort_values(['rhs_exact_spectral_abscissa_sinv', 'n_rpm', 'if_A'], ascending=[False,True,True], kind='mergesort')
    if candidates.empty:
        raise ValueError('No off-grid positive-real-part candidates')

    priority = candidates.drop_duplicates('n_rpm', keep='first')
    remainder = candidates.loc[~candidates.index.isin(priority.index)]
    chosen = pd.concat([priority, remainder]).head(int(max_cases))
    torch_device = torch.device('cpu')
    model = TorchFEA(fea_data, Lgrid, device=torch_device, dtype=torch.float64)
    consistent = DerivativeConsistentFEA(fea_data, Lgrid, device=torch_device, dtype=torch.float64)
    vectors = [np.asarray(fea_data[k], float) for k in ('id_vec','iq_vec','if_vec')]
    widths = np.asarray([v[1]-v[0] for v in vectors], float)
    normscale = np.asarray(STATE_SCALE, float)
    Imax = np.sqrt(3.0) * float(fea_data['ismax'])
    rows, paths = [], []

    def T(a):
        return torch.tensor(np.asarray(a, float), dtype=torch.float64, device=torch_device)

    def evaluate(model_, x, speed, voltage, left, right, track):

        if np.any(x <= left + 1e-10*widths) or np.any(x >= right - 1e-10*widths) or x[0]**2+x[1]**2 >= Imax**2:
            track['outside'] = True
        with torch.no_grad():
            return model_.rhs(T(x).reshape(1,3), T([speed]), T(voltage).reshape(1,3)).numpy()[0]

    times = np.linspace(0.0, float(stop_time_s), int(sample_count))
    for index, case in chosen.iterrows():
        x0 = np.array([case.id_A, case.iq_A, case.if_A], float)
        speed = float(case.n_rpm)
        bounds_idx = np.array([int(np.searchsorted(v, coord, side='right')-1) for v,coord in zip(vectors,x0)])
        left = np.array([vectors[j][k] for j,k in enumerate(bounds_idx)])
        right = left + widths
        if np.any(x0 <= left) or np.any(x0 >= right):
            continue
        with torch.no_grad():
            A = exact_equilibrium_rhs_jacobian(model,T(x0).reshape(1,3),T([speed]))[0].numpy()[0]
            voltage = model.equilibrium_voltage(T(x0).reshape(1,3),T([speed])).numpy()[0]
            B = consistent.flux_jacobian_at(T(x0).reshape(1,3)).numpy()[0]
        we=2*np.pi*fea_data['p']*speed/60
        omega=np.array([[0.,we,0.],[-we,0.,0.],[0.,0.,0.]])
        R=np.diag([fea_data['Rs'],fea_data['Rs'],fea_data['Rf']])
        Acons=np.linalg.solve(B,-R+omega@B)
        eigvals, eigvecs = eig(A)
        j = int(np.argmax(eigvals.real))
        mode = np.real(eigvecs[:,j])
        if np.linalg.norm(mode/normscale) < 1e-8:
            mode = np.imag(eigvecs[:,j])
        mode = mode / np.linalg.norm(mode / normscale)
        amplitude = perturbation_fraction * min(widths/normscale)
        delta = amplitude * mode
        perturbed=x0+delta
        if np.any(perturbed <= left) or np.any(perturbed >= right):
            continue
        solutions={}
        solver_valid={}
        for tag, machine in [('deployed',model),('consistent',consistent)]:
            watch={'outside':False}
            result=solve_ivp(lambda t,x: evaluate(machine,x,speed,voltage,left,right,watch),
                            (float(times[0]),float(times[-1])),perturbed,method='DOP853',
                            t_eval=times,rtol=2e-10,atol=np.array([1e-10,1e-10,1e-12]),
                            max_step=max(stop_time_s/100,1e-7))
            solutions[tag] = result
            solver_valid[tag] = bool(result.success and len(result.t)==len(times) and not watch['outside'])
        valid=bool(all(solver_valid.values()))
        if valid:
            initial = np.linalg.norm(delta/normscale)
            actual=np.linalg.norm((solutions['deployed'].y.T-x0)/normscale,axis=1)/initial
            consistent_norm=np.linalg.norm((solutions['consistent'].y.T-x0)/normscale,axis=1)/initial
            linear=np.array([np.linalg.norm((expm(A*t)@delta)/normscale)/initial for t in times])
            mismatch=float(np.max(np.abs(actual-linear)))
            dep_growth=float(actual[-1]); con_growth=float(consistent_norm[-1]); lin_growth=float(linear[-1])
            for k,t in enumerate(times):
                paths.append({'case':int(case.sample_index) if 'sample_index' in cases.columns else int(index),
                              'time_s':float(t),'deployed_gain':float(actual[k]),
                              'consistent_gain':float(consistent_norm[k]),'linear_gain':float(linear[k])})
        else:
            mismatch=dep_growth=con_growth=lin_growth=np.nan
        rows.append({'case':int(case.sample_index) if 'sample_index' in cases.columns else int(index),
                     'n_rpm':speed,'id_A':x0[0],'iq_A':x0[1],'if_A':x0[2],
                     'alpha_rhs_sinv':float(max(eigvals.real)),
                     'alpha_cons_sinv':float(max(np.linalg.eigvals(Acons).real)),
                     'duration_s':float(times[-1]),'perturbation_scaled_norm':float(np.linalg.norm(delta/normscale)),
                     'strict_same_cell':valid,'deployed_gain':dep_growth,'consistent_gain':con_growth,
                     'linear_gain':lin_growth,'max_abs_gain_difference':mismatch,
                     'solver_success_deployed':bool(solver_valid['deployed']),
                     'solver_success_consistent':bool(solver_valid['consistent'])})
    return pd.DataFrame(rows),pd.DataFrame(paths)


if len(dc_trajectory_paired):
    paired_summary_s4 = paired_nonlinear_table(dc_trajectory_paired, require_full=FULL_RUN)
    paired_summary_s4.to_csv(TAB / 'table_s4_paired_nonlinear.csv', index=False)
else:
    paired_summary_s4 = pd.DataFrame()

if RUN_EXTRA_MODE_CHECK and len(offgrid_exact_df):
    mode_cases, mode_paths = eigenmode_diagnostic(D, L_edge2, offgrid_exact_df,
        max_cases=12 if FULL_RUN else 4)
    mode_cases.to_csv(TAB / 'mode_check_cases.csv', index=False)
    mode_paths.to_csv(TAB / 'mode_check_paths.csv', index=False)
    if not mode_paths.empty:
        q = mode_paths[mode_paths['case'].isin(mode_cases.head(2)['case'])]
        fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
        for ax, (case_id, group) in zip(axes, q.groupby('case', sort=False)):
            ax.plot(group.time_s * 1e3, group.deployed_gain, label='Deployed')
            ax.plot(group.time_s * 1e3, group.consistent_gain, label='Consistent')
            ax.plot(group.time_s * 1e3, group.linear_gain, linestyle='--', label='Linear')
            ax.set(xlabel='Time (ms)', ylabel='Normalized gain', title=f'Case {case_id}')
            ax.legend(fontsize=8)
            ax.grid(alpha=0.2)
        fig.tight_layout()
        plt.close(fig)
else:
    mode_cases, mode_paths = pd.DataFrame(), pd.DataFrame()


## Interface-crossing reference check

In [ ]:
dc_event_aligned_reference_all = pd.DataFrame()
dc_event_aligned_reference_summary = pd.DataFrame()

def _event_grid_spec(D):
    return [np.asarray(D['id_vec'], float), np.asarray(D['iq_vec'], float), np.asarray(D['if_vec'], float)]

def _make_piecewise_events(D, x_start, near_tol_rel=1e-10):
    grids = _event_grid_spec(D)
    events = []
    meta = []
    for d, g in enumerate(grids):
        span = max(float(g[-1] - g[0]), 1.0)
        near_tol = near_tol_rel * span
        for value, label in [(float(g[0]), 'domain_lower'), (float(g[-1]), 'domain_upper')]:
            if abs(float(x_start[d]) - value) <= near_tol:
                continue

            def ev(t, y, dd=d, vv=value):
                return y[dd] - vv
            ev.terminal = True
            ev.direction = 0
            events.append(ev)
            meta.append((d, value, label))
        for value in np.asarray(g[1:-1], float):
            if abs(float(x_start[d]) - float(value)) <= near_tol:
                continue

            def ev(t, y, dd=d, vv=float(value)):
                return y[dd] - vv
            ev.terminal = True
            ev.direction = 0
            events.append(ev)
            meta.append((d, float(value), 'internal'))
    return (events, meta)

def _solve_dc_event_aligned(x0, n_rpm, ueq, D, t_eval, max_step=None):
    t_eval = np.asarray(t_eval, float)
    yout = np.full((len(t_eval), 3), np.nan, float)
    yout[0] = np.asarray(x0, float)
    next_out = 1
    t0 = float(t_eval[0])
    tf = float(t_eval[-1])
    x = np.asarray(x0, float).copy()
    restarts = 0
    interface_events = 0
    domain_contact = False
    success = True
    max_step = float(DC_DOP853_MAX_STEP if max_step is None else max_step)
    grids = _event_grid_spec(D)
    spacing = np.array([np.min(np.diff(g)) for g in grids], float)
    while t0 < tf - 1e-15:
        if restarts > int(EVENT_ALIGNED_MAX_RESTARTS):
            success = False
            break
        events, meta = _make_piecewise_events(D, x, near_tol_rel=EVENT_ALIGNED_NUDGE_REL * 10)
        try:
            sol = solve_ivp(lambda t, y: _dc_rhs_numpy(t, y, n_rpm, ueq, D, None), (t0, tf), x, method='DOP853', rtol=DC_DOP853_RTOL, atol=DC_DOP853_ATOL, max_step=max_step, dense_output=True, events=events)
        except Exception:
            success = False
            break
        if not sol.success:
            success = False
            break
        tend = float(sol.t[-1])
        while next_out < len(t_eval) and t_eval[next_out] <= tend + 1e-14:
            yout[next_out] = np.asarray(sol.sol(float(t_eval[next_out])), float)
            next_out += 1
        hit = []
        for ei, te in enumerate(sol.t_events):
            if len(te):
                hit.append((float(te[0]), ei))
        if not hit:
            x = np.asarray(sol.y[:, -1], float)
            t0 = tend
            break
        hit.sort(key=lambda z: z[0])
        th, ei = hit[0]
        d, value, kind = meta[ei]
        xh = np.asarray(sol.y_events[ei][0], float)
        if kind.startswith('domain_'):
            domain_contact = True
            x = xh
            t0 = th
            break
        interface_events += 1
        restarts += 1
        v = np.asarray(_dc_rhs_numpy(th, xh, n_rpm, ueq, D, None), float)
        xn = xh.copy()
        for dd, g in enumerate(grids):
            internal = np.asarray(g[1:-1], float)
            if len(internal) == 0:
                continue
            jj = int(np.argmin(np.abs(internal - xh[dd])))
            if abs(float(internal[jj] - xh[dd])) <= max(1e-12, EVENT_ALIGNED_NUDGE_REL * max(float(g[-1] - g[0]), 1.0) * 10):
                sgn = np.sign(v[dd])
                if sgn == 0:
                    sgn = 1.0
                xn[dd] += float(sgn) * float(EVENT_ALIGNED_NUDGE_REL) * spacing[dd]
        x = xn
        t0 = th
    if next_out < len(t_eval) and (not domain_contact) and success:
        yout[next_out:] = x
    return {'success': bool(success and np.all(np.isfinite(yout[:max(next_out, 1)]))), 'domain_contact': bool(domain_contact), 'trajectory': yout, 'interface_events': int(interface_events), 'restarts': int(restarts)}
if RUN_EVENT_ALIGNED_REFERENCE_AUDIT and RUN_DERIVATIVE_CONSISTENT_TRAJECTORY_AUDIT and len(dc_reference_pair_summary):
    rows = []
    candidates = dc_reference_pair_summary.head(int(EVENT_ALIGNED_MAX_SCENARIOS)).copy()
    for rr in candidates.itertuples(index=False):
        x0 = np.array([rr.x0_id_A, rr.x0_iq_A, rr.x0_if_A], float)
        ueq = np.array([rr.ueq_d_V, rr.ueq_q_V, rr.ueq_f_V], float)
        n_rpm = float(rr.n_rpm)
        nsteps = int(rr.macro_steps_one_electrical_period)
        t_eval = np.arange(nsteps + 1, dtype=float) * DT
        ev = _solve_dc_event_aligned(x0, n_rpm, ueq, D, t_eval, max_step=DC_DOP853_MAX_STEP)
        xt = torch.tensor(x0[None, :], dtype=torch.float64)
        nt = torch.tensor([n_rpm], dtype=torch.float64)
        ut = torch.tensor(ueq[None, :], dtype=torch.float64)
        hi = max((int(v) for v in DC_REFERENCE_AUDIT_SUBSTEPS))
        rk, contact, crossings, _ = _highres_reference_boundary_audit(fea_consistent_dc, xt, nt, ut, nsteps, hi)
        rk = rk[0]
        rmse = np.nan
        maxerr = np.nan
        if ev['success'] and (not ev['domain_contact']) and (not bool(contact[0])) and np.all(np.isfinite(ev['trajectory'])):
            dd = (rk - ev['trajectory']) / STATE_SCALE[None, :]
            rmse = float(np.sqrt(np.mean(dd * dd)))
            maxerr = float(np.max(np.linalg.norm(dd, axis=1)))
        rows.append({'scenario_id': int(rr.scenario_id), 'n_rpm': n_rpm, 'base_rhs_exact_hurwitz': bool(rr.base_rhs_exact_hurwitz), 'event_aligned_success': bool(ev['success']), 'event_aligned_domain_contact': bool(ev['domain_contact']), 'event_aligned_interface_events': int(ev['interface_events']), 'event_aligned_restarts': int(ev['restarts']), 'high_rk4_substeps': int(hi), 'high_rk4_domain_contact': bool(contact[0]), 'high_rk4_cell_crossings': int(crossings[0]), 'high_rk4_vs_event_aligned_normalized_rmse': rmse, 'high_rk4_vs_event_aligned_max_normalized_state_error': maxerr, 'boundary_policy': DC_BOUNDARY_POLICY})
    dc_event_aligned_reference_all = pd.DataFrame(rows)
    dc_event_aligned_reference_all.to_csv(TAB / 'table04ze_event_aligned_reference_all.csv', index=False)
    strict = dc_event_aligned_reference_all[dc_event_aligned_reference_all.event_aligned_success & ~dc_event_aligned_reference_all.event_aligned_domain_contact & ~dc_event_aligned_reference_all.high_rk4_domain_contact].copy()
    dc_event_aligned_reference_summary = pd.DataFrame([{'audited_scenarios': int(len(dc_event_aligned_reference_all)), 'strict_comparable_scenarios': int(len(strict)), 'median_high_rk4_vs_event_aligned_rmse': float(strict.high_rk4_vs_event_aligned_normalized_rmse.median()) if len(strict) else np.nan, 'p95_high_rk4_vs_event_aligned_rmse': float(strict.high_rk4_vs_event_aligned_normalized_rmse.quantile(0.95)) if len(strict) else np.nan, 'max_high_rk4_vs_event_aligned_rmse': float(strict.high_rk4_vs_event_aligned_normalized_rmse.max()) if len(strict) else np.nan, 'median_event_aligned_interface_events': float(strict.event_aligned_interface_events.median()) if len(strict) else np.nan, 'boundary_policy': DC_BOUNDARY_POLICY, 'interpretation': 'Numerical reference audit for the piecewise-smooth derivative-consistent diagnostic; not transient-FEA/experimental validation.'}])
    dc_event_aligned_reference_summary.to_csv(TAB / 'table04zf_event_aligned_reference_summary.csv', index=False)


## Euler critical time step

In [ ]:
atlas_exc = atlas[atlas['if_A'].isin([2.0, 4.0, 6.0, 8.0])].copy()
atlas_exc['hcrit_us'] = atlas_exc['dtcrit_s'] * 1000000.0
atlas_exc['stable_50us'] = atlas_exc['dtcrit_s'] > DT
stability_table = atlas_exc.groupby(['n_rpm', 'if_A']).agg(points=('hcrit_us', 'size'), hcrit_min_us=('hcrit_us', 'min'), hcrit_median_us=('hcrit_us', 'median'), hcrit_max_us=('hcrit_us', 'max'), stable_50us_fraction=('stable_50us', 'mean')).reset_index()
stability_table.to_csv(TAB / 'table02_stability_atlas.csv', index=False)
sel = atlas[(atlas.n_rpm == 9000) & (atlas.if_A == 6)].copy()
Z = np.full((len(D['iq_vec']), len(D['id_vec'])), np.nan)
for _, r in sel.iterrows():
    Z[int(r.j), int(r.i)] = r.dtcrit_s * 1000000.0
X, Y = np.meshgrid(D['id_vec'], D['iq_vec'])
fig, ax = plt.subplots(figsize=(5.2, 4.0))
cs = ax.contourf(X, Y, np.ma.masked_invalid(Z), levels=18)
fig.colorbar(cs, ax=ax, label='$h_{crit}$ ($\\mu$s)')
th = np.linspace(0, 2 * np.pi, 500)
Rlim = np.sqrt(3) * D['ismax']
ax.plot(Rlim * np.cos(th), Rlim * np.sin(th), 'k--', lw=0.8)
ax.set_xlabel('$i_d$ (A)')
ax.set_ylabel('$i_q$ (A)')
ax.set_aspect('equal', 'box')
ax.set_title('Explicit-Euler critical step: 9000 rpm, $i_f=6$ A')
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Derivative-method sensitivity

In [ ]:
sensitivity_frames = {'edge2': atlas.copy()}
for method in ['edge1', 'spline']:
    Lm = derivative_grids(D, method)
    dfa, _, _ = build_atlas(D, Lm, DT)
    sensitivity_frames[method] = dfa
key = ['n_rpm', 'if_A', 'id_A', 'iq_A']
base_df = sensitivity_frames['edge2'][key + ['dtcrit_s']].rename(columns={'dtcrit_s': 'edge2'})
sens_rows = []
for method in ['edge1', 'spline']:
    m = base_df.merge(sensitivity_frames[method][key + ['dtcrit_s']].rename(columns={'dtcrit_s': method}), on=key)
    rel = np.abs(m[method] - m.edge2) / np.maximum(m.edge2, 1e-15)
    disagree = (m[method] >= DT) != (m.edge2 >= DT)
    sens_rows.append({'derivative_method': method, 'median_relative_hcrit_change': float(np.median(rel)), 'p95_relative_hcrit_change': float(np.percentile(rel, 95)), 'maximum_relative_hcrit_change': float(np.max(rel)), '50us_classification_disagreement_fraction': float(np.mean(disagree))})
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(TAB / 'table03_derivative_sensitivity.csv', index=False)
fig, ax = plt.subplots(figsize=(5.8, 3.6))
for method, dfm in sensitivity_frames.items():
    z = dfm[(dfm.n_rpm == 9000) & (dfm.if_A == 6)].dtcrit_s.to_numpy() * 1000000.0
    ax.hist(z, bins=24, histtype='step', density=True, label=method)
ax.set_xlabel('$h_{crit}$ ($\\mu$s)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Matrix conditioning and voltage limits

In [ ]:
unique_ops = {}
for o in ops:
    key = (int(o['j']), int(o['i']), int(o['k']))
    if key not in unique_ops:
        unique_ops[key] = o
psi_scale = np.array([np.max(np.abs(D['psid'])), np.max(np.abs(D['psiq'])), np.max(np.abs(D['psif']))], dtype=float)
current_scale = np.array([np.sqrt(3.0) * D['ismax'], np.sqrt(3.0) * D['ismax'], D['ifmax']], dtype=float)
Psi_inv = np.diag(1.0 / np.maximum(psi_scale, 1e-15))
I_scale = np.diag(current_scale)
L_rows = []
for (j, i, k), o in unique_ops.items():
    if float(o['if_A']) not in (2.0, 4.0, 6.0, 8.0):
        continue
    Lm = np.asarray(o['L'], dtype=float)
    s = np.linalg.svd(Lm, compute_uv=False)
    Lhat = Psi_inv @ Lm @ I_scale
    shat = np.linalg.svd(Lhat, compute_uv=False)
    asym = np.linalg.norm(Lm - Lm.T, 'fro') / max(np.linalg.norm(Lm, 'fro'), 1e-15)
    sym_min = float(np.min(np.linalg.eigvalsh(0.5 * (Lm + Lm.T))))
    L_rows.append({'if_A': float(o['if_A']), 'id_A': float(o['id_A']), 'iq_A': float(o['iq_A']), 'cond2_raw': float(s[0] / s[-1]), 'sigma_min_raw_H': float(s[-1]), 'sigma_max_raw_H': float(s[0]), 'det_L': float(np.linalg.det(Lm)), 'relative_asymmetry_fro': float(asym), 'lambda_min_symmetric_part_H': sym_min, 'cond2_dimensionless_scaled': float(shat[0] / shat[-1]), 'sigma_min_dimensionless_scaled': float(shat[-1]), 'sigma_max_dimensionless_scaled': float(shat[0])})
L_audit = pd.DataFrame(L_rows)
L_audit.to_csv(TAB / 'table03b_incremental_matrix_audit_all.csv', index=False)
L_audit_summary = pd.DataFrame([{'unique_current_field_points': len(L_audit), 'raw_cond2_median': float(L_audit.cond2_raw.median()), 'raw_cond2_p95': float(L_audit.cond2_raw.quantile(0.95)), 'raw_cond2_max': float(L_audit.cond2_raw.max()), 'raw_sigma_min_H_min': float(L_audit.sigma_min_raw_H.min()), 'raw_sigma_min_H_median': float(L_audit.sigma_min_raw_H.median()), 'scaled_cond2_median': float(L_audit.cond2_dimensionless_scaled.median()), 'scaled_cond2_p95': float(L_audit.cond2_dimensionless_scaled.quantile(0.95)), 'scaled_cond2_max': float(L_audit.cond2_dimensionless_scaled.max()), 'relative_asymmetry_median': float(L_audit.relative_asymmetry_fro.median()), 'relative_asymmetry_p95': float(L_audit.relative_asymmetry_fro.quantile(0.95))}])
L_audit_summary.to_csv(TAB / 'table03c_incremental_matrix_audit_summary.csv', index=False)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.hist(np.log10(L_audit.cond2_dimensionless_scaled.to_numpy()), bins=30)
ax.set_xlabel('$\\log_{10}\\kappa_2(\\widehat{L}_{\\Delta})$')
ax.set_ylabel('Operating points')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')
L_sym = 0.5 * (L_edge2 + np.swapaxes(L_edge2, -1, -2))
atlas_sym, _, _ = build_atlas(D, L_sym, DT)
base_cmp = atlas[['n_rpm', 'if_A', 'id_A', 'iq_A', 'dtcrit_s', 'cont_stable']].copy()
base_cmp = base_cmp.rename(columns={'dtcrit_s': 'dtcrit_primary_s', 'cont_stable': 'cont_stable_primary'})
sym_cmp = atlas_sym[['n_rpm', 'if_A', 'id_A', 'iq_A', 'dtcrit_s', 'cont_stable']].copy()
sym_cmp = sym_cmp.rename(columns={'dtcrit_s': 'dtcrit_sym_s', 'cont_stable': 'cont_stable_sym'})
sym_sens = base_cmp.merge(sym_cmp, on=['n_rpm', 'if_A', 'id_A', 'iq_A'], how='left')
sym_sens['sym_solution_available'] = sym_sens.dtcrit_sym_s.notna()
sym_sens['stable50_primary'] = sym_sens.dtcrit_primary_s >= DT
sym_sens['stable50_sym'] = sym_sens.dtcrit_sym_s >= DT
sym_sens['classification_disagreement'] = np.where(sym_sens.sym_solution_available, sym_sens.stable50_primary != sym_sens.stable50_sym, True)
sym_sens['relative_hcrit_change'] = np.where(sym_sens.sym_solution_available, (sym_sens.dtcrit_sym_s - sym_sens.dtcrit_primary_s) / np.maximum(sym_sens.dtcrit_primary_s, 1e-15), np.nan)
sym_sens.to_csv(TAB / 'table03f_symmetrized_incremental_matrix_sensitivity_all.csv', index=False)
sym_rows = []
for key, gx in [('all', sym_sens)]:
    z = gx[gx.sym_solution_available]
    sym_rows.append({'domain': key, 'points_primary': len(gx), 'sym_solution_fraction': float(gx.sym_solution_available.mean()), 'sym_hurwitz_fraction': float(z.cont_stable_sym.mean()) if len(z) else np.nan, 'median_relative_hcrit_change': float(z.relative_hcrit_change.median()) if len(z) else np.nan, 'p95_abs_relative_hcrit_change': float(z.relative_hcrit_change.abs().quantile(0.95)) if len(z) else np.nan, 'stable50_classification_disagreement_fraction': float(gx.classification_disagreement.mean())})
for (n, iff), gx in sym_sens.groupby(['n_rpm', 'if_A']):
    z = gx[gx.sym_solution_available]
    sym_rows.append({'domain': f'{int(n)}rpm_if{iff:g}A', 'points_primary': len(gx), 'sym_solution_fraction': float(gx.sym_solution_available.mean()), 'sym_hurwitz_fraction': float(z.cont_stable_sym.mean()) if len(z) else np.nan, 'median_relative_hcrit_change': float(z.relative_hcrit_change.median()) if len(z) else np.nan, 'p95_abs_relative_hcrit_change': float(z.relative_hcrit_change.abs().quantile(0.95)) if len(z) else np.nan, 'stable50_classification_disagreement_fraction': float(gx.classification_disagreement.mean())})
sym_summary = pd.DataFrame(sym_rows)
sym_summary.to_csv(TAB / 'table03g_symmetrized_incremental_matrix_sensitivity_summary.csv', index=False)
VDQ_LIMIT_SPWM = float(D['udc'] * np.sqrt(3.0 / 8.0))
VDQ_LIMIT_SVPWM = float(D['udc'] / np.sqrt(2.0))
op_domain_df = atlas.copy().reset_index(drop=True)
op_domain_df['op_index'] = np.arange(len(op_domain_df), dtype=int)
op_domain_df['u_d_eq_V'] = ueq_all[:, 0]
op_domain_df['u_q_eq_V'] = ueq_all[:, 1]
op_domain_df['u_dq_eq_V'] = np.hypot(ueq_all[:, 0], ueq_all[:, 1])
op_domain_df['spwm_limit_V'] = VDQ_LIMIT_SPWM
op_domain_df['svpwm_limit_V'] = VDQ_LIMIT_SVPWM
op_domain_df['voltage_feasible_spwm'] = op_domain_df.u_dq_eq_V <= VDQ_LIMIT_SPWM
op_domain_df['voltage_feasible_svpwm'] = op_domain_df.u_dq_eq_V <= VDQ_LIMIT_SVPWM
op_domain_df['stable_50us'] = op_domain_df.dtcrit_s > DT
op_domain_df.to_csv(TAB / 'table03d_operating_domain_voltage_flags.csv', index=False)
domain_rows = []
for (n, iff), g in op_domain_df[op_domain_df.if_A.isin([2.0, 4.0, 6.0, 8.0])].groupby(['n_rpm', 'if_A']):
    for label, mask_col, limit in [('current_limited', None, np.nan), ('two_level_spwm_linear', 'voltage_feasible_spwm', VDQ_LIMIT_SPWM), ('two_level_svpwm_linear', 'voltage_feasible_svpwm', VDQ_LIMIT_SVPWM)]:
        z = g if mask_col is None else g[g[mask_col]]
        domain_rows.append({'n_rpm': float(n), 'if_A': float(iff), 'domain': label, 'assumed_u_dq_limit_V': limit, 'current_limited_points': len(g), 'retained_points': len(z), 'retained_fraction': float(len(z) / len(g)) if len(g) else np.nan, 'hcrit_min_us': float(z.dtcrit_s.min() * 1000000.0) if len(z) else np.nan, 'hcrit_median_us': float(z.dtcrit_s.median() * 1000000.0) if len(z) else np.nan, 'hcrit_max_us': float(z.dtcrit_s.max() * 1000000.0) if len(z) else np.nan, 'stable_50us_fraction': float(z.stable_50us.mean()) if len(z) else np.nan})
domain_summary = pd.DataFrame(domain_rows)
domain_summary.to_csv(TAB / 'table03e_voltage_feasible_stability_summary.csv', index=False)
g = op_domain_df[(op_domain_df.n_rpm == 9000) & (op_domain_df.if_A == 6.0)].copy()
VZ = np.full((len(D['iq_vec']), len(D['id_vec'])), np.nan)
for _, r in g.iterrows():
    VZ[int(r.j), int(r.i)] = r.u_dq_eq_V
Xv_map, Yv_map = np.meshgrid(D['id_vec'], D['iq_vec'])
fig, ax = plt.subplots(figsize=(5.2, 4.0))
cs = ax.contourf(Xv_map, Yv_map, np.ma.masked_invalid(VZ), levels=18)
fig.colorbar(cs, ax=ax, label='$|u_{dq}^{*}|$ (V)')
if np.nanmin(VZ) <= VDQ_LIMIT_SPWM <= np.nanmax(VZ):
    ax.contour(Xv_map, Yv_map, VZ, levels=[VDQ_LIMIT_SPWM], linewidths=1.2, linestyles='--')
if np.nanmin(VZ) <= VDQ_LIMIT_SVPWM <= np.nanmax(VZ):
    ax.contour(Xv_map, Yv_map, VZ, levels=[VDQ_LIMIT_SVPWM], linewidths=1.2, linestyles='-')
th = np.linspace(0, 2 * np.pi, 500)
Rlim = np.sqrt(3) * D['ismax']
ax.plot(Rlim * np.cos(th), Rlim * np.sin(th), 'k:', lw=0.8)
ax.set_xlabel('$i_d$ (A)')
ax.set_ylabel('$i_q$ (A)')
ax.set_aspect('equal', 'box')
ax.set_title('Equilibrium voltage demand: 9000 rpm, $i_f=6$ A')
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Numerical reference convergence

In [ ]:
check_ids = np.linspace(0, len(eligible) - 1, min(256, len(eligible)), dtype=int)
gi = eligible[check_ids]
with torch.no_grad():
    b = torch.tensor(base_all[gi], device=DEVICE)
    n = torch.tensor(n_all[gi], device=DEVICE)
    u = torch.tensor(ueq_all[gi], device=DEVICE)
    eq_rhs = fea.rhs(b, n, u).cpu().numpy()
print('Maximum equilibrium residual norm:', np.linalg.norm(eq_rhs, axis=1).max())
rng = np.random.default_rng(SEED + 7)
conv_ids = rng.choice(test_ops, size=min(200, len(test_ops)), replace=False)
x0n = np.tile(np.array([20.0 / 30.0, -10.0 / 30.0, 0.2 / 0.35], np.float32), (len(conv_ids), 1))
scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    base = torch.tensor(base_all[conv_ids], device=DEVICE)
    n = torch.tensor(n_all[conv_ids], device=DEVICE)
    u = torch.tensor(ueq_all[conv_ids], device=DEVICE)
    x = base + torch.tensor(x0n, device=DEVICE) * scale
    y10 = rk4_step(fea, x, n, u, DT, 10)
    y20 = rk4_step(fea, x, n, u, DT, 20)
    err = torch.linalg.norm((y10 - y20) / scale, dim=1).cpu().numpy()
conv_df = pd.DataFrame([{'median_normalized_difference': float(np.median(err)), 'p95_normalized_difference': float(np.percentile(err, 95)), 'maximum_normalized_difference': float(np.max(err))}])
conv_df.to_csv(TAB / 'table04_reference_convergence.csv', index=False)
reference_precision_df = pd.DataFrame()
if RUN_REFERENCE_PRECISION_AUDIT:
    fea64_ref = TorchFEA(D, L_edge2, device=torch.device('cpu'), dtype=torch.float64)
    base64 = torch.tensor(base_all[conv_ids], dtype=torch.float64)
    n64 = torch.tensor(n_all[conv_ids], dtype=torch.float64)
    u64 = torch.tensor(ueq_all[conv_ids], dtype=torch.float64)
    scale64 = torch.tensor(STATE_SCALE, dtype=torch.float64)
    x64 = base64 + torch.tensor(x0n, dtype=torch.float64) * scale64
    with torch.no_grad():
        y64 = rk4_step(fea64_ref, x64, n64, u64, DT, 10)
    precision_err = np.linalg.norm((y10.detach().cpu().numpy().astype(np.float64) - y64.cpu().numpy()) / STATE_SCALE[None, :], axis=1)
    reference_precision_df = pd.DataFrame([{'samples': int(len(precision_err)), 'float32_substeps': 10, 'float64_substeps': 10, 'median_normalized_difference': float(np.median(precision_err)), 'p95_normalized_difference': float(np.percentile(precision_err, 95)), 'maximum_normalized_difference': float(np.max(precision_err))}])
    reference_precision_df.to_csv(TAB / 'table04c_reference_precision_audit.csv', index=False)


## Training sample preparation

In [ ]:
Xraw, Ycorr, sample_gids, sample_x0n = generate_one_step_samples(fea, ops, base_all, n_all, ueq_all, eligible, samples_per_op=SAMPLES_PER_OP, seed=SEED + 20, ref_substeps=ONE_STEP_REF_SUBSTEPS, batch_size=4096)
train_mask = np.isin(sample_gids, train_ops)
val_mask = np.isin(sample_gids, val_ops)
test_mask = np.isin(sample_gids, test_ops)
XMU = Xraw[train_mask].mean(axis=0)
XSD = Xraw[train_mask].std(axis=0) + 1e-07
X = (Xraw - XMU) / XSD
Xtr, Ytr = (X[train_mask], Ycorr[train_mask])
Xv, Yv = (X[val_mask], Ycorr[val_mask])
Xt, Yt = (X[test_mask], Ycorr[test_mask])
part = pd.DataFrame([{'partition': 'train', 'operating_points': len(train_ops), 'samples': int(train_mask.sum())}, {'partition': 'validation', 'operating_points': len(val_ops), 'samples': int(val_mask.sum())}, {'partition': 'test', 'operating_points': len(test_ops), 'samples': int(test_mask.sum())}])
part.to_csv(TAB / 'table05_partitions.csv', index=False)
np.savez(OUT / 'preprocessing.npz', XMU=XMU, XSD=XSD, STATE_SCALE=STATE_SCALE, train_ops=train_ops, val_ops=val_ops, test_ops=test_ops, guard_ops=guard_ops)


## Optimization datasets

In [ ]:
def stratified_subsample(ids, max_n, seed):
    if len(ids) <= max_n:
        return np.asarray(ids, dtype=int)
    rng = np.random.default_rng(seed)
    strata = {}
    for gi in ids:
        o = ops[int(gi)]
        key = (int(o['n_rpm']), int(o['if_A']), int(o['rho_euler'] > 1))
        strata.setdefault(key, []).append(int(gi))
    picked = []
    per = max(1, max_n // len(strata))
    for a in strata.values():
        a = np.asarray(a)
        rng.shuffle(a)
        picked.extend(a[:min(per, len(a))].tolist())
    remaining = list(set(map(int, ids)) - set(picked))
    rng.shuffle(remaining)
    picked.extend(remaining[:max(0, max_n - len(picked))])
    return np.asarray(picked[:max_n], dtype=int)
hpo_train_ops = stratified_subsample(train_ops, HPO_TRAIN_OPS, SEED + 30)
hpo_val_ops = stratified_subsample(val_ops, HPO_VAL_OPS, SEED + 31)
train_bank_ops = stratified_subsample(train_ops, TRAIN_ROLLOUT_OPS, SEED + 32)
val_bank_ops = stratified_subsample(val_ops, VAL_ROLLOUT_OPS, SEED + 33)
rng = np.random.default_rng(SEED + 34)
train_x0n = np.clip(rng.normal(0, 0.45, size=(len(train_bank_ops), 3)), -0.9, 0.9).astype(np.float32)
val_x0n = np.clip(rng.normal(0, 0.45, size=(len(val_bank_ops), 3)), -0.9, 0.9).astype(np.float32)
train_bank = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, train_bank_ops, train_x0n, horizon=12, ref_substeps=ROLLOUT_REF_SUBSTEPS)
val_bank = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, val_bank_ops, val_x0n, horizon=40, ref_substeps=ROLLOUT_REF_SUBSTEPS)
htr = np.isin(sample_gids, hpo_train_ops)
hva = np.isin(sample_gids, hpo_val_ops)
HXtr, HYtr = (X[htr], Ycorr[htr])
HXv, HYv = (X[hva], Ycorr[hva])
print('HPO samples:', len(HXtr), len(HXv))


## Hyperparameter optimization

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = {'depth': trial.suggest_int('depth', 2, 4), 'width': trial.suggest_categorical('width', [64, 96, 128, 160]), 'lr': trial.suggest_float('lr', 0.0002, 0.005, log=True), 'weight_decay': trial.suggest_float('weight_decay', 1e-07, 0.003, log=True), 'batch_size': trial.suggest_categorical('batch_size', [128, 256, 512]), 'beta': trial.suggest_float('beta', 0.05, 0.6, log=True), 'lambda_rollout': trial.suggest_categorical('lambda_rollout', [0.1, 0.3, 1.0, 3.0]), 'rollout_horizon': trial.suggest_categorical('rollout_horizon', [4, 8, 12])}
    m, score, _ = train_model(p, True, fea, HXtr, HYtr, HXv, HYv, train_bank, val_bank, base_all, n_all, ueq_all, XMU, XSD, epochs=HPO_EPOCHS, seed=HPO_COMMON_SEED, rollout_updates=ROLLOUT_UPDATES, rollout_batch=ROLLOUT_BATCH, patience=12)
    return score
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
BEST = dict(study.best_params)
print('Best rollout-aware validation score:', study.best_value)
print(json.dumps(BEST, indent=2))
trials = study.trials_dataframe()
trials.to_csv(TAB / 'table06_hpo_trials.csv', index=False)
pd.DataFrame([{'best_validation_score': study.best_value, 'hpo_common_seed': int(HPO_COMMON_SEED), **BEST}]).to_csv(TAB / 'table07_selected_hyperparameters.csv', index=False)
complete = trials[trials.state == 'COMPLETE'].copy()
complete['running_best'] = complete['value'].cummin()
fig, ax = plt.subplots(figsize=(5.8, 3.5))
ax.scatter(complete.number, complete.value, s=24, label='trial')
ax.plot(complete.number, complete.running_best, label='running best')
ax.set_yscale('log')
ax.set_xlabel('Trial')
ax.set_ylabel('Validation score')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Neural network training

In [ ]:
def one_step_mse(model):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(Xt, device=DEVICE)).cpu().numpy()
    return float(np.mean((pred - Yt) ** 2))

def train_family(name, certified, params, seeds, selection='rollout'):
    models = []
    rows = []
    histories = []
    for s in range(seeds):
        m, score, h = train_model(params, certified, fea, Xtr, Ytr, Xv, Yv, train_bank, val_bank, base_all, n_all, ueq_all, XMU, XSD, epochs=FINAL_EPOCHS, seed=SEED + 3000 + 100 * s + (0 if certified else 5000), rollout_updates=ROLLOUT_UPDATES, rollout_batch=ROLLOUT_BATCH, patience=18, selection_metric=selection)
        mse = one_step_mse(m)
        rows.append({'configuration': name, 'seed': s, 'validation_score': score, 'test_one_step_mse': mse})
        h['configuration'] = name
        h['seed'] = s
        histories.append(h)
        torch.save(m.cpu().state_dict(), MOD / f'{name}_seed{s}.pt')
        models.append(m.cpu())
        print(name, s + 1, '/', seeds, 'score=', score, 'one-step=', mse)
    return (models, pd.DataFrame(rows), pd.concat(histories, ignore_index=True))
primary_params = BEST.copy()
one_step_params = BEST.copy()
one_step_params['lambda_rollout'] = 0.0
one_step_params['rollout_horizon'] = 0
primary_models, primary_seed, primary_hist = train_family('certified_rollout', True, primary_params, PRIMARY_SEEDS, 'rollout')
one_models, one_seed, one_hist = train_family('certified_one_step', True, one_step_params, ABLATION_SEEDS, 'one_step')
unc_models, unc_seed, unc_hist = train_family('unconstrained_rollout', False, primary_params, ABLATION_SEEDS, 'rollout')
seed_metrics = pd.concat([primary_seed, one_seed, unc_seed], ignore_index=True)
seed_metrics.to_csv(TAB / 'table08_seed_metrics.csv', index=False)


## Network Jacobian norms

In [ ]:
lip_rows = []
for name, models in [('certified_rollout', primary_models), ('certified_one_step', one_models), ('unconstrained_rollout', unc_models)]:
    for s, m in enumerate(models):
        bound, sigmas = architectural_lipschitz_bound(m)
        vals = empirical_residual_jacobian_norms(m, Xt, n_samples=250, seed=SEED + 500 + s)
        lip_rows.append({'configuration': name, 'seed': s, 'empirical_median': float(np.median(vals)), 'empirical_max': float(np.max(vals)), 'architectural_bound': bound if 'certified' in name else np.nan, 'layer_sigmas': ';'.join((f'{x:.6g}' for x in sigmas))})
        m.cpu()
lip_df = pd.DataFrame(lip_rows)
lip_df.to_csv(TAB / 'table09_lipschitz_audit.csv', index=False)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
groups = [lip_df[lip_df.configuration == c].empirical_max.to_numpy() for c in ['certified_rollout', 'certified_one_step', 'unconstrained_rollout']]
ax.boxplot(groups, tick_labels=['Rollout cert.', 'One-step cert.', 'Unconstrained'])
ax.set_ylabel('Maximum residual Jacobian norm')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')

def coordinate_lipschitz_metrics(model, fea, op_ids, base_all, n_all, ueq_all, xmu_np, xsd_np, n_samples=300, eps=0.001, seed=SEED):
    rng = np.random.default_rng(seed)
    ids = rng.choice(np.asarray(op_ids, dtype=int), size=min(int(n_samples), len(op_ids)), replace=False)
    x0n = rng.uniform(-0.45, 0.45, size=(len(ids), 3)).astype(np.float32)
    base = torch.tensor(base_all[ids], dtype=torch.float32, device=fea.device)
    n = torch.tensor(n_all[ids], dtype=torch.float32, device=fea.device)
    u = torch.tensor(ueq_all[ids], dtype=torch.float32, device=fea.device)
    xn = torch.tensor(x0n, dtype=torch.float32, device=fea.device)
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    xmu = torch.tensor(xmu_np, dtype=torch.float32, device=fea.device)
    xsd = torch.tensor(xsd_np, dtype=torch.float32, device=fea.device)
    Jz_cols = []
    Jc_cols = []
    model = model.to(fea.device).eval()
    with torch.no_grad():
        for d in range(3):
            e = torch.zeros_like(xn)
            e[:, d] = float(eps)

            def z_and_c(xnn):
                xx = base + xnn * scale
                z, _ = features_torch(fea, xx, base, n, u, xmu, xsd)
                c = model(z)
                return (z, c)
            zp, cp = z_and_c(xn + e)
            zm, cm = z_and_c(xn - e)
            Jz_cols.append(((zp - zm) / (2.0 * float(eps))).cpu().numpy())
            Jc_cols.append(((cp - cm) / (2.0 * float(eps))).cpu().numpy())
    Jz = np.stack(Jz_cols, axis=2)
    Jc = np.stack(Jc_cols, axis=2)
    nz = np.linalg.svd(Jz, compute_uv=False)[:, 0]
    nc = np.linalg.svd(Jc, compute_uv=False)[:, 0]
    bound, _ = architectural_lipschitz_bound(model)
    model.cpu()
    return {'network_input_architectural_bound': float(bound), 'feature_jacobian_median': float(np.median(nz)), 'feature_jacobian_max': float(np.max(nz)), 'chain_rule_local_bound_median': float(bound * np.median(nz)), 'chain_rule_local_bound_max': float(bound * np.max(nz)), 'state_to_residual_jacobian_median': float(np.median(nc)), 'state_to_residual_jacobian_max': float(np.max(nc)), 'samples': int(len(ids)), 'finite_difference_eps_normalized': float(eps)}
coord_lip_rows = []
for name, models in [('certified_rollout', primary_models), ('certified_one_step', one_models), ('unconstrained_rollout', unc_models)]:
    for si, m in enumerate(models):
        rec = coordinate_lipschitz_metrics(m, fea, test_ops, base_all, n_all, ueq_all, XMU, XSD, LIP_COORD_SAMPLES, LIP_COORD_EPS, SEED + 900 + si)
        rec.update({'configuration': name, 'seed': si})
        if 'unconstrained' in name:
            rec['network_input_architectural_bound'] = np.nan
            rec['chain_rule_local_bound_median'] = np.nan
            rec['chain_rule_local_bound_max'] = np.nan
        coord_lip_rows.append(rec)
coord_lip_df = pd.DataFrame(coord_lip_rows)
coord_lip_df.to_csv(TAB / 'table09b_coordinate_aware_lipschitz_audit.csv', index=False)


## Canonical trajectory comparison

In [ ]:
canonical_x0n = np.tile(np.array([20.0 / 30.0, -10.0 / 30.0, 0.2 / 0.35], np.float32), (len(test_ops), 1))
main_bank = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, test_ops, canonical_x0n, horizon=MAIN_HORIZON, ref_substeps=ROLLOUT_REF_SUBSTEPS)

def family_rollout(models, name, bank, max_models=None):
    rows = []
    use = models if max_models is None else models[:max_models]
    for s, m in enumerate(use):
        m = m.to(DEVICE)
        r = eval_bank(m, fea, bank, base_all, n_all, ueq_all, XMU, XSD)
        m.cpu()
        for k, (gi, v) in enumerate(zip(bank['gids'], r)):
            o = ops[int(gi)]
            rows.append({'configuration': name, 'seed': s, 'scenario_index': k, 'op_index': int(gi), 'n_rpm': o['n_rpm'], 'if_A': o['if_A'], 'id_A': o['id_A'], 'iq_A': o['iq_A'], 'rmse': float(v)})
    return pd.DataFrame(rows)
roll_primary = family_rollout(primary_models, 'certified_rollout', main_bank)
roll_one = family_rollout(one_models, 'certified_one_step', main_bank)
roll_unc = family_rollout(unc_models, 'unconstrained_rollout', main_bank)
classical = []
for method in ['euler', 'heun', 'rk4']:
    r = eval_classical_bank(fea, main_bank, base_all, n_all, ueq_all, method)
    for gi, v in zip(main_bank['gids'], r):
        o = ops[int(gi)]
        classical.append({'method': method, 'op_index': int(gi), 'n_rpm': o['n_rpm'], 'if_A': o['if_A'], 'rmse': float(v)})
classical = pd.DataFrame(classical)
roll_all = pd.concat([roll_primary, roll_one, roll_unc], ignore_index=True)
roll_all.to_csv(TAB / 'table09_all_canonical_rollouts.csv', index=False)
classical.to_csv(TAB / 'table10_nonlinear_classical_rollouts.csv', index=False)
seedavg = roll_all.groupby(['configuration', 'op_index', 'n_rpm', 'if_A']).rmse.mean().reset_index()
pivot = seedavg.pivot_table(index=['op_index', 'n_rpm', 'if_A'], columns='configuration', values='rmse').reset_index()
cp = classical.pivot_table(index=['op_index', 'n_rpm', 'if_A'], columns='method', values='rmse').reset_index()
main_eval = pivot.merge(cp, on=['op_index', 'n_rpm', 'if_A'])
main_eval.to_csv(TAB / 'table11_canonical_seed_averaged.csv', index=False)


## One electrical period comparison

In [ ]:
domain_flags = op_domain_df[['op_index', 'u_dq_eq_V', 'voltage_feasible_spwm', 'voltage_feasible_svpwm']].copy()
main_eval = main_eval.merge(domain_flags, on='op_index', how='left')
main_eval.to_csv(TAB / 'table11_canonical_seed_averaged.csv', index=False)
domain_roll_rows = []
for domain, mask in [('current_limited', np.ones(len(main_eval), dtype=bool)), ('two_level_spwm_linear', main_eval.voltage_feasible_spwm.to_numpy(bool)), ('two_level_svpwm_linear', main_eval.voltage_feasible_svpwm.to_numpy(bool))]:
    d = main_eval.loc[mask]
    if len(d) == 0:
        continue
    domain_roll_rows.append({'domain': domain, 'test_operating_points': len(d), 'median_euler_rmse': float(d.euler.median()), 'median_heun_rmse': float(d.heun.median()), 'median_rk4_rmse': float(d.rk4.median()), 'median_certified_rollout_rmse': float(d.certified_rollout.median()), 'corrected_better_than_euler_fraction': float(np.mean(d.certified_rollout < d.euler))})
domain_rollout_summary = pd.DataFrame(domain_roll_rows)
domain_rollout_summary.to_csv(TAB / 'table11b_voltage_domain_rollout_summary.csv', index=False)

def _in_fea_domain_torch(fea, x, tol=1e-07):
    return fea.in_domain(x, tol=tol)

def _ref_domain_exit_flags(fea, ref):
    lo = np.array([fea.id0, fea.iq0, fea.if0], float)
    hi = np.array([float(fea.idv[-1]), float(fea.iqv[-1]), float(fea.ifv[-1])], float)
    bad = np.any((ref < lo[None, None, :] - 1e-07) | (ref > hi[None, None, :] + 1e-07), axis=2)
    return np.any(bad, axis=1)

def _ref_min_boundary_margin_normalized(fea, ref):
    lo = np.array([fea.id0, fea.iq0, fea.if0], float)
    hi = np.array([float(fea.idv[-1]), float(fea.iqv[-1]), float(fea.ifv[-1])], float)
    d1 = (ref - lo[None, None, :]) / STATE_SCALE[None, None, :]
    d2 = (hi[None, None, :] - ref) / STATE_SCALE[None, None, :]
    return np.min(np.minimum(d1, d2), axis=(1, 2))

def eval_corrected_bank_with_domain(model, fea, bank, base_all, n_all, ueq_all, xmu_np, xsd_np, batch_size=2048, problem_cell_lookup=None):
    model.eval()
    rms = []
    exits = []
    firsts = []
    component_rms = []
    pcontacts = []
    pcounts = []
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    xmu = torch.tensor(xmu_np, device=fea.device)
    xsd = torch.tensor(xsd_np, device=fea.device)
    lookup = problem_cell_lookup.to(fea.device) if problem_cell_lookup is not None else None
    gi_all = bank['gids']
    for st in range(0, len(gi_all), batch_size):
        en = min(st + batch_size, len(gi_all))
        gi = gi_all[st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        x = base + torch.tensor(bank['x0n'][st:en], device=fea.device) * scale
        ref = torch.tensor(bank['ref'][st:en], device=fea.device)
        se = torch.zeros(len(gi), device=fea.device)
        se_phys = torch.zeros((len(gi), 3), device=fea.device)
        ex = torch.zeros(len(gi), dtype=torch.bool, device=fea.device)
        first = torch.full((len(gi),), -1, dtype=torch.int64, device=fea.device)
        pc = problem_cell_mask_torch(fea, x, n, lookup)
        pcount = pc.to(torch.int64)
        with torch.no_grad():
            for t in range(bank['horizon']):
                bad0 = ~_in_fea_domain_torch(fea, x)
                new = bad0 & ~ex
                first[new] = t
                ex |= bad0
                x = corrected_step(model, fea, x, base, n, u, xmu, xsd)
                bad1 = ~_in_fea_domain_torch(fea, x)
                new = bad1 & ~ex
                first[new] = t + 1
                ex |= bad1
                ps = problem_cell_mask_torch(fea, x, n, lookup)
                pc |= ps
                pcount += ps.to(torch.int64)
                dphys = x - ref[:, t + 1]
                d = dphys / scale
                se += torch.sum(d * d, dim=1)
                se_phys += dphys * dphys
        rms.append(torch.sqrt(se / (bank['horizon'] * 3)).cpu().numpy())
        component_rms.append(torch.sqrt(se_phys / bank['horizon']).cpu().numpy())
        exits.append(ex.cpu().numpy())
        firsts.append(first.cpu().numpy())
        pcontacts.append(pc.cpu().numpy())
        pcounts.append(pcount.cpu().numpy())
    return (np.concatenate(rms), np.concatenate(exits), np.concatenate(firsts), np.concatenate(component_rms), np.concatenate(pcontacts), np.concatenate(pcounts))

def eval_classical_bank_with_domain(fea, bank, base_all, n_all, ueq_all, method='euler', batch_size=2048, problem_cell_lookup=None):
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    out = []
    exits = []
    firsts = []
    component_rms = []
    pcontacts = []
    pcounts = []
    gi_all = bank['gids']
    lookup = problem_cell_lookup.to(fea.device) if problem_cell_lookup is not None else None
    for st in range(0, len(gi_all), batch_size):
        en = min(st + batch_size, len(gi_all))
        gi = gi_all[st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        x = base + torch.tensor(bank['x0n'][st:en], device=fea.device) * scale
        ref = torch.tensor(bank['ref'][st:en], device=fea.device)
        se = torch.zeros(len(gi), device=fea.device)
        se_phys = torch.zeros((len(gi), 3), device=fea.device)
        ex = torch.zeros(len(gi), dtype=torch.bool, device=fea.device)
        first = torch.full((len(gi),), -1, dtype=torch.int64, device=fea.device)
        pc = problem_cell_mask_torch(fea, x, n, lookup)
        pcount = pc.to(torch.int64)
        with torch.no_grad():
            for t in range(bank['horizon']):
                bad = ~_in_fea_domain_torch(fea, x)
                new = bad & ~ex
                first[new] = t
                ex |= bad
                if method == 'euler':
                    k1 = fea.rhs(x, n, u)
                    xn = x + DT * k1
                    badstage = ~_in_fea_domain_torch(fea, xn)
                    new = badstage & ~ex
                    first[new] = t + 1
                    ex |= badstage
                    x = xn
                elif method == 'heun':
                    k1 = fea.rhs(x, n, u)
                    xp = x + DT * k1
                    badstage = ~_in_fea_domain_torch(fea, xp)
                    new = badstage & ~ex
                    first[new] = t + 1
                    ex |= badstage
                    k2 = fea.rhs(xp, n, u)
                    x = x + 0.5 * DT * (k1 + k2)
                elif method == 'rk4':
                    k1 = fea.rhs(x, n, u)
                    x2 = x + 0.5 * DT * k1
                    badstage = ~_in_fea_domain_torch(fea, x2)
                    new = badstage & ~ex
                    first[new] = t + 1
                    ex |= badstage
                    k2 = fea.rhs(x2, n, u)
                    x3 = x + 0.5 * DT * k2
                    badstage = ~_in_fea_domain_torch(fea, x3)
                    new = badstage & ~ex
                    first[new] = t + 1
                    ex |= badstage
                    k3 = fea.rhs(x3, n, u)
                    x4 = x + DT * k3
                    badstage = ~_in_fea_domain_torch(fea, x4)
                    new = badstage & ~ex
                    first[new] = t + 1
                    ex |= badstage
                    k4 = fea.rhs(x4, n, u)
                    x = x + DT / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
                else:
                    raise ValueError(method)
                bad1 = ~_in_fea_domain_torch(fea, x)
                new = bad1 & ~ex
                first[new] = t + 1
                ex |= bad1
                ps = problem_cell_mask_torch(fea, x, n, lookup)
                pc |= ps
                pcount += ps.to(torch.int64)
                dphys = x - ref[:, t + 1]
                d = dphys / scale
                se += torch.sum(d * d, dim=1)
                se_phys += dphys * dphys
        out.append(torch.sqrt(se / (bank['horizon'] * 3)).cpu().numpy())
        component_rms.append(torch.sqrt(se_phys / bank['horizon']).cpu().numpy())
        exits.append(ex.cpu().numpy())
        firsts.append(first.cpu().numpy())
        pcontacts.append(pc.cpu().numpy())
        pcounts.append(pcount.cpu().numpy())
    return (np.concatenate(out), np.concatenate(exits), np.concatenate(firsts), np.concatenate(component_rms), np.concatenate(pcontacts), np.concatenate(pcounts))
long_rows = []
long_ref_rows = []
problem_lookup = UNSTABLE_EXACT_CELL_LOOKUP_TORCH if RUN_ROLLOUT_CELL_CONTACT_AUDIT and UNSTABLE_EXACT_CELL_LOOKUP_TORCH is not None else None
for spd in sorted(np.unique(n_all[test_ops])):
    ids = np.array([int(gi) for gi in test_ops if float(ops[int(gi)]['n_rpm']) == float(spd)], dtype=int)
    if len(ids) == 0:
        continue
    Te = 60.0 / (D['p'] * float(spd))
    horizon = max(1, int(np.ceil(LONG_PERIODS * Te / DT)))
    x0 = np.tile(np.array([20.0 / 30.0, -10.0 / 30.0, 0.2 / 0.35], np.float32), (len(ids), 1))
    bank = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, ids, x0, horizon=horizon, ref_substeps=ROLLOUT_REF_SUBSTEPS, problem_cell_lookup=problem_lookup)
    ref_exit = bank['ref_domain_contact']
    ref_margin = _ref_min_boundary_margin_normalized(fea, bank['ref'])
    for gi, ex, fs, mg, pc, pcnt in zip(ids, ref_exit, bank['ref_first_domain_contact_step'], ref_margin, bank['ref_nonhurwitz_cell_center_contact'], bank['ref_nonhurwitz_cell_center_endpoint_count']):
        long_ref_rows.append({'op_index': int(gi), 'n_rpm': float(spd), 'horizon_steps': horizon, 'electrical_period_ms': Te * 1000.0, 'reference_domain_contact': bool(ex), 'first_reference_domain_contact_step': int(fs), 'reference_min_boundary_margin_normalized': float(mg), 'nonhurwitz_cell_center_contact': bool(pc), 'nonhurwitz_cell_center_endpoint_count': int(pcnt)})
    for si, m in enumerate(primary_models):
        m = m.to(DEVICE)
        rr, ex, first, comp, pcon, pcnt = eval_corrected_bank_with_domain(m, fea, bank, base_all, n_all, ueq_all, XMU, XSD, problem_cell_lookup=problem_lookup)
        m.cpu()
        for gi, v, e0, f0, cr, pc, pn in zip(ids, rr, ex, first, comp, pcon, pcnt):
            o = ops[int(gi)]
            long_rows.append({'configuration': 'certified_rollout', 'seed': si, 'op_index': int(gi), 'n_rpm': o['n_rpm'], 'if_A': o['if_A'], 'id_A': o['id_A'], 'iq_A': o['iq_A'], 'horizon_steps': horizon, 'electrical_period_ms': Te * 1000.0, 'rmse': float(v), 'rmse_id_A': float(cr[0]), 'rmse_iq_A': float(cr[1]), 'rmse_if_A': float(cr[2]), 'domain_contact': bool(e0), 'first_domain_contact_step': int(f0), 'nonhurwitz_cell_center_contact': bool(pc), 'nonhurwitz_cell_center_endpoint_count': int(pn)})
    for method in ['euler', 'heun', 'rk4']:
        rr, ex, first, comp, pcon, pcnt = eval_classical_bank_with_domain(fea, bank, base_all, n_all, ueq_all, method, problem_cell_lookup=problem_lookup)
        for gi, v, e0, f0, cr, pc, pn in zip(ids, rr, ex, first, comp, pcon, pcnt):
            o = ops[int(gi)]
            long_rows.append({'configuration': method, 'seed': -1, 'op_index': int(gi), 'n_rpm': o['n_rpm'], 'if_A': o['if_A'], 'id_A': o['id_A'], 'iq_A': o['iq_A'], 'horizon_steps': horizon, 'electrical_period_ms': Te * 1000.0, 'rmse': float(v), 'rmse_id_A': float(cr[0]), 'rmse_iq_A': float(cr[1]), 'rmse_if_A': float(cr[2]), 'domain_contact': bool(e0), 'first_domain_contact_step': int(f0), 'nonhurwitz_cell_center_contact': bool(pc), 'nonhurwitz_cell_center_endpoint_count': int(pn)})
long_df = pd.DataFrame(long_rows)
long_ref_df = pd.DataFrame(long_ref_rows)
long_df.to_csv(TAB / 'table11c_one_electrical_period_rollouts.csv', index=False)
long_ref_df.to_csv(TAB / 'table11d_one_electrical_period_reference_domain_audit.csv', index=False)
learn = long_df[long_df.configuration == 'certified_rollout'].groupby(['op_index', 'n_rpm', 'if_A']).agg(rmse=('rmse', 'mean'), domain_contact=('domain_contact', 'mean')).reset_index()
learn['configuration'] = 'certified_rollout'
classic = long_df[long_df.configuration != 'certified_rollout'].copy()
classic['domain_contact'] = classic.domain_contact.astype(float)
long_agg = pd.concat([learn, classic[['op_index', 'n_rpm', 'if_A', 'rmse', 'domain_contact', 'configuration']]], ignore_index=True)
summary_rows = []
for cfg, gx in long_agg.groupby('configuration'):
    strict = gx[gx.domain_contact == 0]
    summary_rows.append({'configuration': cfg, 'operating_points': len(gx), 'median_rmse_all': float(gx.rmse.median()), 'domain_contact_fraction': float(np.mean(gx.domain_contact > 0)), 'strict_domain_points': len(strict), 'median_rmse_strict_domain': float(strict.rmse.median()) if len(strict) else np.nan})
for (cfg, spd), gx in long_agg.groupby(['configuration', 'n_rpm']):
    strict = gx[gx.domain_contact == 0]
    summary_rows.append({'configuration': cfg, 'n_rpm': float(spd), 'operating_points': len(gx), 'median_rmse_all': float(gx.rmse.median()), 'domain_contact_fraction': float(np.mean(gx.domain_contact > 0)), 'strict_domain_points': len(strict), 'median_rmse_strict_domain': float(strict.rmse.median()) if len(strict) else np.nan})
long_summary = pd.DataFrame(summary_rows)
long_summary.to_csv(TAB / 'table11e_one_electrical_period_summary.csv', index=False)
fig, ax = plt.subplots(figsize=(6.0, 3.8))
order = ['euler', 'certified_rollout', 'heun', 'rk4']
vals = [long_agg[long_agg.configuration == c].rmse.to_numpy() for c in order]
ax.boxplot(vals, tick_labels=['Euler', 'Certified', 'Heun', 'RK4'], showfliers=False)
ax.set_yscale('log')
ax.set_ylabel('One-electrical-period normalized RMSE')
ax.grid(axis='y', which='both', alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Common subset and bootstrap

In [ ]:
def two_way_seed_cluster_bootstrap(seed_level_df, value_col, seed_col='seed', cluster_cols=('n_rpm', 'if_A', 'spatial_block'), B=50000, seed=SEED, batch_size=512):
    d = seed_level_df.dropna(subset=[value_col]).copy()
    seeds = np.array(sorted(d[seed_col].unique()))
    d['cluster_key'] = list(zip(*[d[c] for c in cluster_cols]))
    clusters = list(dict.fromkeys(d.cluster_key.tolist()))
    mat = np.full((len(seeds), len(clusters)), np.nan, float)
    for i, sd in enumerate(seeds):
        med = d[d[seed_col] == sd].groupby('cluster_key')[value_col].median()
        for j, key in enumerate(clusters):
            if key in med.index:
                mat[i, j] = float(med.at[key])
    point = float(np.nanmedian(mat))
    rng = np.random.default_rng(seed)
    boots = np.empty(int(B), float)
    pos = 0
    while pos < int(B):
        m = min(int(batch_size), int(B) - pos)
        ri = rng.integers(0, len(seeds), size=(m, len(seeds)))
        ci = rng.integers(0, len(clusters), size=(m, len(clusters)))
        sampled = mat[ri[:, :, None], ci[:, None, :]]
        boots[pos:pos + m] = np.nanmedian(sampled, axis=(1, 2))
        pos += m
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return {'training_seeds': int(len(seeds)), 'spatial_speed_field_clusters': int(len(clusters)), 'median_difference': point, 'ci_low': float(lo), 'ci_high': float(hi), 'bootstrap_replicates': int(B), 'bootstrap_rng_seed': int(seed), 'estimator': 'median of seed-by-cluster medians', 'resampling_design': 'two-way crossed resampling of training seeds and spatial speed-field clusters', 'conditioning': 'observed paired subset supplied to the bootstrap'}

def crossed_bootstrap_rng_sensitivity(seed_level_df, value_col, base_seed, repeats, B, seed_col='seed', cluster_cols=('n_rpm', 'if_A', 'spatial_block')):
    rows = []
    for r in range(int(repeats)):
        bs = int(base_seed + 10000 * r)
        rec = two_way_seed_cluster_bootstrap(seed_level_df, value_col, seed_col=seed_col, cluster_cols=cluster_cols, B=B, seed=bs)
        rec['rng_repeat'] = int(r + 1)
        rows.append(rec)
    return pd.DataFrame(rows)
ref_status = long_ref_df[['op_index', 'reference_domain_contact', 'reference_min_boundary_margin_normalized']].drop_duplicates('op_index')
classical_long = long_df[long_df.configuration.isin(['euler', 'heun', 'rk4'])]
class_contact = classical_long.pivot(index='op_index', columns='configuration', values='domain_contact').reset_index()
class_contact = class_contact.rename(columns={c: f'{c}_domain_contact' for c in ['euler', 'heun', 'rk4']})
learn_contact = long_df[long_df.configuration == 'certified_rollout'].groupby('op_index').domain_contact.max().reset_index(name='certified_any_seed_domain_contact')
common_flags = ref_status.merge(class_contact, on='op_index', how='inner').merge(learn_contact, on='op_index', how='inner')
common_flags['common_strict_domain'] = ~(common_flags.reference_domain_contact.astype(bool) | common_flags.euler_domain_contact.astype(bool) | common_flags.heun_domain_contact.astype(bool) | common_flags.rk4_domain_contact.astype(bool) | common_flags.certified_any_seed_domain_contact.astype(bool))
common_ids = common_flags.loc[common_flags.common_strict_domain, 'op_index'].to_numpy(int)
classic_rmse = classical_long.pivot(index=['op_index', 'n_rpm', 'if_A'], columns='configuration', values='rmse').reset_index()
learn_seed = long_df[long_df.configuration == 'certified_rollout'][['seed', 'op_index', 'n_rpm', 'if_A', 'id_A', 'iq_A', 'rmse', 'domain_contact']].copy()
learn_seed = learn_seed[learn_seed.op_index.isin(common_ids)].copy()
common_seed = learn_seed.merge(classic_rmse, on=['op_index', 'n_rpm', 'if_A'], how='left')
common_seed = common_seed.rename(columns={'rmse': 'corrected_rmse'})
common_seed['spatial_block'] = [OP_BLOCK.get(int(gi), -1) for gi in common_seed.op_index]
common_seed['diff_corrected_minus_euler'] = common_seed.corrected_rmse - common_seed.euler
common_seed['diff_corrected_minus_rk4'] = common_seed.corrected_rmse - common_seed.rk4
common_seed.to_csv(TAB / 'table11f_common_strict_seed_level.csv', index=False)
common_avg = common_seed.groupby(['op_index', 'n_rpm', 'if_A']).agg(corrected_rmse=('corrected_rmse', 'mean'), euler_rmse=('euler', 'first'), heun_rmse=('heun', 'first'), rk4_rmse=('rk4', 'first')).reset_index()
common_rows = []
for label, g in [('Overall', common_avg)]:
    common_rows.append({'speed': label, 'points': int(len(g)), 'median_corrected_rmse': float(g.corrected_rmse.median()), 'median_euler_rmse': float(g.euler_rmse.median()), 'median_heun_rmse': float(g.heun_rmse.median()), 'median_rk4_rmse': float(g.rk4_rmse.median()), 'corrected_better_than_euler_fraction': float(np.mean(g.corrected_rmse < g.euler_rmse))})
for speed, g in common_avg.groupby('n_rpm'):
    common_rows.append({'speed': int(speed), 'points': int(len(g)), 'median_corrected_rmse': float(g.corrected_rmse.median()), 'median_euler_rmse': float(g.euler_rmse.median()), 'median_heun_rmse': float(g.heun_rmse.median()), 'median_rk4_rmse': float(g.rk4_rmse.median()), 'corrected_better_than_euler_fraction': float(np.mean(g.corrected_rmse < g.euler_rmse))})
common_summary = pd.DataFrame(common_rows)
common_summary.to_csv(TAB / 'table11g_common_strict_summary.csv', index=False)
if RUN_ROLLOUT_CELL_CONTACT_AUDIT and 'nonhurwitz_cell_center_contact' in long_df.columns:
    contact_rows = []
    ref_all = long_ref_df.copy()
    for scope, ids_scope in [('all_test', set(long_ref_df.op_index.astype(int))), ('common_strict', set(map(int, common_ids)))]:
        rr = ref_all[ref_all.op_index.astype(int).isin(ids_scope)]
        contact_rows.append({'configuration': 'high_resolution_reference', 'scope': scope, 'trajectories': int(len(rr)), 'contact_fraction': float(rr.nonhurwitz_cell_center_contact.mean()) if len(rr) else np.nan, 'median_contacted_endpoints': float(rr.nonhurwitz_cell_center_endpoint_count.median()) if len(rr) else np.nan})
        for cfg, z0 in long_df.groupby('configuration'):
            z = z0[z0.op_index.astype(int).isin(ids_scope)].copy()
            if cfg == 'certified_rollout':
                z = z.groupby('op_index').agg(nonhurwitz_cell_center_contact=('nonhurwitz_cell_center_contact', 'max'), nonhurwitz_cell_center_endpoint_count=('nonhurwitz_cell_center_endpoint_count', 'median')).reset_index()
            contact_rows.append({'configuration': cfg, 'scope': scope, 'trajectories': int(len(z)), 'contact_fraction': float(z.nonhurwitz_cell_center_contact.mean()) if len(z) else np.nan, 'median_contacted_endpoints': float(z.nonhurwitz_cell_center_endpoint_count.median()) if len(z) else np.nan})
    rollout_representation_contact_summary = pd.DataFrame(contact_rows)
    rollout_representation_contact_summary.to_csv(TAB / 'table11n_rollout_nonhurwitz_cell_contact_summary.csv', index=False)
phys_rows = []
for cfg in ['euler', 'certified_rollout', 'heun', 'rk4']:
    g = long_df[(long_df.configuration == cfg) & long_df.op_index.isin(common_ids)].copy()
    if cfg == 'certified_rollout':
        g = g.groupby(['op_index', 'n_rpm']).agg(rmse_id_A=('rmse_id_A', 'mean'), rmse_iq_A=('rmse_iq_A', 'mean'), rmse_if_A=('rmse_if_A', 'mean')).reset_index()
    for label, z in [('Overall', g)] + [(f'{int(spd)} rpm', q) for spd, q in g.groupby('n_rpm')]:
        phys_rows.append({'configuration': cfg, 'scope': label, 'points': int(len(z)), 'median_rmse_id_A': float(z.rmse_id_A.median()), 'median_rmse_iq_A': float(z.rmse_iq_A.median()), 'median_rmse_if_A': float(z.rmse_if_A.median())})
physical_component_summary = pd.DataFrame(phys_rows)
physical_component_summary.to_csv(TAB / 'table11m_common_strict_physical_component_rmse.csv', index=False)
common_voltage = common_avg.merge(op_domain_df[['op_index', 'voltage_feasible_spwm', 'voltage_feasible_svpwm']], on='op_index', how='left')
voltage_rows = []
for name, mask in [('two_level_spwm_linear', common_voltage.voltage_feasible_spwm.astype(bool)), ('two_level_svpwm_linear', common_voltage.voltage_feasible_svpwm.astype(bool))]:
    g = common_voltage.loc[mask]
    voltage_rows.append({'domain': name, 'points': int(len(g)), 'median_euler_rmse': float(g.euler_rmse.median()) if len(g) else np.nan, 'median_corrected_rmse': float(g.corrected_rmse.median()) if len(g) else np.nan, 'median_heun_rmse': float(g.heun_rmse.median()) if len(g) else np.nan, 'median_rk4_rmse': float(g.rk4_rmse.median()) if len(g) else np.nan, 'corrected_better_than_euler_fraction': float(np.mean(g.corrected_rmse < g.euler_rmse)) if len(g) else np.nan})
common_voltage_summary = pd.DataFrame(voltage_rows)
common_voltage_summary.to_csv(TAB / 'table11h_common_strict_voltage_summary.csv', index=False)
selection = common_flags.merge(op_domain_df[['op_index', 'n_rpm', 'if_A', 'id_A', 'iq_A', 'dtcrit_s', 'u_dq_eq_V', 'voltage_feasible_spwm', 'voltage_feasible_svpwm']], on='op_index', how='left').merge(classic_rmse[['op_index', 'euler']].rename(columns={'euler': 'euler_one_period_rmse'}), on='op_index', how='left')
selection.to_csv(TAB / 'table11i_common_strict_selection_flags.csv', index=False)
selection_rows = []
for retained, g in selection.groupby('common_strict_domain'):
    selection_rows.append({'subset': 'retained' if retained else 'excluded', 'points': int(len(g)), 'hcrit_median_us': float(g.dtcrit_s.median() * 1000000.0), 'u_dq_eq_median_V': float(g.u_dq_eq_V.median()), 'euler_one_period_rmse_median': float(g.euler_one_period_rmse.median()), 'spwm_feasible_fraction': float(g.voltage_feasible_spwm.mean()), 'svpwm_feasible_fraction': float(g.voltage_feasible_svpwm.mean()), 'reference_contact_fraction': float(g.reference_domain_contact.mean()), 'euler_contact_fraction': float(g.euler_domain_contact.mean()), 'heun_contact_fraction': float(g.heun_domain_contact.mean()), 'rk4_contact_fraction': float(g.rk4_domain_contact.mean()), 'certified_any_seed_contact_fraction': float(g.certified_any_seed_domain_contact.mean())})
    for speed, z in g.groupby('n_rpm'):
        selection_rows.append({'subset': ('retained' if retained else 'excluded') + f'_{int(speed)}rpm', 'points': int(len(z)), 'hcrit_median_us': float(z.dtcrit_s.median() * 1000000.0), 'u_dq_eq_median_V': float(z.u_dq_eq_V.median()), 'euler_one_period_rmse_median': float(z.euler_one_period_rmse.median()), 'spwm_feasible_fraction': float(z.voltage_feasible_spwm.mean()), 'svpwm_feasible_fraction': float(z.voltage_feasible_svpwm.mean()), 'reference_contact_fraction': float(z.reference_domain_contact.mean()), 'euler_contact_fraction': float(z.euler_domain_contact.mean()), 'heun_contact_fraction': float(z.heun_domain_contact.mean()), 'rk4_contact_fraction': float(z.rk4_domain_contact.mean()), 'certified_any_seed_contact_fraction': float(z.certified_any_seed_domain_contact.mean())})
selection_summary = pd.DataFrame(selection_rows)
selection_summary.to_csv(TAB / 'table11j_common_strict_selection_summary.csv', index=False)
period_crossed_rows = []
for col, label in [('diff_corrected_minus_euler', 'certified rollout - Euler'), ('diff_corrected_minus_rk4', 'certified rollout - RK4')]:
    rec = two_way_seed_cluster_bootstrap(common_seed, col, B=CROSSED_BOOTSTRAP_B, seed=SEED + 181)
    rec['comparison'] = label
    period_crossed_rows.append(rec)
one_period_crossed_df = pd.DataFrame(period_crossed_rows)
one_period_crossed_df.to_csv(TAB / 'table19c_one_period_two_way_seed_spatial_block_bootstrap.csv', index=False)
rng_frames = []
for col, label in [('diff_corrected_minus_euler', 'certified rollout - Euler'), ('diff_corrected_minus_rk4', 'certified rollout - RK4')]:
    z = crossed_bootstrap_rng_sensitivity(common_seed, col, base_seed=SEED + 181, repeats=BOOTSTRAP_RNG_REPEATS, B=CROSSED_BOOTSTRAP_B)
    z['comparison'] = label
    rng_frames.append(z)
one_period_bootstrap_rng_sensitivity = pd.concat(rng_frames, ignore_index=True)
one_period_bootstrap_rng_sensitivity.to_csv(TAB / 'table19d_one_period_two_way_bootstrap_rng_sensitivity.csv', index=False)
if RUN_LEAVE_ONE_SEED_OUT_BOOTSTRAP:
    loso_rows = []
    for col, label in [('diff_corrected_minus_euler', 'certified rollout - Euler'), ('diff_corrected_minus_rk4', 'certified rollout - RK4')]:
        for omitted in sorted(common_seed.seed.unique()):
            z = common_seed[common_seed.seed != omitted]
            rec = two_way_seed_cluster_bootstrap(z, col, B=CROSSED_BOOTSTRAP_B, seed=SEED + 281 + int(omitted))
            rec['comparison'] = label
            rec['omitted_training_seed'] = int(omitted)
            loso_rows.append(rec)
    one_period_leave_one_seed_out = pd.DataFrame(loso_rows)
    one_period_leave_one_seed_out.to_csv(TAB / 'table19f_one_period_leave_one_seed_out_bootstrap.csv', index=False)


## Independent DOP853 reference

In [ ]:
def interp_numpy_single(V, x, D):
    idv, iqv, ifv = (D['id_vec'], D['iq_vec'], D['if_vec'])
    did = float(idv[1] - idv[0])
    diq = float(iqv[1] - iqv[0])
    dif = float(ifv[1] - ifv[0])
    uid = np.clip((x[0] - idv[0]) / did, 0.0, len(idv) - 1 - 1e-12)
    uiq = np.clip((x[1] - iqv[0]) / diq, 0.0, len(iqv) - 1 - 1e-12)
    uif = np.clip((x[2] - ifv[0]) / dif, 0.0, len(ifv) - 1 - 1e-12)
    i0, q0, f0 = (int(np.floor(uid)), int(np.floor(uiq)), int(np.floor(uif)))
    i1, q1, f1 = (i0 + 1, q0 + 1, f0 + 1)
    wi, wq, wf = (uid - i0, uiq - q0, uif - f0)
    c000 = V[q0, i0, f0]
    c001 = V[q0, i0, f1]
    c010 = V[q0, i1, f0]
    c011 = V[q0, i1, f1]
    c100 = V[q1, i0, f0]
    c101 = V[q1, i0, f1]
    c110 = V[q1, i1, f0]
    c111 = V[q1, i1, f1]
    c00 = c000 * (1 - wi) + c010 * wi
    c01 = c001 * (1 - wi) + c011 * wi
    c10 = c100 * (1 - wi) + c110 * wi
    c11 = c101 * (1 - wi) + c111 * wi
    c0 = c00 * (1 - wq) + c10 * wq
    c1 = c01 * (1 - wq) + c11 * wq
    return c0 * (1 - wf) + c1 * wf

def rhs_numpy_single(x, n_rpm, ueq, D, Lgrid):
    flux_grid = np.stack([D['psid'], D['psiq'], D['psif']], axis=-1)
    psi = interp_numpy_single(flux_grid, x, D)
    L = interp_numpy_single(Lgrid, x, D)
    we = 2.0 * np.pi * D['p'] * float(n_rpm) / 60.0
    b = np.array([ueq[0] - D['Rs'] * x[0] + we * psi[1], ueq[1] - D['Rs'] * x[1] - we * psi[0], ueq[2] - D['Rf'] * x[2]], dtype=float)
    return np.linalg.solve(L, b)

def rk4_numpy_trajectory(x0, n_rpm, ueq, D, Lgrid, horizon, macro_h, substeps):
    y = np.asarray(x0, dtype=float).copy()
    out = [y.copy()]
    hh = float(macro_h) / int(substeps)
    for _ in range(int(horizon)):
        for _ in range(int(substeps)):
            k1 = rhs_numpy_single(y, n_rpm, ueq, D, Lgrid)
            k2 = rhs_numpy_single(y + 0.5 * hh * k1, n_rpm, ueq, D, Lgrid)
            k3 = rhs_numpy_single(y + 0.5 * hh * k2, n_rpm, ueq, D, Lgrid)
            k4 = rhs_numpy_single(y + hh * k3, n_rpm, ueq, D, Lgrid)
            y = y + hh / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
        out.append(y.copy())
    return np.asarray(out)

def in_numpy_domain(x, D, tol=1e-07):
    lo = np.array([D['id_vec'][0], D['iq_vec'][0], D['if_vec'][0]], float)
    hi = np.array([D['id_vec'][-1], D['iq_vec'][-1], D['if_vec'][-1]], float)
    return bool(np.all(x >= lo - tol) and np.all(x <= hi + tol))
independent_reference_df = pd.DataFrame()
independent_reference_summary = pd.DataFrame()
if RUN_INDEPENDENT_REFERENCE_AUDIT and len(common_ids):
    margin_table = ref_status[ref_status.op_index.isin(common_ids) & (ref_status.reference_min_boundary_margin_normalized >= DOP853_MIN_MARGIN_NORMALIZED)].copy()
    if len(margin_table) == 0:
        margin_table = ref_status[ref_status.op_index.isin(common_ids)].copy()
    margin_table['margin_bin'] = pd.qcut(margin_table.reference_min_boundary_margin_normalized, q=min(5, len(margin_table)), duplicates='drop')
    rng = np.random.default_rng(SEED + 182)
    ordered = []
    for _, z in margin_table.groupby('margin_bin', observed=True, sort=False):
        a = z.op_index.to_numpy(int).copy()
        rng.shuffle(a)
        ordered.extend(a.tolist())
    ordered = list(dict.fromkeys(ordered))
    remaining = [int(x) for x in margin_table.sort_values('reference_min_boundary_margin_normalized', ascending=False).op_index if int(x) not in set(ordered)]
    candidate_ids = np.asarray((ordered + remaining)[:min(REFERENCE_AUDIT_MAX_ATTEMPTS, len(margin_table))], dtype=int)
    audit_rows = []
    valid_count = 0
    x0_offset = np.array([20.0, -10.0, 0.2], dtype=float)
    for gi in candidate_ids:
        gi = int(gi)
        n_rpm = float(n_all[gi])
        Te = 60.0 / (D['p'] * n_rpm)
        horizon = max(1, int(np.ceil(LONG_PERIODS * Te / DT)))
        t_eval = np.arange(horizon + 1, dtype=float) * DT
        x0 = np.asarray(base_all[gi], dtype=float) + x0_offset
        ueq = np.asarray(ueq_all[gi], dtype=float)
        contact = [False]

        def fun(t, y):
            if not in_numpy_domain(y, D):
                contact[0] = True
            return rhs_numpy_single(y, n_rpm, ueq, D, L_edge2)
        sol = solve_ivp(fun, (0.0, float(t_eval[-1])), x0, method='DOP853', t_eval=t_eval, rtol=DOP853_RTOL, atol=DOP853_ATOL)
        if not sol.success or sol.y.shape[1] != len(t_eval):
            audit_rows.append({'op_index': gi, 'n_rpm': n_rpm, 'solver_success': False, 'dop853_domain_contact': bool(contact[0]), 'message': str(sol.message)})
            continue
        dop = sol.y.T
        rk10 = rk4_numpy_trajectory(x0, n_rpm, ueq, D, L_edge2, horizon, DT, 10)
        rk20 = rk4_numpy_trajectory(x0, n_rpm, ueq, D, L_edge2, horizon, DT, 20)
        d10 = (rk10 - dop) / STATE_SCALE[None, :]
        d20 = (rk20 - dop) / STATE_SCALE[None, :]
        rmse10 = float(np.sqrt(np.mean(d10 * d10)))
        rmse20 = float(np.sqrt(np.mean(d20 * d20)))
        max10 = float(np.max(np.linalg.norm(d10, axis=1)))
        max20 = float(np.max(np.linalg.norm(d20, axis=1)))
        is_valid = bool(not contact[0])
        audit_rows.append({'op_index': gi, 'n_rpm': n_rpm, 'if_A': float(ops[gi]['if_A']), 'id_A': float(ops[gi]['id_A']), 'iq_A': float(ops[gi]['iq_A']), 'horizon_steps': int(horizon), 'electrical_period_ms': float(Te * 1000.0), 'solver_success': True, 'dop853_domain_contact': bool(contact[0]), 'rk4_10sub_normalized_rmse_vs_dop853': rmse10, 'rk4_20sub_normalized_rmse_vs_dop853': rmse20, 'rk4_10sub_max_normalized_state_error_vs_dop853': max10, 'rk4_20sub_max_normalized_state_error_vs_dop853': max20, 'message': str(sol.message)})
        if is_valid:
            valid_count += 1
            if valid_count >= int(REFERENCE_AUDIT_TARGET_VALID):
                break
    independent_reference_df = pd.DataFrame(audit_rows)
    independent_reference_df.to_csv(TAB / 'table11k_independent_dop853_reference_audit_all.csv', index=False)
    ok = independent_reference_df[independent_reference_df.solver_success.fillna(False) & ~independent_reference_df.dop853_domain_contact.fillna(True)].copy()
    independent_reference_summary = pd.DataFrame([{'target_domain_valid_points': int(REFERENCE_AUDIT_TARGET_VALID), 'attempted_points': int(len(independent_reference_df)), 'eligible_margin_filtered_points': int(len(margin_table)), 'maximum_attempts': int(REFERENCE_AUDIT_MAX_ATTEMPTS), 'minimum_reference_boundary_margin_normalized': float(DOP853_MIN_MARGIN_NORMALIZED), 'successful_domain_valid_points': int(len(ok)), 'target_reached': bool(len(ok) >= int(REFERENCE_AUDIT_TARGET_VALID)), 'dop853_rtol': float(DOP853_RTOL), 'rk4_10sub_median_normalized_rmse_vs_dop853': float(ok.rk4_10sub_normalized_rmse_vs_dop853.median()) if len(ok) else np.nan, 'rk4_10sub_p95_normalized_rmse_vs_dop853': float(ok.rk4_10sub_normalized_rmse_vs_dop853.quantile(0.95)) if len(ok) else np.nan, 'rk4_10sub_max_normalized_rmse_vs_dop853': float(ok.rk4_10sub_normalized_rmse_vs_dop853.max()) if len(ok) else np.nan, 'rk4_20sub_median_normalized_rmse_vs_dop853': float(ok.rk4_20sub_normalized_rmse_vs_dop853.median()) if len(ok) else np.nan, 'rk4_20sub_p95_normalized_rmse_vs_dop853': float(ok.rk4_20sub_normalized_rmse_vs_dop853.quantile(0.95)) if len(ok) else np.nan, 'rk4_20sub_max_normalized_rmse_vs_dop853': float(ok.rk4_20sub_normalized_rmse_vs_dop853.max()) if len(ok) else np.nan}])
    independent_reference_summary.to_csv(TAB / 'table11l_independent_dop853_reference_audit_summary.csv', index=False)


## Multiple perturbation check

In [ ]:
rob_gids, rob_x0n, rob_labels, rob_amps = scenario_bank_ids(ops, test_ops, amplitudes=(0.25, 0.75), random_dirs=2, seed=SEED + 60)
rob_bank = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, rob_gids, rob_x0n, horizon=ROBUST_HORIZON, ref_substeps=ROLLOUT_REF_SUBSTEPS)
robust_frames = []
for models, name in [(primary_models, 'certified_rollout'), (one_models, 'certified_one_step'), (unc_models, 'unconstrained_rollout')]:
    d = family_rollout(models, name, rob_bank, max_models=min(ROBUST_MODEL_SEEDS, len(models)))
    d['direction'] = np.tile(rob_labels, min(ROBUST_MODEL_SEEDS, len(models)))
    d['amplitude'] = np.tile(rob_amps, min(ROBUST_MODEL_SEEDS, len(models)))
    robust_frames.append(d)
robust = pd.concat(robust_frames, ignore_index=True)
rob_euler = eval_classical_bank(fea, rob_bank, base_all, n_all, ueq_all, 'euler')
rob_euler_df = pd.DataFrame({'scenario_index': np.arange(len(rob_gids)), 'op_index': rob_gids, 'direction': rob_labels, 'amplitude': rob_amps, 'euler_rmse': rob_euler})
rob_euler_df.to_csv(TAB / 'table13_multi_perturbation_euler.csv', index=False)
robust.to_csv(TAB / 'table12_multi_perturbation_rollouts.csv', index=False)
rob_avg = robust.groupby(['configuration', 'op_index', 'n_rpm', 'if_A', 'direction', 'amplitude']).rmse.mean().reset_index()
rob_op = rob_avg.groupby(['configuration', 'op_index', 'n_rpm', 'if_A']).rmse.median().reset_index()
rob_piv = rob_op.pivot_table(index=['op_index', 'n_rpm', 'if_A'], columns='configuration', values='rmse').reset_index()
rob_piv['diff_primary_minus_onestep'] = rob_piv.certified_rollout - rob_piv.certified_one_step
rob_piv['diff_primary_minus_unconstrained'] = rob_piv.certified_rollout - rob_piv.unconstrained_rollout
rob_piv.to_csv(TAB / 'table13b_multi_perturbation_operating_point_summary.csv', index=False)


## Instantaneous full-step Jacobian

In [ ]:
def _normalized_map(model, fea, xn, base, n, u, xmu, xsd, corrected=True):
    scale = torch.as_tensor(STATE_SCALE, dtype=xn.dtype, device=xn.device)
    x = base + xn * scale
    if corrected:
        y = corrected_step(model, fea, x, base, n, u, xmu, xsd)
    else:
        y = euler_step(fea, x, n, u, DT)
    return (y - base) / scale

def _jacobian_probe_domain_valid(fea, xn, base, eps):
    scale = torch.as_tensor(STATE_SCALE, dtype=xn.dtype, device=xn.device)
    valid = fea.in_domain(base + xn * scale)
    for d in range(3):
        e = torch.zeros_like(xn)
        e[:, d] = float(eps)
        valid &= fea.in_domain(base + (xn + e) * scale)
        valid &= fea.in_domain(base + (xn - e) * scale)
    return valid

def _map_jacobian_batch(model, fea, xn, base, n, u, xmu, xsd, eps, corrected=True):
    cols = []
    with torch.no_grad():
        for d in range(3):
            e = torch.zeros_like(xn)
            e[:, d] = float(eps)
            pp = _normalized_map(model, fea, xn + e, base, n, u, xmu, xsd, corrected=corrected)
            pm = _normalized_map(model, fea, xn - e, base, n, u, xmu, xsd, corrected=corrected)
            cols.append(((pp - pm) / (2.0 * float(eps))).cpu().numpy())
    return np.stack(cols, axis=2)

def deployed_map_metrics(model, fea, bank, base_all, n_all, ueq_all, xmu_np, xsd_np, eps=0.001, batch_size=2048, include_euler=True):
    xmu = torch.tensor(xmu_np, dtype=torch.float32, device=fea.device)
    xsd = torch.tensor(xsd_np, dtype=torch.float32, device=fea.device)
    rho_c, sig_c, rho_e, sig_e, probe_valid = ([], [], [], [], [])
    for st in range(0, len(bank['gids']), batch_size):
        en = min(st + batch_size, len(bank['gids']))
        gi = bank['gids'][st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        xn = torch.tensor(bank['x0n'][st:en], device=fea.device)
        probe_valid.append(_jacobian_probe_domain_valid(fea, xn, base, eps).cpu().numpy())
        Jc = _map_jacobian_batch(model, fea, xn, base, n, u, xmu, xsd, eps, corrected=True)
        rho_c.append(np.max(np.abs(np.linalg.eigvals(Jc)), axis=1))
        sig_c.append(np.linalg.svd(Jc, compute_uv=False)[:, 0])
        if include_euler:
            Je = _map_jacobian_batch(model, fea, xn, base, n, u, xmu, xsd, eps, corrected=False)
            rho_e.append(np.max(np.abs(np.linalg.eigvals(Je)), axis=1))
            sig_e.append(np.linalg.svd(Je, compute_uv=False)[:, 0])
    return {'rho_corrected': np.concatenate(rho_c), 'sigma_corrected': np.concatenate(sig_c), 'rho_euler': np.concatenate(rho_e) if include_euler else None, 'sigma_euler': np.concatenate(sig_e) if include_euler else None, 'jacobian_probe_domain_valid': np.concatenate(probe_valid)}
keep = np.isin(rob_labels, ['dom+', 'dom-', 'rand1'])
keep_idx = np.where(keep)[0]
map_bank = {'gids': rob_gids[keep], 'x0n': rob_x0n[keep], 'ref': rob_bank['ref'][keep], 'horizon': rob_bank['horizon']}
map_labels = rob_labels[keep]
map_amps = rob_amps[keep]
pd.DataFrame([{'analysis': 'multi_perturbation_rollout', 'directions': 'dom+, dom-, id, iq, if, rand1, rand2', 'amplitudes': '0.25, 0.75', 'scenarios_per_test_point': 14, 'total_scenarios': int(len(rob_gids))}, {'analysis': 'full_map_jacobian', 'directions': 'dom+, dom-, rand1', 'amplitudes': '0.25, 0.75', 'scenarios_per_test_point': 6, 'total_scenarios': int(len(map_bank['gids'])), 'network_seeds': int(min(ROBUST_MODEL_SEEDS, len(primary_models)))}]).to_csv(TAB / 'table14a_scenario_definition.csv', index=False)
map_rows = []
for s, m in enumerate(primary_models[:min(ROBUST_MODEL_SEEDS, len(primary_models))]):
    m = m.to(DEVICE).eval()
    met = deployed_map_metrics(m, fea, map_bank, base_all, n_all, ueq_all, XMU, XSD, eps=FULL_MAP_EPS, batch_size=2048, include_euler=True)
    m.cpu()
    for orig_idx, gi, lab, amp, rc, sc, re, se, probe_ok in zip(keep_idx, map_bank['gids'], map_labels, map_amps, met['rho_corrected'], met['sigma_corrected'], met['rho_euler'], met['sigma_euler'], met['jacobian_probe_domain_valid']):
        o = ops[int(gi)]
        map_rows.append({'seed': s, 'scenario_index': int(orig_idx), 'op_index': int(gi), 'n_rpm': o['n_rpm'], 'if_A': o['if_A'], 'direction': str(lab), 'amplitude': float(amp), 'finite_difference_eps_normalized': FULL_MAP_EPS, 'rho_corrected': float(rc), 'sigma_max_corrected': float(sc), 'rho_euler': float(re), 'sigma_max_euler': float(se), 'jacobian_probe_domain_valid': bool(probe_ok)})
map_df = pd.DataFrame(map_rows)
map_df.to_csv(TAB / 'table14_full_map_jacobian_metrics.csv', index=False)
map_avg = map_df.groupby(['scenario_index', 'op_index', 'n_rpm', 'if_A', 'direction', 'amplitude']).agg(rho_corrected=('rho_corrected', 'median'), sigma_max_corrected=('sigma_max_corrected', 'median'), rho_euler=('rho_euler', 'median'), sigma_max_euler=('sigma_max_euler', 'median'), jacobian_probe_domain_valid=('jacobian_probe_domain_valid', 'min')).reset_index()
prim_scen = robust[robust.configuration == 'certified_rollout'].groupby(['scenario_index', 'op_index']).rmse.mean().reset_index(name='corrected_rmse')
eul_scen = rob_euler_df[['scenario_index', 'op_index', 'euler_rmse']]
map_avg = map_avg.merge(prim_scen, on=['scenario_index', 'op_index']).merge(eul_scen, on=['scenario_index', 'op_index'])
map_avg['rollout_improved'] = map_avg.corrected_rmse < map_avg.euler_rmse
map_avg['rho_crossed_inward'] = (map_avg.rho_euler >= 1) & (map_avg.rho_corrected < 1)
map_avg['rho_crossed_outward'] = (map_avg.rho_euler < 1) & (map_avg.rho_corrected >= 1)
map_avg['euler_local_2norm_contractive'] = map_avg.sigma_max_euler < 1
map_avg['corrected_local_2norm_contractive'] = map_avg.sigma_max_corrected < 1
map_avg['sigma_crossed_inward'] = (map_avg.sigma_max_euler >= 1) & (map_avg.sigma_max_corrected < 1)
map_avg['sigma_crossed_outward'] = (map_avg.sigma_max_euler < 1) & (map_avg.sigma_max_corrected >= 1)
map_avg.to_csv(TAB / 'table15_full_map_seed_averaged_metrics.csv', index=False)
summary_rows = [{'scenarios': len(map_avg), 'euler_rho_ge_1_fraction': float(np.mean(map_avg.rho_euler >= 1)), 'corrected_rho_ge_1_fraction': float(np.mean(map_avg.rho_corrected >= 1)), 'rho_inward_crossing_among_euler_rho_ge_1': float(np.mean(map_avg.loc[map_avg.rho_euler >= 1, 'rho_corrected'] < 1)) if np.any(map_avg.rho_euler >= 1) else np.nan, 'rho_outward_crossing_among_euler_rho_lt_1': float(np.mean(map_avg.loc[map_avg.rho_euler < 1, 'rho_corrected'] >= 1)) if np.any(map_avg.rho_euler < 1) else np.nan, 'euler_sigma_max_ge_1_fraction': float(np.mean(map_avg.sigma_max_euler >= 1)), 'corrected_sigma_max_ge_1_fraction': float(np.mean(map_avg.sigma_max_corrected >= 1)), 'sigma_inward_crossing_among_euler_sigma_ge_1': float(np.mean(map_avg.loc[map_avg.sigma_max_euler >= 1, 'sigma_max_corrected'] < 1)) if np.any(map_avg.sigma_max_euler >= 1) else np.nan, 'sigma_outward_crossing_among_euler_sigma_lt_1': float(np.mean(map_avg.loc[map_avg.sigma_max_euler < 1, 'sigma_max_corrected'] >= 1)) if np.any(map_avg.sigma_max_euler < 1) else np.nan, 'rollout_improved_fraction': float(np.mean(map_avg.rollout_improved)), 'rho_inward_crossing_with_rollout_improvement_fraction': float(np.mean(map_avg.loc[map_avg.rho_crossed_inward, 'rollout_improved'])) if np.any(map_avg.rho_crossed_inward) else np.nan, 'sigma_inward_crossing_with_rollout_improvement_fraction': float(np.mean(map_avg.loc[map_avg.sigma_crossed_inward, 'rollout_improved'])) if np.any(map_avg.sigma_crossed_inward) else np.nan}]
map_summary = pd.DataFrame(summary_rows)
map_summary.to_csv(TAB / 'table15b_full_map_jacobian_summary.csv', index=False)
strict_map = map_avg[map_avg.jacobian_probe_domain_valid.astype(bool)].copy()
strict_map_summary = pd.DataFrame([{'scenarios': int(len(strict_map)), 'unique_operating_points': int(strict_map.op_index.nunique()) if len(strict_map) else 0, 'euler_rho_ge_1_fraction': float(np.mean(strict_map.rho_euler >= 1)) if len(strict_map) else np.nan, 'corrected_rho_ge_1_fraction': float(np.mean(strict_map.rho_corrected >= 1)) if len(strict_map) else np.nan, 'euler_sigma_max_ge_1_fraction': float(np.mean(strict_map.sigma_max_euler >= 1)) if len(strict_map) else np.nan, 'corrected_sigma_max_ge_1_fraction': float(np.mean(strict_map.sigma_max_corrected >= 1)) if len(strict_map) else np.nan, 'rollout_improved_fraction': float(np.mean(strict_map.rollout_improved)) if len(strict_map) else np.nan}])
strict_map_summary.to_csv(TAB / 'table15bb_full_map_jacobian_strict_domain_summary.csv', index=False)
fig, ax = plt.subplots(figsize=(5.0, 4.3))
ax.scatter(map_avg.rho_euler, map_avg.rho_corrected, s=10, alpha=0.35)
lim = max(1.2, float(max(map_avg.rho_euler.max(), map_avg.rho_corrected.max())))
ax.plot([0, lim], [0, lim], 'k--', lw=0.8)
ax.axhline(1, color='k', lw=0.8)
ax.axvline(1, color='k', lw=0.8)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel('$\\rho(D\\Phi_{\\mathrm{Euler}})$')
ax.set_ylabel('$\\rho(D\\Phi_{\\mathrm{corrected}})$')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')
fig, ax = plt.subplots(figsize=(5.0, 4.3))
ax.scatter(map_avg.sigma_max_euler, map_avg.sigma_max_corrected, s=10, alpha=0.35)
lim = max(1.2, float(max(map_avg.sigma_max_euler.max(), map_avg.sigma_max_corrected.max())))
ax.plot([0, lim], [0, lim], 'k--', lw=0.8)
ax.axhline(1, color='k', lw=0.8)
ax.axvline(1, color='k', lw=0.8)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel('$\\sigma_{\\max}(D\\Phi_{\\mathrm{Euler}})$')
ax.set_ylabel('$\\sigma_{\\max}(D\\Phi_{\\mathrm{corrected}})$')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')
rng = np.random.default_rng(SEED + 71)
subset_idx = rng.choice(len(map_bank['gids']), size=min(EPS_SENSITIVITY_SCENARIOS, len(map_bank['gids'])), replace=False)
eps_bank = {'gids': map_bank['gids'][subset_idx], 'x0n': map_bank['x0n'][subset_idx], 'ref': map_bank['ref'][subset_idx], 'horizon': map_bank['horizon']}
m = primary_models[0].to(DEVICE).eval()
eps_results = {}
for eps in EPS_SENSITIVITY:
    eps_results[float(eps)] = deployed_map_metrics(m, fea, eps_bank, base_all, n_all, ueq_all, XMU, XSD, eps=float(eps), batch_size=2048, include_euler=True)
m.cpu()
base_eps = float(FULL_MAP_EPS)
if base_eps not in eps_results:
    m = primary_models[0].to(DEVICE).eval()
    eps_results[base_eps] = deployed_map_metrics(m, fea, eps_bank, base_all, n_all, ueq_all, XMU, XSD, eps=base_eps, batch_size=2048, include_euler=True)
    m.cpu()
b = eps_results[base_eps]
eps_rows = []
for eps, met in sorted(eps_results.items()):
    for key in ['rho_corrected', 'sigma_corrected', 'rho_euler', 'sigma_euler']:
        rel = np.abs(met[key] - b[key]) / np.maximum(np.abs(b[key]), 1e-12)
        threshold_disagreement = np.mean((met[key] >= 1) != (b[key] >= 1))
        eps_rows.append({'eps_normalized': eps, 'metric': key, 'samples': len(rel), 'median_relative_change': float(np.median(rel)), 'p95_relative_change': float(np.percentile(rel, 95)), 'maximum_relative_change': float(np.max(rel)), 'unit_threshold_classification_disagreement': float(threshold_disagreement), 'median_absolute_distance_from_unit_at_baseline': float(np.median(np.abs(b[key] - 1.0))), 'near_unit_fraction_baseline_abs_lt_0p02': float(np.mean(np.abs(b[key] - 1.0) < 0.02))})
eps_sensitivity_df = pd.DataFrame(eps_rows)
eps_sensitivity_df.to_csv(TAB / 'table15c_full_map_epsilon_sensitivity.csv', index=False)


## Finite-time Jacobian products

In [ ]:
def finite_time_product_metrics(model, fea, bank, base_all, n_all, ueq_all, xmu_np, xsd_np, steps=20, eps=0.001, batch_size=768):
    xmu = torch.tensor(xmu_np, dtype=torch.float32, device=fea.device)
    xsd = torch.tensor(xsd_np, dtype=torch.float32, device=fea.device)
    scale = torch.tensor(STATE_SCALE, dtype=torch.float32, device=fea.device)
    rows = []
    for st in range(0, len(bank['gids']), batch_size):
        en = min(st + batch_size, len(bank['gids']))
        gi = bank['gids'][st:en]
        base = torch.tensor(base_all[gi], device=fea.device)
        n = torch.tensor(n_all[gi], device=fea.device)
        u = torch.tensor(ueq_all[gi], device=fea.device)
        xn_c = torch.tensor(bank['x0n'][st:en], device=fea.device)
        xn_e = xn_c.clone()
        B = len(gi)
        Pc = np.repeat(np.eye(3)[None, :, :], B, axis=0)
        Pe = np.repeat(np.eye(3)[None, :, :], B, axis=0)
        contact_c = np.zeros(B, dtype=bool)
        contact_e = np.zeros(B, dtype=bool)
        for _ in range(int(steps)):
            valid_c = _jacobian_probe_domain_valid(fea, xn_c, base, eps).cpu().numpy()
            valid_e = _jacobian_probe_domain_valid(fea, xn_e, base, eps).cpu().numpy()
            contact_c |= ~valid_c
            contact_e |= ~valid_e
            Jc = _map_jacobian_batch(model, fea, xn_c, base, n, u, xmu, xsd, eps, corrected=True)
            Je = _map_jacobian_batch(model, fea, xn_e, base, n, u, xmu, xsd, eps, corrected=False)
            Pc = np.einsum('bij,bjk->bik', Jc, Pc)
            Pe = np.einsum('bij,bjk->bik', Je, Pe)
            with torch.no_grad():
                xn_c = _normalized_map(model, fea, xn_c, base, n, u, xmu, xsd, corrected=True)
                xn_e = _normalized_map(model, fea, xn_e, base, n, u, xmu, xsd, corrected=False)
                contact_c |= (~fea.in_domain(base + xn_c * scale)).cpu().numpy()
                contact_e |= (~fea.in_domain(base + xn_e * scale)).cpu().numpy()
        sc = np.linalg.svd(Pc, compute_uv=False)[:, 0]
        se = np.linalg.svd(Pe, compute_uv=False)[:, 0]
        rc = np.max(np.abs(np.linalg.eigvals(Pc)), axis=1)
        re = np.max(np.abs(np.linalg.eigvals(Pe)), axis=1)
        for local, gidx in enumerate(gi):
            rows.append({'op_index': int(gidx), 'finite_time_steps': int(steps), 'finite_time_ms': float(steps * DT * 1000.0), 'sigma_max_product_corrected': float(sc[local]), 'sigma_max_product_euler': float(se[local]), 'rho_product_corrected': float(rc[local]), 'rho_product_euler': float(re[local]), 'geometric_gain_per_step_corrected': float(sc[local] ** (1.0 / steps)), 'geometric_gain_per_step_euler': float(se[local] ** (1.0 / steps)), 'domain_contact_corrected': bool(contact_c[local]), 'domain_contact_euler': bool(contact_e[local]), 'strict_domain_both': bool(not contact_c[local] and (not contact_e[local]))})
    return pd.DataFrame(rows)
ft_frames = []
for s, m in enumerate(primary_models[:min(FINITE_TIME_MODEL_SEEDS, len(primary_models))]):
    m = m.to(DEVICE).eval()
    d = finite_time_product_metrics(m, fea, map_bank, base_all, n_all, ueq_all, XMU, XSD, steps=FINITE_TIME_STEPS, eps=FULL_MAP_EPS)
    d['seed'] = s
    d['scenario_index'] = keep_idx
    d['direction'] = map_labels
    d['amplitude'] = map_amps
    ft_frames.append(d)
    m.cpu()
finite_time_df = pd.concat(ft_frames, ignore_index=True)
finite_time_df.to_csv(TAB / 'table15d_finite_time_jacobian_products_all.csv', index=False)
finite_time_avg = finite_time_df.groupby(['scenario_index', 'op_index', 'direction', 'amplitude']).agg(sigma_max_product_corrected=('sigma_max_product_corrected', 'median'), sigma_max_product_euler=('sigma_max_product_euler', 'median'), rho_product_corrected=('rho_product_corrected', 'median'), rho_product_euler=('rho_product_euler', 'median'), geometric_gain_per_step_corrected=('geometric_gain_per_step_corrected', 'median'), geometric_gain_per_step_euler=('geometric_gain_per_step_euler', 'median'), domain_contact_corrected=('domain_contact_corrected', 'max'), domain_contact_euler=('domain_contact_euler', 'max'), strict_domain_both=('strict_domain_both', 'min')).reset_index()
ft_summary = pd.DataFrame([{'scenarios': len(finite_time_avg), 'unique_operating_points': int(finite_time_avg.op_index.nunique()), 'steps': FINITE_TIME_STEPS, 'horizon_ms': FINITE_TIME_STEPS * DT * 1000.0, 'median_sigma_product_euler': float(finite_time_avg.sigma_max_product_euler.median()), 'median_sigma_product_corrected': float(finite_time_avg.sigma_max_product_corrected.median()), 'p95_sigma_product_euler': float(finite_time_avg.sigma_max_product_euler.quantile(0.95)), 'p95_sigma_product_corrected': float(finite_time_avg.sigma_max_product_corrected.quantile(0.95)), 'corrected_lower_sigma_product_fraction': float(np.mean(finite_time_avg.sigma_max_product_corrected < finite_time_avg.sigma_max_product_euler)), 'euler_sigma_product_gt_1_fraction': float(np.mean(finite_time_avg.sigma_max_product_euler > 1)), 'corrected_sigma_product_gt_1_fraction': float(np.mean(finite_time_avg.sigma_max_product_corrected > 1))}])
ft_summary.to_csv(TAB / 'table15e_finite_time_jacobian_product_summary.csv', index=False)
strict_ft = finite_time_avg[finite_time_avg.strict_domain_both.astype(bool)].copy()
ft_strict_summary = pd.DataFrame([{'scenarios': int(len(strict_ft)), 'unique_operating_points': int(strict_ft.op_index.nunique()) if len(strict_ft) else 0, 'steps': FINITE_TIME_STEPS, 'horizon_ms': FINITE_TIME_STEPS * DT * 1000.0, 'median_sigma_product_euler': float(strict_ft.sigma_max_product_euler.median()) if len(strict_ft) else np.nan, 'median_sigma_product_corrected': float(strict_ft.sigma_max_product_corrected.median()) if len(strict_ft) else np.nan, 'euler_product_gt1_fraction': float(np.mean(strict_ft.sigma_max_product_euler > 1)) if len(strict_ft) else np.nan, 'corrected_product_gt1_fraction': float(np.mean(strict_ft.sigma_max_product_corrected > 1)) if len(strict_ft) else np.nan, 'corrected_lower_product_gain_fraction': float(np.mean(strict_ft.sigma_max_product_corrected < strict_ft.sigma_max_product_euler)) if len(strict_ft) else np.nan}])
ft_strict_summary.to_csv(TAB / 'table15f_finite_time_jacobian_product_strict_domain_summary.csv', index=False)
fig, ax = plt.subplots(figsize=(5.0, 4.3))
x = np.maximum(finite_time_avg.sigma_max_product_euler.to_numpy(), 1e-12)
y = np.maximum(finite_time_avg.sigma_max_product_corrected.to_numpy(), 1e-12)
ax.scatter(x, y, s=10, alpha=0.35)
lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$\\sigma_{\\max}(\\prod D\\Phi_{\\mathrm{Euler}})$')
ax.set_ylabel('$\\sigma_{\\max}(\\prod D\\Phi_{\\mathrm{corrected}})$')
ax.grid(alpha=0.2, which='both')
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Linear solver comparison

In [ ]:
methods = ['Euler', 'Heun', 'RK4', 'Backward Euler', 'Trapezoidal', 'Exact exponential']
rows = []
I = np.eye(3)
for gi in test_ops:
    A = ops[int(gi)]['A']
    Z = DT * A
    E = expm(Z)
    mats = {'Euler': I + Z, 'Heun': I + Z + 0.5 * (Z @ Z), 'RK4': I + Z + 0.5 * (Z @ Z) + Z @ Z @ Z / 6.0 + Z @ Z @ Z @ Z / 24.0, 'Backward Euler': np.linalg.inv(I - Z), 'Trapezoidal': np.linalg.solve(I - 0.5 * Z, I + 0.5 * Z), 'Exact exponential': E}
    for name, M in mats.items():
        rows.append({'op_index': int(gi), 'n_rpm': ops[int(gi)]['n_rpm'], 'if_A': ops[int(gi)]['if_A'], 'method': name, 'spectral_radius': float(np.max(np.abs(np.linalg.eigvals(M)))), 'relative_matrix_error': float(np.linalg.norm(M - E, 'fro') / np.linalg.norm(E, 'fro'))})
linbench = pd.DataFrame(rows)
linbench.to_csv(TAB / 'table16_linear_solver_benchmark_all.csv', index=False)
lin_summary = linbench.groupby('method').agg(stable_fraction=('spectral_radius', lambda x: np.mean(np.asarray(x) < 1)), median_relative_matrix_error=('relative_matrix_error', 'median'), p95_relative_matrix_error=('relative_matrix_error', lambda x: np.percentile(x, 95))).reset_index()
lin_summary.to_csv(TAB / 'table17_linear_solver_benchmark_summary.csv', index=False)
work = pd.DataFrame([['Euler', 1, 0, 'one right-hand-side evaluation'], ['Heun', 2, 0, 'two right-hand-side evaluations'], ['RK4', 4, 0, 'four right-hand-side evaluations'], ['Backward Euler', 1, 1, 'nonlinear or linear solve required in a changing model'], ['Trapezoidal', 2, 1, 'implicit solve required in a changing model'], ['Exact exponential', 0, 1, 'matrix-exponential setup for each local Jacobian'], ['Residual-corrected Euler', 1, 0, 'one right-hand-side evaluation plus one network evaluation']], columns=['method', 'rhs_evaluations_per_step', 'implicit_or_matrix_setup', 'work_note'])
work.to_csv(TAB / 'table18_algorithmic_work.csv', index=False)

def explicit_stability_polynomial(z, method):
    if method == 'Euler':
        return 1.0 + z
    if method == 'Heun':
        return 1.0 + z + 0.5 * z * z
    if method == 'RK4':
        return 1.0 + z + 0.5 * z * z + z ** 3 / 6.0 + z ** 4 / 24.0
    raise ValueError(method)

def critical_step_explicit(eigs, method, h_start=1e-09, h_cap=10.0, bisect_iter=70):
    eigs = np.asarray(eigs, dtype=complex)
    if np.any(np.real(eigs) >= 0):
        return 0.0
    if method == 'Euler':
        return dtcrit_from_eigs(eigs)

    def stable(h):
        return bool(np.max(np.abs(explicit_stability_polynomial(h * eigs, method))) < 1.0)
    lo = 0.0
    hi = max(float(h_start), 1e-12)
    while hi < h_cap and stable(hi):
        lo = hi
        hi *= 2.0
    if hi >= h_cap and stable(h_cap):
        return float(h_cap)
    hi = min(hi, h_cap)
    for _ in range(int(bisect_iter)):
        mid = 0.5 * (lo + hi)
        if stable(mid):
            lo = mid
        else:
            hi = mid
    return float(lo)
solver_step_atlas = pd.DataFrame()
solver_step_summary = pd.DataFrame()
direct_solver_step_atlas = pd.DataFrame()
direct_solver_step_summary = pd.DataFrame()
direct_solver_step_comparison = pd.DataFrame()
if RUN_SOLVER_STEP_ATLAS:
    solver_step_rows = []
    for gi, o in enumerate(ops):
        eigs = np.linalg.eigvals(o['A'])
        for method in ['Euler', 'Heun', 'RK4']:
            hc = critical_step_explicit(eigs, method)
            solver_step_rows.append({'op_index': int(gi), 'n_rpm': float(o['n_rpm']), 'if_A': float(o['if_A']), 'id_A': float(o['id_A']), 'iq_A': float(o['iq_A']), 'method': method, 'linearization_location': 'differential_map_grid_knot', 'hcrit_s': hc, 'hcrit_us': hc * 1000000.0, 'stable_at_50us': bool(DT < hc)})
    solver_step_atlas = pd.DataFrame(solver_step_rows)
    solver_step_atlas.to_csv(TAB / 'table17b_explicit_solver_critical_step_atlas.csv', index=False)
    solver_step_summary = solver_step_atlas.groupby(['method', 'n_rpm']).agg(points=('op_index', 'size'), hcrit_min_us=('hcrit_us', 'min'), hcrit_median_us=('hcrit_us', 'median'), hcrit_max_us=('hcrit_us', 'max'), stable_at_50us_fraction=('stable_at_50us', 'mean')).reset_index()
    solver_step_summary.to_csv(TAB / 'table17c_explicit_solver_critical_step_summary.csv', index=False)
    if RUN_DIRECT_RHS_JACOBIAN_AUDIT and len(rhs_jacobian_audit):
        grid_base = rhs_jacobian_audit[np.isclose(rhs_jacobian_audit.eps_normalized, DIRECT_RHS_JAC_PRIMARY_EPS)].sort_values('op_index')
        grid_rows = []
        for _, r in grid_base.iterrows():
            for method in ['Euler', 'Heun', 'RK4']:
                tag = method.lower()
                hc = float(r[f'hcrit_rhs_fd_{tag}_s'])
                grid_rows.append({'op_index': int(r.op_index), 'n_rpm': float(r.n_rpm), 'if_A': float(r.if_A), 'id_A': float(r.id_A), 'iq_A': float(r.iq_A), 'method': method, 'finite_difference_eps_normalized': float(DIRECT_RHS_JAC_PRIMARY_EPS), 'linearization_location': 'grid_knot_symmetric_fd', 'hcrit_s': hc, 'hcrit_us': hc * 1000000.0, 'stable_at_50us': bool(DT < hc)})
        grid_solver_step_atlas = pd.DataFrame(grid_rows)
        grid_solver_step_atlas.to_csv(TAB / 'table17d_explicit_solver_critical_step_grid_knot_fd_atlas.csv', index=False)
        grid_solver_step_summary = grid_solver_step_atlas.groupby(['method', 'n_rpm']).agg(points=('op_index', 'size'), hcrit_min_us=('hcrit_us', 'min'), hcrit_median_us=('hcrit_us', 'median'), hcrit_max_us=('hcrit_us', 'max'), stable_at_50us_fraction=('stable_at_50us', 'mean')).reset_index()
        grid_solver_step_summary.to_csv(TAB / 'table17e_explicit_solver_critical_step_grid_knot_fd_summary.csv', index=False)
        a = solver_step_atlas.rename(columns={'hcrit_us': 'hcrit_differential_map_us', 'stable_at_50us': 'stable50_differential_map'})[['op_index', 'n_rpm', 'if_A', 'method', 'hcrit_differential_map_us', 'stable50_differential_map']]
        b = grid_solver_step_atlas.rename(columns={'hcrit_us': 'hcrit_grid_knot_fd_us', 'stable_at_50us': 'stable50_grid_knot_fd'})[['op_index', 'method', 'hcrit_grid_knot_fd_us', 'stable50_grid_knot_fd']]
        grid_solver_step_comparison = a.merge(b, on=['op_index', 'method'], how='inner')
        grid_solver_step_comparison['relative_hcrit_difference'] = np.abs(grid_solver_step_comparison.hcrit_grid_knot_fd_us - grid_solver_step_comparison.hcrit_differential_map_us) / np.maximum(np.abs(grid_solver_step_comparison.hcrit_grid_knot_fd_us), 1e-15)
        grid_solver_step_comparison['stable50_agreement'] = grid_solver_step_comparison.stable50_grid_knot_fd == grid_solver_step_comparison.stable50_differential_map
        grid_solver_step_comparison.to_csv(TAB / 'table17f_explicit_solver_critical_step_grid_knot_comparison.csv', index=False)


## Cluster-based statistics

In [ ]:
stat_rows = []
main_eval['diff_primary_minus_euler'] = main_eval.certified_rollout - main_eval.euler
main_eval['diff_primary_minus_rk4'] = main_eval.certified_rollout - main_eval.rk4
for col, label in [('diff_primary_minus_euler', 'certified rollout - Euler'), ('diff_primary_minus_rk4', 'certified rollout - RK4')]:
    z = cluster_stats(main_eval, col, ('n_rpm', 'if_A'), B=CROSSED_BOOTSTRAP_B, seed=SEED + 80)
    z['comparison'] = label
    stat_rows.append(z)
for col, label in [('diff_primary_minus_onestep', 'rollout-trained - one-step-trained'), ('diff_primary_minus_unconstrained', 'certified - unconstrained')]:
    z = cluster_stats(rob_piv, col, ('n_rpm', 'if_A'), B=CROSSED_BOOTSTRAP_B, seed=SEED + 81)
    z['comparison'] = label
    stat_rows.append(z)
cluster_df = pd.DataFrame(stat_rows)
cluster_df.to_csv(TAB / 'table19_cluster_aware_statistics.csv', index=False)
euler_by_op = classical[classical.method == 'euler'][['op_index', 'rmse']].rename(columns={'rmse': 'euler_rmse'})
rk4_by_op = classical[classical.method == 'rk4'][['op_index', 'rmse']].rename(columns={'rmse': 'rk4_rmse'})
seed_block = roll_primary.merge(euler_by_op, on='op_index', how='left').merge(rk4_by_op, on='op_index', how='left')
seed_block['spatial_block'] = [OP_BLOCK.get(int(gi), -1) for gi in seed_block.op_index]
seed_block['diff_corrected_minus_euler'] = seed_block.rmse - seed_block.euler_rmse
seed_block['diff_corrected_minus_rk4'] = seed_block.rmse - seed_block.rk4_rmse
crossed_rows = []
for col, label in [('diff_corrected_minus_euler', 'certified rollout - Euler'), ('diff_corrected_minus_rk4', 'certified rollout - RK4')]:
    rec = two_way_seed_cluster_bootstrap(seed_block, col, B=CROSSED_BOOTSTRAP_B, seed=SEED + 180)
    rec['comparison'] = label
    crossed_rows.append(rec)
fixed_duration_crossed_df = pd.DataFrame(crossed_rows)
fixed_duration_crossed_df.to_csv(TAB / 'table19b_two_way_seed_spatial_block_bootstrap.csv', index=False)
fixed_rng_frames = []
for col, label in [('diff_corrected_minus_euler', 'certified rollout - Euler'), ('diff_corrected_minus_rk4', 'certified rollout - RK4')]:
    z = crossed_bootstrap_rng_sensitivity(seed_block, col, base_seed=SEED + 180, repeats=BOOTSTRAP_RNG_REPEATS, B=CROSSED_BOOTSTRAP_B)
    z['comparison'] = label
    fixed_rng_frames.append(z)
fixed_duration_bootstrap_rng_sensitivity = pd.concat(fixed_rng_frames, ignore_index=True)
fixed_duration_bootstrap_rng_sensitivity.to_csv(TAB / 'table19e_fixed_duration_two_way_bootstrap_rng_sensitivity.csv', index=False)


## Cross-speed transfer

In [ ]:
loso_rows = []
if RUN_LOSO:
    rng = np.random.default_rng(SEED + 90)
    for held in D['n_vec']:
        held_ids = np.array([gi for gi in eligible if ops[int(gi)]['n_rpm'] == held], dtype=int)
        remain = np.array([gi for gi in eligible if ops[int(gi)]['n_rpm'] != held], dtype=int)
        strata = {}
        for gi in remain:
            o = ops[int(gi)]
            strata.setdefault((int(o['if_A']), int(o['rho_euler'] > 1)), []).append(int(gi))
        tr = []
        va = []
        for a in strata.values():
            a = np.asarray(a)
            rng.shuffle(a)
            cut = max(1, int(0.8 * len(a)))
            tr += a[:cut].tolist()
            va += a[cut:].tolist()
        tr = np.asarray(tr, dtype=int)
        va = np.asarray(va, dtype=int)
        sm_tr = np.isin(sample_gids, tr)
        sm_va = np.isin(sample_gids, va)
        mu = Xraw[sm_tr].mean(0)
        sd = Xraw[sm_tr].std(0) + 1e-07
        xtr = (Xraw[sm_tr] - mu) / sd
        ytr = Ycorr[sm_tr]
        xv = (Xraw[sm_va] - mu) / sd
        yv = Ycorr[sm_va]
        tb_ops = stratified_subsample(tr, min(500, len(tr)), SEED + int(held))
        vb_ops = stratified_subsample(va, min(250, len(va)), SEED + int(held) + 1)
        tx = np.clip(rng.normal(0, 0.4, size=(len(tb_ops), 3)), -0.8, 0.8).astype(np.float32)
        vx = np.clip(rng.normal(0, 0.4, size=(len(vb_ops), 3)), -0.8, 0.8).astype(np.float32)
        tb = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, tb_ops, tx, horizon=12, ref_substeps=ROLLOUT_REF_SUBSTEPS)
        vb = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, vb_ops, vx, horizon=30, ref_substeps=ROLLOUT_REF_SUBSTEPS)
        testx = np.tile(np.array([20 / 30, -10 / 30, 0.2 / 0.35], np.float32), (len(held_ids), 1))
        hb = make_rollout_bank(fea, ops, base_all, n_all, ueq_all, held_ids, testx, horizon=40, ref_substeps=ROLLOUT_REF_SUBSTEPS)
        for s in range(LOSO_SEEDS):
            m, score, _ = train_model(primary_params, True, fea, xtr, ytr, xv, yv, tb, vb, base_all, n_all, ueq_all, mu, sd, epochs=LOSO_EPOCHS, seed=SEED + 6000 + int(held) + s, rollout_updates=ROLLOUT_UPDATES, rollout_batch=ROLLOUT_BATCH, patience=12)
            rr = eval_bank(m, fea, hb, base_all, n_all, ueq_all, mu, sd)
            ee = eval_classical_bank(fea, hb, base_all, n_all, ueq_all, 'euler')
            loso_rows.append({'held_speed_rpm': int(held), 'seed': s, 'test_points': len(held_ids), 'median_corrected_rmse': float(np.median(rr)), 'median_euler_rmse': float(np.median(ee)), 'corrected_better_fraction': float(np.mean(rr < ee))})
            m.cpu()
loso_df = pd.DataFrame(loso_rows)
loso_df.to_csv(TAB / 'table20_leave_one_speed_out.csv', index=False)


## Platform-dependent timing

In [ ]:
def benchmark_step_times(model, cpu_fea, base_all, n_all, ueq_all, op_ids,
                         xmu_np, xsd_np, current_limit_rms_A, repeats=100, batch_sizes=(1, 256)):
    """Measure CPU-only software latency without moving the trained analysis model."""
    timing_device = torch.device('cpu')
    if cpu_fea.device.type != 'cpu':
        raise ValueError('CPU timing requires a CPU-resident FEA representation.')
    # Legacy spectral_norm modules do not always support deepcopy after forward().
    # Reconstruct an independent CPU model from its state without mutating the
    # trained analysis model or advancing the experiment's CPU random state.
    linear_layers = [layer for layer in model.net if isinstance(layer, torch.nn.Linear)]
    if len(linear_layers) < 2:
        raise ValueError('Timing model must have an input layer and an output layer.')
    certified = (hasattr(linear_layers[0], 'parametrizations')
                 and hasattr(linear_layers[0].parametrizations, 'weight'))
    with torch.random.fork_rng(devices=[]):
        measured_model = ResidualNet(width=linear_layers[0].out_features,
                                     depth=len(linear_layers)-1,
                                     beta=model.beta, certified=certified)
        cpu_state = {key: value.detach().cpu().clone()
                     for key, value in model.state_dict().items()}
        measured_model.load_state_dict(cpu_state, strict=True)
        measured_model.to(timing_device).eval()
    xmu = torch.as_tensor(xmu_np, dtype=torch.float32, device=timing_device)
    xsd = torch.as_tensor(xsd_np, dtype=torch.float32, device=timing_device)
    scale = torch.as_tensor(STATE_SCALE, dtype=torch.float32, device=timing_device)
    # Each repetition times one step from a fixed, checked starting state.
    # Chaining outputs would eventually leave the tabulated FEA domain and
    # change the workload across methods. The CPU timing is a software benchmark
    # at retained current-limited operating points, not a trajectory experiment.
    if repeats < 1 or any(int(size) < 1 for size in batch_sizes):
        raise ValueError('Timing repeats and batch sizes must be positive.')
    candidates = np.asarray(op_ids, dtype=int)
    if not len(candidates):
        raise ValueError('No candidate operating points for CPU timing.')
    current_limit = float(current_limit_rms_A) * math.sqrt(3.0)

    def valid_state(values):
        return (cpu_fea.in_domain(values) & torch.isfinite(values).all(dim=1)
                & (torch.linalg.vector_norm(values[:, :2], dim=1) <= current_limit))

    with torch.no_grad():
        cb = torch.as_tensor(base_all[candidates], dtype=torch.float32, device=timing_device)
        cn = torch.as_tensor(n_all[candidates], dtype=torch.float32, device=timing_device)
        cu = torch.as_tensor(ueq_all[candidates], dtype=torch.float32, device=timing_device)
        cx = cb + 0.2 * scale
        keep = valid_state(cx)
        # Only compute RHS evaluations for initially admissible states.
        candidates = candidates[keep.cpu().numpy()]
        cb, cn, cu, cx = cb[keep], cn[keep], cu[keep], cx[keep]
        if not len(candidates):
            raise ValueError('No current- and table-valid starting states for CPU timing.')

        k1 = cpu_fea.rhs(cx, cn, cu)
        euler_next = cx + DT * k1
        heun_next = cx + 0.5 * DT * (k1 + cpu_fea.rhs(euler_next, cn, cu))
        mid2 = cx + 0.5 * DT * k1
        k2 = cpu_fea.rhs(mid2, cn, cu)
        mid3 = cx + 0.5 * DT * k2
        k3 = cpu_fea.rhs(mid3, cn, cu)
        mid4 = cx + DT * k3
        k4 = cpu_fea.rhs(mid4, cn, cu)
        rk4_next = cx + DT * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
        corrected_next = corrected_step(measured_model, cpu_fea, cx, cb, cn, cu, xmu, xsd)
        admissible = torch.ones(len(candidates), dtype=torch.bool, device=timing_device)
        for state in (cx, euler_next, heun_next, mid2, mid3, mid4,
                      rk4_next, corrected_next):
            admissible &= valid_state(state)
        candidates = candidates[admissible.cpu().numpy()]
        if not len(candidates):
            raise ValueError('No one-step-valid CPU timing points across all four solvers.')

    rng = np.random.default_rng(SEED + 120)
    rows = []
    for B in batch_sizes:
        B = int(B)
        ids = rng.choice(candidates, size=B, replace=B > len(candidates))
        base = torch.as_tensor(base_all[ids], dtype=torch.float32, device=timing_device)
        n = torch.as_tensor(n_all[ids], dtype=torch.float32, device=timing_device)
        u = torch.as_tensor(ueq_all[ids], dtype=torch.float32, device=timing_device)
        x = base + 0.2 * scale
        methods = {
            'Euler': lambda z: euler_step(cpu_fea, z, n, u, DT),
            'Heun': lambda z: heun_step(cpu_fea, z, n, u, DT),
            'RK4': lambda z: rk4_macro(cpu_fea, z, n, u, DT),
            'Corrected Euler': lambda z: corrected_step(measured_model, cpu_fea, z, base, n, u, xmu, xsd),
        }
        for name, fun in methods.items():
            with torch.no_grad():
                for _ in range(5):
                    _ = fun(x)
                start = time.perf_counter()
                for _ in range(int(repeats)):
                    _ = fun(x)  # independent single-step evaluations; no state drift
                elapsed = time.perf_counter() - start
            rows.append({'device': 'cpu', 'batch_size': B, 'method': name,
                         'repeats': int(repeats),
                         'timing_protocol': 'fixed_input_single_step',
                         'domain_checked': True,
                         'eligible_operating_points': int(len(candidates)),
                         'mean_time_per_batch_us': elapsed / repeats * 1e6,
                         'mean_time_per_state_us': elapsed / repeats / B * 1e6})
    return pd.DataFrame(rows)

if RUN_TIMING:
    cpu_fea_for_timing = TorchFEA(D, L_edge2, device=torch.device('cpu'))
    timing_df = benchmark_step_times(primary_models[0], cpu_fea_for_timing, base_all,
                                     n_all, ueq_all, test_ops, XMU, XSD, current_limit_rms_A=D['ismax'], repeats=TIMING_REPEATS)
    if not timing_df.device.eq('cpu').all():
        raise AssertionError('CPU timing must never contain GPU timings.')
    timing_df.to_csv(TAB / 'table20b_platform_specific_step_timing.csv', index=False)
    del cpu_fea_for_timing
else:
    timing_df = pd.DataFrame()


## Euler step coverage

In [ ]:
plot_df = main_eval.copy()
fig, ax = plt.subplots(figsize=(6.2, 3.8))
vals = [plot_df.euler, plot_df.heun, plot_df.rk4, plot_df.certified_one_step, plot_df.certified_rollout, plot_df.unconstrained_rollout]
ax.boxplot(vals, tick_labels=['Euler', 'Heun', 'RK4', 'One-step\nSN', 'Rollout\nSN', 'Rollout\nfree'], showfliers=False)
ax.set_yscale('log')
ax.set_ylabel('Normalized rollout RMSE')
ax.grid(axis='y', which='both', alpha=0.2)
fig.tight_layout()
plt.close(fig)
plt.close('all')
cov = []
for n in sorted(atlas_exc.n_rpm.unique()):
    z = atlas_exc[atlas_exc.n_rpm == n].hcrit_us.to_numpy()
    for h in [10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200]:
        cov.append({'n_rpm': n, 'step_us': h, 'stable_fraction': np.mean(z > h)})
cov = pd.DataFrame(cov)
cov.to_csv(TAB / 'table21_fixed_step_coverage.csv', index=False)
fig, ax = plt.subplots(figsize=(5.8, 3.6))
for n in sorted(cov.n_rpm.unique()):
    s = cov[cov.n_rpm == n]
    ax.plot(s.step_us, 100 * s.stable_fraction, marker='o', label=f'{int(n)} rpm')
ax.set_xscale('log')
ax.set_xlabel('Fixed step (µs)')
ax.set_ylabel('Locally stable points (%)')
ax.set_ylim(-2, 102)
ax.grid(which='both', alpha=0.2)
ax.legend()
fig.tight_layout()
plt.close(fig)
plt.close('all')


## Cell invertibility and structural checks

In [ ]:
from scipy.stats import spearmanr
from itertools import product
INTERFACE_SAMPLE_COUNT = 600 if FULL_RUN else 120
CELL_POSITION_SAMPLE_COUNT = 250 if FULL_RUN else 35
CERTIFY_ALL_CELLS = True
CELL_CERTIFICATION_BATCH = 256
STATE_COORDINATES = ('id_A', 'iq_A', 'if_A')

def symmetric_part(A):
    return 0.5 * (A + np.swapaxes(A, -1, -2))

def cellwise_flux_inductance(X, want_L_grad=False, force_cells=None):
    X = np.asarray(X, dtype=np.float64).reshape(-1, 3)
    aa = [np.asarray(D[k], dtype=float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    for d, a in enumerate(aa):
        if np.any((X[:, d] < a[0] - 1e-11) | (X[:, d] > a[-1] + 1e-11)):
            raise ValueError('Out-of-table evaluation is not permitted')
    ix = []
    t = []
    h = []
    for d, a in enumerate(aa):
        if force_cells is None:
            k = np.clip(np.searchsorted(a, X[:, d], side='right') - 1, 0, len(a) - 2)
        else:
            k = np.asarray(force_cells, dtype=int).reshape(-1, 3)[:, d]
            if len(k) != len(X) or np.any(k < 0) or np.any(k > len(a) - 2):
                raise ValueError('Incorrect forced cell indices')
        hd = a[k + 1] - a[k]
        ix.append(k)
        h.append(hd)
        t.append(np.clip((X[:, d] - a[k]) / hd, 0.0, 1.0))
    flux_grid = np.stack([D['psid'], D['psiq'], D['psif']], axis=-1)
    vt = []
    lt = []
    w = []
    dw = []
    for bi, bq, bf in product((0, 1), repeat=3):
        bits = (bi, bq, bf)
        f = flux_grid[ix[1] + bq, ix[0] + bi, ix[2] + bf]
        mat = L_edge2[ix[1] + bq, ix[0] + bi, ix[2] + bf]
        weights = [t[d] if bits[d] else 1 - t[d] for d in range(3)]
        weights_d = [np.full(len(X), 1.0 / h[d]) if bits[d] else np.full(len(X), -1.0 / h[d]) for d in range(3)]
        W = weights[0] * weights[1] * weights[2]
        DW = np.stack([weights_d[d] * weights[(d + 1) % 3] * weights[(d + 2) % 3] for d in range(3)], axis=1)
        vt.append(f)
        lt.append(mat)
        w.append(W)
        dw.append(DW)
    Fv = np.stack(vt, axis=1)
    Lv = np.stack(lt, axis=1)
    WW = np.stack(w, axis=1)
    DD = np.stack(dw, axis=1)
    flux = np.einsum('nv,nvi->ni', WW, Fv)
    Lfd = np.einsum('nv,nvij->nij', WW, Lv)
    Dic = np.einsum('nvc,nvi->nic', DD, Fv)
    if want_L_grad:
        GL = np.einsum('nvc,nvij->nijc', DD, Lv)
        return (flux, Lfd, Dic, GL)
    return (flux, Lfd, Dic)

def evaluate_stability_metrics(X, speed):
    X = np.asarray(X, dtype=float).reshape(-1, 3)
    speed = np.broadcast_to(np.asarray(speed, dtype=float), len(X))
    flux, L, Dic, GL = cellwise_flux_inductance(X, True)
    Rp = np.diag([D['Rs'], D['Rs'], D['Rf']])
    J = np.array([[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
    we = 2 * np.pi * D['p'] * speed / 60
    Om = we[:, None, None] * J
    svL = np.linalg.svd(L, compute_uv=False)[:, -1]
    svD = np.linalg.svd(Dic, compute_uv=False)[:, -1]
    inverse_valid = (svL > 1e-12) & (svD > 1e-12)
    if not inverse_valid.all():
        raise np.linalg.LinAlgError(f'{(~inverse_valid).sum()} sampled singular matrices; no defined ODE at those points')
    IL = np.linalg.inv(L)
    ID = np.linalg.inv(Dic)
    A_rhs = IL @ (-Rp + Om @ Dic)
    A_cons = ID @ (-Rp + Om @ Dic)
    A_map = IL @ (-Rp + Om @ L)

    def rh3(A):
        tr = np.trace(A, axis1=1, axis2=2)
        a1 = -tr
        a2 = 0.5 * (tr * tr - np.trace(A @ A, axis1=1, axis2=2))
        a3 = -np.linalg.det(A)
        delta = a1 * a2 - a3
        yes = (a1 > 0) & (a2 > 0) & (a3 > 0) & (delta > 0)
        return (yes, a1, a2, a3, delta)
    rh_rhs, rh_a1, rh_a2, rh_a3, rh_delta = rh3(A_rhs)
    rh_cons, *_ = rh3(A_cons)
    alpha_rhs = np.linalg.eigvals(A_rhs).real.max(axis=1)
    alpha_cons = np.linalg.eigvals(A_cons).real.max(axis=1)
    alpha_map = np.linalg.eigvals(A_map).real.max(axis=1)
    damping_D = np.linalg.eigvalsh(symmetric_part(Rp @ ID))[:, 0]
    damping_L = np.linalg.eigvalsh(symmetric_part(Rp @ IL))[:, 0]
    rotation = np.linalg.eigvalsh(symmetric_part(J @ Dic @ IL))[:, -1]
    beta = -damping_L + we * rotation
    beta_cons = -damping_D
    beta_map = -damping_L
    tol = 1e-07 * np.maximum(1.0, np.abs(alpha_rhs))
    if np.any(alpha_rhs > beta + tol) or np.any(alpha_cons > beta_cons + tol) or np.any(alpha_map > beta_map + tol):
        raise AssertionError('Logarithmic-norm eigenvalue bound violated: check matrix conventions')
    curl_01 = GL[:, :, 1, 0] - GL[:, :, 0, 1]
    curl_02 = GL[:, :, 2, 0] - GL[:, :, 0, 2]
    curl_12 = GL[:, :, 2, 1] - GL[:, :, 1, 2]
    curl = np.sqrt(np.sum(curl_01 ** 2 + curl_02 ** 2 + curl_12 ** 2, axis=1))
    mismatch = np.linalg.norm(L - Dic, axis=(1, 2)) / np.maximum(np.linalg.norm(Dic, axis=(1, 2)), 1e-14)
    rows = pd.DataFrame({'id_A': X[:, 0], 'iq_A': X[:, 1], 'if_A': X[:, 2], 'n_rpm': speed, 'alpha_deployed_sinv': alpha_rhs, 'alpha_consistent_sinv': alpha_cons, 'alpha_frozen_map_sinv': alpha_map, 'structured_bound_sinv': beta, 'consistent_bound_sinv': beta_cons, 'frozen_map_bound_sinv': beta_map, 'rotation_term_sinv': we * rotation, 'damping_L_sinv': damping_L, 'damping_D_sinv': damping_D, 'operator_mismatch_rel_fro': mismatch, 'fd_integrability_curl_fro': curl, 'svmin_L_FD': svL, 'svmin_D_IC': svD, 'rhs_nonhurwitz': alpha_rhs >= 0, 'consistent_nonhurwitz': alpha_cons >= 0, 'rh_rhs_hurwitz': rh_rhs, 'rh_cons_hurwitz': rh_cons, 'rh_a1': rh_a1, 'rh_a2': rh_a2, 'rh_a3': rh_a3, 'rh_delta': rh_delta, 'consistent_pd_certificate': damping_D > 0, 'deployed_structural_hurwitz_certificate': beta < 0, 'frozen_map_pd_certificate': damping_L > 0, 'deployed_certificate_unresolved': beta >= 0})
    if np.any(rh_rhs != (alpha_rhs < 0)) or np.any(rh_cons != (alpha_cons < 0)):
        raise AssertionError('Routh-Hurwitz coefficients disagree with independent eigensolver')
    return rows

def summarize_pointwise_stability(frame):
    rows = []
    for scope, g in [('all', frame)] + [(f'{int(speed)}rpm', gg) for speed, gg in frame.groupby('n_rpm')]:
        rr = {'scope': scope, 'n': len(g), 'deployed_nonhurwitz_fraction': g.rhs_nonhurwitz.mean(), 'consistent_nonhurwitz_fraction': g.consistent_nonhurwitz.mean(), 'consistent_pd_cert_fraction': g.consistent_pd_certificate.mean(), 'deployed_structural_cert_fraction': g.deployed_structural_hurwitz_certificate.mean(), 'uncertified_but_hurwitz_count': int((g.deployed_certificate_unresolved & ~g.rhs_nonhurwitz).sum()), 'min_sv_L_FD': g.svmin_L_FD.min(), 'min_sv_D_IC': g.svmin_D_IC.min(), 'median_curl_L_FD': g.fd_integrability_curl_fro.median(), 'median_operator_mismatch': g.operator_mismatch_rel_fro.median(), 'rho_mismatch_vs_alpha': g.operator_mismatch_rel_fro.corr(g.alpha_deployed_sinv, method='spearman'), 'rho_structured_bound_vs_alpha': g.structured_bound_sinv.corr(g.alpha_deployed_sinv, method='spearman'), 'max_alpha_minus_upper_bound': float((g.alpha_deployed_sinv - g.structured_bound_sinv).max()), 'routh_hurwitz_rhs_agreement': float((g.rh_rhs_hurwitz == ~g.rhs_nonhurwitz).mean())}
        rows.append(rr)
    return pd.DataFrame(rows)

def evaluate_cell_certificates(parent):
    parents = parent[['id_cell', 'q_cell', 'if_cell']].drop_duplicates().reset_index(drop=True)
    va = [np.asarray(D[k], float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    out = []
    for start in range(0, len(parents), CELL_CERTIFICATION_BATCH):
        b = parents.iloc[start:start + CELL_CERTIFICATION_BATCH]
        idx = b[['id_cell', 'q_cell', 'if_cell']].to_numpy(int)
        xlow = np.stack([va[d][idx[:, d]] for d in range(3)], axis=1)
        xhi = np.stack([va[d][idx[:, d] + 1] for d in range(3)], axis=1)
        xc = 0.5 * (xlow + xhi)
        _, Lc, Dc = cellwise_flux_inductance(xc)
        corners = np.concatenate([np.stack([xhi[:, d] if bit[d] else xlow[:, d] for d in range(3)], axis=1) for bit in product((0, 1), repeat=3)], axis=0)
        _, LL, DD = cellwise_flux_inductance(corners)
        LL = LL.reshape(8, len(b), 3, 3).transpose(1, 0, 2, 3)
        DD = DD.reshape(8, len(b), 3, 3).transpose(1, 0, 2, 3)
        flux_grid = np.stack([D['psid'], D['psiq'], D['psif']], axis=-1)
        one_cell_D = []
        for bit in product((0, 1), repeat=3):
            bi, bq, bf = bit
            ii = idx[:, 0]
            qq = idx[:, 1]
            kk = idx[:, 2]
            dd = (flux_grid[qq + bq, ii + 1, kk + bf] - flux_grid[qq + bq, ii, kk + bf]) / (xhi[:, 0] - xlow[:, 0])[:, None]
            dq = (flux_grid[qq + 1, ii + bi, kk + bf] - flux_grid[qq, ii + bi, kk + bf]) / (xhi[:, 1] - xlow[:, 1])[:, None]
            df = (flux_grid[qq + bq, ii + bi, kk + 1] - flux_grid[qq + bq, ii + bi, kk]) / (xhi[:, 2] - xlow[:, 2])[:, None]
            one_cell_D.append(np.stack([dd, dq, df], axis=2))
        DD = np.stack(one_cell_D, axis=1)
        rL = np.linalg.svd(LL - Lc[:, None], compute_uv=False)[:, :, 0].max(axis=1)
        rD = np.linalg.svd(DD - Dc[:, None], compute_uv=False)[:, :, 0].max(axis=1)
        sL = np.linalg.svd(Lc, compute_uv=False)[:, -1]
        sD = np.linalg.svd(Dc, compute_uv=False)[:, -1]
        tmp = b.copy()
        tmp['L_cell_min_sv_lower_bound'] = sL - rL
        tmp['D_cell_min_sv_lower_bound'] = sD - rD
        tmp['L_cell_certified_nonsingular'] = sL - rL > 1e-10
        tmp['D_cell_certified_nonsingular'] = sD - rD > 1e-10
        tmp['D_cell_unresolved'] = ~tmp['D_cell_certified_nonsingular']
        tmp['L_cell_unresolved'] = ~tmp['L_cell_certified_nonsingular']
        out.append(tmp)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def evaluate_bernstein_certificates(parent):
    aa = [np.asarray(D[k], float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    frac = np.linspace(0.0, 1.0, 4)
    from math import comb
    BB = np.array([[comb(3, j) * v ** j * (1 - v) ** (3 - j) for j in range(4)] for v in frac])
    ib = np.linalg.inv(BB)
    ff = np.array(list(product(frac, repeat=3)), float)
    frames = []
    for start in range(0, len(parent), 32):
        b = parent.iloc[start:start + 32]
        ix = b[['id_cell', 'q_cell', 'if_cell']].to_numpy(int)
        lo = np.stack([aa[d][ix[:, d]] for d in range(3)], axis=1)
        hi = np.stack([aa[d][ix[:, d] + 1] for d in range(3)], axis=1)
        xp = lo[:, None, :] + (hi - lo)[:, None, :] * ff[None, :, :]
        repeated = np.repeat(ix, len(ff), axis=0)
        _, LL, DD = cellwise_flux_inductance(xp.reshape(-1, 3), force_cells=repeated)
        detL = np.linalg.det(LL).reshape(len(b), 4, 4, 4)
        detD = np.linalg.det(DD).reshape(len(b), 4, 4, 4)
        coL = np.einsum('ai,bj,ck,nijk->nabc', ib, ib, ib, detL, optimize=True)
        coD = np.einsum('ai,bj,ck,nijk->nabc', ib, ib, ib, detD, optimize=True)
        z = b.copy()
        for label, cf in [('L', coL), ('D', coD)]:
            c = cf.reshape(len(b), -1)
            mn = c.min(axis=1)
            mx = c.max(axis=1)
            eps = 1e-08 * np.max(np.abs(c), axis=1)
            pos = mn > eps
            neg = mx < -eps
            z[f'{label}_bernstein_det_sign_certified'] = pos | neg
            z[f'{label}_bernstein_det_sign'] = np.where(pos, 1, np.where(neg, -1, 0))
            z[f'{label}_bernstein_min_coef'] = mn
            z[f'{label}_bernstein_max_coef'] = mx
        frames.append(z)
    return pd.concat(frames, ignore_index=True)

def evaluate_cell_position_sensitivity(centers, nparents):
    sel = centers[['id_cell', 'q_cell', 'if_cell', 'n_rpm']].drop_duplicates().sort_values(['n_rpm', 'if_cell', 'q_cell', 'id_cell'])
    selected = []
    for speed, group in sel.groupby('n_rpm'):
        ix = np.linspace(0, len(group) - 1, min(nparents, len(group)), dtype=int)
        selected.append(group.iloc[np.unique(ix)])
    parents = pd.concat(selected, ignore_index=True)
    coords = [np.asarray(D[k], float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    xlist = []
    nlist = []
    met = []
    Rlim2 = 3 * D['ismax'] ** 2
    for b in parents.itertuples(index=False):
        idx = (int(b.id_cell), int(b.q_cell), int(b.if_cell))
        for ff in product((0.05, 0.5, 0.95), repeat=3):
            x = [coords[d][idx[d]] + ff[d] * (coords[d][idx[d] + 1] - coords[d][idx[d]]) for d in range(3)]
            if x[0] ** 2 + x[1] ** 2 > Rlim2:
                continue
            xlist.append(x)
            nlist.append(b.n_rpm)
            met.append({'id_cell': idx[0], 'q_cell': idx[1], 'if_cell': idx[2], 'within_cell_min_fraction': round(min(min(ff), 1 - max(ff)), 4), 'within_cell_center': all((abs(v - 0.5) < 1e-10 for v in ff)), 'near_any_face': any((v != 0.5 for v in ff))})
    f = evaluate_stability_metrics(np.array(xlist), np.array(nlist))
    for k, v in pd.DataFrame(met).items():
        f[k] = v.to_numpy()
    return f

def evaluate_interface_directions(parent, max_n=600):
    a = [np.asarray(D[k], float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    parent = parent.sort_values(['n_rpm', 'if_cell', 'q_cell', 'id_cell'])
    if len(parent) > max_n:
        parent = parent.iloc[np.linspace(0, len(parent) - 1, max_n, dtype=int)]
    Rs = np.diag([D['Rs'], D['Rs'], D['Rf']])
    J = np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 0]], float)
    records = []
    for b in parent.itertuples(index=False):
        ix = (int(b.id_cell), int(b.q_cell), int(b.if_cell))
        lo = np.array([a[d][ix[d]] for d in range(3)])
        hi = np.array([a[d][ix[d] + 1] for d in range(3)])
        xc = (lo + hi) / 2
        we = 2 * np.pi * D['p'] * float(b.n_rpm) / 60
        Om = we * J
        psibase = cellwise_flux_inductance(xc)[0][0]
        ueq = Rs @ xc - Om @ psibase
        for axis in range(3):
            face = xc.copy()
            face[axis] = hi[axis]
            if ix[axis] + 1 >= len(a[axis]) - 1:
                continue
            xp = face.copy()
            xm = face.copy()
            delta = 1e-07 * (hi[axis] - lo[axis])
            xp[axis] += delta
            xm[axis] -= delta
            if face[0] ** 2 + face[1] ** 2 > 3 * D['ismax'] ** 2:
                continue
            pp, ll, dp = cellwise_flux_inductance(np.array([xm, xp]))
            u = ueq - Rs @ face + Om @ pp[0]
            try:
                vm = np.linalg.solve(dp[0], u)
                vp = np.linalg.solve(dp[1], u)
            except np.linalg.LinAlgError:
                continue
            ln, rn = (vm[axis], vp[axis])
            label = 'attracting' if ln > 0 and rn < 0 else 'repelling' if ln < 0 and rn > 0 else 'same_direction_or_tangent'
            records.append({'n_rpm': float(b.n_rpm), 'id_cell': ix[0], 'q_cell': ix[1], 'if_cell': ix[2], 'axis': STATE_COORDINATES[axis], 'left_normal_speed': ln, 'right_normal_speed': rn, 'one_sided_normal_discontinuity': abs(rn - ln), 'direction_class': label})
    return pd.DataFrame(records)
if not (RUN_OFFGRID_JACOBIAN_AUDIT and len(offgrid_exact_df)):
    raise RuntimeError('Run the knot and cell-interior Jacobian analysis before this section.')
STRUCTURAL_INPUT = offgrid_exact_df.copy()
if not set(STATE_COORDINATES).issubset(STRUCTURAL_INPUT.columns):
    raise KeyError('Expected offgrid center columns are missing')
STRUCTURAL_INPUT['id_cell'] = np.searchsorted(D['id_vec'], STRUCTURAL_INPUT.id_A, side='right') - 1
STRUCTURAL_INPUT['q_cell'] = np.searchsorted(D['iq_vec'], STRUCTURAL_INPUT.iq_A, side='right') - 1
STRUCTURAL_INPUT['if_cell'] = np.searchsorted(D['if_vec'], STRUCTURAL_INPUT.if_A, side='right') - 1
X_struct = STRUCTURAL_INPUT[list(STATE_COORDINATES)].to_numpy(dtype=float)
N_struct = STRUCTURAL_INPUT.n_rpm.to_numpy(dtype=float)
structural_points = evaluate_stability_metrics(X_struct, N_struct)
structural_points['center_source_row'] = np.arange(len(structural_points))
structural_summary = summarize_pointwise_stability(structural_points)
structural_points.to_csv(TAB / 'table04structural_structural_pointwise.csv', index=False)
structural_summary.to_csv(TAB / 'table04structural_structural_by_speed.csv', index=False)
if 'Aexact' in globals() and len(Aexact) == len(structural_points):
    alpha_original = np.linalg.eigvals(np.asarray(Aexact, dtype=float)).real.max(axis=1)
    max_alpha_err = float(np.max(np.abs(alpha_original - structural_points.alpha_deployed_sinv.to_numpy())))
    if not np.allclose(alpha_original, structural_points.alpha_deployed_sinv.to_numpy(), rtol=0.0002, atol=5e-05):
        raise AssertionError('Exact-Jacobian results disagree with the baseline computation')
if CERTIFY_ALL_CELLS:
    cmeta = []
    for q in range(len(D['iq_vec']) - 1):
        for i in range(len(D['id_vec']) - 1):
            for k in range(len(D['if_vec']) - 1):
                x0 = 0.5 * (D['id_vec'][i] + D['id_vec'][i + 1])
                y0 = 0.5 * (D['iq_vec'][q] + D['iq_vec'][q + 1])
                if x0 * x0 + y0 * y0 <= 3 * D['ismax'] ** 2:
                    cmeta.append((i, q, k))
    parent_cells = pd.DataFrame(cmeta, columns=['id_cell', 'q_cell', 'if_cell'])
else:
    parent_cells = STRUCTURAL_INPUT[['id_cell', 'q_cell', 'if_cell']].drop_duplicates()
cell_certificates = evaluate_cell_certificates(parent_cells)
determinant_certificates = evaluate_bernstein_certificates(parent_cells)
cell_certificates = cell_certificates.merge(determinant_certificates, on=['id_cell', 'q_cell', 'if_cell'], validate='one_to_one')
cell_certificates['L_cell_any_sufficient_nonsingular'] = cell_certificates.L_cell_certified_nonsingular | cell_certificates.L_bernstein_det_sign_certified
cell_certificates['D_cell_any_sufficient_nonsingular'] = cell_certificates.D_cell_certified_nonsingular | cell_certificates.D_bernstein_det_sign_certified
cell_certificates.to_csv(TAB / 'table04structural_cell_inverse_certificates.csv', index=False)
cell_position_results = evaluate_cell_position_sensitivity(STRUCTURAL_INPUT, CELL_POSITION_SAMPLE_COUNT)
cell_position_results.to_csv(TAB / 'table04structural_27point_spatial_audit.csv', index=False)
cell_position_results['within_cell_min_fraction'] = cell_position_results['within_cell_min_fraction'].round(3)
cell_position_summary = cell_position_results.groupby(['n_rpm', 'within_cell_min_fraction'], dropna=False).agg(n=('rhs_nonhurwitz', 'size'), nonhurwitz_fraction=('rhs_nonhurwitz', 'mean'), median_mismatch=('operator_mismatch_rel_fro', 'median'), structural_certification_fraction=('deployed_structural_hurwitz_certificate', 'mean')).reset_index()
cell_position_summary.to_csv(TAB / 'table04structural_spatial_summary.csv', index=False)
interface_directions = evaluate_interface_directions(STRUCTURAL_INPUT[['id_cell', 'q_cell', 'if_cell', 'n_rpm']], INTERFACE_SAMPLE_COUNT)
interface_directions.to_csv(TAB / 'table04structural_interface_direction_screen.csv', index=False)
psi_struct = cellwise_flux_inductance(X_struct)[0]
we = 2 * np.pi * D['p'] * N_struct / 60
u5 = np.column_stack([D['Rs'] * X_struct[:, 0] - we * psi_struct[:, 1], D['Rs'] * X_struct[:, 1] + we * psi_struct[:, 0], D['Rf'] * X_struct[:, 2]])
voltage_dq = np.linalg.norm(u5[:, :2], axis=1)
structural_points['equilibrium_stator_voltage_V'] = voltage_dq
structural_points['conditional_SPWM'] = voltage_dq <= D['udc'] * np.sqrt(3 / 8)
structural_points['conditional_SVPWM'] = voltage_dq <= D['udc'] / np.sqrt(2)
voltage_strata_rows = []
for label, mask in [('all', np.ones(len(structural_points), bool)), ('SPWM_valid', structural_points.conditional_SPWM.to_numpy(bool)), ('SPWM_excluded', ~structural_points.conditional_SPWM.to_numpy(bool)), ('SVPWM_valid', structural_points.conditional_SVPWM.to_numpy(bool)), ('SVPWM_excluded', ~structural_points.conditional_SVPWM.to_numpy(bool))]:
    sub = structural_points[mask]
    if not len(sub):
        continue
    voltage_strata_rows.append({'subset': label, 'n': len(sub), 'nonhurwitz_fraction': sub.rhs_nonhurwitz.mean(), 'operator_mismatch_median': sub.operator_mismatch_rel_fro.median(), 'structured_hurwitz_certificate_fraction': sub.deployed_structural_hurwitz_certificate.mean(), 'minimum_sv_L_FD': sub.svmin_L_FD.min()})
voltage_strata = pd.DataFrame(voltage_strata_rows)
voltage_strata.to_csv(TAB / 'table04structural_voltage_mismatch_strata.csv', index=False)
structural_points.to_csv(TAB / 'table04structural_structural_pointwise.csv', index=False)
certification_status = {'finite_center_points_checked': len(structural_points), 'finite_parent_cells_tested': len(cell_certificates), 'pointwise_D_positive_definite_certificate_count': int(structural_points.consistent_pd_certificate.sum()), 'pointwise_deployed_structural_certificate_count': int(structural_points.deployed_structural_hurwitz_certificate.sum()), 'L_entire_cell_certified_nonsingular_count': int(cell_certificates.L_cell_any_sufficient_nonsingular.sum()), 'D_entire_cell_certified_nonsingular_count': int(cell_certificates.D_cell_any_sufficient_nonsingular.sum()), 'unresolved_cell_L_count': int((~cell_certificates.L_cell_any_sufficient_nonsingular).sum()), 'unresolved_cell_D_count': int((~cell_certificates.D_cell_any_sufficient_nonsingular).sum()), 'interface_screens_evaluated': len(interface_directions), 'interface_attracting_screen_count': int((interface_directions.direction_class == 'attracting').sum()) if len(interface_directions) else 0, 'intracell_27point_screens': len(cell_position_results), 'methodological_limitations': ['Sufficient spectral bounds; beta >= 0 is INCONCLUSIVE, not unstable.', 'Positive-definiteness checked at finite points only; no universal theorem unless certified on domain.', 'Floating Bernstein checks are numerical screening; the separate exact dyadic integer Bernstein calculations establish stored-data cellwide signs.', 'CURL measures compatibility within a cell; integrability may fail across interfaces and global potential is NOT asserted.', 'One-sided interface direction screen is heuristic and is NOT an existence/uniqueness proof.', 'Finite-sample conditional percentages are not geometric prevalence measures.', 'Full nonlinear ODE, solver, and K-step contraction are not established from local eigenvalue tests.']}
(OUT / 'structural_certification_status.json').write_text(json.dumps(certification_status, indent=2), encoding='utf-8')
import scipy, sys, platform
independent_checks = {'finite_sample_exact_routh_hurwitz_agreement': bool((structural_points.rh_rhs_hurwitz == ~structural_points.rhs_nonhurwitz).all()), 'finite_sample_eigenvalue_bound_violations': int((structural_points.alpha_deployed_sinv > structural_points.structured_bound_sinv + 1e-07).sum()), 'finite_sample_nonzero_curl_count_at_1e_minus_10': int((structural_points.fd_integrability_curl_fro > 1e-10).sum()), 'pointwise_hurwitz_rh_count': int(structural_points.rh_rhs_hurwitz.sum()), 'pointwise_hurwitz_spectral_count': int((~structural_points.rhs_nonhurwitz).sum()), 'center_admissible_cells_L_numeric_nonsingular_certified': bool(cell_certificates.L_cell_any_sufficient_nonsingular.all()), 'center_admissible_cells_D_numeric_nonsingular_certified': bool(cell_certificates.D_cell_any_sufficient_nonsingular.all()), 'matrix_inverse_cell_evidence_is_interval_proof': False, 'global_injectivity_or_global_ODE_uniqueness_proved': False, 'structural_sufficient_hurwitz_test_never_certifies': bool((~structural_points.deployed_structural_hurwitz_certificate).all()), 'offgrid_center_count': len(structural_points), 'distinct_current_valid_parent_cells': len(cell_certificates), 'software': {'python': sys.version.split()[0], 'numpy': np.__version__, 'scipy': scipy.__version__, 'pandas': pd.__version__}, 'sources': {'dataset': 'Zenodo 10.5281/zenodo.18822007', 'primary_code': 'FEA-informed EESM stability analysis notebook'}, 'limitations': ['Routh-Hurwitz is applied to finite evaluation points, not proven uniformly across all states.', 'Floating-point Bernstein signs are screening only; exact dyadic certificate is computed separately.', 'Interface directional tests are necessary research screens, not uniqueness theorems.', 'No neural models retrained; earlier trained results are reused only with their original qualifiers.']}
(OUT / 'structural_validation_checks.json').write_text(json.dumps(independent_checks, indent=2))
for m in ('L', 'D'):
    cf = cell_certificates
    largest = np.maximum(np.abs(cf[f'{m}_bernstein_min_coef']), np.abs(cf[f'{m}_bernstein_max_coef']))
    relative = cf[f'{m}_bernstein_min_coef'] / largest
    independent_checks[f'{m}_min_bernstein_positive_relative_margin'] = float(np.nanmin(relative))
(OUT / 'structural_validation_checks.json').write_text(json.dumps(independent_checks, indent=2))
import math, itertools, time, datetime
from collections import Counter
cell_verification_start = time.time()
BITS = list(itertools.product((0, 1), repeat=3))
BITIDS = [b[0] * 16 + b[1] * 4 + b[2] for b in BITS]

def dyadic_integer_representation(a):
    a = np.asarray(a, dtype=np.float64)
    powers = []
    for v in a.flat:
        if not np.isfinite(v):
            raise ValueError('Non-finite nodal FEA coefficient')
        de = float(v).as_integer_ratio()[1]
        powers.append(de.bit_length() - 1)
    exponent = max(powers)
    scale = 1 << exponent
    vals = np.asarray([int(float(v).as_integer_ratio()[0]) << exponent - (float(v).as_integer_ratio()[1].bit_length() - 1) for v in a.flat], dtype=object).reshape(a.shape)
    return (vals, exponent)

def determinant_bernstein_numerators(corners):
    out = [0] * 64
    for b, c in enumerate(BITS):
        u = corners[b][1]
        for d, e in enumerate(BITS):
            v = corners[d][2]
            cross = (u[1] * v[2] - u[2] * v[1], u[2] * v[0] - u[0] * v[2], u[0] * v[1] - u[1] * v[0])
            for f, g in enumerate(BITS):
                w = corners[f][0]
                value = w[0] * cross[0] + w[1] * cross[1] + w[2] * cross[2]
                ks = (c[0] + e[0] + g[0], c[1] + e[1] + g[1], c[2] + e[2] + g[2])
                out[ks[0] * 16 + ks[1] * 4 + ks[2]] += value
    return out

def exact_coefficient_sign(arr):
    all_positive = all((v > 0 for v in arr))
    all_negative = all((v < 0 for v in arr))
    return int(all_positive) - int(all_negative)

def current_limit_intersecting_cells(data):
    idaxis, iqaxis, ifaxis = (np.asarray(data[k], float) for k in ('id_vec', 'iq_vec', 'if_vec'))
    limit2 = 3 * float(data['ismax']) ** 2
    rows = []
    for i in range(len(idaxis) - 1):
        lx, hx = idaxis[i:i + 2]
        dx = 0 if lx <= 0 <= hx else min(abs(lx), abs(hx))
        for q in range(len(iqaxis) - 1):
            ly, hy = iqaxis[q:q + 2]
            dy = 0 if ly <= 0 <= hy else min(abs(ly), abs(hy))
            if dx * dx + dy * dy <= limit2:
                center_valid = ((lx + hx) * 0.5) ** 2 + ((ly + hy) * 0.5) ** 2 <= limit2
                for k in range(len(ifaxis) - 1):
                    rows.append((i, q, k, center_valid))
    return pd.DataFrame(rows, columns=['id_cell', 'q_cell', 'if_cell', 'center_current_valid'])

def certify_current_limited_cells():
    if not np.allclose(np.diff(D['id_vec']), 50) or not np.allclose(np.diff(D['iq_vec']), 50) or (not np.allclose(np.diff(D['if_vec']), 2)):
        raise RuntimeError('This exact integer proof assumes 50-A stator and 2-A field grid increments; generalize common denominators for different data.')
    gpsi, epsi = dyadic_integer_representation(np.stack([D['psid'], D['psiq'], D['psif']], axis=-1))
    gl, el = dyadic_integer_representation(L_edge2)
    cells = current_limit_intersecting_cells(D)
    results = []
    for z, (i, q, k, center_valid) in enumerate(cells.itertuples(index=False, name=None)):
        ps = {bit: tuple((int(x) for x in gpsi[q + bit[1], i + bit[0], k + bit[2]])) for bit in BITS}
        cl = []
        cd = []
        for bi, bq, bf in BITS:
            bit = (bi, bq, bf)
            raw_l = gl[q + bq, i + bi, k + bf]
            cl.append([[int(raw_l[r, c]) for c in range(3)] for r in range(3)])
            dcols = []
            for r in range(3):
                dd = ps[1, bq, bf][r] - ps[0, bq, bf][r]
                dq = ps[bi, 1, bf][r] - ps[bi, 0, bf][r]
                df = 25 * (ps[bi, bq, 1][r] - ps[bi, bq, 0][r])
                dcols.append([dd, dq, df])
            cd.append(dcols)
        co_l = determinant_bernstein_numerators(cl)
        co_d = determinant_bernstein_numerators(cd)
        sl = exact_coefficient_sign(co_l)
        sd = exact_coefficient_sign(co_d)
        results.append((i, q, k, bool(center_valid), sl, sd, len([a for a in co_l if a == 0]), len([a for a in co_d if a == 0])))
    report = pd.DataFrame(results, columns=['id_cell', 'q_cell', 'if_cell', 'center_current_valid', 'L_exact_bernstein_det_sign', 'D_exact_bernstein_det_sign', 'L_zero_bernstein_numerators', 'D_zero_bernstein_numerators'])
    report['L_exactly_proven_nonsingular'] = report.L_exact_bernstein_det_sign.ne(0)
    report['D_exactly_proven_nonsingular'] = report.D_exact_bernstein_det_sign.ne(0)
    report.to_csv(TAB / 'table_v7_exact_bernstein_cell_proof.csv', index=False)
    return (report, el, epsi)
exact_cell_certificates, L_dyadic_exponent, psi_dyadic_exponent = certify_current_limited_cells()
assert determinant_bernstein_numerators([[[1, 0, 0], [0, 1, 0], [0, 0, 1]] for _ in BITS])
assert exact_coefficient_sign(determinant_bernstein_numerators([[[1, 0, 0], [0, 1, 0], [0, 0, 1]] for _ in BITS])) == 1
assert exact_coefficient_sign(determinant_bernstein_numerators([[[-1, 0, 0], [0, 1, 0], [0, 0, 1]] for _ in BITS])) == -1
assert exact_coefficient_sign(determinant_bernstein_numerators([[[0, 0, 0], [0, 1, 0], [0, 0, 1]] for _ in BITS])) == 0

def evaluate_adjacent_cell_interfaces(max_per_cell=9):
    grids = [np.asarray(D[k], float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    cc = exact_cell_certificates
    parents = cc[cc.center_current_valid].sort_values(['if_cell', 'q_cell', 'id_cell'])
    speed_values = np.asarray(D['n_vec'], float)
    stator_lim2 = 3 * D['ismax'] ** 2
    facepairs = []
    for row in parents.itertuples(index=False):
        a = np.array([row.id_cell, row.q_cell, row.if_cell], dtype=int)
        for axis in range(3):
            b = a.copy()
            b[axis] += 1
            if b[axis] >= len(grids[axis]) - 1:
                continue
            lo = np.array([grids[d][a[d]] for d in range(3)])
            hi = np.array([grids[d][a[d] + 1] for d in range(3)])
            xc = (lo + hi) / 2
            for uv in itertools.product((0.05, 0.5, 0.95), repeat=2):
                ff = [0.5, 0.5, 0.5]
                free = [x for x in range(3) if x != axis]
                ff[free[0]] = uv[0]
                ff[free[1]] = uv[1]
                ff[axis] = 1
                face = lo + (hi - lo) * ff
                if face[0] * face[0] + face[1] * face[1] > stator_lim2:
                    continue
                facepairs.append((a.copy(), b.copy(), axis, xc, face))
    if not facepairs:
        return pd.DataFrame()
    fr = np.array([r[4] for r in facepairs])
    cen = np.array([r[3] for r in facepairs])
    axes = np.array([r[2] for r in facepairs])
    N = len(fr)
    idxmin = np.array([r[0] for r in facepairs])
    idxplus = np.array([r[1] for r in facepairs])
    R = np.diag([D['Rs'], D['Rs'], D['Rf']])
    J = np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 0]], float)
    rec = []
    for st in range(0, N, 3000):
        ed = min(st + 3000, N)
        ff = fr[st:ed]
        ccen = cen[st:ed]
        fluxf, _, dminus = cellwise_flux_inductance(ff, force_cells=idxmin[st:ed])
        fluxp, _, dplus = cellwise_flux_inductance(ff, force_cells=idxplus[st:ed])
        if not np.allclose(fluxf, fluxp, rtol=1e-09, atol=1e-09):
            raise AssertionError('Flux not continuous across interface')
        fluxc = cellwise_flux_inductance(ccen)[0]
        for n in speed_values:
            we = 2 * np.pi * D['p'] * n / 60
            F = (ccen - ff) @ R.T + we * (fluxf - fluxc) @ J.T
            vm = np.linalg.solve(dminus, F[..., None])[..., 0]
            vp = np.linalg.solve(dplus, F[..., None])[..., 0]
            m = vm[np.arange(len(ff)), axes[st:ed]]
            p = vp[np.arange(len(ff)), axes[st:ed]]
            eps = 1e-08 * np.maximum(1, np.maximum(abs(m), abs(p)))
            typ = np.where((m > eps) & (p < -eps), 'attracting', np.where((m < -eps) & (p > eps), 'repelling', np.where((np.abs(m) <= eps) | (np.abs(p) <= eps), 'tangent', 'same_direction')))
            for j in range(len(ff)):
                rec.append((n, *idxmin[st + j].tolist(), int(axes[st + j]), m[j], p[j], typ[j], *ff[j].tolist()))
    result = pd.DataFrame(rec, columns=['n_rpm', 'id_cell', 'q_cell', 'if_cell', 'interface_axis', 'left_normal_velocity', 'right_normal_velocity', 'direction', 'id_A', 'iq_A', 'if_A'])
    result.to_csv(TAB / 'table_v7_exhaustive_interface_directions.csv', index=False)
    return result
interface_check_results = evaluate_adjacent_cell_interfaces()

def evaluate_nonlinear_vector_field(X, n_rpm, u_eq, mode='deployed'):
    X = np.asarray(X, dtype=float).reshape(-1, 3)
    ps, L, Di = cellwise_flux_inductance(X)
    Rp = np.array([D['Rs'], D['Rs'], D['Rf']], float)
    Om = 2 * np.pi * D['p'] * n_rpm / 60 * np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 0]], float)
    F = np.asarray(u_eq, float)[None, :] - X * Rp[None, :] + ps @ Om.T
    return np.linalg.solve(L if mode == 'deployed' else Di, F[..., None])[..., 0]

def nonlinear_rk4_step(x, n, u, h, mode):
    try:
        k1 = evaluate_nonlinear_vector_field(x, n, u, mode)
        k2 = evaluate_nonlinear_vector_field(x + 0.5 * h * k1, n, u, mode)
        k3 = evaluate_nonlinear_vector_field(x + 0.5 * h * k2, n, u, mode)
        k4 = evaluate_nonlinear_vector_field(x + h * k3, n, u, mode)
        y = x + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return y
    except (ValueError, np.linalg.LinAlgError):
        raise

def evaluate_trajectory_exposure(nbases=12, period_fraction=0.1):
    rng = np.random.default_rng(31075)
    frame = structural_points
    selected = []
    for n, g in frame.groupby('n_rpm'):
        for flag, sub in g.groupby('rhs_nonhurwitz'):
            if len(sub):
                selected.extend(sub.iloc[np.linspace(0, len(sub) - 1, min(nbases // 5, len(sub)), dtype=int)].index.tolist())
    rec = []
    for idx in selected:
        row = frame.loc[idx]
        base = np.array([row.id_A, row.iq_A, row.if_A], float)
        n = float(row.n_rpm)
        ome = 2 * np.pi * D['p'] * n / 60
        ps = cellwise_flux_inductance(base)[0][0]
        Rp = np.array([D['Rs'], D['Rs'], D['Rf']])
        Omega = ome * np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 0]], float)
        u = Rp * base - Omega @ ps
        for perturb in [np.array([1.0, -1.0, 0.01]), np.array([-1.0, 1.0, -0.01])]:
            init = base + perturb
            if not (D['id_vec'][0] <= init[0] <= D['id_vec'][-1] and D['iq_vec'][0] <= init[1] <= D['iq_vec'][-1] and (D['if_vec'][0] < init[2] < D['if_vec'][-1]) and (init[0] ** 2 + init[1] ** 2 <= 3 * D['ismax'] ** 2)):
                continue
            T = period_fraction * 60 / (D['p'] * n)
            for mode in ['deployed', 'consistent']:
                pairs = []
                for h_div in (1, 2):
                    h = min(2.5e-05 / h_div, T / 8)
                    steps = int(np.ceil(T / h))
                    h = T / steps
                    x = init[None, :].copy()
                    status = 'completed'
                    try:
                        for _ in range(steps):
                            x = nonlinear_rk4_step(x, n, u, h, mode)
                            if x[0, 0] ** 2 + x[0, 1] ** 2 > 3 * D['ismax'] ** 2 or not D['if_vec'][0] <= x[0, 2] <= D['if_vec'][-1]:
                                status = 'outside_operating_domain'
                                break
                    except (ValueError, np.linalg.LinAlgError):
                        status = 'outside_table_or_singular'
                    pairs.append((x[0].copy(), status, int(steps)))
                if all((p[1] == 'completed' for p in pairs)):
                    err = float(np.linalg.norm((pairs[0][0] - pairs[1][0]) / np.array([30.0, 30.0, 0.35])))
                    gain = float(np.linalg.norm((pairs[1][0] - base) / np.array([30.0, 30.0, 0.35])) / np.linalg.norm((init - base) / np.array([30.0, 30.0, 0.35])))
                else:
                    err = np.nan
                    gain = np.nan
                rec.append({'n_rpm': n, 'base_index': idx, 'rhs_nonhurwitz': bool(row.rhs_nonhurwitz), 'representation': mode, 'run_status_coarse': pairs[0][1], 'run_status_fine': pairs[1][1], 'end_state_difference_RK4_h_vs_h2_normalized': err, 'perturbation_gain': gain, 'target_duration_s': T, 'initial_id_A': init[0], 'initial_iq_A': init[1], 'initial_if_A': init[2]})
    out = pd.DataFrame(rec)
    out.to_csv(TAB / 'table_v7_nonlinear_exposure.csv', index=False)
    return out
nonlinear_trajectory_results = evaluate_trajectory_exposure()
normalization_matrix = np.diag([30.0, 30.0, 0.35])
inverse_normalization_matrix = np.linalg.inv(normalization_matrix)
Xr = structural_points[['id_A', 'iq_A', 'if_A']].to_numpy()
ns = structural_points.n_rpm.to_numpy()
psi, Lf, Df = cellwise_flux_inductance(Xr)
Rmat = np.diag([D['Rs'], D['Rs'], D['Rf']])
Jmat = np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 0]], float)
ww = 2 * np.pi * D['p'] * ns / 60
Am = np.linalg.solve(Lf, -Rmat[None, :, :] + ww[:, None, None] * (Jmat @ Df))
Ae = np.eye(3)[None, :, :] + 5e-05 * (inverse_normalization_matrix @ Am @ normalization_matrix)
full_euler_sigma = np.linalg.svd(Ae, compute_uv=False)[:, 0]
contraction_test = {'fixed_step_s': 5e-05, 'n_points': int(len(full_euler_sigma)), 'local_Euler_induced_norm_min': float(full_euler_sigma.min()), 'local_Euler_induced_norm_max': float(full_euler_sigma.max()), 'points_proving_no_uniform_Euclidean_contraction_at_50us': int((full_euler_sigma >= 1).sum()), 'global_contraction_proved': False}
(TAB / 'table_v7_pointwise_Euler_induced_norm.csv').write_text('induced_2norm\n' + '\n'.join(map(str, full_euler_sigma)) + '\n')
domain_verification_summary = {'source_dataset_doi': '10.5281/zenodo.18822007', 'exact_dyadic_ieee754_bernstein_certificate': True, 'exact_proof_excludes_original_fea_uncertainty': True, 'cells_intersecting_current_limit': int(len(exact_cell_certificates)), 'L_nonsingular_exactly_proven_cells': int(exact_cell_certificates.L_exactly_proven_nonsingular.sum()), 'D_nonsingular_exactly_proven_cells': int(exact_cell_certificates.D_exactly_proven_nonsingular.sum()), 'L_unresolved_cells': int((~exact_cell_certificates.L_exactly_proven_nonsingular).sum()), 'D_unresolved_cells': int((~exact_cell_certificates.D_exactly_proven_nonsingular).sum()), 'dyadic_L_2_power_denominator': int(L_dyadic_exponent), 'dyadic_psi_2_power_denominator': int(psi_dyadic_exponent), 'interface_point_speed_checks': len(interface_check_results), 'interface_attracting_checks': int((interface_check_results.direction == 'attracting').sum()), 'interface_repelling_checks': int((interface_check_results.direction == 'repelling').sum()), 'nonlinear_trajectory_checks': len(nonlinear_trajectory_results), 'complete_nonlinear_trajectory_checks': int((nonlinear_trajectory_results.run_status_fine == 'completed').sum()), 'classical_local_contraction_counterexample': contraction_test, 'global_solution_existence_uniqueness_proved': False, 'local_flux_coordinate_wellposedness_under_clarke_assumptions': True, 'complete_continuous_domain_hurwitz_proved': False, 'full_K_step_uniform_contraction_proved': False, 'integrability_of_L_FD_proved': False, 'conclusion': 'Exact cell invertibility may be proved from stored nodal data; it alone does not establish global one-to-one flux map, global ODE uniqueness, or physical machine stability.'}
(OUT / 'domain_verification_summary.json').write_text(json.dumps(domain_verification_summary, ensure_ascii=False, indent=2))

def clarke_column_separation_test():
    axes = [np.asarray(D[k], float) for k in ('id_vec', 'iq_vec', 'if_vec')]
    nodes = []
    for i in range(len(axes[0])):
        for q in range(len(axes[1])):
            if axes[0][i] ** 2 + axes[1][q] ** 2 > 3 * D['ismax'] ** 2:
                continue
            for k in range(len(axes[2])):
                nodes.append((i, q, k))
    max_column_disagreement = 0.0
    convex_status = []
    for idx in nodes:
        boxes = list(itertools.product(*[[c for c in (j - 1, j) if 0 <= c < len(axes[d]) - 1] for d, j in enumerate(idx)]))
        X = np.tile(np.array([axes[d][idx[d]] for d in range(3)], float), (len(boxes), 1))
        mat = cellwise_flux_inductance(X, force_cells=np.array(boxes, dtype=int))[2]
        for j in range(3):
            for s in (idx[j] - 1, idx[j]):
                group = [p for p, b in enumerate(boxes) if b[j] == s]
                if len(group) > 1:
                    deviation = np.max(np.abs(mat[group, :, j] - mat[group[0], :, j]))
                    max_column_disagreement = max(max_column_disagreement, float(deviation))
        dets = np.linalg.det(mat)
        convex_status.append(bool(np.all(dets > 0)))
    return {'n_grid_junctions_checked': len(nodes), 'max_one_sided_column_dependency_error': max_column_disagreement, 'all_grid_junction_incident_determinants_positive_numerically': bool(all(convex_status))}
clarke_check_results = clarke_column_separation_test()
assert clarke_check_results['max_one_sided_column_dependency_error'] < 1e-12, 'Column-separability assumption failed'
assert exact_cell_certificates.D_exact_bernstein_det_sign.eq(1).all(), 'Clarke determinant orientation hypothesis failed'
assert clarke_check_results['all_grid_junction_incident_determinants_positive_numerically']
wellposedness_conditions = {'source': 'stored FEA nodal values interpreted as exact dyadic real data', 'assumptions': 'continuous piecewise-trilinear flux maps on aligned rectangular grid; constant rotational speed; continuous external input; trajectories restricted to interior of supplied current-limit and FEA table', 'all_intersecting_cells_det_D_strictly_positive_by_exact_bernstein': True, 'one_sided_columns_separable_at_grid_junctions_verified': clarke_check_results, 'clarke_generalized_jacobians_nonsingular_inside_domain_under_column_separability_and_exact_cell_sign_assumptions': True, 'psi_has_local_bilipschitz_inverse_inside_domain': True, 'unique_local_flux_coordinate_ODE_solution_under_stated_assumptions_until_domain_exit': True, 'all_time_global_solutions_or_physical_machine_stability_proved': False, 'statement': 'The branchwise current ODE is well-posed via unique flux-coordinate dynamics locally, while inside the open tabulated/current-limited domain, under continuous time input and fixed speed. The result does not justify sampled knot-matrix Hurwitz as nonlinear equilibrium stability, and does not extend beyond the FEA domain.'}
(OUT / 'wellposedness_conditions.json').write_text(json.dumps(wellposedness_conditions, indent=2), encoding='utf-8')
domain_verification_summary.update({'clarke_local_inverse_wellposedness_proved_under_stated_assumptions': True, 'proof_scope': 'local uniqueness until exit, continuous u(t), fixed speed, open tabulated current-limited domain'})
(OUT / 'domain_verification_summary.json').write_text(json.dumps(domain_verification_summary, indent=2), encoding='utf-8')
wellposedness_conditions['deployed_current_ODE_locally_unique_by_Picard_Lindelof'] = True
(OUT / 'wellposedness_conditions.json').write_text(json.dumps(wellposedness_conditions, indent=2), encoding='utf-8')
import matplotlib.pyplot as plt
figout = OUT / 'figures_v7'
figout.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
a = structural_points
ax.scatter(a['structured_bound_sinv'], a['alpha_deployed_sinv'], s=3, alpha=0.28, rasterized=True)
ax.axhline(0, ls='--', lw=1)
ax.set(xlabel='Sufficient upper bound on spectral abscissa (s$^{-1}$)', ylabel='Computed spectral abscissa (s$^{-1}$)', title='Structured bound versus local spectral abscissa')
fig.tight_layout()
plt.close(fig)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.hist(full_euler_sigma, bins=45)
ax.axvline(1, ls='--', lw=1)
ax.set(xlabel='Normalized equilibrium-local Euler Jacobian induced 2-norm', ylabel='Number of sampled equilibria', title='Local contraction counterexample, h = 50 μs')
fig.tight_layout()
plt.close(fig)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(structural_summary.loc[structural_summary.scope != 'all', 'scope'], structural_summary.loc[structural_summary.scope != 'all', 'deployed_nonhurwitz_fraction'] * 100, 'o-')
ax.set(xlabel='Mechanical speed', ylabel='Non-Hurwitz share of finite sample (%)', title='Speed-stratified representation-dependent classification')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.close(fig)
domain_verification_summary['generated_figures'] = sorted((p.name for p in figout.glob('*.png')))
(OUT / 'domain_verification_summary.json').write_text(json.dumps(domain_verification_summary, indent=2), encoding='utf-8')


## Structural and voltage strata

In [ ]:
from scipy.stats import rankdata
if len(structural_points) < 30 or len(cell_position_results) < 30:
    raise RuntimeError('Run the representation and structural analyses before this section.')

def _roc_auc_binary(y, score):
    y = np.asarray(y, dtype=bool)
    x = np.asarray(score, dtype=float)
    ok = np.isfinite(x)
    y, x = (y[ok], x[ok])
    pos = int(y.sum())
    neg = len(y) - pos
    if pos == 0 or neg == 0:
        return float('nan')
    r = rankdata(x, method='average')
    return float((r[y].sum() - pos * (pos + 1) / 2) / (pos * neg))
screening_points = structural_points.copy()
screening_points['id_cell'] = np.searchsorted(D['id_vec'], screening_points.id_A, side='right') - 1
screening_points['q_cell'] = np.searchsorted(D['iq_vec'], screening_points.iq_A, side='right') - 1
screening_points['if_cell'] = np.searchsorted(D['if_vec'], screening_points.if_A, side='right') - 1
screening_points['spatial_cell_key'] = list(zip(screening_points.id_cell, screening_points.q_cell, screening_points.if_cell))
screening_cols = {'structured_bound': 'structured_bound_sinv', 'scalar_operator_mismatch': 'operator_mismatch_rel_fro', 'mechanical_speed': 'n_rpm'}
audit_rows = []
for k, g in [('all', screening_points)] + [(f'{int(n)}rpm', x) for n, x in screening_points.groupby('n_rpm')]:
    for label, col in screening_cols.items():
        audit_rows.append({'stratum': k, 'score': label, 'n': int(len(g)), 'nonhurwitz_n': int(g.rhs_nonhurwitz.sum()), 'roc_auc_nonhurwitz': _roc_auc_binary(g.rhs_nonhurwitz, g[col]), 'spearman_vs_spectral_abscissa': float(g[col].corr(g.alpha_deployed_sinv, method='spearman')) if g[col].nunique() > 1 else float('nan'), 'mathematical_hurwitz_certificate': bool(label == 'structured_bound'), 'certificate_count': int(g.deployed_structural_hurwitz_certificate.sum()) if label == 'structured_bound' else 0})
structural_screening = pd.DataFrame(audit_rows)
structural_screening.to_csv(TAB / 'diagnostic_structural_roc_auc_by_speed.csv', index=False)
rng = np.random.default_rng(int(SEED) + 22026)
bootstrap_draws = 1000 if FULL_RUN else 100
cluster_ids = pd.factorize(screening_points['spatial_cell_key'].astype(str))[0]
clusters = [np.flatnonzero(cluster_ids == cid) for cid in np.unique(cluster_ids)]
bootstrap_rows = []
for b in range(bootstrap_draws):
    chosen = rng.integers(len(clusters), size=len(clusters))
    sample_idx = np.concatenate([clusters[k] for k in chosen])
    g = screening_points.iloc[sample_idx]
    for label, col in screening_cols.items():
        if label == 'mechanical_speed':
            continue
        bootstrap_rows.append((b, label, _roc_auc_binary(g.rhs_nonhurwitz, g[col])))
bootstrap_raw = pd.DataFrame(bootstrap_rows, columns=['bootstrap', 'score', 'roc_auc'])
bootstrap_raw.to_csv(TAB / 'diagnostic_cluster_bootstrap_auc_replicates.csv', index=False)
structural_auc_ci = bootstrap_raw.groupby('score').roc_auc.agg(n_resamples='count', p025=lambda x: float(np.nanpercentile(x, 2.5)), median='median', p975=lambda x: float(np.nanpercentile(x, 97.5))).reset_index()
structural_auc_ci.to_csv(TAB / 'diagnostic_cluster_bootstrap_auc_ci.csv', index=False)
strata_rows = []
for (speed, x), g in screening_points.groupby(['n_rpm', 'conditional_SPWM']):
    strata_rows.append({'modulation': 'SPWM', 'n_rpm': speed, 'voltage_feasible': bool(x), 'n': int(len(g)), 'nonhurwitz_n': int(g.rhs_nonhurwitz.sum()), 'nonhurwitz_fraction': float(g.rhs_nonhurwitz.mean()), 'median_relative_mismatch': float(g.operator_mismatch_rel_fro.median()), 'median_spectral_abscissa_sinv': float(g.alpha_deployed_sinv.median())})
for (speed, x), g in screening_points.groupby(['n_rpm', 'conditional_SVPWM']):
    strata_rows.append({'modulation': 'SVPWM', 'n_rpm': speed, 'voltage_feasible': bool(x), 'n': int(len(g)), 'nonhurwitz_n': int(g.rhs_nonhurwitz.sum()), 'nonhurwitz_fraction': float(g.rhs_nonhurwitz.mean()), 'median_relative_mismatch': float(g.operator_mismatch_rel_fro.median()), 'median_spectral_abscissa_sinv': float(g.alpha_deployed_sinv.median())})
for mod, flag in [('SPWM', 'conditional_SPWM'), ('SVPWM', 'conditional_SVPWM')]:
    for status, g in screening_points.groupby(flag):
        strata_rows.append({'modulation': mod, 'n_rpm': 'all', 'voltage_feasible': bool(status), 'n': int(len(g)), 'nonhurwitz_n': int(g.rhs_nonhurwitz.sum()), 'nonhurwitz_fraction': float(g.rhs_nonhurwitz.mean()), 'median_relative_mismatch': float(g.operator_mismatch_rel_fro.median()), 'median_spectral_abscissa_sinv': float(g.alpha_deployed_sinv.median())})
voltage_feasibility_by_speed = pd.DataFrame(strata_rows)
voltage_feasibility_by_speed.to_csv(TAB / 'diagnostic_voltage_feasibility_speed_strata.csv', index=False)
key = ['n_rpm', 'id_cell', 'q_cell', 'if_cell']
rs = cell_position_results.copy()
center = rs.loc[rs.within_cell_center.astype(bool)].groupby(key).agg(center_nonhurwitz=('rhs_nonhurwitz', 'mean'), center_mismatch=('operator_mismatch_rel_fro', 'mean')).reset_index()
near = rs.loc[np.isclose(rs.within_cell_min_fraction, 0.05)].groupby(key).agg(near_n=('rhs_nonhurwitz', 'size'), near_nonhurwitz_fraction=('rhs_nonhurwitz', 'mean'), near_mismatch_mean=('operator_mismatch_rel_fro', 'mean')).reset_index()
paired = center.merge(near, on=key, validate='one_to_one', how='inner')
paired['paired_nonhurwitz_difference'] = paired.near_nonhurwitz_fraction - paired.center_nonhurwitz
paired['paired_mismatch_difference'] = paired.near_mismatch_mean - paired.center_mismatch
paired.to_csv(TAB / 'diagnostic_paired_center_near_face_by_cell_speed.csv', index=False)
paired_summary = []
for speed, g in [('all', paired)] + [(int(n), x) for n, x in paired.groupby('n_rpm')]:
    paired_summary.append({'n_rpm': speed, 'paired_cell_speed_groups': len(g), 'center_nonhurwitz_equal_cell_fraction': float(g.center_nonhurwitz.mean()), 'near_face_nonhurwitz_equal_cell_fraction': float(g.near_nonhurwitz_fraction.mean()), 'near_minus_center_fraction': float(g.paired_nonhurwitz_difference.mean()), 'center_mismatch_equal_cell_mean': float(g.center_mismatch.mean()), 'near_face_mismatch_equal_cell_mean': float(g.near_mismatch_mean.mean())})
paired_position_summary = pd.DataFrame(paired_summary)
paired_position_summary.to_csv(TAB / 'diagnostic_paired_spatial_by_speed.csv', index=False)
paired_ci = []
for _, g in [('all', paired)] + [(f'{int(n)}rpm', x) for n, x in paired.groupby('n_rpm')]:
    values = g.paired_nonhurwitz_difference.to_numpy(dtype=float)
    bs = np.mean(values[rng.integers(len(values), size=(bootstrap_draws, len(values)))], axis=1)
    paired_ci.append({'stratum': 'all' if len(g) == len(paired) else f'{int(g.n_rpm.iloc[0])}rpm', 'n_paired': len(g), 'difference_mean': float(values.mean()), 'bootstrap_p025': float(np.quantile(bs, 0.025)), 'bootstrap_p975': float(np.quantile(bs, 0.975))})
pd.DataFrame(paired_ci).to_csv(TAB / 'diagnostic_paired_spatial_bootstrap_ci.csv', index=False)
bench_rows = []
for op in test_ops:
    entry = ops[int(op)]
    A = np.asarray(entry['A'], dtype=float)
    for h in (2e-05, 5e-05):
        Z = h * A
        I3 = np.eye(3)
        E = expm(Z)
        transitions = {'Euler': I3 + Z, 'Heun': I3 + Z + 0.5 * (Z @ Z), 'RK4': I3 + Z + 0.5 * (Z @ Z) + Z @ Z @ Z / 6 + Z @ Z @ Z @ Z / 24, 'Backward Euler': np.linalg.solve(I3 - Z, I3), 'Trapezoidal': np.linalg.solve(I3 - 0.5 * Z, I3 + 0.5 * Z), 'Exact exponential': E}
        for method, matrix in transitions.items():
            bench_rows.append({'op_index': int(op), 'n_rpm': float(entry['n_rpm']), 'step_us': 1000000.0 * h, 'method': method, 'spectral_radius': float(max(abs(np.linalg.eigvals(matrix)))), 'stable_linear': bool(max(abs(np.linalg.eigvals(matrix))) < 1), 'relative_matrix_error': float(np.linalg.norm(matrix - E, 'fro') / np.linalg.norm(E, 'fro')), 'comparison_set': 'same_guarded_test_set_as_table16'})
solver_step_comparison = pd.DataFrame(bench_rows)
solver_step_comparison.to_csv(TAB / 'diagnostic_matched_guarded_20_50us_solver_benchmark.csv', index=False)
solver_step_comparison_summary = solver_step_comparison.groupby(['step_us', 'method']).agg(n=('op_index', 'size'), stable_fraction=('stable_linear', 'mean'), median_relative_matrix_error=('relative_matrix_error', 'median'), p95_relative_matrix_error=('relative_matrix_error', lambda s: float(np.percentile(s, 95)))).reset_index()
solver_step_comparison_summary.to_csv(TAB / 'diagnostic_matched_guarded_20_50us_solver_summary.csv', index=False)
old_euler = linbench.loc[linbench.method == 'Euler', ['op_index', 'spectral_radius']].set_index('op_index')
new_euler = solver_step_comparison.loc[(solver_step_comparison.method == 'Euler') & np.isclose(solver_step_comparison.step_us, 50), ['op_index', 'spectral_radius']].set_index('op_index')
assert np.allclose(old_euler.sort_index().spectral_radius.to_numpy(), new_euler.sort_index().spectral_radius.to_numpy(), rtol=1e-12, atol=1e-12)
assert (screening_points.alpha_deployed_sinv <= screening_points.structured_bound_sinv + 1e-06).all()
assert (screening_points.rh_rhs_hurwitz == ~screening_points.rhs_nonhurwitz).all()
assert screening_points.conditional_SPWM.le(screening_points.conditional_SVPWM).all()
fig, ax = plt.subplots(figsize=(6.4, 4.1))
for modulation, symbol in [('SPWM', 'o-'), ('SVPWM', 's--')]:
    subset = voltage_feasibility_by_speed[(voltage_feasibility_by_speed.modulation == modulation) & (voltage_feasibility_by_speed.n_rpm != 'all')]
    for feasible, suffix in [(True, 'within mask'), (False, 'outside mask')]:
        g = subset[subset.voltage_feasible == feasible].copy()
        if len(g):
            ax.plot(g.n_rpm.astype(float), 100 * g.nonhurwitz_fraction, symbol, label=modulation + ' ' + suffix)
ax.set(xlabel='Mechanical speed (rpm)', ylabel='Non-Hurwitz fraction in finite sample (%)')
ax.legend(fontsize=8)
fig.tight_layout()
plt.close(fig)
fig, ax = plt.subplots(figsize=(6.4, 4.1))
for label, s in structural_screening[structural_screening.stratum != 'all'].groupby('score'):
    ax.plot(s.stratum.str.replace('rpm', '').astype(int), s.roc_auc_nonhurwitz, marker='o', label=label)
ax.set(xlabel='Mechanical speed (rpm)', ylabel='ROC-AUC for non-Hurwitz classification')
ax.legend(fontsize=8)
fig.tight_layout()
plt.close(fig)
diagnostic_analysis_manifest = {'pointwise_count': int(len(screening_points)), 'current_limited_cell_count': int(len(exact_cell_certificates)), 'rigorous_data_assumption': 'stored IEEE-754 FEA numbers interpreted as exact dyadic constants', 'clarke_interface_assumption': 'tensor-product trilinear flux map; local interior inverse only', 'full_operating_domain_hurwitz_certified': False, 'spwm_svpwm_masks_are_switching_level_guarantees': False, 'roc_auc_is_sufficient_stability_certificate': False, 'comparison_set': 'original test_ops, not grid-knot atlas', 'results': sorted([f.name for f in TAB.glob('diagnostic_*.csv')]), 'full_run': bool(FULL_RUN)}
(OUT / 'diagnostic_analysis_manifest.json').write_text(json.dumps(diagnostic_analysis_manifest, ensure_ascii=False, indent=2), encoding='utf-8')


## Scientific consistency checks

Evaluate internal numerical invariants and record the observed metrics. These checks do not require an external reference CSV and do not modify calculated results.


In [ ]:
from datetime import datetime, timezone

REPORT_DIR = BASE_DIR / (RESULTS_OUT.name + '_Validation')
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def _read_analysis_csv(name):
    path = TAB / name
    return (pd.read_csv(path), name) if path.is_file() else (None, name)


def _result_at(df, column, selector=None):
    if df is None or column not in df.columns:
        return None
    subset = df
    for key, value in (selector or {}).items():
        if key not in subset.columns:
            return None
        if isinstance(value, (int, float)):
            subset = subset[np.isclose(pd.to_numeric(subset[key], errors='coerce'), value)]
        else:
            subset = subset[subset[key].astype(str).str.casefold().eq(str(value).casefold())]
    if len(subset) != 1:
        return None
    number = pd.to_numeric(pd.Series([subset.iloc[0][column]]), errors='coerce').iloc[0]
    return float(number) if pd.notna(number) else None


def collected_metrics():
    # Measure values without imposing reference target numbers.
    readings = []
    def put(claim, source, value):
        readings.append({'claim': claim, 'source': source, 'observed': value})
    df, name = _read_analysis_csv('table04i_offgrid_exact_within_cell_summary.csv')
    put('Off-grid sample n', name, _result_at(df, 'points', {'scope': 'all'}))
    put('Deployed non-Hurwitz count (off-grid)', name, _result_at(df, 'rhs_exact_nonhurwitz_points', {'scope': 'all'}))
    put('Consistent sample Hurwitz share', name, _result_at(df, 'consistent_baseline_hurwitz_fraction_on_available', {'scope': 'all'}))
    df, name = _read_analysis_csv('table04q6_exhaustive_intracell_summary.csv')
    for label, col in [('Admissible parent cells','admissible_parent_cells'),
                       ('Finite-design retained points','retained_design_points_inside_current_limit'),
                       ('Deployed non-Hurwitz finite design','deployed_nonhurwitz_points'),
                       ('Consistent non-Hurwitz finite design','consistent_nonhurwitz_points')]:
        put(label, name, _result_at(df, col))
    df, name = _read_analysis_csv('table_v7_exact_bernstein_cell_proof.csv')
    put('Exact Bernstein examined cells', name, len(df) if df is not None else None)
    if df is not None and {'L_exact_bernstein_det_sign','D_exact_bernstein_det_sign'}.issubset(df):
        put('L_FD exact coefficient positivity', name, int(df.L_exact_bernstein_det_sign.eq(1).sum()))
        put('D_IC exact coefficient positivity', name, int(df.D_exact_bernstein_det_sign.eq(1).sum()))
    df, name = _read_analysis_csv('table_v7_exhaustive_interface_directions.csv')
    put('Sampled face-speed tests', name, len(df) if df is not None else None)
    if df is not None and 'direction' in df:
        put('Attracting face-speed samples', name, int(df.direction.eq('attracting').sum()))
        put('Repelling face-speed samples', name, int(df.direction.eq('repelling').sum()))
    df, name = _read_analysis_csv('table_v7_nonlinear_exposure.csv')
    put('Matched short-horizon trajectories', name, len(df) if df is not None else None)
    if df is not None and 'run_status_fine' in df:
        put('Completed fine RK4 checks', name, int(df.run_status_fine.eq('completed').sum()))
    df, name = _read_analysis_csv('table11g_common_strict_summary.csv')
    for label, col in [('Strict domain one-period paired cases','points'),
                       ('Strict subset Heun median RMSE','median_heun_rmse'),
                       ('Strict subset RK4 median RMSE','median_rk4_rmse')]:
        put(label, name, _result_at(df, col, {'speed':'Overall'}))
    df, name = _read_analysis_csv('diagnostic_structural_roc_auc_by_speed.csv')
    for label, score in [('Structural-score ROC-AUC','structured_bound'),
                         ('Scalar mismatch ROC-AUC','scalar_operator_mismatch')]:
        put(label, name, _result_at(df, 'roc_auc_nonhurwitz', {'stratum':'all','score':score}))
    df, name = _read_analysis_csv('diagnostic_paired_spatial_by_speed.csv')
    put('Paired spatial cell-speed groups', name, _result_at(df,'paired_cell_speed_groups',{'n_rpm':'all'}))
    put('Paired near-minus-center non-Hurwitz fraction', name,
        _result_at(df,'near_minus_center_fraction',{'n_rpm':'all'}))
    df, name = _read_analysis_csv('diagnostic_matched_guarded_20_50us_solver_summary.csv')
    for label, step, col in [('Matched 20 us Euler tested points',20,'n'),
                             ('Matched 20 us Euler stable fraction',20,'stable_fraction'),
                             ('Matched 50 us Euler stable fraction',50,'stable_fraction')]:
        put(label,name,_result_at(df,col,{'step_us':step,'method':'Euler'}))
    paired, name = _read_analysis_csv('table04t_derivative_consistent_trajectory_paired.csv')
    if paired is not None:
        comp = paired_nonlinear_table(paired, require_full=FULL_RUN)
        for rec in comp.itertuples(index=False):
            label = rec[0]
            if label in ('Euler','Heun','RK4'):
                for suffix, value in [('scenario count',rec[1]),('strict pairs',rec[2]),
                                      ('non-Hurwitz strict',rec[4]),('reference difference',rec[5])]:
                    put(label+' '+suffix,name, float(value))
    return pd.DataFrame(readings)


def scientific_validation(result_dir):
    # Validate scientific invariants independently of reported reference values.
    checks=[]
    def record(name, passed, evidence):
        checks.append({'check':name,'status':'PASS' if passed else 'FAIL','details':evidence})
    def required_csv(name, columns):
        data, _ = _read_analysis_csv(name)
        if data is None:
            record('Required analysis: '+name,False,'CSV missing')
            return None
        missing = sorted(set(columns)-set(data.columns))
        good = not missing and len(data)>0
        record('Required analysis: '+name,good,'rows='+str(len(data))+'; missing='+str(missing))
        return data if good else None
    off=required_csv('table04i_offgrid_exact_within_cell_summary.csv',
                     ('scope','points','rhs_exact_nonhurwitz_points'))
    if off is not None:
        a=off[off.scope.astype(str).eq('all')]
        good=(len(a)==1 and 0<=float(a.iloc[0].rhs_exact_nonhurwitz_points)<=float(a.iloc[0].points))
        record('Off-grid incidence bounds',good,'non-Hurwitz count between 0 and sample count')
    intra=required_csv('table04q6_exhaustive_intracell_summary.csv',
                       ('retained_design_points_inside_current_limit','deployed_nonhurwitz_points'))
    if intra is not None and len(intra)==1:
        q=intra.iloc[0];record('Interior incidence bounds',
                0<=float(q.deployed_nonhurwitz_points)<=float(q.retained_design_points_inside_current_limit),
                'finite-design count within retained locations')
    exact=required_csv('table_v7_exact_bernstein_cell_proof.csv',
                       ('L_exact_bernstein_det_sign','D_exact_bernstein_det_sign'))
    if exact is not None:
        for col in ('L_exact_bernstein_det_sign','D_exact_bernstein_det_sign'):
            passed=bool(exact[col].notna().all() and exact[col].eq(1).all())
            record('Exact Bernstein positivity: '+col,passed,
                   f'{int(exact[col].eq(1).sum())}/{len(exact)} positive')
    structure=required_csv('table04structural_structural_pointwise.csv',
                          ('alpha_deployed_sinv','structured_bound_sinv','rhs_nonhurwitz'))
    if structure is not None:
        a=pd.to_numeric(structure.alpha_deployed_sinv,errors='coerce').to_numpy(dtype=float)
        b=pd.to_numeric(structure.structured_bound_sinv,errors='coerce').to_numpy(dtype=float)
        finite=bool(np.isfinite(a).all() and np.isfinite(b).all())
        record('Finite structural eigenvalue/bound values',finite,f'{len(structure)} rows')
        if finite:
            violations=int(np.count_nonzero(a>b+1e-8*np.maximum(1,np.abs(b))))
            record('Structured spectral upper bound',violations==0,f'{violations} violations')
    report=pd.DataFrame(checks)
    result_dir=Path(result_dir);result_dir.mkdir(parents=True,exist_ok=True)
    report.to_csv(result_dir/'scientific_validation.csv',index=False)
    status='PASS' if bool((report.status=='PASS').all()) else 'FAIL'
    (result_dir/'scientific_validation.json').write_text(
        json.dumps({'status':status,'checks':checks},indent=2,ensure_ascii=False),encoding='utf-8')
    return {'status':status,'checked':len(checks),'passed':int(report.status.eq('PASS').sum()),
            'failures':report[report.status=='FAIL'].check.tolist()}


def _notebook_identity():
    specified = os.environ.get('EESM_NOTEBOOK_PATH', '').strip()
    if specified:
        candidate = Path(specified).expanduser()
        notebook_path = candidate if candidate.is_file() else None
        identity_note = ('Using EESM_NOTEBOOK_PATH.' if notebook_path is not None else
                         'EESM_NOTEBOOK_PATH does not refer to an accessible file.')
    else:
        # Avoid recording the hash of an unrelated notebook when several copies exist.
        available = {
            str(path.resolve()): path
            for directory in (Path.cwd(), BASE_DIR)
            for path in directory.glob('EESM_LUT*.ipynb')
            if path.is_file()
        }
        notebook_path = next(iter(available.values())) if len(available) == 1 else None
        identity_note = ('Unique local EESM_LUT notebook located automatically.'
                         if notebook_path is not None else
                         'No unique local notebook found; set EESM_NOTEBOOK_PATH for a SHA-256 record.')
    sha = hashlib.sha256(notebook_path.read_bytes()).hexdigest() if notebook_path else None
    filename = notebook_path.name if notebook_path else None
    commit = None
    if notebook_path is not None:
        try:
            tracked = subprocess.run(
                ['git', '-C', str(notebook_path.parent), 'ls-files', '--error-unmatch', notebook_path.name],
                capture_output=True, text=True, timeout=3, check=False)
            if tracked.returncode == 0:
                git = subprocess.run(
                    ['git', '-C', str(notebook_path.parent), 'rev-parse', 'HEAD'],
                    capture_output=True, text=True, timeout=3, check=False)
                commit = git.stdout.strip() if git.returncode == 0 else None
        except (OSError, subprocess.TimeoutExpired):
            pass
    return {'code_version': CODE_VERSION, 'notebook_sha256': sha,
            'notebook_file': filename, 'git_commit': commit,
            'identity_note': identity_note}


metrics=collected_metrics()
metrics.to_csv(REPORT_DIR/'observed_metrics.csv',index=False)
scientific_report=scientific_validation(REPORT_DIR)

run_manifest={'created_utc':datetime.now(timezone.utc).isoformat(),
              'code':_notebook_identity(),
              'input_sha256':INPUT_CHECKSUMS,
              'seed':int(MASTER_SEED),
              'full_run':bool(FULL_RUN),
              'python':platform.python_version(),
              'platform':platform.platform(),
              'device':str(DEVICE),
              'torch_deterministic_algorithms':bool(torch.are_deterministic_algorithms_enabled()),
              'torch_determinism_warn_only':bool(torch.is_deterministic_algorithms_warn_only_enabled())
                     if hasattr(torch,'is_deterministic_algorithms_warn_only_enabled') else 'not_available',
              'libraries':{'numpy':np.__version__,'scipy':scipy.__version__,
                           'pandas':pd.__version__,'torch':torch.__version__,
                           'optuna':optuna.__version__},
              'settings':{'dt_seconds':float(DT),'state_scale':STATE_SCALE.tolist(),
                          'primary_seeds':int(PRIMARY_SEEDS),
                          'bootstrap_replicates':int(CROSSED_BOOTSTRAP_B),
                          'bootstrap_rng_repeats':int(BOOTSTRAP_RNG_REPEATS),
                          'figure_dpi':int(FIGURE_DPI)},
              'scientific_validation':scientific_report,
              'limitations':['Exact Bernstein positivity is conditional on stored numeric coefficients.',
                             'Local Hurwitz and solver screening do not establish physical/global stability.',
                             'In-sample ROC-AUC is not a prospective classifier.']}
(REPORT_DIR/'run_manifest.json').write_text(json.dumps(run_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Scientific checks:',scientific_report['status'],scientific_report['passed'],'/',scientific_report['checked'])
print('Audit reports:',REPORT_DIR)
if scientific_report['status']!='PASS':
    raise RuntimeError('Scientific validation failed; inspect '+str(REPORT_DIR/'scientific_validation.csv'))

## Selected figure and table outputs

With `FULL_RUN=True`, the results ZIP contains **Figures 2–6 and Figures S1–S3** (8 PNG files at 600 dpi) and **Tables 1–9 and Tables S1–S2** (11 CSV files) under `Results/Figures` and `Results/Tables`. Figure 1 is intentionally excluded. Two supporting CSV files are stored separately under `Results/Diagnostics`. All values are obtained from the preceding analysis; the outputs are not populated from a reference document. CPU timings use independent one-step evaluations from admissible fixed inputs and are platform dependent.


In [ ]:
"""Export analysis tables, figures, and result summaries."""


def export_results(work, destination, D, settings):
    work, destination = Path(work), Path(destination)
    if not settings.get('FULL_RUN', True):
        raise RuntimeError('Result export requires FULL_RUN=True.')
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError('The result output directory must be empty.')
    destination.mkdir(parents=True, exist_ok=True)
    selected = destination / 'Results'
    for folder in (selected / 'Figures', selected / 'Tables', selected / 'Diagnostics'):
        folder.mkdir(parents=True, exist_ok=True)
    def read(name, *alternates):
        for f in (name,)+alternates:
            path=work/'tables'/f
            if path.is_file():
                return pd.read_csv(path)
        raise FileNotFoundError('Required analysis result missing: '+name)
    def row(frame, **where):
        q=frame
        for column,value in where.items():
            q=q[np.isclose(q[column].astype(float),value)] if isinstance(value,(int,float)) else q[q[column].astype(str)==str(value)]
        if len(q)!=1:
            raise ValueError('Expected one result row: '+str(where))
        return q.iloc[0]
    def one(name):
        q=read(name)
        if len(q)!=1:
            raise ValueError('Expected a single summary: '+name)
        return q.iloc[0]
    def table(df, name):
        path=selected/'Tables'/name
        df.to_csv(path,index=False)
        return path
    def finish(fig,name):
        fig.tight_layout()
        payload=io.BytesIO()
        fig.savefig(payload,bbox_inches='tight',format='png',dpi=int(settings['FIGURE_DPI']))
        path=selected/'Figures'/name
        temporary=path.with_suffix('.tmp')
        with temporary.open('wb') as target:
            target.write(payload.getvalue())
            target.flush()
            os.fsync(target.fileno())
        temporary.replace(path)
        plt.close(fig)
    def panels(axes):
        for label,ax in zip('abcd',np.asarray(axes).ravel()):
            ax.text(-.10,1.04,'('+label+')',transform=ax.transAxes,
                    fontsize=11,fontweight='bold',va='bottom')
    def diagonal(ax, x, y):
        lo=min(np.min(x),np.min(y)); hi=max(np.max(x),np.max(y))
        ax.plot([lo,hi],[lo,hi],'k--',lw=.8)
        ax.set_xscale('log');ax.set_yscale('log')
    def coverage_plot(ax,coverage):
        for speed,g in coverage.groupby('n_rpm'):
            ax.plot(g.step_us,100*g.stable_fraction,marker='o',ms=3,label=f'{int(speed)} rpm')
        ax.set_xscale('log');ax.set_ylim(-2,102)
        ax.set(xlabel='Fixed step (µs)',ylabel='Locally stable grid-knot points (%)')
        ax.grid(which='both',alpha=.2);ax.legend(fontsize=7)

    dataset=one('table01_dataset_audit.csv')
    partition=read('table05a_spatial_partition_audit.csv')
    distances=one('table05b_spatial_partition_distances.csv')
    counts=dict(zip(partition.partition,partition.operating_points.astype(int)))
    offgrid=read('table04h_offgrid_exact_within_cell_jacobian_all.csv')
    structural=read('table04structural_structural_pointwise.csv','table04v5_structural_pointwise.csv')
    exact=read('table_v7_exact_bernstein_cell_proof.csv')
    interfaces=read('table_v7_exhaustive_interface_directions.csv')
    nonlinear=read('table_v7_nonlinear_exposure.csv')
    norms=read('table_v7_pointwise_Euler_induced_norm.csv')
    stability=read('table02_stability_atlas.csv')
    domains=read('table03e_voltage_feasible_stability_summary.csv')
    screening=read('diagnostic_structural_roc_auc_by_speed.csv')
    auc_ci=read('diagnostic_cluster_bootstrap_auc_ci.csv')
    masks=read('diagnostic_voltage_feasibility_speed_strata.csv')
    benchmark=read('diagnostic_matched_guarded_20_50us_solver_summary.csv')
    common=read('table11g_common_strict_summary.csv')
    phys=read('table11m_common_strict_physical_component_rmse.csv')
    map_summary=one('table15b_full_map_jacobian_summary.csv')
    finite=one('table15e_finite_time_jacobian_product_summary.csv')
    finite_strict=one('table15f_finite_time_jacobian_product_strict_domain_summary.csv')
    scale=np.asarray(settings['STATE_SCALE'],float)
    dt=float(settings['DT'])
    speeds=np.asarray(D['n_vec'],float)
    periods=60/(float(D['p'])*speeds)*1000
    table1=pd.DataFrame([
        ['Pole pairs',int(dataset.pole_pairs),'Grid points/field level',int(dataset.nominal_id_iq_grid_points)],
        ['DC-link voltage',f'{dataset.dc_link_V:g} V','Atlas Jacobians',int(dataset.local_models)],
        ['Stator resistance, Rs',f'{1000*dataset.Rs_ohm:g} mΩ','Neural operating points',sum(counts.values())],
        ['Field resistance, Rf',f'{dataset.Rf_ohm:g} Ω','Train / val. / test',f'{counts["train"]} / {counts["validation"]} / {counts["test"]}'],
        ['Max. phase RMS current',f'{dataset.phase_current_limit_rms_A:g} A','Guard points',counts['guard']],
        ['Max. field current',f'{dataset.field_current_max_A:g} A','Lattice/guard',f'{settings["SPATIAL_NBINS"]}×{settings["SPATIAL_NBINS"]} / {settings["SPATIAL_GUARD_A"]:g} A'],
        ['Macro step, h',f'{dt*1e6:g} µs','Min. active separation',f'{min(distances.train_validation_min_current_plane_distance_A,distances.train_test_min_current_plane_distance_A,distances.validation_test_min_current_plane_distance_A):g} A'],
        ['Primary long horizon',f'1 period ({periods.min():.2f}–{periods.max():.1f} ms)','Full-map FD step',f'{settings["FULL_MAP_EPS"]:g} (normalized)'],
        ['Jacobian-product horizon',f'{settings["FINITE_TIME_STEPS"]} steps ({settings["FINITE_TIME_STEPS"]*dt*1000:g} ms)','Grid-knot FD check',f'{sum(counts.values()):,} points'],
        ['Primary network seeds',settings['PRIMARY_SEEDS'],'State scale, S',f'diag({scale[0]:g} A, {scale[1]:g} A, {scale[2]:g} A)'],
        ['Off-grid audit',f'{len(offgrid)} (if=1,3,5,7,9 A)','Crossed bootstrap',f'{settings["CROSSED_BOOTSTRAP_B"]}×{settings["BOOTSTRAP_RNG_REPEATS"]} + {settings["PRIMARY_SEEDS"]} LOSO'],
    ],columns=['Machine/setting','Value','Computational setting','Computational value'])
    table(table1,'Table_1_Machine_Computational_Settings.csv')
    t2=stability[np.isclose(stability.if_A,6)].sort_values('n_rpm')
    t2=t2[['n_rpm','hcrit_min_us','hcrit_median_us','hcrit_max_us','stable_50us_fraction']].copy()
    t2.columns=['Speed (rpm)','Min. (µs)','Median (µs)','Max. (µs)','Pass at 50 µs (%)']
    t2['Pass at 50 µs (%)']*=100
    table(t2,'Table_2_Euler_Critical_Step_if6A.csv')
    nonhurwitz=int(structural.rhs_nonhurwitz.sum())
    agreements=int((structural.rh_rhs_hurwitz==(~structural.rhs_nonhurwitz)).sum())
    table3=pd.DataFrame([
        ['Routh–Hurwitz vs eigenvalues',f'{len(structural):,} centers',f'{agreements:,} agreement; {nonhurwitz} non-Hurwitz A_RHS'],
        ['Exact det(L_FD) Bernstein sign',f'{len(exact):,} cells',f'{int((exact.L_exact_bernstein_det_sign==1).sum()):,} positive'],
        ['Exact det(D_IC) Bernstein sign',f'{len(exact):,} cells',f'{int((exact.D_exact_bernstein_det_sign==1).sum()):,} positive'],
        ['Face–speed normal-direction screen',f'{len(interfaces):,} configurations',f'{int(interfaces.direction.eq("attracting").sum())} attracting; {int(interfaces.direction.eq("repelling").sum())} repelling'],
        ['Short nonlinear RK4 trajectories',f'{len(nonlinear)} trajectories',f'{int(nonlinear.run_status_fine.eq("completed").sum())} completed'],
        ['Euler 2-norm < 1 at 50 µs',f'{len(norms):,} centers',f'{int((norms.induced_2norm<1).sum())} satisfy'],
        ['Structured bound Hurwitz certificates',f'{len(structural):,} centers',f'{int(structural.deployed_structural_hurwitz_certificate.sum())} certified'],
    ],columns=['Audit','Scope','Outcome'])
    table(table3,'Table_3_Cell_Invertibility_Jacobian_Verification.csv')
    # Five structural scores/conditions in the source Table 4 (not a confidence-interval table).
    coupling_for_table4 = one('table04p_representation_mismatch_stability_coupling_summary.csv')
    score_rows = []
    for code, label, interpretation in (
        ('structured_bound', 'Structured matrix-measure bound', 'In-sample ranking'),
        ('scalar_operator_mismatch', 'Relative operator mismatch', 'In-sample ranking'),
    ):
        score = row(screening, stratum='all', score=code)
        score_rows.append([label, float(score.roc_auc_nonhurwitz), interpretation])
    score_rows.extend([
        ['Structured Hurwitz condition', f'{int(structural.deployed_structural_hurwitz_certificate.sum())} / {len(structural):,}', 'Sufficient test inconclusive'],
        ['Positive-definiteness conditions',
         f'{int((structural.consistent_pd_certificate.astype(bool) & structural.frozen_map_pd_certificate.astype(bool)).sum())} / {len(structural):,}',
         'Both matrix families'],
        ['Bauer–Fike condition',
         f'{int(coupling_for_table4.bauer_fike_certified_hurwitz_points)} / {len(structural):,}',
         'Sufficient test inconclusive'],
    ])
    table(pd.DataFrame(score_rows, columns=['Score or condition', 'ROC-AUC / count', 'Interpretation']),
          'Table_4_Structural_Indicators.csv')
    t5=[]
    for speed in (5000,7000,9000):
        line=[speed]
        for domain in ('current_limited','two_level_spwm_linear','two_level_svpwm_linear'):
            r=row(domains,n_rpm=speed,if_A=6,domain=domain)
            line.append(f'{int(r.retained_points)} ({100*r.retained_fraction:.1f}) / {r.hcrit_median_us:.2f} / {100*r.stable_50us_fraction:.1f}')
        t5.append(line)
    table(pd.DataFrame(t5,columns=['Speed (rpm)','Current-limited FEA: n (%) / median hcrit (µs) / pass at 50 µs (%)','Two-level SPWM','Two-level SVPWM']),
          'Table_5_Conditional_Voltage_Sensitivity_if6A.csv')
    t6=[]
    for mod in ('SPWM','SVPWM'):
        for speed in speeds:
            q=masks[(masks.modulation==mod)&(masks.n_rpm.astype(str)==str(float(speed)))]
            if q.empty:
                q=masks[(masks.modulation==mod)&(pd.to_numeric(masks.n_rpm,errors='coerce')==speed)]
            inside=q[q.voltage_feasible.astype(bool)];outside=q[~q.voltage_feasible.astype(bool)]
            ni=int(inside.n.sum());no=int(outside.n.sum())
            pi=float(inside.nonhurwitz_n.sum()/ni*100) if ni else np.nan
            po=float(outside.nonhurwitz_n.sum()/no*100) if no else np.nan
            t6.append([int(speed),mod,f'{ni}/{no}',pi,po])
    table(pd.DataFrame(t6,columns=['Speed (rpm)','Mask','Inside/outside (n)','Inside non-Hurwitz (%)','Outside non-Hurwitz (%)']),
          'Table_6_Speed_Modulation_NonHurwitz.csv')
    method_order=['Euler','Heun','RK4','Backward Euler','Trapezoidal','Exact exponential']
    # Table 7 is the strict-domain one-period trajectory RMSE, not the linear benchmark.
    c7=common.copy()
    c7['_sort']=pd.to_numeric(c7.speed,errors='coerce').fillna(-1)
    c7=c7.sort_values('_sort')
    t7=pd.DataFrame({
        'Speed':c7.speed.astype(str).replace({'Overall':'Overall'}),
        'n':c7.points.astype(int),
        'Euler':c7.median_euler_rmse,
        'Corr. Euler':c7.median_corrected_rmse,
        'Heun':c7.median_heun_rmse,
        'RK4':c7.median_rk4_rmse,
        'Corr. better (%)':100*c7.corrected_better_than_euler_fraction,
    })
    table(t7,'Table_7_One_Period_Common_Strict_RMSE.csv')
    # Table 8 is the paired 20/50 µs matrix-propagator benchmark.
    t8_rows=[]
    for method in method_order:
        r20=row(benchmark,method=method,step_us=20)
        r50=row(benchmark,method=method,step_us=50)
        t8_rows.append([method,100*r20.stable_fraction,r20.median_relative_matrix_error,
                        100*r50.stable_fraction,r50.median_relative_matrix_error])
    table(pd.DataFrame(t8_rows,columns=['Method','Pass20 (%)','Error20','Pass50 (%)','Error50']),
          'Table_8_Matched_20_50us_Solver_Benchmark.csv')
    # Supplementary Table S2 shares exactly the same 50 µs numerical benchmark.
    table(pd.DataFrame([
        [method,100*row(benchmark,method=method,step_us=50).stable_fraction,
         row(benchmark,method=method,step_us=50).median_relative_matrix_error,
         row(benchmark,method=method,step_us=50).p95_relative_matrix_error]
        for method in method_order],
        columns=['Method','Stable (%)','Median rel. error','P95 error']),
        'Table_S2_50us_Propagator_Upper_Tail.csv')
    # Supplementary Table S1: two independent sensitivity designs, all vs strict domains.
    table(pd.DataFrame([
        ['Instantaneous ρ ≥ 1',100*map_summary.euler_rho_ge_1_fraction,100*map_summary.corrected_rho_ge_1_fraction],
        ['Instantaneous σmax ≥ 1',100*map_summary.euler_sigma_max_ge_1_fraction,100*map_summary.corrected_sigma_max_ge_1_fraction],
        ['20-step median σmax, all',finite.median_sigma_product_euler,finite.median_sigma_product_corrected],
        ['20-step P95 σmax, all',finite.p95_sigma_product_euler,finite.p95_sigma_product_corrected],
        ['20-step σmax > 1, all',100*finite.euler_sigma_product_gt_1_fraction,100*finite.corrected_sigma_product_gt_1_fraction],
        [f'20-step median σmax, strict (n={int(finite_strict.scenarios)})',finite_strict.median_sigma_product_euler,finite_strict.median_sigma_product_corrected],
        ['20-step σmax > 1, strict',100*finite_strict.euler_product_gt1_fraction,100*finite_strict.corrected_product_gt1_fraction],
        ['Lower product gain, strict',np.nan,f'{100*finite_strict.corrected_lower_product_gain_fraction:.1f}% of scenarios'],
    ],columns=['Metric','Euler','Corr. Euler']),'Table_S1_Normalized_Map_Sensitivity.csv')

    sens=read('table03_derivative_sensitivity.csv')
    knot=read('table04b_grid_knot_rhs_fd_consistency_summary.csv')
    kprimary=knot[np.isclose(knot.eps_normalized,settings['DIRECT_RHS_JAC_PRIMARY_EPS'])]
    kprimary=kprimary[kprimary.scope.str.contains('interior',case=False)]
    coupling=one('table04p_representation_mismatch_stability_coupling_summary.csv')
    exhaustive=one('table04q6_exhaustive_intracell_summary.csv')
    interpolation=read('table04ab_interpolation_family_sensitivity_summary.csv')
    grid=read('table04ad_grid_resolution_sensitivity_summary.csv')
    independent=one('table11l_independent_dop853_reference_audit_summary.csv')
    dcref=one('table04za_derivative_consistent_reference_convergence_summary.csv')
    symmetry=row(read('table03g_symmetrized_incremental_matrix_sensitivity_summary.csv'),domain='all')
    voltage=read('table04j_offgrid_exact_voltage_feasibility.csv')
    selection=read('table11j_common_strict_selection_summary.csv')
    overall=row(common,speed='Overall')
    crossed=read('table19c_one_period_two_way_seed_spatial_block_bootstrap.csv')
    boot=crossed[crossed.comparison.str.contains('Euler',case=False)].iloc[0]
    losob=read('table19f_one_period_leave_one_seed_out_bootstrap.csv')
    losob=losob[losob.comparison.str.contains('Euler',case=False)]
    crossing=int(((losob.ci_low<=0)&(losob.ci_high>=0)).sum())
    contact=read('table11n_rollout_nonhurwitz_cell_contact_summary.csv')
    transfer=read('table20_leave_one_speed_out.csv')
    timing=read('table20b_platform_specific_step_timing.csv')
    timing.to_csv(selected / 'Diagnostics' / 'CPU_Timing_Detail.csv', index=False)
    if 'device' not in timing.columns or not timing.device.astype(str).str.lower().eq('cpu').all():
        raise AssertionError('Table 9 requires verified CPU timing measurements.')
    if not np.isfinite(timing.mean_time_per_batch_us.to_numpy(dtype=float)).all() or (timing.mean_time_per_batch_us<=0).any():
        raise AssertionError('Invalid timing measurements; no timings are substituted.')
    time1=timing[timing.batch_size==1]
    if len(time1)!=4 or set(time1.method)!= {'Euler','Corrected Euler','Heun','RK4'}:
        raise AssertionError('Incomplete batch-1 CPU timing data for Table 9.')
    retained=row(selection,subset='retained');excluded=row(selection,subset='excluded')
    spline=row(sens,derivative_method='spline');edge=row(sens,derivative_method='edge1')

    knot_text='; '.join(f'{r.get("field_scope",r.get("scope","stratum"))}: {100*r["euler_stable50_agreement_fraction"]:.2f}%' for _,r in kprimary.iterrows())
    interp_text='; '.join(f'{r.interpolation_family}: {100*r.deployed_nonhurwitz_fraction:.2f}% vs {100*r.consistent_nonhurwitz_fraction:.2f}%' for _,r in interpolation.iterrows())
    grid_text='; '.join(f'{int(r.coarsen_factor_id_iq)}: {100*r.deployed_nonhurwitz_fraction:.2f}% vs {100*r.consistent_nonhurwitz_fraction:.2f}%' for _,r in grid.iterrows())
    mod_text='; '.join(f'{r.domain}: {100*r.rhs_exact_nonhurwitz_fraction:.2f}%' for _,r in voltage[voltage.speed_scope.astype(str)=='all'].iterrows())
    contact_text='; '.join(f'{r.configuration}: {100*r.contact_fraction:.1f}%' for _,r in contact[contact.scope=='all_test'].iterrows())
    timing_text='; '.join(f'{r.method}: {r.mean_time_per_batch_us/1000:.3f} ms' for _,r in time1.iterrows())
    # Table 4 uses the combined condition; record the individual counts as well.
    pd_detail = pd.DataFrame([
        {'matrix': 'D', 'positive_definiteness_count': int(structural.consistent_pd_certificate.sum()),
         'tested_centers': len(structural)},
        {'matrix': 'L', 'positive_definiteness_count': int(structural.frozen_map_pd_certificate.sum()),
         'tested_centers': len(structural)},
        {'matrix': 'D and L', 'positive_definiteness_count': int((structural.consistent_pd_certificate.astype(bool) & structural.frozen_map_pd_certificate.astype(bool)).sum()),
         'tested_centers': len(structural)},
    ])
    pd_detail.to_csv(selected / 'Diagnostics' / 'Table_4_Positive_Definiteness_Detail.csv', index=False)
    # Nine-row consolidated audit with the same scope as source Table 9.
    geometric_cell_count=int(round(float(exhaustive.admissible_parent_cells)/len(speeds)))
    if geometric_cell_count*len(speeds)!=int(exhaustive.admissible_parent_cells):
        raise AssertionError('Extended cell–speed count cannot be factored by available speeds.')
    mask_totals=[]
    for mod in ('SPWM','SVPWM'):
        q=masks[(masks.modulation==mod)&(masks.voltage_feasible.astype(bool))]
        if q.n_rpm.astype(str).eq('all').any():
            q=q[q.n_rpm.astype(str).eq('all')]
        den=int(q.n.sum());num=int(q.nonhurwitz_n.sum())
        mask_totals.append(f'{100*num/den:.2f}%' if den else 'N/A')
    t9=pd.DataFrame([
        ['Off-grid Jacobian',f'{len(offgrid):,} centers',
         (f'{nonhurwitz} non-Hurwitz RHS; none formal or consistent' if (offgrid.model_hurwitz.astype(bool).all() and offgrid.consistent_hurwitz.astype(bool).all()) else f'{nonhurwitz} non-Hurwitz RHS; {int((~offgrid.model_hurwitz.astype(bool)).sum())} formal; {int((~offgrid.consistent_hurwitz.astype(bool)).sum())} consistent')],
        ['Extended design',f'{geometric_cell_count:,} cells × {len(speeds)} speeds; {int(exhaustive.retained_design_points_inside_current_limit):,} valid locations',
         f'{int(exhaustive.deployed_nonhurwitz_points):,} non-Hurwitz RHS ({100*exhaustive.deployed_nonhurwitz_point_fraction:.2f}%); consistent {100*exhaustive.consistent_nonhurwitz_point_fraction:.0f}%'],
        ['Invertibility',f'{len(exact):,} relevant cells',
         ('Positive determinant certificates for both fields' if exact.L_exact_bernstein_det_sign.eq(1).all() and exact.D_exact_bernstein_det_sign.eq(1).all() else f'Positive determinant certificates: L {int((exact.L_exact_bernstein_det_sign==1).sum())}/{len(exact)}; D {int((exact.D_exact_bernstein_det_sign==1).sum())}/{len(exact)}')],
        ['Structural bound',f'{len(structural):,} centers',
         f'ROC-AUC {row(screening,stratum="all",score="structured_bound").roc_auc_nonhurwitz:.4f}; {int(structural.deployed_structural_hurwitz_certificate.sum())} Hurwitz certificates'],
        ['Voltage masks','SPWM / SVPWM',f'Non-Hurwitz inside: {mask_totals[0]} / {mask_totals[1]}'],
        ['Classical solvers',f'{int(overall.points):,} common strict points',
         f'Median E/C/H/RK4: {overall.median_euler_rmse:.4g} / {overall.median_corrected_rmse:.4g} / {overall.median_heun_rmse:.4g} / {overall.median_rk4_rmse:.4g}'],
        ['Residual correction',f'{settings["PRIMARY_SEEDS"]} seeds; selected subset',
         f'{100*overall.corrected_better_than_euler_fraction:.2f}% improve; {crossing}/{len(losob)} seed-omission intervals include zero'],
        ['Finite-time sensitivity',f'{int(finite_strict.scenarios):,} strict products',
         ('All product norms exceed one' if float(finite_strict.euler_product_gt1_fraction)==1.0 and float(finite_strict.corrected_product_gt1_fraction)==1.0 else f'Euler {100*finite_strict.euler_product_gt1_fraction:.1f}%, corrected {100*finite_strict.corrected_product_gt1_fraction:.1f}% product norms > 1')],
        ['Timing','Batch 1; measured CPU',
         'E/C/H/RK4: '+ ' / '.join(
          f'{float(row(time1,method=label).mean_time_per_batch_us)/1000:.3f}'
          for label in ('Euler','Corrected Euler','Heun','RK4')) + ' ms'],
    ],columns=['Audit','Scope','Principal result'])
    table(t9,'Table_9_Consolidated_Audit.csv')
    # Supporting calculations remain in the temporary analysis workspace.
    fig,axs=plt.subplots(2,2,figsize=(10,8));panels(axs)
    ax=axs[0,0]
    op=read('table03d_operating_domain_voltage_flags.csv')
    q=op[np.isclose(op.n_rpm,9000)&np.isclose(op.if_A,6)]
    Z=np.full((len(D['iq_vec']),len(D['id_vec'])),np.nan)
    for _,r in q.iterrows():Z[int(r.j),int(r.i)]=r.dtcrit_s*1e6
    X,Y=np.meshgrid(D['id_vec'],D['iq_vec'])
    cs=ax.contourf(X,Y,np.ma.masked_invalid(Z),levels=18);fig.colorbar(cs,ax=ax,label='Euler critical step (µs)')
    angle=np.linspace(0,2*np.pi,500);rlim=np.sqrt(3)*float(D['ismax'])
    ax.plot(rlim*np.cos(angle),rlim*np.sin(angle),'k--',lw=.8)
    ax.set(xlabel='$i_d$ (A)',ylabel='$i_q$ (A)',title='9000 rpm, $i_f=6$ A');ax.set_aspect('equal','box')
    ax=axs[0,1]

    for method in ('edge2','edge1','spline'):
        raw=settings['sensitivity_frames'][method]
        z=raw[np.isclose(raw.n_rpm,9000)&np.isclose(raw.if_A,6)].dtcrit_s.to_numpy()*1e6
        ax.hist(z,bins=24,histtype='step',density=True,label=method)
    ax.set(xlabel='Euler critical step (µs)',ylabel='Density');ax.grid(alpha=.2);ax.legend(fontsize=8)
    ax=axs[1,0];a=offgrid.rhs_exact_spectral_abscissa_sinv.to_numpy()
    ax.hist(a[np.isfinite(a)],bins=50);ax.axvline(0,color='k',ls='--',lw=1)
    ax.set(xlabel='Audited RHS spectral abscissa (s$^{-1}$)',ylabel='Off-grid equilibria');ax.grid(alpha=.2)
    ax.text(.03,.96,f'Non-Hurwitz: {int((~offgrid.rhs_exact_hurwitz).sum())}/{len(offgrid)}',transform=ax.transAxes,va='top',fontsize=8)
    ax=axs[1,1]
    q=offgrid[offgrid.model_hurwitz&offgrid.rhs_exact_hurwitz&(offgrid.hcrit_model_euler_s>0)&(offgrid.hcrit_rhs_exact_euler_s>0)]
    x=q.hcrit_model_euler_s.to_numpy()*1e6;y=q.hcrit_rhs_exact_euler_s.to_numpy()*1e6
    ax.scatter(x,y,s=7,alpha=.3);diagonal(ax,x,y)
    ax.set(xlabel='Frozen-map Euler critical step (µs)',ylabel='Audited RHS Euler critical step (µs)');ax.grid(which='both',alpha=.2)
    finish(fig,'Figure_2_Local_Stability_Consistency.png')

    fig,axs=plt.subplots(2,2,figsize=(10,8));panels(axs)
    coupling_points=read('table04o_representation_mismatch_stability_coupling_all.csv')
    ax=axs[0,0]
    for flip,label,marker in [(False,'No Hurwitz flip','o'),(True,'Hurwitz flip','x')]:
        z=coupling_points[coupling_points.consistent_to_deployed_hurwitz_flip==flip]
        ax.scatter(z.relative_D_operator_mismatch_fro,z.alpha_deployed_sinv,s=9,alpha=.4,marker=marker,label=label)
    ax.axhline(0,color='k',ls='--',lw=.8);ax.set(xlabel='Relative derivative-operator mismatch',ylabel='Audited spectral abscissa (s$^{-1}$)');ax.legend(fontsize=8);ax.grid(alpha=.2)
    persistence=read('table04q2_intracell_cell_level_persistence.csv')
    ax=axs[0,1];ax.hist(persistence.offcenter_nonhurwitz_fraction.dropna(),bins=np.linspace(0,1,18))
    ax.set(xlabel='Within-cell off-center non-Hurwitz fraction',ylabel='Sampled cells');ax.grid(axis='y',alpha=.2)
    pairs=read('table04t_derivative_consistent_trajectory_paired.csv')
    ax=axs[1,0];values=[]
    for solver,z in pairs[pairs.paired_strict].groupby('solver'):
        z=z[(z.deployed>0)&(z.derivative_consistent>0)]
        ax.scatter(z.deployed,z.derivative_consistent,s=15,alpha=.5,label=solver.upper())
        values.extend(z.deployed);values.extend(z.derivative_consistent)
    if values:diagonal(ax,np.array(values),np.array(values))
    ax.set(xlabel='RMSE: audited representation',ylabel='RMSE: derivative-consistent');ax.legend(fontsize=8);ax.grid(which='both',alpha=.2)
    conditioning=read('table04v_derivative_consistent_conditioning_all.csv')
    ax=axs[1,1]
    x=conditioning['I_Dhpsi_cond2_dimensionless'].to_numpy();y=conditioning['D_Ipsi_cond2_dimensionless'].to_numpy()
    ax.scatter(x,y,s=7,alpha=.3);diagonal(ax,x,y)
    ax.set(xlabel='Scaled κ₂ of interpolated derivative map',ylabel='Scaled κ₂ of flux-interpolant derivative');ax.grid(which='both',alpha=.2)
    finish(fig,'Figure_3_Representation_Robustness.png')

    fig,ax=plt.subplots(figsize=(6.3,4.8))
    q=op[np.isclose(op.n_rpm,9000)&np.isclose(op.if_A,6)]
    V=np.full_like(Z,np.nan)
    for _,r in q.iterrows():V[int(r.j),int(r.i)]=r.u_dq_eq_V
    cs=ax.contourf(X,Y,np.ma.masked_invalid(V),levels=18);fig.colorbar(cs,ax=ax,label='Equilibrium voltage magnitude (V)')
    for limit,style,label in [(D['udc']*np.sqrt(3/8),'--','SPWM'),(D['udc']/np.sqrt(2),'-','SVPWM')]:
        if np.nanmin(V)<=limit<=np.nanmax(V):
            lines=ax.contour(X,Y,V,levels=[limit],colors='k',linestyles=style,linewidths=1)
            ax.clabel(lines,fmt={limit:label},fontsize=8)
    ax.plot(rlim*np.cos(angle),rlim*np.sin(angle),'k:',lw=.9)
    ax.set(xlabel='$i_d$ (A)',ylabel='$i_q$ (A)',title='9000 rpm, $i_f=6$ A');ax.set_aspect('equal','box')
    finish(fig,'Figure_4_Conditional_Voltage_Domain.png')

    common_points=read('table11f_common_strict_seed_level.csv')
    avg=common_points.groupby(['op_index','n_rpm','if_A'])[['corrected_rmse','euler','heun','rk4']].mean().reset_index()
    long_results=read('table11c_one_electrical_period_rollouts.csv')
    long_average=long_results.groupby(['configuration','op_index','n_rpm','if_A']).rmse.mean().reset_index()
    heldout=long_average.pivot(index=['op_index','n_rpm','if_A'],columns='configuration',values='rmse')
    fig,axs=plt.subplots(1,2,figsize=(10,4.4));panels(axs)
    names=[('euler','Euler'),('corrected_rmse','Corrected Euler'),('heun','Heun'),('rk4','RK4')]
    axs[0].boxplot([heldout[col] for col in ['euler','certified_rollout','heun','rk4']],tick_labels=[label for _,label in names],showfliers=False)
    axs[0].set_yscale('log');axs[0].set_ylabel('One-period normalized RMSE');axs[0].tick_params(axis='x',rotation=18);axs[0].grid(axis='y',which='both',alpha=.2)
    for col,label in names:
        g=avg.groupby('n_rpm')[col].median();axs[1].plot(g.index,g.values,marker='o',ms=4,label=label)
    axs[1].set(xlabel='Mechanical speed (rpm)',ylabel='Median normalized RMSE');axs[1].set_yscale('log');axs[1].grid(which='both',alpha=.2);axs[1].legend(fontsize=8)
    finish(fig,'Figure_5_One_Period_Solver_Errors.png')
    coverage=read('table21_fixed_step_coverage.csv')
    fig,ax=plt.subplots(figsize=(6.3,4.3));coverage_plot(ax,coverage)
    finish(fig,'Figure_6_Euler_Fixed_Step_Screening.png')
    # Supplementary Figure S1. Spatial partition and residual-network norm distributions.
    coords=read('table05c_spatial_coordinate_partition.csv')
    lip=read('table09_lipschitz_audit.csv')
    fig,axs=plt.subplots(1,2,figsize=(10,4.6));panels(axs)
    ax=axs[0]
    for part in ('train','validation','test','guard'):
        subset=coords[coords.partition==part]
        if not subset.empty:
            ax.scatter(subset.id_A,subset.iq_A,s=12,alpha=.8,label=part)
    ax.set(xlabel='$i_d$ (A)',ylabel='$i_q$ (A)')
    ax.set_aspect('equal','box');ax.grid(alpha=.2);ax.legend(fontsize=8)
    ax=axs[1]
    variants=('certified_rollout','certified_one_step','unconstrained_rollout')
    groups=[lip.loc[lip.configuration==v,'empirical_max'].dropna().to_numpy() for v in variants]
    if any(len(g)==0 for g in groups):
        raise AssertionError('A network training variant is missing from S1 diagnostic data.')
    ax.boxplot(groups,tick_labels=['Rollout-constrained','One-step-constrained','Unconstrained'])
    ax.set_ylabel('Maximum residual-network Jacobian norm')
    ax.grid(axis='y',alpha=.2);ax.tick_params(axis='x',rotation=15)
    finish(fig,'Figure_S1_Secondary_Learning_Audit.png')

    # Supplementary Figure S2. Per-scenario, seed-median instantaneous Jacobian metrics.
    instantaneous=read('table15_full_map_seed_averaged_metrics.csv')
    fig,axs=plt.subplots(1,2,figsize=(10,4.5));panels(axs)
    for ax,fields,limlabel in zip(axs,
         (('rho_euler','rho_corrected'),('sigma_max_euler','sigma_max_corrected')),
         (r'\rho(D\Phi)',r'\sigma_{\max}(D\Phi)')):
        x=instantaneous[fields[0]].to_numpy(dtype=float)
        y=instantaneous[fields[1]].to_numpy(dtype=float)
        valid=np.isfinite(x)&np.isfinite(y)
        if not np.any(valid):
            raise AssertionError('No finite instantaneous-map pairs for S2.')
        x,y=x[valid],y[valid]
        ax.scatter(x,y,s=9,alpha=.3)
        bound=max(1.2,float(np.max(x)),float(np.max(y)))
        ax.plot([0,bound],[0,bound],'k--',lw=.9)
        ax.axhline(1,color='gray',lw=.8);ax.axvline(1,color='gray',lw=.8)
        ax.set(xlim=(0,bound),ylim=(0,bound),xlabel='$'+limlabel+r'_{\mathrm{Euler}}$',
               ylabel='$'+limlabel+r'_{\mathrm{corrected}}$')
        ax.grid(alpha=.2)
    finish(fig,'Figure_S2_Instantaneous_Normalized_Map.png')

    # Supplementary Figure S3. Seed-median paired *strict-domain* 20-step gains.
    products=read('table15d_finite_time_jacobian_products_all.csv')
    strict_gains=products.groupby(['scenario_index','op_index'],as_index=False).agg(
        euler=('sigma_max_product_euler','median'),
        corrected=('sigma_max_product_corrected','median'),
        strict=('strict_domain_both','min'))
    strict_gains=strict_gains[strict_gains.strict.astype(bool)]
    if len(strict_gains)!=int(finite_strict.scenarios):
        raise AssertionError('S3 strict-domain sample size does not match Table S1.')
    x=strict_gains.euler.to_numpy(dtype=float)
    y=strict_gains.corrected.to_numpy(dtype=float)
    good=np.isfinite(x)&np.isfinite(y)&(x>0)&(y>0)
    if not np.all(good):
        raise AssertionError('Nonpositive or nonfinite 20-step gains in strict subset.')
    fig,ax=plt.subplots(figsize=(6.1,5.4))
    ax.scatter(x,y,s=8,alpha=.3)
    lo=min(float(x.min()),float(y.min()));hi=max(float(x.max()),float(y.max()))
    ax.plot([lo,hi],[lo,hi],'k--',lw=.8)
    ax.set_xscale('log');ax.set_yscale('log')
    ax.set(xlabel=r'$\sigma_{\max}(\prod D\Phi_{\mathrm{Euler}})$',
           ylabel=r'$\sigma_{\max}(\prod D\Phi_{\mathrm{corrected}})$')
    ax.grid(which='both',alpha=.2)
    finish(fig,'Figure_S3_Strict_20Step_Jacobian_Products.png')

    files=sorted(p.relative_to(destination).as_posix() for p in destination.rglob('*') if p.is_file())
    counted=[p for p in files if p.startswith('Results/')]
    if any(p.endswith('.pdf') for p in files):
        raise AssertionError('PDF output is disabled.')
    expected_figures={f'Figure_{n}_' for n in range(2,7)} | {f'Figure_S{n}_' for n in range(1,4)}
    expected_tables={f'Table_{n}_' for n in range(1,10)} | {f'Table_S{n}_' for n in range(1,3)}
    png=[Path(p).name for p in files if p.startswith('Results/Figures/') and p.endswith('.png')]
    csv=[Path(p).name for p in files if p.startswith('Results/Tables/') and p.endswith('.csv')]
    diagnostics=[Path(p).name for p in files if p.startswith('Results/Diagnostics/')]
    if len(png)!=8 or any(sum(name.startswith(prefix) for name in png)!=1 for prefix in expected_figures):
        raise AssertionError('Expected exactly Figures 2–6 and Figures S1–S3 (8 PNG files).')
    if len(csv)!=11 or any(sum(name.startswith(prefix) for name in csv)!=1 for prefix in expected_tables):
        raise AssertionError('Expected exactly Tables 1–9 and Tables S1–S2 (11 CSV files).')
    if set(diagnostics) != {'CPU_Timing_Detail.csv', 'Table_4_Positive_Definiteness_Detail.csv'}:
        raise AssertionError('Expected exactly two supplemental diagnostic CSV files.')
    return {'output_directory':str(destination),'files':files,'table_count':11,'diagnostic_count':2,
            'figure_count':8,'figure_dpi':int(settings['FIGURE_DPI']),'figure_format':'PNG',
        'timing_device':'cpu','timing_samples':len(timing)}

OUTPUT_SETTINGS = {
    'FULL_RUN': FULL_RUN,
    'FIGURE_DPI': FIGURE_DPI,
    'STATE_SCALE': STATE_SCALE,
    'DT': DT,
    'SPATIAL_NBINS': SPATIAL_NBINS,
    'SPATIAL_GUARD_A': SPATIAL_GUARD_A,
    'FULL_MAP_EPS': FULL_MAP_EPS,
    'FINITE_TIME_STEPS': FINITE_TIME_STEPS,
    'PRIMARY_SEEDS': PRIMARY_SEEDS,
    'CROSSED_BOOTSTRAP_B': CROSSED_BOOTSTRAP_B,
    'BOOTSTRAP_RNG_REPEATS': BOOTSTRAP_RNG_REPEATS,
    'DIRECT_RHS_JAC_PRIMARY_EPS': DIRECT_RHS_JAC_PRIMARY_EPS,
    'sensitivity_frames': sensitivity_frames,
}
if FULL_RUN and scientific_report['status'] != 'PASS':
    raise RuntimeError('Scientific checks failed; see the separate validation report.')
result_outputs = export_results(OUT, RESULTS_OUT, D, OUTPUT_SETTINGS) if FULL_RUN else {'files': []}
print('Output directory:', RESULTS_OUT if FULL_RUN else OUT)
for result_file in result_outputs['files']:
    print(result_file)
    result_path = RESULTS_OUT / result_file
    if result_path.suffix.lower() == '.csv':
        print('  Preview:', result_path.name)
        display(pd.read_csv(result_path))
    elif result_path.suffix.lower() == '.png':
        from IPython.display import Image
        display(Image(filename=str(result_path), width=900))
print('Scientific checks:', scientific_report['passed'], '/', scientific_report['checked'])
print('Packages:', {k: v for k,v in {'Python':sys.version.split()[0], 'NumPy':np.__version__, 'SciPy':scipy.__version__, 'Pandas':pd.__version__, 'PyTorch':torch.__version__, 'Optuna':optuna.__version__}.items()})
print('Source SHA-256:', INPUT_CHECKSUMS)
# Optional numerical evidence archive; the primary results archive is unchanged.
if EXPORT_AUDIT_EVIDENCE and FULL_RUN:
    import zipfile as _evidence_zipfile
    evidence_path = BASE_DIR / 'EESM_LUT_Audit_Evidence.zip'
    evidence_names = [
        'table_v7_exact_bernstein_cell_proof.csv',
        'table_v7_exhaustive_interface_directions.csv',
        'table04h_offgrid_exact_within_cell_jacobian_all.csv',
        'table04q6_exhaustive_intracell_summary.csv',
        'diagnostic_structural_roc_auc_by_speed.csv',
        'diagnostic_cluster_bootstrap_auc_ci.csv',
        'diagnostic_matched_guarded_20_50us_solver_summary.csv',
        'table04t_derivative_consistent_trajectory_paired.csv',
    ]
    evidence_meta = {
        'input_sha256': INPUT_CHECKSUMS,
        'random_seed': int(MASTER_SEED),
        'python': sys.version.split()[0],
        'packages': {'numpy':np.__version__, 'scipy':scipy.__version__,
                     'pandas':pd.__version__, 'torch':torch.__version__,
                     'optuna':optuna.__version__},
        'scientific_validation': scientific_report['status'],
        'code_identity': run_manifest['code'],
        'scope': 'Exact positivity is conditional on the stored IEEE-754 values; '
                 'sampled classifications do not prove physical or global stability.',
    }
    with _evidence_zipfile.ZipFile(evidence_path, 'w', _evidence_zipfile.ZIP_DEFLATED) as archive:
        archive.writestr('audit_metadata.json', json.dumps(evidence_meta, indent=2))
        for name in evidence_names:
            source = TAB / name
            if not source.is_file():
                raise FileNotFoundError('Audit evidence missing: ' + name)
            archive.write(source, arcname='Checks/' + name)
    print('Optional audit evidence archive:', evidence_path)

result_archive = shutil.make_archive(str(RESULTS_OUT if FULL_RUN else BASE_DIR / 'EESM_LUT_Quick_Check'), 'zip', RESULTS_OUT if FULL_RUN else OUT)
validation_archive = shutil.make_archive(str(REPORT_DIR), 'zip', REPORT_DIR)
print('Results archive:', result_archive)
print('Validation archive:', validation_archive)
_results_workspace.cleanup()
try:
    from google.colab import files as _colab_output
except ImportError:
    print('Archives are saved at the paths shown above.')
else:
    _colab_output.download(result_archive)
    _colab_output.download(validation_archive)
